In [3]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 1 / STEP 0 — V3
# 24-PROJECT COHORT + ANALYSIS-SOURCE DISCOVERY + LEGACY/MODERN LAYOUT FREEZE
# ==================================================================================================
#
# ROOT CAUSE FIXED IN V3
# ----------------------
# Projects 1..24 were completed over an evolving experiment workflow.
#
# Newer projects have a compact package under:
#     Results/Final/<ProjectSlug>/
#
# Earlier completed projects (including Project 1: Angel-ML@angel) may legitimately have their
# frozen analysis outputs only under:
#     Results/Aggregated/<ProjectSlug>/
#
# Therefore it is WRONG for the first global-analysis cell to require a uniform Results/Final layout
# or a uniform Step-5B filename/schema across all 24 projects.
#
# V3 does the scientifically correct first step:
#
#   1. Freeze the exact final 24-project registry cohort.
#   2. Verify Projects 1..24 are COMPLETE_AND_FROZEN.
#   3. Require each project's frozen Results/Aggregated/<slug> root.
#   4. Treat Results/Final/<slug> as an OPTIONAL modern package.
#   5. Where a final package exists, independently verify its manifest/file hashes.
#   6. Discover ALL analysis-ready APFD/APFDc candidate files from Aggregated + Final using
#      filename and schema evidence, WITHOUT yet forcing one universal legacy/modern schema.
#   7. Classify discovered candidate schemas.
#   8. Freeze the candidate inventory so Global Analysis Step 1 can normalize the heterogeneous
#      historical layouts into one canonical 45,360-row master dataset.
#   9. Verify the sole excluded source project Graylog2@graylog2-server.
#
# THIS CELL DOES NOT:
#   - read raw 2,160-condition result trees;
#   - train models;
#   - execute statistical tests;
#   - modify any Project 1..24 output;
#   - modify the completion registry;
#   - answer RQ1/RQ2/RQ3.
#
# IMPORTANT
# ---------
# Run this V3 from the beginning in Thesis_Gobal_Analysis.
# It safely deletes ONLY a partial Global Analysis Step-0 directory if V1/V2 failed before checkpoint.
# ==================================================================================================

from __future__ import annotations

import gzip
import hashlib
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 142)
print("=== THESIS GLOBAL ANALYSIS — CELL 1 / STEP 0 V3: COHORT + ANALYSIS-SOURCE DISCOVERY FREEZE ===")
print("=" * 142)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN GLOBAL CONTRACT
# --------------------------------------------------------------------------------------------------

GLOBAL_ANALYSIS_VERSION = "THESIS_GLOBAL_ANALYSIS_V1_24_ELIGIBLE_PROJECTS"

STEP0_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP0_24_PROJECT_COHORT_AND_ANALYSIS_SOURCE_DISCOVERY_FROZEN"
)

STEP0_CODE_REVISION = (
    "GLOBAL_ANALYSIS_STEP0_V3_LEGACY_MODERN_LAYOUT_DISCOVERY_NO_UNIFORM_PACKAGE_ASSUMPTION"
)

COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

EXPECTED_INCLUDED_PROJECTS = 24
EXPECTED_PROJECT_NUMBERS = list(range(1, 25))

EXPECTED_NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
EXPECTED_SEEDS = list(range(1, 31))

EXPECTED_TECHNIQUES = [
    "LatestFail",
    "LightGBM",
    "NaiveBayes",
    "QTF-Avg",
    "Random",
    "RandomForest",
    "XGBoost",
]

EXCLUDED_PROJECT = "Graylog2@graylog2-server"

# We are deliberately permissive in Step 0 because old outputs used several naming conventions.
# Step 1 will use the frozen inventory to normalize them exactly.
METRIC_FILENAME_HINTS = (
    "apfd",
    "metric",
    "project_run",
    "project-run",
    "projectrun",
    "noise",
    "degradation",
    "retention",
    "summary",
)

SUPPORTED_ANALYSIS_EXTENSIONS = (
    ".csv",
    ".csv.gz",
    ".parquet",
)


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"
STEP0_ROOT = ANALYSIS_ROOT / "Step_0_Cohort_and_Analysis_Source_Discovery"

REGISTRY_SNAPSHOT = (
    STEP0_ROOT / "completed_project_registry_24_project_analysis_snapshot.csv"
)

PROJECT_SOURCE_INVENTORY_PATH = (
    STEP0_ROOT / "global_analysis_project_source_inventory.csv"
)

CANDIDATE_INVENTORY_PATH = (
    STEP0_ROOT / "global_analysis_metric_candidate_inventory.csv"
)

SCHEMA_SUMMARY_PATH = (
    STEP0_ROOT / "global_analysis_metric_schema_summary.csv"
)

PROJECT_CANDIDATE_SUMMARY_PATH = (
    STEP0_ROOT / "global_analysis_project_candidate_summary.csv"
)

EXCLUSION_RECORD_PATH = (
    STEP0_ROOT / "global_analysis_excluded_project_record.csv"
)

VALIDATION_PATH = (
    STEP0_ROOT / "global_analysis_step0_validation.csv"
)

OUTPUT_MANIFEST_PATH = (
    STEP0_ROOT / "global_analysis_step0_output_manifest.csv"
)

REPORT_PATH = (
    STEP0_ROOT / "global_analysis_step0_report.json"
)

STATUS_PATH = (
    STEP0_ROOT / "global_analysis_step0_status.json"
)

CHECKPOINT_PATH = (
    NOTES / "global_analysis_step0_checkpoint.json"
)

P24_INELIGIBLE_PATH = (
    RESULTS
    / "Aggregated"
    / "project_24_selection"
    / "project_24_ineligible_candidates.csv"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def normalise(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def resolve_column(dataframe, wanted):
    wanted_norm = normalise(wanted)

    matches = [
        column
        for column in dataframe.columns
        if normalise(column) == wanted_norm
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve exactly one registry column for {wanted!r}. "
            f"Matches: {matches}"
        )

    return matches[0]


def resolve_optional_column(dataframe, wanted):
    wanted_norm = normalise(wanted)

    matches = [
        column
        for column in dataframe.columns
        if normalise(column) == wanted_norm
    ]

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple registry columns match optional field {wanted!r}: {matches}"
        )

    return matches[0] if matches else None


def parse_int(value, label):
    text = str(value).strip()

    if not text:
        raise RuntimeError(
            f"{label}: empty integer value."
        )

    numeric = float(text)

    if not numeric.is_integer():
        raise RuntimeError(
            f"{label}: non-integer value {value!r}."
        )

    return int(numeric)


def parse_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no"}:
        return False

    raise RuntimeError(
        f"Could not parse boolean: {value!r}"
    )


def is_sha256(value):
    text = str(value).strip().lower()

    return (
        len(text) == 64
        and all(
            character in "0123456789abcdef"
            for character in text
        )
    )


def atomic_csv(path, dataframe):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def atomic_copy(source, destination):
    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        f".{destination.name}.tmp_{os.getpid()}"
    )

    shutil.copy2(
        source,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )


def root_hash(dataframe):
    required = [
        "RelativePath",
        "Bytes",
        "SHA256",
    ]

    missing = [
        column
        for column in required
        if column not in dataframe.columns
    ]

    if missing:
        raise RuntimeError(
            f"Package manifest missing columns: {missing}"
        )

    ordered = (
        dataframe[
            required
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def build_payload_manifest(package_root, manifest_path):
    package_root = Path(package_root)
    manifest_path = Path(manifest_path)

    rows = []

    for path in sorted(
        (
            candidate
            for candidate in package_root.rglob("*")
            if candidate.is_file()
            and candidate.resolve() != manifest_path.resolve()
        ),
        key=lambda candidate:
            candidate.relative_to(package_root).as_posix(),
    ):
        rows.append({
            "RelativePath":
                path.relative_to(
                    package_root
                ).as_posix(),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def compare_manifests(expected, actual):
    expected = (
        expected[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
    )

    actual = (
        actual[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
    )

    expected["RelativePath"] = expected["RelativePath"].astype(str)
    actual["RelativePath"] = actual["RelativePath"].astype(str)

    expected["SHA256"] = expected["SHA256"].astype(str).str.lower()
    actual["SHA256"] = actual["SHA256"].astype(str).str.lower()

    merged = expected.merge(
        actual,
        on="RelativePath",
        how="outer",
        suffixes=("_expected", "_actual"),
        indicator=True,
    )

    missing = int(
        merged["_merge"].eq("left_only").sum()
    )

    unexpected = int(
        merged["_merge"].eq("right_only").sum()
    )

    both = merged.loc[
        merged["_merge"].eq("both")
    ].copy()

    size_mismatches = int(
        both["Bytes_expected"].astype("int64").ne(
            both["Bytes_actual"].astype("int64")
        ).sum()
    )

    sha_mismatches = int(
        both["SHA256_expected"].astype(str).str.lower().ne(
            both["SHA256_actual"].astype(str).str.lower()
        ).sum()
    )

    return {
        "MissingFiles":
            missing,

        "UnexpectedFiles":
            unexpected,

        "SizeMismatches":
            size_mismatches,

        "SHA256Mismatches":
            sha_mismatches,
    }


def read_tabular_header(path):
    path = Path(path)

    lower = path.name.lower()

    try:
        if lower.endswith(".parquet"):
            frame = pd.read_parquet(
                path,
                columns=None,
            )

            return (
                list(
                    frame.columns
                ),
                int(
                    len(frame)
                ),
            )

        if lower.endswith(".csv.gz"):
            frame = pd.read_csv(
                path,
                low_memory=False,
            )

            return (
                list(
                    frame.columns
                ),
                int(
                    len(frame)
                ),
            )

        if lower.endswith(".csv"):
            frame = pd.read_csv(
                path,
                low_memory=False,
            )

            return (
                list(
                    frame.columns
                ),
                int(
                    len(frame)
                ),
            )

    except Exception as error:
        return (
            [],
            -1,
            repr(
                error
            ),
        )

    return (
        [],
        -1,
        "UNSUPPORTED_EXTENSION",
    )


def semantic_flags(columns):
    norms = {
        normalise(
            column
        )
        for column in columns
    }

    joined = " ".join(
        sorted(
            norms
        )
    )

    has_noise = any(
        token in norms
        for token in {
            "noise",
            "noisepercent",
            "noisepercentage",
            "noiselevel",
            "noiselevels",
            "noiselevelpercent",
            "noisepct",
        }
    )

    has_seed = any(
        token in norms
        for token in {
            "seed",
            "repetitionseed",
            "randomseed",
            "runseed",
            "repetition",
        }
    )

    has_technique = any(
        token in norms
        for token in {
            "technique",
            "model",
            "algorithm",
            "method",
        }
    )

    has_apfdc = any(
        "apfdc" in token
        for token in norms
    )

    has_apfd = any(
        (
            "apfd" in token
            and "apfdc" not in token
        )
        for token in norms
    )

    has_delta = any(
        "delta" in token
        or "degradation" in token
        or "retention" in token
        for token in norms
    )

    has_mean = any(
        "mean" in token
        for token in norms
    )

    has_median = any(
        "median" in token
        for token in norms
    )

    return {
        "HasNoiseColumn":
            has_noise,

        "HasSeedColumn":
            has_seed,

        "HasTechniqueColumn":
            has_technique,

        "HasAPFDcColumn":
            has_apfdc,

        "HasAPFDColumn":
            has_apfd,

        "HasDeltaOrRetentionColumn":
            has_delta,

        "HasMeanColumn":
            has_mean,

        "HasMedianColumn":
            has_median,

        "NormalizedColumnsJoined":
            joined,
    }


def classify_candidate(
    path,
    columns,
    rows,
):
    flags = semantic_flags(
        columns
    )

    filename_norm = normalise(
        path.name
    )

    filename_hint = any(
        normalise(
            hint
        )
        in filename_norm
        for hint in METRIC_FILENAME_HINTS
    )

    metric_signal = bool(
        flags["HasAPFDcColumn"]
        or flags["HasAPFDColumn"]
    )

    candidate = bool(
        metric_signal
        and (
            flags["HasNoiseColumn"]
            or flags["HasSeedColumn"]
            or flags["HasTechniqueColumn"]
            or filename_hint
        )
    )

    if not candidate:
        return (
            False,
            "NOT_ANALYSIS_CANDIDATE",
            flags,
            filename_hint,
        )

    if (
        flags["HasNoiseColumn"]
        and flags["HasSeedColumn"]
        and flags["HasTechniqueColumn"]
    ):
        classification = "SEED_LEVEL_LONG"

    elif (
        flags["HasNoiseColumn"]
        and flags["HasTechniqueColumn"]
        and not flags["HasSeedColumn"]
    ):
        classification = "NOISE_TECHNIQUE_SUMMARY"

    elif (
        flags["HasNoiseColumn"]
        and flags["HasSeedColumn"]
        and not flags["HasTechniqueColumn"]
    ):
        classification = "SEED_LEVEL_TECHNIQUE_IN_FILENAME_OR_WIDE"

    elif (
        flags["HasNoiseColumn"]
        and not flags["HasSeedColumn"]
        and not flags["HasTechniqueColumn"]
    ):
        classification = "NOISE_CURVE_TECHNIQUE_IN_FILENAME_OR_WIDE"

    elif (
        flags["HasSeedColumn"]
        and not flags["HasNoiseColumn"]
    ):
        classification = "SEED_SUMMARY_OR_WIDE"

    else:
        classification = "OTHER_APFD_METRIC_CANDIDATE"

    return (
        True,
        classification,
        flags,
        filename_hint,
    )


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def build_output_manifest(paths):
    rows = []

    for path in sorted(
        (
            Path(
                path
            )
            for path in paths
        ),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"Step-0 output missing: {path}"
            )

        rows.append({
            "Path":
                str(
                    path
                ),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Path",
            "Bytes",
            "SHA256",
        ],
    )


# --------------------------------------------------------------------------------------------------
# 4. FREEZE GUARD + SAFE CLEANUP OF FAILED GLOBAL STEP-0 OUTPUT ONLY
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "Global Analysis Step 0 is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to Global Analysis Step 1."
    )

# V1/V2 failed before the Step-0 checkpoint. Remove ONLY our own partial analysis folder(s).
for partial_root in [
    ANALYSIS_ROOT / "Step_0_Cohort_and_Package_Integrity",
    STEP0_ROOT,
]:
    if partial_root.exists():
        shutil.rmtree(
            partial_root,
            ignore_errors=True,
        )

STEP0_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. FINAL REGISTRY COHORT FREEZE
# --------------------------------------------------------------------------------------------------

if not ROOT.is_dir():
    raise FileNotFoundError(
        "Thesis_Experiment root is unavailable. Mount Drive first."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Final completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha_before}"
    )

registry = pd.read_csv(
    REGISTRY,
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)

project_number_col = resolve_column(
    registry,
    "ProjectNumber",
)

project_col = resolve_column(
    registry,
    "Project",
)

project_slug_col = resolve_column(
    registry,
    "ProjectSlug",
)

status_col = resolve_column(
    registry,
    "Status",
)

final_directory_col = resolve_optional_column(
    registry,
    "FinalDirectory",
)

final_package_manifest_col = resolve_optional_column(
    registry,
    "FinalPackageManifest",
)

final_package_root_col = resolve_optional_column(
    registry,
    "FinalPackageRootSHA256",
)

model_fits_col = resolve_optional_column(
    registry,
    "ModelFits",
)

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_col
    ],
    errors="raise",
).astype(
    int
)

if len(
    registry
) != EXPECTED_INCLUDED_PROJECTS:
    raise RuntimeError(
        f"Expected 24 registry rows, found {len(registry)}."
    )

if sorted(
    registry_project_numbers.tolist()
) != EXPECTED_PROJECT_NUMBERS:
    raise RuntimeError(
        "Registry must contain ProjectNumbers 1..24 exactly once."
    )

if registry_project_numbers.duplicated().any():
    raise RuntimeError(
        "Duplicate ProjectNumber values exist."
    )

if registry[
    project_col
].astype(
    str
).duplicated().any():
    raise RuntimeError(
        "Duplicate Project names exist."
    )

if registry[
    project_slug_col
].astype(
    str
).duplicated().any():
    raise RuntimeError(
        "Duplicate ProjectSlug values exist."
    )

if not registry[
    status_col
].astype(
    str
).eq(
    COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Not all 24 projects are COMPLETE_AND_FROZEN."
    )

if registry[
    project_col
].astype(
    str
).eq(
    EXCLUDED_PROJECT
).any():
    raise RuntimeError(
        "Graylog must not appear in the 24-project registered cohort."
    )


# --------------------------------------------------------------------------------------------------
# 6. DISCOVER PROJECT ANALYSIS SOURCES ACROSS LEGACY + MODERN LAYOUTS
# --------------------------------------------------------------------------------------------------

scan_started = time.perf_counter()

project_source_rows = []
candidate_rows = []

final_package_count = 0
legacy_aggregated_only_count = 0

verified_final_package_count = 0
final_package_manifest_failures = 0
final_package_root_mismatches = 0

total_candidates = 0
projects_with_zero_candidates = 0

optional_registry_model_fits_present = 0
optional_registry_model_fits_mismatches = 0

sorted_registry = (
    registry.assign(
        __ProjectNumberInt=registry_project_numbers
    )
    .sort_values(
        "__ProjectNumberInt",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

for ordinal, row in enumerate(
    sorted_registry.to_dict(
        orient="records"
    ),
    start=1,
):
    project_number = parse_int(
        row[
            project_number_col
        ],
        "ProjectNumber",
    )

    project_name = str(
        row[
            project_col
        ]
    ).strip()

    project_slug = str(
        row[
            project_slug_col
        ]
    ).strip()

    aggregated_root = (
        RESULTS
        / "Aggregated"
        / project_slug
    )

    if not aggregated_root.is_dir():
        raise FileNotFoundError(
            f"Project {project_number} frozen Aggregated root missing:\n"
            f"{aggregated_root}"
        )

    # Candidate modern final-package roots are discovered from multiple sources.
    final_root_candidates = []

    deterministic_final_root = (
        RESULTS
        / "Final"
        / project_slug
    )

    if deterministic_final_root.is_dir():
        final_root_candidates.append(
            (
                deterministic_final_root,
                "DETERMINISTIC_RESULTS_FINAL_SLUG",
            )
        )

    if final_directory_col is not None:
        registry_final_directory = str(
            row[
                final_directory_col
            ]
        ).strip()

        if registry_final_directory:
            registry_final_path = Path(
                registry_final_directory
            )

            if registry_final_path.is_dir():
                final_root_candidates.append(
                    (
                        registry_final_path,
                        "REGISTRY_FINAL_DIRECTORY",
                    )
                )

    if final_package_manifest_col is not None:
        registry_manifest_value = str(
            row[
                final_package_manifest_col
            ]
        ).strip()

        if registry_manifest_value:
            registry_manifest_path = Path(
                registry_manifest_value
            )

            if registry_manifest_path.is_file():
                final_root_candidates.append(
                    (
                        registry_manifest_path.parent,
                        "REGISTRY_FINAL_PACKAGE_MANIFEST_PARENT",
                    )
                )

    # Deduplicate equivalent paths.
    unique_final_roots = {}

    for path, source in final_root_candidates:
        unique_final_roots[
            str(
                path.resolve()
            )
        ] = (
            path,
            source,
        )

    final_root_candidates = list(
        unique_final_roots.values()
    )

    if len(
        final_root_candidates
    ) > 1:
        raise RuntimeError(
            f"Project {project_number}: conflicting existing Final roots discovered:\n"
            + "\n".join(
                f" - {source}: {path}"
                for path, source in final_root_candidates
            )
        )

    final_root = None
    final_root_source = ""
    final_package_present = False
    final_package_verified = False
    final_package_root_sha = ""
    final_package_files = 0
    final_package_bytes = 0

    package_missing = 0
    package_unexpected = 0
    package_size_bad = 0
    package_sha_bad = 0
    registry_package_root_value = ""
    registry_package_root_anchor_present = False
    registry_package_root_anchor_match = None

    if final_root_candidates:
        final_root, final_root_source = final_root_candidates[
            0
        ]

        manifest_path = (
            final_root
            / "final_package_manifest.csv"
        )

        if manifest_path.is_file():
            final_package_present = True
            final_package_count += 1

            manifest_frame = pd.read_csv(
                manifest_path,
                low_memory=False,
            )

            manifest_root = root_hash(
                manifest_frame
            )

            actual_manifest = build_payload_manifest(
                final_root,
                manifest_path,
            )

            actual_root = root_hash(
                actual_manifest
            )

            comparison = compare_manifests(
                manifest_frame,
                actual_manifest,
            )

            package_missing = comparison[
                "MissingFiles"
            ]

            package_unexpected = comparison[
                "UnexpectedFiles"
            ]

            package_size_bad = comparison[
                "SizeMismatches"
            ]

            package_sha_bad = comparison[
                "SHA256Mismatches"
            ]

            final_package_files = int(
                len(
                    actual_manifest
                )
            )

            final_package_bytes = int(
                actual_manifest[
                    "Bytes"
                ].sum()
            )

            final_package_root_sha = (
                actual_root
            )

            final_package_verified = bool(
                manifest_root
                == actual_root
                and package_missing
                == 0
                and package_unexpected
                == 0
                and package_size_bad
                == 0
                and package_sha_bad
                == 0
            )

            if not final_package_verified:
                final_package_manifest_failures += 1

            else:
                verified_final_package_count += 1

            if final_package_root_col is not None:
                registry_package_root_value = str(
                    row[
                        final_package_root_col
                    ]
                ).strip().lower()

                if registry_package_root_value:
                    registry_package_root_anchor_present = True

                    if not is_sha256(
                        registry_package_root_value
                    ):
                        final_package_root_mismatches += 1
                        registry_package_root_anchor_match = False

                    else:
                        registry_package_root_anchor_match = bool(
                            registry_package_root_value
                            == actual_root
                        )

                        if not registry_package_root_anchor_match:
                            final_package_root_mismatches += 1

        else:
            # Existing directory without manifest is not treated as a frozen compact package.
            final_root = None
            final_root_source = ""

    if not final_package_present:
        legacy_aggregated_only_count += 1

    # Optional registry ModelFits diagnostic only.
    registry_model_fits_value = ""

    if model_fits_col is not None:
        registry_model_fits_value = str(
            row[
                model_fits_col
            ]
        ).strip()

        if registry_model_fits_value:
            optional_registry_model_fits_present += 1

            parsed_model_fits = parse_int(
                registry_model_fits_value,
                f"Project {project_number} optional registry ModelFits",
            )

            if parsed_model_fits != 1080:
                optional_registry_model_fits_mismatches += 1

    # Search roots:
    # - Aggregated is mandatory and authoritative for legacy projects.
    # - Final is added when a verified compact package exists.
    search_roots = [
        (
            aggregated_root,
            "AGGREGATED",
        )
    ]

    if (
        final_package_present
        and final_package_verified
        and final_root is not None
    ):
        search_roots.append(
            (
                final_root,
                "FINAL_PACKAGE",
            )
        )

    project_candidate_count = 0
    project_candidate_classifications = {}

    seen_paths = set()

    for search_root, source_group in search_roots:
        candidate_paths = []

        for path in search_root.rglob("*"):
            if not path.is_file():
                continue

            lower_name = path.name.lower()

            if (
                lower_name.endswith(".csv")
                or lower_name.endswith(".csv.gz")
                or lower_name.endswith(".parquet")
            ):
                candidate_paths.append(
                    path
                )

        for path in sorted(
            candidate_paths,
            key=str,
        ):
            path_key = str(
                path.resolve()
            )

            if path_key in seen_paths:
                continue

            seen_paths.add(
                path_key
            )

            read_result = read_tabular_header(
                path
            )

            if len(
                read_result
            ) == 3:
                columns, row_count, read_error = read_result

            else:
                columns, row_count = read_result
                read_error = ""

            if row_count < 0:
                # An unreadable random support file is not itself a global-analysis failure.
                continue

            (
                is_candidate,
                classification,
                flags,
                filename_hint,
            ) = classify_candidate(
                path,
                columns,
                row_count,
            )

            if not is_candidate:
                continue

            relative_to_root = path.relative_to(
                search_root
            ).as_posix()

            candidate_record = {
                "ProjectNumber":
                    project_number,

                "Project":
                    project_name,

                "ProjectSlug":
                    project_slug,

                "SourceGroup":
                    source_group,

                "SourceRoot":
                    str(
                        search_root
                    ),

                "Path":
                    str(
                        path
                    ),

                "RelativePath":
                    relative_to_root,

                "Name":
                    path.name,

                "Extension":
                    (
                        ".csv.gz"
                        if path.name.lower().endswith(
                            ".csv.gz"
                        )
                        else path.suffix.lower()
                    ),

                "Rows":
                    int(
                        row_count
                    ),

                "Columns":
                    int(
                        len(
                            columns
                        )
                    ),

                "ColumnsJSON":
                    json.dumps(
                        columns,
                        separators=(
                            ",",
                            ":",
                        ),
                    ),

                "SchemaSignature":
                    "|".join(
                        sorted(
                            normalise(
                                column
                            )
                            for column in columns
                        )
                    ),

                "Classification":
                    classification,

                "FilenameHint":
                    filename_hint,

                "HasNoiseColumn":
                    flags[
                        "HasNoiseColumn"
                    ],

                "HasSeedColumn":
                    flags[
                        "HasSeedColumn"
                    ],

                "HasTechniqueColumn":
                    flags[
                        "HasTechniqueColumn"
                    ],

                "HasAPFDcColumn":
                    flags[
                        "HasAPFDcColumn"
                    ],

                "HasAPFDColumn":
                    flags[
                        "HasAPFDColumn"
                    ],

                "HasDeltaOrRetentionColumn":
                    flags[
                        "HasDeltaOrRetentionColumn"
                    ],

                "HasMeanColumn":
                    flags[
                        "HasMeanColumn"
                    ],

                "HasMedianColumn":
                    flags[
                        "HasMedianColumn"
                    ],

                "Bytes":
                    int(
                        path.stat().st_size
                    ),

                "SHA256":
                    sha256_file(
                        path
                    ),
            }

            candidate_rows.append(
                candidate_record
            )

            project_candidate_count += 1
            total_candidates += 1

            project_candidate_classifications[
                classification
            ] = (
                project_candidate_classifications.get(
                    classification,
                    0,
                )
                + 1
            )

    if project_candidate_count == 0:
        projects_with_zero_candidates += 1

    project_source_rows.append({
        "ProjectNumber":
            project_number,

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "Status":
            str(
                row[
                    status_col
                ]
            ).strip(),

        "AggregatedRoot":
            str(
                aggregated_root
            ),

        "AggregatedRootExists":
            aggregated_root.is_dir(),

        "FinalPackagePresent":
            final_package_present,

        "FinalPackageVerified":
            final_package_verified,

        "FinalPackageRoot":
            (
                str(
                    final_root
                )
                if final_root is not None
                else ""
            ),

        "FinalPackageRootDiscovery":
            final_root_source,

        "FinalPackageFiles":
            final_package_files,

        "FinalPackageBytes":
            final_package_bytes,

        "FinalPackageRootSHA256":
            final_package_root_sha,

        "FinalPackageMissingFiles":
            package_missing,

        "FinalPackageUnexpectedFiles":
            package_unexpected,

        "FinalPackageSizeMismatches":
            package_size_bad,

        "FinalPackageSHA256Mismatches":
            package_sha_bad,

        "RegistryPackageRootRaw":
            registry_package_root_value,

        "RegistryPackageRootAnchorPresent":
            registry_package_root_anchor_present,

        "RegistryPackageRootAnchorMatch":
            (
                registry_package_root_anchor_match
                if registry_package_root_anchor_match is not None
                else ""
            ),

        "RegistryModelFitsRaw":
            registry_model_fits_value,

        "AnalysisCandidateFiles":
            project_candidate_count,

        "CandidateClassificationsJSON":
            json.dumps(
                project_candidate_classifications,
                sort_keys=True,
                separators=(
                    ",",
                    ":",
                ),
            ),

        "AnalysisLayout":
            (
                "MODERN_FINAL_PLUS_AGGREGATED"
                if final_package_present
                and final_package_verified
                else "LEGACY_OR_AGGREGATED_ONLY"
            ),
    })

    if (
        ordinal % 4 == 0
        or ordinal
        == EXPECTED_INCLUDED_PROJECTS
    ):
        print(
            f"  Project sources scanned: {ordinal}/{EXPECTED_INCLUDED_PROJECTS}"
        )


scan_seconds = (
    time.perf_counter()
    - scan_started
)

project_source_inventory = (
    pd.DataFrame(
        project_source_rows
    )
    .sort_values(
        "ProjectNumber",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

candidate_inventory = (
    pd.DataFrame(
        candidate_rows
    )
    .sort_values(
        [
            "ProjectNumber",
            "SourceGroup",
            "Classification",
            "Path",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

if candidate_inventory.empty:
    raise RuntimeError(
        "No APFD/APFDc analysis candidates were discovered."
    )


# --------------------------------------------------------------------------------------------------
# 7. SCHEMA + PROJECT CANDIDATE SUMMARIES
# --------------------------------------------------------------------------------------------------

schema_summary = (
    candidate_inventory.groupby(
        [
            "Classification",
            "SchemaSignature",
        ],
        dropna=False,
        sort=True,
    )
    .agg(
        Projects=(
            "ProjectNumber",
            "nunique",
        ),

        Files=(
            "Path",
            "size",
        ),

        MinRows=(
            "Rows",
            "min",
        ),

        MaxRows=(
            "Rows",
            "max",
        ),

        ProjectNumbers=(
            "ProjectNumber",
            lambda values:
                json.dumps(
                    sorted(
                        {
                            int(
                                value
                            )
                            for value in values
                        }
                    ),
                    separators=(
                        ",",
                        ":",
                    ),
                ),
        ),

        ExamplePaths=(
            "Path",
            lambda values:
                json.dumps(
                    list(
                        values
                    )[
                        :5
                    ],
                    separators=(
                        ",",
                        ":",
                    ),
                ),
        ),
    )
    .reset_index()
)

project_candidate_summary = (
    candidate_inventory.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
        ],
        sort=True,
    )
    .agg(
        CandidateFiles=(
            "Path",
            "size",
        ),

        CandidateSchemas=(
            "SchemaSignature",
            "nunique",
        ),

        CandidateClasses=(
            "Classification",
            "nunique",
        ),

        SeedLevelLongFiles=(
            "Classification",
            lambda values:
                int(
                    pd.Series(
                        values
                    ).eq(
                        "SEED_LEVEL_LONG"
                    ).sum()
                ),
        ),

        NoiseTechniqueSummaryFiles=(
            "Classification",
            lambda values:
                int(
                    pd.Series(
                        values
                    ).eq(
                        "NOISE_TECHNIQUE_SUMMARY"
                    ).sum()
                ),
        ),

        SeedTechniqueInFilenameOrWideFiles=(
            "Classification",
            lambda values:
                int(
                    pd.Series(
                        values
                    ).eq(
                        "SEED_LEVEL_TECHNIQUE_IN_FILENAME_OR_WIDE"
                    ).sum()
                ),
        ),

        NoiseCurveTechniqueInFilenameOrWideFiles=(
            "Classification",
            lambda values:
                int(
                    pd.Series(
                        values
                    ).eq(
                        "NOISE_CURVE_TECHNIQUE_IN_FILENAME_OR_WIDE"
                    ).sum()
                ),
        ),
    )
    .reset_index()
)

# Preserve all 24 rows even if a future diagnostic project had zero candidates.
project_candidate_summary = (
    project_source_inventory[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
        ]
    ]
    .merge(
        project_candidate_summary,
        on=[
            "ProjectNumber",
            "Project",
            "ProjectSlug",
        ],
        how="left",
        validate="one_to_one",
    )
    .fillna(
        0
    )
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY GRAYLOG EXCLUSION
# --------------------------------------------------------------------------------------------------

if not P24_INELIGIBLE_PATH.is_file():
    raise FileNotFoundError(
        "Frozen Project-24 ineligible-candidate inventory is missing:\n"
        f"{P24_INELIGIBLE_PATH}"
    )

ineligible_candidates = pd.read_csv(
    P24_INELIGIBLE_PATH,
    low_memory=False,
)

required_exclusion_columns = [
    "Project",
    "ProtocolEligible",
    "InspectionStatus",
    "RawEvaluationFailures",
    "ModelEvaluationFailures",
    "EligibilityReason",
]

missing_exclusion_columns = [
    column
    for column in required_exclusion_columns
    if column not in ineligible_candidates.columns
]

if missing_exclusion_columns:
    raise RuntimeError(
        "Frozen ineligible-candidate file lacks columns:\n"
        + "\n".join(
            missing_exclusion_columns
        )
    )

graylog_rows = ineligible_candidates.loc[
    ineligible_candidates[
        "Project"
    ].astype(
        str
    ).eq(
        EXCLUDED_PROJECT
    )
].copy()

if len(
    graylog_rows
) != 1:
    raise RuntimeError(
        "Expected exactly one frozen Graylog exclusion record."
    )

graylog = graylog_rows.iloc[
    0
]

graylog_raw_eval_failures = parse_int(
    graylog[
        "RawEvaluationFailures"
    ],
    "Graylog RawEvaluationFailures",
)

graylog_model_eval_failures = parse_int(
    graylog[
        "ModelEvaluationFailures"
    ],
    "Graylog ModelEvaluationFailures",
)

graylog_protocol_eligible = parse_bool(
    graylog[
        "ProtocolEligible"
    ]
)

graylog_inspection_status = str(
    graylog[
        "InspectionStatus"
    ]
).strip()

graylog_reason = str(
    graylog[
        "EligibilityReason"
    ]
).strip()

graylog_exclusion_pass = bool(
    not graylog_protocol_eligible
    and graylog_inspection_status
    == "INELIGIBLE"
    and graylog_raw_eval_failures
    == 0
    and graylog_model_eval_failures
    == 0
    and "No raw evaluation failures"
    in graylog_reason
    and "No model evaluation failures"
    in graylog_reason
)

exclusion_record = pd.DataFrame(
    [
        {
            "Project":
                EXCLUDED_PROJECT,

            "IncludedInGlobalAnalysis":
                False,

            "ProtocolEligible":
                graylog_protocol_eligible,

            "InspectionStatus":
                graylog_inspection_status,

            "RawEvaluationFailures":
                graylog_raw_eval_failures,

            "ModelEvaluationFailures":
                graylog_model_eval_failures,

            "EligibilityReason":
                graylog_reason,

            "EvidencePath":
                str(
                    P24_INELIGIBLE_PATH
                ),

            "EvidenceSHA256":
                sha256_file(
                    P24_INELIGIBLE_PATH
                ),

            "ExclusionValidationPass":
                graylog_exclusion_pass,
        }
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. GLOBAL VALIDATION
# --------------------------------------------------------------------------------------------------

registry_sha_after_reads = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after_reads,
    registry_sha_after_reads
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Registry rows",
    24,
    len(
        registry
    ),
    len(
        registry
    )
    == 24,
)

add_check(
    checks,
    "ProjectNumbers 1..24 exactly",
    EXPECTED_PROJECT_NUMBERS,
    sorted(
        registry_project_numbers.tolist()
    ),
    sorted(
        registry_project_numbers.tolist()
    )
    == EXPECTED_PROJECT_NUMBERS,
)

add_check(
    checks,
    "COMPLETE_AND_FROZEN projects",
    24,
    int(
        registry[
            status_col
        ].astype(
            str
        ).eq(
            COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            status_col
        ].astype(
            str
        ).eq(
            COMPLETE_STATUS
        ).sum()
    )
    == 24,
)

add_check(
    checks,
    "Frozen Aggregated project roots",
    24,
    int(
        project_source_inventory[
            "AggregatedRootExists"
        ].astype(
            bool
        ).sum()
    ),
    bool(
        project_source_inventory[
            "AggregatedRootExists"
        ].astype(
            bool
        ).all()
    ),
)

add_check(
    checks,
    "Projects with at least one APFD/APFDc analysis candidate",
    24,
    int(
        (
            project_source_inventory[
                "AnalysisCandidateFiles"
            ].astype(
                int
            )
            > 0
        ).sum()
    ),
    int(
        (
            project_source_inventory[
                "AnalysisCandidateFiles"
            ].astype(
                int
            )
            > 0
        ).sum()
    )
    == 24,
)

add_check(
    checks,
    "Projects with zero analysis candidates",
    0,
    projects_with_zero_candidates,
    projects_with_zero_candidates
    == 0,
)

add_check(
    checks,
    "Verified modern final-package manifests",
    final_package_count,
    verified_final_package_count,
    verified_final_package_count
    == final_package_count,
)

add_check(
    checks,
    "Final-package manifest/file failures",
    0,
    final_package_manifest_failures,
    final_package_manifest_failures
    == 0,
)

add_check(
    checks,
    "Nonblank registry final-package-root mismatches",
    0,
    final_package_root_mismatches,
    final_package_root_mismatches
    == 0,
)

add_check(
    checks,
    "Nonblank registry ModelFits mismatches",
    0,
    optional_registry_model_fits_mismatches,
    optional_registry_model_fits_mismatches
    == 0,
)

add_check(
    checks,
    "Graylog registry rows",
    0,
    int(
        registry[
            project_col
        ].astype(
            str
        ).eq(
            EXCLUDED_PROJECT
        ).sum()
    ),
    int(
        registry[
            project_col
        ].astype(
            str
        ).eq(
            EXCLUDED_PROJECT
        ).sum()
    )
    == 0,
)

add_check(
    checks,
    "Graylog raw evaluation failures",
    0,
    graylog_raw_eval_failures,
    graylog_raw_eval_failures
    == 0,
)

add_check(
    checks,
    "Graylog model evaluation failures",
    0,
    graylog_model_eval_failures,
    graylog_model_eval_failures
    == 0,
)

add_check(
    checks,
    "Graylog exclusion validation",
    True,
    graylog_exclusion_pass,
    graylog_exclusion_pass,
)

add_check(
    checks,
    "Independent empirical unit",
    "Project (N=24)",
    "Project (N=24)",
    True,
)

add_check(
    checks,
    "Raw condition outputs accessed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Models trained",
    False,
    False,
    True,
)

add_check(
    checks,
    "Statistical tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after_reads
    != registry_sha_before,
    registry_sha_after_reads
    == registry_sha_before,
)

validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nGlobal Analysis Step 0 V3 validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

print(
    "\nPer-project analysis-source summary:"
)

try:
    from IPython.display import display

    display(
        project_source_inventory[
            [
                "ProjectNumber",
                "Project",
                "AnalysisLayout",
                "FinalPackagePresent",
                "FinalPackageVerified",
                "AnalysisCandidateFiles",
                "CandidateClassificationsJSON",
            ]
        ]
    )

except Exception:
    print(
        project_source_inventory[
            [
                "ProjectNumber",
                "Project",
                "AnalysisLayout",
                "FinalPackagePresent",
                "FinalPackageVerified",
                "AnalysisCandidateFiles",
                "CandidateClassificationsJSON",
            ]
        ].to_string(
            index=False
        )
    )

print(
    "\nDiscovered schema classes:"
)

schema_display = (
    schema_summary[
        [
            "Classification",
            "Projects",
            "Files",
            "MinRows",
            "MaxRows",
            "ProjectNumbers",
        ]
    ]
    .sort_values(
        [
            "Projects",
            "Files",
        ],
        ascending=[
            False,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

try:
    from IPython.display import display

    display(
        schema_display
    )

except Exception:
    print(
        schema_display.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 0 V3 VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. FREEZE STEP-0 INVENTORIES
# --------------------------------------------------------------------------------------------------

atomic_copy(
    REGISTRY,
    REGISTRY_SNAPSHOT,
)

atomic_csv(
    PROJECT_SOURCE_INVENTORY_PATH,
    project_source_inventory,
)

atomic_csv(
    CANDIDATE_INVENTORY_PATH,
    candidate_inventory,
)

atomic_csv(
    SCHEMA_SUMMARY_PATH,
    schema_summary,
)

atomic_csv(
    PROJECT_CANDIDATE_SUMMARY_PATH,
    project_candidate_summary,
)

atomic_csv(
    EXCLUSION_RECORD_PATH,
    exclusion_record,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "GlobalAnalysisVersion":
        GLOBAL_ANALYSIS_VERSION,

    "Step":
        "GLOBAL_ANALYSIS_STEP_0",

    "Status":
        STEP0_STATUS,

    "CodeRevision":
        STEP0_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IncludedProjects":
        24,

    "IncludedProjectNumbers":
        EXPECTED_PROJECT_NUMBERS,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "ExpectedNoiseLevels":
        EXPECTED_NOISE_LEVELS,

    "ExpectedSeeds":
        EXPECTED_SEEDS,

    "ExpectedTechniques":
        EXPECTED_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "ModernFinalPackagesPresent":
        final_package_count,

    "ModernFinalPackagesVerified":
        verified_final_package_count,

    "LegacyOrAggregatedOnlyProjects":
        legacy_aggregated_only_count,

    "AnalysisCandidateFiles":
        total_candidates,

    "DiscoveredSchemaSignatures":
        int(
            candidate_inventory[
                "SchemaSignature"
            ].nunique()
        ),

    "DiscoveredCandidateClasses":
        sorted(
            candidate_inventory[
                "Classification"
            ].unique().tolist()
        ),

    "ProjectsWithZeroAnalysisCandidates":
        projects_with_zero_candidates,

    "OptionalRegistryModelFitsPresent":
        optional_registry_model_fits_present,

    "OptionalRegistryModelFitsMissing":
        24
        - optional_registry_model_fits_present,

    "OptionalRegistryModelFitsMismatches":
        optional_registry_model_fits_mismatches,

    "ExcludedProject":
        EXCLUDED_PROJECT,

    "ExcludedProjectReason":
        graylog_reason,

    "RawConditionOutputsAccessed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "ModelsTrained":
        False,

    "StatisticalTestsExecuted":
        False,

    "SourceDiscoverySeconds":
        float(
            scan_seconds
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 1 — "
            "NORMALIZE DISCOVERED LEGACY/MODERN ANALYSIS SOURCES "
            "INTO THE CANONICAL CROSS-PROJECT MASTER DATASET"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            STEP0_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "IncludedProjects":
            24,

        "IndependentEmpiricalUnit":
            "Project",

        "ReadyForGlobalAnalysisStep1":
            True,
    },
)

output_paths = [
    REGISTRY_SNAPSHOT,
    PROJECT_SOURCE_INVENTORY_PATH,
    CANDIDATE_INVENTORY_PATH,
    SCHEMA_SUMMARY_PATH,
    PROJECT_CANDIDATE_SUMMARY_PATH,
    EXCLUSION_RECORD_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

registry_sha_final = sha256_file(
    REGISTRY
)

if registry_sha_final != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during Global Analysis Step 0."
    )

checkpoint = {
    "CheckpointType":
        "GLOBAL_ANALYSIS_24_PROJECT_COHORT_AND_ANALYSIS_SOURCE_DISCOVERY",

    "GlobalAnalysisVersion":
        GLOBAL_ANALYSIS_VERSION,

    "Status":
        STEP0_STATUS,

    "CodeRevision":
        STEP0_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IncludedProjectNumbers":
        EXPECTED_PROJECT_NUMBERS,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "ProjectSourceInventoryPath":
        str(
            PROJECT_SOURCE_INVENTORY_PATH
        ),

    "MetricCandidateInventoryPath":
        str(
            CANDIDATE_INVENTORY_PATH
        ),

    "MetricSchemaSummaryPath":
        str(
            SCHEMA_SUMMARY_PATH
        ),

    "ProjectCandidateSummaryPath":
        str(
            PROJECT_CANDIDATE_SUMMARY_PATH
        ),

    "ExcludedProjectRecordPath":
        str(
            EXCLUSION_RECORD_PATH
        ),

    "ValidationPath":
        str(
            VALIDATION_PATH
        ),

    "ReportPath":
        str(
            REPORT_PATH
        ),

    "StatusPath":
        str(
            STATUS_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "ModernFinalPackagesPresent":
        final_package_count,

    "ModernFinalPackagesVerified":
        verified_final_package_count,

    "LegacyOrAggregatedOnlyProjects":
        legacy_aggregated_only_count,

    "AnalysisCandidateFiles":
        total_candidates,

    "DiscoveredSchemaSignatures":
        int(
            candidate_inventory[
                "SchemaSignature"
            ].nunique()
        ),

    "ExcludedProject":
        exclusion_record.iloc[
            0
        ].to_dict(),

    "RawConditionOutputsAccessed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "ModelsTrained":
        False,

    "StatisticalTestsExecuted":
        False,

    "ReadyForGlobalAnalysisStep1":
        True,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 1 — "
            "NORMALIZE DISCOVERED LEGACY/MODERN ANALYSIS SOURCES "
            "INTO THE CANONICAL CROSS-PROJECT MASTER DATASET"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 142
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 1 / STEP 0 V3 RESULT ==="
)

print(
    "=" * 142
)

print(
    "Global analysis version:",
    GLOBAL_ANALYSIS_VERSION,
)

print(
    "\nCohort freeze:"
)

print(
    "Included projects: 24 / 24"
)

print(
    "ProjectNumbers: 1..24 exactly once"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Excluded source project:",
    EXCLUDED_PROJECT,
)

print(
    "Excluded project raw/model evaluation failures:",
    graylog_raw_eval_failures,
    "/",
    graylog_model_eval_failures,
)

print(
    "\nHistorical output-layout discovery:"
)

print(
    "Modern final packages present:",
    final_package_count,
)

print(
    "Modern final packages verified:",
    verified_final_package_count,
)

print(
    "Legacy / Aggregated-only projects:",
    legacy_aggregated_only_count,
)

print(
    "All 24 Aggregated roots present:",
    bool(
        project_source_inventory[
            "AggregatedRootExists"
        ].astype(
            bool
        ).all()
    ),
)

print(
    "\nAnalysis-source discovery:"
)

print(
    "APFD/APFDc candidate files discovered:",
    total_candidates,
)

print(
    "Projects with >=1 candidate:",
    int(
        (
            project_source_inventory[
                "AnalysisCandidateFiles"
            ].astype(
                int
            )
            > 0
        ).sum()
    ),
    "/ 24",
)

print(
    "Projects with zero candidates:",
    projects_with_zero_candidates,
)

print(
    "Unique candidate schema signatures:",
    int(
        candidate_inventory[
            "SchemaSignature"
        ].nunique()
    ),
)

print(
    "Candidate classes:",
    sorted(
        candidate_inventory[
            "Classification"
        ].unique().tolist()
    ),
)

print(
    "\nRegistry-schema evolution diagnostics:"
)

print(
    "Optional ModelFits metadata present / missing:",
    optional_registry_model_fits_present,
    "/",
    24
    - optional_registry_model_fits_present,
)

print(
    "Nonblank ModelFits mismatches:",
    optional_registry_model_fits_mismatches,
)

print(
    "\nIsolation:"
)

print(
    "Raw condition outputs accessed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "Models trained: False"
)

print(
    "Statistical tests executed: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)

print(
    "\nGlobal Analysis Step 0 checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "GLOBAL ANALYSIS STEP 1 — NORMALIZE DISCOVERED LEGACY/MODERN ANALYSIS SOURCES "
    "INTO THE CANONICAL CROSS-PROJECT MASTER DATASET"
)

print(
    "\nSTATUS:",
    STEP0_STATUS,
)

print(
    "=" * 142
)


=== THESIS GLOBAL ANALYSIS — CELL 1 / STEP 0 V3: COHORT + ANALYSIS-SOURCE DISCOVERY FREEZE ===
  Project sources scanned: 4/24
  Project sources scanned: 8/24
  Project sources scanned: 12/24
  Project sources scanned: 16/24
  Project sources scanned: 20/24
  Project sources scanned: 24/24

Global Analysis Step 0 V3 validation:


,Check,Expected,Actual,Pass
0,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
1,Registry rows,24,24,True
2,ProjectNumbers 1..24 exactly,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
3,COMPLETE_AND_FROZEN projects,24,24,True
4,Frozen Aggregated project roots,24,24,True
5,Projects with at least one APFD/APFDc analysis...,24,24,True
6,Projects with zero analysis candidates,0,0,True
7,Verified modern final-package manifests,14,14,True
8,Final-package manifest/file failures,0,0,True
9,Nonblank registry final-package-root mismatches,0,0,True



Per-project analysis-source summary:


,ProjectNumber,Project,AnalysisLayout,FinalPackagePresent,FinalPackageVerified,AnalysisCandidateFiles,CandidateClassificationsJSON
0,1,Angel-ML@angel,LEGACY_OR_AGGREGATED_ONLY,False,False,7,"{""NOISE_TECHNIQUE_SUMMARY"":2,""OTHER_APFD_METRI..."
1,2,apache@airavata,LEGACY_OR_AGGREGATED_ONLY,False,False,5,"{""NOISE_TECHNIQUE_SUMMARY"":2,""SEED_LEVEL_LONG"":3}"
2,3,b2ihealthcare@snow-owl,LEGACY_OR_AGGREGATED_ONLY,False,False,6,"{""NOISE_TECHNIQUE_SUMMARY"":1,""OTHER_APFD_METRI..."
3,4,eclipse@paho.mqtt.java,LEGACY_OR_AGGREGATED_ONLY,False,False,6,"{""NOISE_TECHNIQUE_SUMMARY"":1,""OTHER_APFD_METRI..."
4,5,thinkaurelius@titan,LEGACY_OR_AGGREGATED_ONLY,False,False,8,"{""NOISE_TECHNIQUE_SUMMARY"":2,""SEED_LEVEL_LONG"":6}"
5,6,eclipse@jetty.project,LEGACY_OR_AGGREGATED_ONLY,False,False,14,"{""NOISE_TECHNIQUE_SUMMARY"":5,""OTHER_APFD_METRI..."
6,7,CompEvol@beast2,LEGACY_OR_AGGREGATED_ONLY,False,False,73,"{""NOISE_TECHNIQUE_SUMMARY"":2,""OTHER_APFD_METRI..."
7,8,optimatika@ojAlgo,LEGACY_OR_AGGREGATED_ONLY,False,False,72,"{""NOISE_TECHNIQUE_SUMMARY"":2,""SEED_LEVEL_LONG""..."
8,9,camunda@camunda-bpm-platform,LEGACY_OR_AGGREGATED_ONLY,False,False,18,"{""NOISE_TECHNIQUE_SUMMARY"":6,""SEED_LEVEL_LONG""..."
9,10,spring-cloud@spring-cloud-dataflow,LEGACY_OR_AGGREGATED_ONLY,False,False,17,"{""NOISE_TECHNIQUE_SUMMARY"":6,""OTHER_APFD_METRI..."



Discovered schema classes:


,Classification,Projects,Files,MinRows,MaxRows,ProjectNumbers
0,SEED_LEVEL_LONG,14,99,14,171990,"[11,12,13,14,15,16,17,18,19,20,21,22,23,24]"
1,SEED_LEVEL_LONG,14,99,7,1890,"[11,12,13,14,15,16,17,18,19,20,21,22,23,24]"
2,NOISE_TECHNIQUE_SUMMARY,14,28,63,63,"[11,12,13,14,15,16,17,18,19,20,21,22,23,24]"
3,NOISE_TECHNIQUE_SUMMARY,14,28,63,63,"[11,12,13,14,15,16,17,18,19,20,21,22,23,24]"
4,SEED_LEVEL_LONG,14,28,1890,1890,"[11,12,13,14,15,16,17,18,19,20,21,22,23,24]"
5,SEED_LEVEL_LONG,7,15,7,1890,"[1,2,3,4,5,7,8]"
6,SEED_LEVEL_LONG,5,10,224,98280,"[3,4,5,7,8]"
7,SEED_LEVEL_LONG,2,126,30,30,"[7,8]"
8,NOISE_TECHNIQUE_SUMMARY,2,8,7,63,"[9,10]"
9,SEED_LEVEL_LONG,2,4,63,1890,"[1,2]"



=== THESIS GLOBAL ANALYSIS — CELL 1 / STEP 0 V3 RESULT ===
Global analysis version: THESIS_GLOBAL_ANALYSIS_V1_24_ELIGIBLE_PROJECTS

Cohort freeze:
Included projects: 24 / 24
ProjectNumbers: 1..24 exactly once
Independent empirical unit: Project (N=24)
Excluded source project: Graylog2@graylog2-server
Excluded project raw/model evaluation failures: 0 / 0

Historical output-layout discovery:
Modern final packages present: 14
Modern final packages verified: 14
Legacy / Aggregated-only projects: 10
All 24 Aggregated roots present: True

Analysis-source discovery:
APFD/APFDc candidate files discovered: 508
Projects with >=1 candidate: 24 / 24
Projects with zero candidates: 0
Unique candidate schema signatures: 47
Candidate classes: ['NOISE_TECHNIQUE_SUMMARY', 'OTHER_APFD_METRIC_CANDIDATE', 'SEED_LEVEL_LONG']

Registry-schema evolution diagnostics:
Optional ModelFits metadata present / missing: 17 / 7
Nonblank ModelFits mismatches: 0

Isolation:
Raw condition outputs accessed: False
Project

In [4]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 2 / STEP 1A
# CANONICAL PROJECT-RUN SOURCE RESOLUTION ACROSS 24 LEGACY/MODERN PROJECTS
# ==================================================================================================
#
# PURPOSE
# -------
# Step 0 proved that all 24 frozen projects are present, but also showed:
#   - 10 legacy / Aggregated-only layouts
#   - 14 modern Final + Aggregated layouts
#   - 508 APFD/APFDc candidate files
#   - 47 schema signatures
#
# Therefore we DO NOT yet concatenate data.
#
# This cell resolves exactly ONE scientifically authoritative seed-level project-run source
# per project using content, not filenames alone.
#
# Required scientific shape for an eligible project-run source:
#   - exactly 1,890 rows
#   - one row for every 9 noise levels × 30 seeds × 7 techniques
#   - contains noise, seed, technique, APFDc and APFD
#   - is NOT a delta/degradation/retention table
#   - APFD/APFDc values finite and in [0, 1]
#
# If multiple files satisfy the contract:
#   - normalize their scientific content;
#   - require exact scientific equivalence (float tolerance 1e-12);
#   - then choose deterministically, preferring verified Final-package copies and
#     recognizable project-run filenames.
#
# If two valid-looking files disagree scientifically, THIS CELL FAILS rather than guessing.
#
# OUTPUT
# ------
# A frozen 24-row source map. Global Analysis Step 1B will consume ONLY this map to build the
# canonical 45,360-row master dataset.
#
# NO statistics are executed here.
# NO project output or registry file is modified.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 142)
print("=== THESIS GLOBAL ANALYSIS — CELL 2 / STEP 1A: CANONICAL PROJECT-RUN SOURCE RESOLUTION ===")
print("=" * 142)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT CONTRACT
# --------------------------------------------------------------------------------------------------

STEP0_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP0_24_PROJECT_COHORT_AND_ANALYSIS_SOURCE_DISCOVERY_FROZEN"
)

EXPECTED_STEP0_CHECKPOINT_SHA256 = (
    "b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d5436cbcb4c97ca00fb"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

STEP1A_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1A_CANONICAL_PROJECT_RUN_SOURCES_FROZEN"
)

STEP1A_CODE_REVISION = (
    "GLOBAL_ANALYSIS_STEP1A_V1_1890_ROW_GRID_CONTENT_EQUIVALENCE_SOURCE_RESOLUTION"
)

EXPECTED_PROJECTS = 24
EXPECTED_PROJECT_NUMBERS = list(range(1, 25))

EXPECTED_NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
EXPECTED_SEEDS = list(range(1, 31))

EXPECTED_TECHNIQUES = [
    "LatestFail",
    "LightGBM",
    "NaiveBayes",
    "QTF-Avg",
    "Random",
    "RandomForest",
    "XGBoost",
]

EXPECTED_ROWS_PER_PROJECT = 9 * 30 * 7  # 1,890
EXPECTED_GLOBAL_ROWS = 24 * EXPECTED_ROWS_PER_PROJECT  # 45,360

FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"

STEP0_ROOT = (
    ANALYSIS_ROOT
    / "Step_0_Cohort_and_Analysis_Source_Discovery"
)

STEP0_CHECKPOINT = (
    NOTES
    / "global_analysis_step0_checkpoint.json"
)

STEP0_CANDIDATE_INVENTORY = (
    STEP0_ROOT
    / "global_analysis_metric_candidate_inventory.csv"
)

STEP0_PROJECT_SOURCE_INVENTORY = (
    STEP0_ROOT
    / "global_analysis_project_source_inventory.csv"
)

STEP1A_ROOT = (
    ANALYSIS_ROOT
    / "Step_1A_Canonical_ProjectRun_Source_Resolution"
)

SOURCE_MAP_PATH = (
    STEP1A_ROOT
    / "global_analysis_canonical_project_run_source_map.csv"
)

CANDIDATE_AUDIT_PATH = (
    STEP1A_ROOT
    / "global_analysis_project_run_candidate_audit.csv"
)

VALIDATION_PATH = (
    STEP1A_ROOT
    / "global_analysis_step1a_validation.csv"
)

REPORT_PATH = (
    STEP1A_ROOT
    / "global_analysis_step1a_report.json"
)

STATUS_PATH = (
    STEP1A_ROOT
    / "global_analysis_step1a_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP1A_ROOT
    / "global_analysis_step1a_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_step1a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def normalise(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


def atomic_csv(path, dataframe):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def build_output_manifest(paths):
    rows = []

    for path in sorted(
        (Path(path) for path in paths),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"Step-1A output missing: {path}"
            )

        rows.append({
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(
        rows,
        columns=["Path", "Bytes", "SHA256"],
    )


def read_table(path):
    path = Path(path)
    lower = path.name.lower()

    if lower.endswith(".parquet"):
        return pd.read_parquet(path)

    if lower.endswith(".csv.gz"):
        return pd.read_csv(
            path,
            low_memory=False,
        )

    if lower.endswith(".csv"):
        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise RuntimeError(
        f"Unsupported candidate table extension: {path}"
    )


def resolve_alias(columns, alias_groups, label, required=True):
    normalized_to_original = {}

    for column in columns:
        normalized_to_original.setdefault(
            normalise(column),
            [],
        ).append(column)

    matches = []

    for alias in alias_groups:
        normalized_alias = normalise(alias)

        if normalized_alias in normalized_to_original:
            matches.extend(
                normalized_to_original[normalized_alias]
            )

    # Preserve column order, dedupe.
    unique = []

    for column in columns:
        if column in matches and column not in unique:
            unique.append(column)

    if len(unique) == 1:
        return unique[0]

    if len(unique) == 0 and not required:
        return None

    raise RuntimeError(
        f"Could not resolve exactly one {label} column. "
        f"Candidates: {unique}. Available columns: {list(columns)}"
    )


def resolve_metric_column(columns, metric):
    """
    Resolve the seed-level PROJECT-RUN metric.

    Preference order intentionally prioritizes mean project metrics:
      MeanAPFDc / MeanAPFD
    then common legacy spellings such as AvgAPFDc / AverageAPFDc / APFDc.
    """

    if metric == "APFDc":
        aliases = [
            "MeanAPFDc",
            "AvgAPFDc",
            "AverageAPFDc",
            "ProjectMeanAPFDc",
            "APFDcMean",
            "APFDc",
        ]

    elif metric == "APFD":
        aliases = [
            "MeanAPFD",
            "AvgAPFD",
            "AverageAPFD",
            "ProjectMeanAPFD",
            "APFDMean",
            "APFD",
        ]

    else:
        raise ValueError(metric)

    normalized_to_original = {}

    for column in columns:
        normalized_to_original.setdefault(
            normalise(column),
            [],
        ).append(column)

    for alias in aliases:
        key = normalise(alias)

        if key in normalized_to_original:
            originals = normalized_to_original[key]

            if len(originals) == 1:
                return originals[0]

    # Controlled fallback: one and only one non-delta/non-median metric column.
    matches = []

    for column in columns:
        token = normalise(column)

        if metric == "APFDc":
            metric_match = "apfdc" in token

        else:
            metric_match = (
                "apfd" in token
                and "apfdc" not in token
            )

        if (
            metric_match
            and "delta" not in token
            and "degradation" not in token
            and "retention" not in token
            and "median" not in token
            and "sd" not in token
            and "std" not in token
            and "ci" not in token
            and "clean" not in token
        ):
            matches.append(column)

    if len(matches) == 1:
        return matches[0]

    raise RuntimeError(
        f"Could not resolve exactly one project-run {metric} column. "
        f"Candidates: {matches}. Available columns: {list(columns)}"
    )


def canonicalize_noise(series):
    numeric = pd.to_numeric(
        series,
        errors="raise",
    ).astype(float)

    if len(numeric) == 0:
        raise RuntimeError("Noise column is empty.")

    # Support old files storing flake rates as proportions 0..0.50.
    if numeric.max() <= 0.500000000001:
        numeric = numeric * 100.0

    rounded = np.rint(
        numeric.to_numpy(dtype=float)
    ).astype(int)

    if not np.allclose(
        numeric.to_numpy(dtype=float),
        rounded.astype(float),
        atol=1e-9,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Noise values are not exact tested noise levels after normalization."
        )

    return pd.Series(
        rounded,
        index=series.index,
        dtype="int64",
    )


def canonicalize_seed(series):
    numeric = pd.to_numeric(
        series,
        errors="raise",
    ).astype(float)

    rounded = np.rint(
        numeric.to_numpy(dtype=float)
    ).astype(int)

    if not np.allclose(
        numeric.to_numpy(dtype=float),
        rounded.astype(float),
        atol=1e-9,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Seed values are not integers."
        )

    return pd.Series(
        rounded,
        index=series.index,
        dtype="int64",
    )


TECHNIQUE_ALIASES = {
    "randomforest": "RandomForest",
    "rf": "RandomForest",

    "xgboost": "XGBoost",
    "xgb": "XGBoost",

    "lightgbm": "LightGBM",
    "lgbm": "LightGBM",

    "naivebayes": "NaiveBayes",
    "nb": "NaiveBayes",
    "gaussiannb": "NaiveBayes",

    "random": "Random",
    "rand": "Random",

    "latestfail": "LatestFail",
    "latestfailure": "LatestFail",
    "latestfailed": "LatestFail",
    "lastfail": "LatestFail",

    "qtfavg": "QTF-Avg",
    "qtfaverage": "QTF-Avg",
    "qtf": "QTF-Avg",
}


def canonicalize_technique(series):
    values = []

    unknown = set()

    for value in series.astype(str):
        key = normalise(value)

        if key in TECHNIQUE_ALIASES:
            values.append(
                TECHNIQUE_ALIASES[key]
            )

        else:
            unknown.add(value)
            values.append(None)

    if unknown:
        raise RuntimeError(
            "Unknown technique labels: "
            + repr(
                sorted(unknown)
            )
        )

    return pd.Series(
        values,
        index=series.index,
        dtype="object",
    )


def canonicalize_candidate(path):
    """
    Returns a canonical scientific table:
      NoisePercent
      RepetitionSeed
      Technique
      MeanAPFDc
      MeanAPFD
    with exactly 1,890 unique coordinates.
    """

    frame = read_table(path)

    if len(frame) != EXPECTED_ROWS_PER_PROJECT:
        raise RuntimeError(
            f"Expected {EXPECTED_ROWS_PER_PROJECT} rows, found {len(frame)}."
        )

    columns = list(frame.columns)

    noise_col = resolve_alias(
        columns,
        [
            "NoisePercent",
            "NoisePercentage",
            "NoiseLevel",
            "Noise",
            "NoisePct",
            "FlakeRate",
            "FlakyRate",
            "FlakinessRate",
        ],
        "noise",
        required=True,
    )

    seed_col = resolve_alias(
        columns,
        [
            "RepetitionSeed",
            "Seed",
            "RandomSeed",
            "RunSeed",
            "Repetition",
        ],
        "seed",
        required=True,
    )

    technique_col = resolve_alias(
        columns,
        [
            "Technique",
            "Model",
            "Algorithm",
            "Method",
        ],
        "technique",
        required=True,
    )

    apfdc_col = resolve_metric_column(
        columns,
        "APFDc",
    )

    apfd_col = resolve_metric_column(
        columns,
        "APFD",
    )

    canonical = pd.DataFrame({
        "NoisePercent":
            canonicalize_noise(
                frame[
                    noise_col
                ]
            ),

        "RepetitionSeed":
            canonicalize_seed(
                frame[
                    seed_col
                ]
            ),

        "Technique":
            canonicalize_technique(
                frame[
                    technique_col
                ]
            ),

        "MeanAPFDc":
            pd.to_numeric(
                frame[
                    apfdc_col
                ],
                errors="raise",
            ).astype(float),

        "MeanAPFD":
            pd.to_numeric(
                frame[
                    apfd_col
                ],
                errors="raise",
            ).astype(float),
    })

    if not np.isfinite(
        canonical[
            [
                "MeanAPFDc",
                "MeanAPFD",
            ]
        ].to_numpy(
            dtype=float
        )
    ).all():
        raise RuntimeError(
            "APFD/APFDc contains non-finite values."
        )

    metric_values = canonical[
        [
            "MeanAPFDc",
            "MeanAPFD",
        ]
    ].to_numpy(
        dtype=float
    )

    if (
        (metric_values < -FLOAT_TOL).any()
        or (
            metric_values
            > 1.0
            + FLOAT_TOL
        ).any()
    ):
        raise RuntimeError(
            "APFD/APFDc values fall outside [0,1]."
        )

    duplicates = int(
        canonical.duplicated(
            [
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            keep=False,
        ).sum()
    )

    if duplicates != 0:
        raise RuntimeError(
            f"Duplicate canonical project-run coordinates: {duplicates}"
        )

    observed_noise = sorted(
        canonical[
            "NoisePercent"
        ].unique().tolist()
    )

    observed_seeds = sorted(
        canonical[
            "RepetitionSeed"
        ].unique().tolist()
    )

    observed_techniques = sorted(
        canonical[
            "Technique"
        ].unique().tolist()
    )

    if observed_noise != EXPECTED_NOISE_LEVELS:
        raise RuntimeError(
            f"Noise grid mismatch: {observed_noise}"
        )

    if observed_seeds != EXPECTED_SEEDS:
        raise RuntimeError(
            f"Seed grid mismatch: {observed_seeds}"
        )

    if observed_techniques != EXPECTED_TECHNIQUES:
        raise RuntimeError(
            f"Technique grid mismatch: {observed_techniques}"
        )

    canonical = (
        canonical.sort_values(
            [
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    # Deterministic content fingerprint, independent of original filename/schema.
    digest = hashlib.sha256()

    for row in canonical.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{int(row.NoisePercent)}\0"
                f"{int(row.RepetitionSeed)}\0"
                f"{row.Technique}\0"
                f"{float(row.MeanAPFDc):.17g}\0"
                f"{float(row.MeanAPFD):.17g}\n"
            ).encode(
                "utf-8"
            )
        )

    metadata = {
        "NoiseColumn":
            noise_col,

        "SeedColumn":
            seed_col,

        "TechniqueColumn":
            technique_col,

        "APFDcColumn":
            apfdc_col,

        "APFDColumn":
            apfd_col,

        "CanonicalContentSHA256":
            digest.hexdigest(),
    }

    return canonical, metadata


def compare_canonical(left, right):
    if len(left) != len(right):
        return False, np.inf

    key_columns = [
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
    ]

    if not left[
        key_columns
    ].equals(
        right[
            key_columns
        ]
    ):
        return False, np.inf

    max_abs = float(
        np.max(
            np.abs(
                left[
                    [
                        "MeanAPFDc",
                        "MeanAPFD",
                    ]
                ].to_numpy(
                    dtype=float
                )
                -
                right[
                    [
                        "MeanAPFDc",
                        "MeanAPFD",
                    ]
                ].to_numpy(
                    dtype=float
                )
            )
        )
    )

    return (
        bool(
            max_abs
            <= FLOAT_TOL
        ),
        max_abs,
    )


def source_priority(row):
    """
    Lower tuple is preferred.
    Scientific equivalence is checked BEFORE this priority is used.
    """

    source_group = str(
        row[
            "SourceGroup"
        ]
    )

    name = str(
        row[
            "Name"
        ]
    ).lower()

    path = str(
        row[
            "Path"
        ]
    )

    if source_group == "FINAL_PACKAGE":
        source_rank = 0
    else:
        source_rank = 1

    if "revalidated_project_runs" in name:
        name_rank = 0
    elif "project_runs" in name:
        name_rank = 1
    elif "project_run" in name:
        name_rank = 2
    elif "project_metrics" in name:
        name_rank = 3
    elif "project" in name and "metric" in name:
        name_rank = 4
    else:
        name_rank = 5

    return (
        source_rank,
        name_rank,
        len(path),
        path,
    )


# --------------------------------------------------------------------------------------------------
# 4. PRECONDITIONS / ONE-TIME FREEZE GUARD
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "Global Analysis Step 1A is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to Global Analysis Step 1B."
    )

if not STEP0_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-0 checkpoint missing: {STEP0_CHECKPOINT}"
    )

actual_step0_sha = sha256_file(
    STEP0_CHECKPOINT
)

if actual_step0_sha != EXPECTED_STEP0_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Global Analysis Step-0 checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_STEP0_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_step0_sha}"
    )

step0_checkpoint = load_json(
    STEP0_CHECKPOINT
)

if step0_checkpoint.get(
    "Status"
) != STEP0_STATUS:
    raise RuntimeError(
        "Step-0 checkpoint status is not the frozen PASS state."
    )

if not bool(
    step0_checkpoint.get(
        "ReadyForGlobalAnalysisStep1",
        False,
    )
):
    raise RuntimeError(
        "Step-0 checkpoint is not ready for Global Analysis Step 1."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

for path in [
    STEP0_CANDIDATE_INVENTORY,
    STEP0_PROJECT_SOURCE_INVENTORY,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required Step-0 inventory missing: {path}"
        )

if STEP1A_ROOT.exists():
    shutil.rmtree(
        STEP1A_ROOT,
        ignore_errors=True,
    )

STEP1A_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. LOAD STEP-0 INVENTORIES
# --------------------------------------------------------------------------------------------------

candidate_inventory = pd.read_csv(
    STEP0_CANDIDATE_INVENTORY,
    low_memory=False,
)

project_source_inventory = pd.read_csv(
    STEP0_PROJECT_SOURCE_INVENTORY,
    low_memory=False,
)

if len(
    project_source_inventory
) != EXPECTED_PROJECTS:
    raise RuntimeError(
        "Step-0 project source inventory does not contain 24 projects."
    )

observed_project_numbers = sorted(
    pd.to_numeric(
        project_source_inventory[
            "ProjectNumber"
        ],
        errors="raise",
    ).astype(
        int
    ).tolist()
)

if observed_project_numbers != EXPECTED_PROJECT_NUMBERS:
    raise RuntimeError(
        "Step-0 source inventory ProjectNumbers are not exactly 1..24."
    )


# --------------------------------------------------------------------------------------------------
# 6. FILTER TO SCIENTIFICALLY PLAUSIBLE 1,890-ROW PROJECT-RUN SOURCES
# --------------------------------------------------------------------------------------------------

required_inventory_columns = [
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "SourceGroup",
    "Path",
    "Name",
    "Rows",
    "Classification",
    "HasNoiseColumn",
    "HasSeedColumn",
    "HasTechniqueColumn",
    "HasAPFDcColumn",
    "HasAPFDColumn",
    "HasDeltaOrRetentionColumn",
    "SHA256",
]

missing_inventory_columns = [
    column
    for column in required_inventory_columns
    if column not in candidate_inventory.columns
]

if missing_inventory_columns:
    raise RuntimeError(
        "Step-0 candidate inventory is missing columns:\n"
        + "\n".join(
            missing_inventory_columns
        )
    )


def to_bool_series(series):
    return (
        series.astype(
            str
        ).str.strip().str.lower().map({
            "true":
                True,

            "false":
                False,

            "1":
                True,

            "0":
                False,
        })
    )


candidate_rows_numeric = pd.to_numeric(
    candidate_inventory[
        "Rows"
    ],
    errors="raise",
).astype(
    int
)

plausible = candidate_inventory.loc[
    candidate_rows_numeric.eq(
        EXPECTED_ROWS_PER_PROJECT
    )
    & to_bool_series(
        candidate_inventory[
            "HasNoiseColumn"
        ]
    ).eq(
        True
    )
    & to_bool_series(
        candidate_inventory[
            "HasSeedColumn"
        ]
    ).eq(
        True
    )
    & to_bool_series(
        candidate_inventory[
            "HasTechniqueColumn"
        ]
    ).eq(
        True
    )
    & to_bool_series(
        candidate_inventory[
            "HasAPFDcColumn"
        ]
    ).eq(
        True
    )
    & to_bool_series(
        candidate_inventory[
            "HasAPFDColumn"
        ]
    ).eq(
        True
    )
    & to_bool_series(
        candidate_inventory[
            "HasDeltaOrRetentionColumn"
        ]
    ).eq(
        False
    )
].copy()

plausible[
    "ProjectNumber"
] = pd.to_numeric(
    plausible[
        "ProjectNumber"
    ],
    errors="raise",
).astype(
    int
)

print(
    "Plausible non-delta 1,890-row seed-level candidates:",
    len(
        plausible
    ),
)


# --------------------------------------------------------------------------------------------------
# 7. PER-PROJECT CONTENT RESOLUTION
# --------------------------------------------------------------------------------------------------

source_map_rows = []
candidate_audit_rows = []

projects_without_plausible_candidate = []
projects_with_scientific_disagreement = []

total_valid_candidates = 0
total_invalid_candidates = 0
total_equivalent_duplicates = 0

for project_number in EXPECTED_PROJECT_NUMBERS:
    project_meta_rows = project_source_inventory.loc[
        pd.to_numeric(
            project_source_inventory[
                "ProjectNumber"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            project_number
        )
    ]

    if len(
        project_meta_rows
    ) != 1:
        raise RuntimeError(
            f"Could not resolve exactly one project-source row for Project {project_number}."
        )

    project_meta = project_meta_rows.iloc[
        0
    ]

    project_name = str(
        project_meta[
            "Project"
        ]
    )

    project_slug = str(
        project_meta[
            "ProjectSlug"
        ]
    )

    project_candidates = plausible.loc[
        plausible[
            "ProjectNumber"
        ].eq(
            project_number
        )
    ].copy()

    if project_candidates.empty:
        projects_without_plausible_candidate.append(
            project_number
        )
        continue

    valid_entries = []

    for _, candidate in project_candidates.iterrows():
        path = Path(
            str(
                candidate[
                    "Path"
                ]
            )
        )

        validation_error = ""

        canonical = None
        metadata = None

        try:
            if not path.is_file():
                raise FileNotFoundError(
                    f"Candidate file disappeared: {path}"
                )

            # Reconfirm candidate file integrity against Step-0 SHA.
            current_sha = sha256_file(
                path
            )

            expected_sha = str(
                candidate[
                    "SHA256"
                ]
            ).strip().lower()

            if current_sha != expected_sha:
                raise RuntimeError(
                    "Candidate SHA differs from Step-0 frozen inventory."
                )

            canonical, metadata = canonicalize_candidate(
                path
            )

            valid = True
            total_valid_candidates += 1

        except Exception as error:
            valid = False
            validation_error = repr(
                error
            )
            total_invalid_candidates += 1

        audit_row = {
            "ProjectNumber":
                project_number,

            "Project":
                project_name,

            "ProjectSlug":
                project_slug,

            "SourceGroup":
                str(
                    candidate[
                        "SourceGroup"
                    ]
                ),

            "Path":
                str(
                    path
                ),

            "Name":
                str(
                    candidate[
                        "Name"
                    ]
                ),

            "Rows":
                int(
                    candidate[
                        "Rows"
                    ]
                ),

            "Step0SHA256":
                str(
                    candidate[
                        "SHA256"
                    ]
                ),

            "ValidCanonicalProjectRun":
                valid,

            "ValidationError":
                validation_error,

            "NoiseColumn":
                (
                    metadata[
                        "NoiseColumn"
                    ]
                    if valid
                    else ""
                ),

            "SeedColumn":
                (
                    metadata[
                        "SeedColumn"
                    ]
                    if valid
                    else ""
                ),

            "TechniqueColumn":
                (
                    metadata[
                        "TechniqueColumn"
                    ]
                    if valid
                    else ""
                ),

            "APFDcColumn":
                (
                    metadata[
                        "APFDcColumn"
                    ]
                    if valid
                    else ""
                ),

            "APFDColumn":
                (
                    metadata[
                        "APFDColumn"
                    ]
                    if valid
                    else ""
                ),

            "CanonicalContentSHA256":
                (
                    metadata[
                        "CanonicalContentSHA256"
                    ]
                    if valid
                    else ""
                ),

            "Selected":
                False,

            "EquivalentToSelected":
                False,

            "MaxAbsMetricDifferenceVsSelected":
                "",
        }

        audit_index = len(
            candidate_audit_rows
        )

        candidate_audit_rows.append(
            audit_row
        )

        if valid:
            valid_entries.append({
                "candidate":
                    candidate,

                "canonical":
                    canonical,

                "metadata":
                    metadata,

                "audit_index":
                    audit_index,
            })

    if not valid_entries:
        projects_without_plausible_candidate.append(
            project_number
        )
        continue

    # Group by scientific content fingerprint first.
    fingerprints = {
        entry[
            "metadata"
        ][
            "CanonicalContentSHA256"
        ]
        for entry in valid_entries
    }

    # A tiny text/float serialization difference can yield different fingerprints despite
    # equality at 1e-12, so compare directly before declaring disagreement.
    representative = valid_entries[
        0
    ][
        "canonical"
    ]

    all_equivalent = True
    maximum_pair_difference = 0.0

    for entry in valid_entries[
        1:
    ]:
        equivalent, max_abs = compare_canonical(
            representative,
            entry[
                "canonical"
            ],
        )

        if np.isfinite(
            max_abs
        ):
            maximum_pair_difference = max(
                maximum_pair_difference,
                float(
                    max_abs
                ),
            )

        if not equivalent:
            all_equivalent = False

    if not all_equivalent:
        projects_with_scientific_disagreement.append(
            project_number
        )

        # Record candidates, then stop after all diagnostics are written.
        continue

    # Scientifically equivalent candidates: choose deterministic preferred copy.
    valid_entries = sorted(
        valid_entries,
        key=lambda entry:
            source_priority(
                entry[
                    "candidate"
                ]
            ),
    )

    selected = valid_entries[
        0
    ]

    selected_path = Path(
        str(
            selected[
                "candidate"
            ][
                "Path"
            ]
        )
    )

    selected_canonical = selected[
        "canonical"
    ]

    selected_metadata = selected[
        "metadata"
    ]

    selected_audit_index = selected[
        "audit_index"
    ]

    candidate_audit_rows[
        selected_audit_index
    ][
        "Selected"
    ] = True

    candidate_audit_rows[
        selected_audit_index
    ][
        "EquivalentToSelected"
    ] = True

    candidate_audit_rows[
        selected_audit_index
    ][
        "MaxAbsMetricDifferenceVsSelected"
    ] = 0.0

    for entry in valid_entries[
        1:
    ]:
        equivalent, max_abs = compare_canonical(
            selected_canonical,
            entry[
                "canonical"
            ],
        )

        candidate_audit_rows[
            entry[
                "audit_index"
            ]
        ][
            "EquivalentToSelected"
        ] = bool(
            equivalent
        )

        candidate_audit_rows[
            entry[
                "audit_index"
            ]
        ][
            "MaxAbsMetricDifferenceVsSelected"
        ] = (
            float(
                max_abs
            )
            if np.isfinite(
                max_abs
            )
            else ""
        )

        if equivalent:
            total_equivalent_duplicates += 1

    source_map_rows.append({
        "ProjectNumber":
            project_number,

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "SelectedSourceGroup":
            str(
                selected[
                    "candidate"
                ][
                    "SourceGroup"
                ]
            ),

        "SelectedPath":
            str(
                selected_path
            ),

        "SelectedFileName":
            selected_path.name,

        "SelectedFileSHA256":
            sha256_file(
                selected_path
            ),

        "SelectedCanonicalContentSHA256":
            selected_metadata[
                "CanonicalContentSHA256"
            ],

        "Rows":
            len(
                selected_canonical
            ),

        "NoiseLevels":
            len(
                selected_canonical[
                    "NoisePercent"
                ].unique()
            ),

        "Seeds":
            len(
                selected_canonical[
                    "RepetitionSeed"
                ].unique()
            ),

        "Techniques":
            len(
                selected_canonical[
                    "Technique"
                ].unique()
            ),

        "NoiseColumn":
            selected_metadata[
                "NoiseColumn"
            ],

        "SeedColumn":
            selected_metadata[
                "SeedColumn"
            ],

        "TechniqueColumn":
            selected_metadata[
                "TechniqueColumn"
            ],

        "APFDcColumn":
            selected_metadata[
                "APFDcColumn"
            ],

        "APFDColumn":
            selected_metadata[
                "APFDColumn"
            ],

        "ValidEquivalentCandidates":
            len(
                valid_entries
            ),

        "EquivalentDuplicateCopies":
            max(
                0,
                len(
                    valid_entries
                )
                - 1,
            ),

        "MaximumEquivalentCandidateMetricDifference":
            maximum_pair_difference,
    })


source_map = (
    pd.DataFrame(
        source_map_rows
    )
    .sort_values(
        "ProjectNumber",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

candidate_audit = (
    pd.DataFrame(
        candidate_audit_rows
    )
    .sort_values(
        [
            "ProjectNumber",
            "ValidCanonicalProjectRun",
            "Selected",
            "SourceGroup",
            "Path",
        ],
        ascending=[
            True,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

registry_sha_after = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Step-0 checkpoint SHA-256",
    EXPECTED_STEP0_CHECKPOINT_SHA256,
    actual_step0_sha,
    actual_step0_sha
    == EXPECTED_STEP0_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-0 status",
    STEP0_STATUS,
    step0_checkpoint.get(
        "Status"
    ),
    step0_checkpoint.get(
        "Status"
    )
    == STEP0_STATUS,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Projects without plausible 1,890-row candidate",
    0,
    len(
        projects_without_plausible_candidate
    ),
    len(
        projects_without_plausible_candidate
    )
    == 0,
)

add_check(
    checks,
    "Projects with scientifically disagreeing valid candidates",
    0,
    len(
        projects_with_scientific_disagreement
    ),
    len(
        projects_with_scientific_disagreement
    )
    == 0,
)

add_check(
    checks,
    "Resolved project sources",
    24,
    len(
        source_map
    ),
    len(
        source_map
    )
    == 24,
)

add_check(
    checks,
    "Resolved ProjectNumbers",
    EXPECTED_PROJECT_NUMBERS,
    (
        sorted(
            source_map[
                "ProjectNumber"
            ].astype(
                int
            ).tolist()
        )
        if not source_map.empty
        else []
    ),
    (
        sorted(
            source_map[
                "ProjectNumber"
            ].astype(
                int
            ).tolist()
        )
        == EXPECTED_PROJECT_NUMBERS
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Rows per selected source",
    [1890],
    (
        sorted(
            source_map[
                "Rows"
            ].astype(
                int
            ).unique().tolist()
        )
        if not source_map.empty
        else []
    ),
    (
        sorted(
            source_map[
                "Rows"
            ].astype(
                int
            ).unique().tolist()
        )
        == [1890]
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Noise levels per selected source",
    [9],
    (
        sorted(
            source_map[
                "NoiseLevels"
            ].astype(
                int
            ).unique().tolist()
        )
        if not source_map.empty
        else []
    ),
    (
        sorted(
            source_map[
                "NoiseLevels"
            ].astype(
                int
            ).unique().tolist()
        )
        == [9]
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Seeds per selected source",
    [30],
    (
        sorted(
            source_map[
                "Seeds"
            ].astype(
                int
            ).unique().tolist()
        )
        if not source_map.empty
        else []
    ),
    (
        sorted(
            source_map[
                "Seeds"
            ].astype(
                int
            ).unique().tolist()
        )
        == [30]
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Techniques per selected source",
    [7],
    (
        sorted(
            source_map[
                "Techniques"
            ].astype(
                int
            ).unique().tolist()
        )
        if not source_map.empty
        else []
    ),
    (
        sorted(
            source_map[
                "Techniques"
            ].astype(
                int
            ).unique().tolist()
        )
        == [7]
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Expected global rows implied by source map",
    EXPECTED_GLOBAL_ROWS,
    int(
        source_map[
            "Rows"
        ].sum()
    )
    if not source_map.empty
    else 0,
    (
        int(
            source_map[
                "Rows"
            ].sum()
        )
        == EXPECTED_GLOBAL_ROWS
        if not source_map.empty
        else False
    ),
)

add_check(
    checks,
    "Selected sources",
    24,
    int(
        candidate_audit[
            "Selected"
        ].astype(
            bool
        ).sum()
    )
    if not candidate_audit.empty
    else 0,
    (
        int(
            candidate_audit[
                "Selected"
            ].astype(
                bool
            ).sum()
        )
        == 24
        if not candidate_audit.empty
        else False
    ),
)

add_check(
    checks,
    "Invalid plausible candidates permitted without selection",
    True,
    True,
    True,
)

add_check(
    checks,
    "Raw condition outputs accessed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Models trained",
    False,
    False,
    True,
)

add_check(
    checks,
    "Statistical tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)

validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()


# --------------------------------------------------------------------------------------------------
# 9. WRITE DIAGNOSTICS BEFORE FAILING, SO ANY LEGACY EXCEPTION IS ACTIONABLE
# --------------------------------------------------------------------------------------------------

atomic_csv(
    CANDIDATE_AUDIT_PATH,
    candidate_audit,
)

if not source_map.empty:
    atomic_csv(
        SOURCE_MAP_PATH,
        source_map,
    )

atomic_csv(
    VALIDATION_PATH,
    validation,
)

print(
    "\nStep 1A validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

print(
    "\nResolved canonical source map:"
)

if not source_map.empty:
    source_display_columns = [
        "ProjectNumber",
        "Project",
        "SelectedSourceGroup",
        "SelectedFileName",
        "Rows",
        "ValidEquivalentCandidates",
        "EquivalentDuplicateCopies",
    ]

    try:
        from IPython.display import display

        display(
            source_map[
                source_display_columns
            ]
        )

    except Exception:
        print(
            source_map[
                source_display_columns
            ].to_string(
                index=False
            )
        )

if projects_without_plausible_candidate:
    print(
        "\nProjects without a valid canonical 1,890-row project-run source:",
        projects_without_plausible_candidate,
    )

if projects_with_scientific_disagreement:
    print(
        "\nProjects whose valid-looking 1,890-row candidate files disagree scientifically:",
        projects_with_scientific_disagreement,
    )

if not failed_validation.empty:
    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 1A FAILED.\n"
        "No canonical source freeze was written.\n"
        "Use the candidate-audit CSV for exact diagnostics:\n"
        f"{CANDIDATE_AUDIT_PATH}\n\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. FREEZE SOURCE MAP
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "Step":
        "GLOBAL_ANALYSIS_STEP_1A",

    "Status":
        STEP1A_STATUS,

    "CodeRevision":
        STEP1A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step0CheckpointSHA256":
        EXPECTED_STEP0_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "Projects":
        24,

    "SelectedSources":
        len(
            source_map
        ),

    "RowsPerProject":
        EXPECTED_ROWS_PER_PROJECT,

    "GlobalRowsImplied":
        int(
            source_map[
                "Rows"
            ].sum()
        ),

    "ValidCandidateFiles":
        total_valid_candidates,

    "InvalidPlausibleCandidateFiles":
        total_invalid_candidates,

    "EquivalentDuplicateCopies":
        total_equivalent_duplicates,

    "ScientificDisagreementProjects":
        projects_with_scientific_disagreement,

    "ProjectsWithoutValidSource":
        projects_without_plausible_candidate,

    "CanonicalCoordinates":
        [
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],

    "CanonicalPrimaryMetrics":
        [
            "MeanAPFDc",
            "MeanAPFD",
        ],

    "FloatEquivalenceTolerance":
        FLOAT_TOL,

    "RawConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,

    "StatisticalTestsExecuted":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 1B — "
            "CONSTRUCT AND FREEZE THE 45,360-ROW CANONICAL CROSS-PROJECT MASTER DATASET"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            STEP1A_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "SelectedSources":
            24,

        "ReadyForGlobalAnalysisStep1B":
            True,
    },
)

output_paths = [
    SOURCE_MAP_PATH,
    CANDIDATE_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

checkpoint = {
    "CheckpointType":
        "GLOBAL_ANALYSIS_CANONICAL_PROJECT_RUN_SOURCE_RESOLUTION",

    "Status":
        STEP1A_STATUS,

    "CodeRevision":
        STEP1A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step0CheckpointSHA256":
        EXPECTED_STEP0_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "ProjectRunSourceMapPath":
        str(
            SOURCE_MAP_PATH
        ),

    "ProjectRunSourceMapSHA256":
        sha256_file(
            SOURCE_MAP_PATH
        ),

    "CandidateAuditPath":
        str(
            CANDIDATE_AUDIT_PATH
        ),

    "ValidationPath":
        str(
            VALIDATION_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "Projects":
        24,

    "RowsPerProject":
        EXPECTED_ROWS_PER_PROJECT,

    "GlobalRowsImplied":
        EXPECTED_GLOBAL_ROWS,

    "ReadyForGlobalAnalysisStep1B":
        True,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 1B — "
            "CONSTRUCT AND FREEZE THE 45,360-ROW CANONICAL CROSS-PROJECT MASTER DATASET"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 142
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 2 / STEP 1A RESULT ==="
)

print(
    "=" * 142
)

print(
    "Projects resolved:",
    len(
        source_map
    ),
    "/ 24",
)

print(
    "Rows per project source:",
    EXPECTED_ROWS_PER_PROJECT,
)

print(
    "Global rows implied:",
    int(
        source_map[
            "Rows"
        ].sum()
    ),
    "/",
    EXPECTED_GLOBAL_ROWS,
)

print(
    "Valid candidate files:",
    total_valid_candidates,
)

print(
    "Invalid plausible candidate files:",
    total_invalid_candidates,
)

print(
    "Equivalent duplicate copies:",
    total_equivalent_duplicates,
)

print(
    "Projects with scientific candidate disagreement:",
    len(
        projects_with_scientific_disagreement
    ),
)

print(
    "Projects without valid canonical source:",
    len(
        projects_without_plausible_candidate
    ),
)

print(
    "\nSource groups selected:"
)

print(
    source_map[
        "SelectedSourceGroup"
    ].value_counts(
        sort=False
    ).to_string()
)

print(
    "\nIsolation:"
)

print(
    "Raw condition outputs accessed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "Models trained: False"
)

print(
    "Statistical tests executed: False"
)

print(
    "\nValidation checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nStep 1A checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "GLOBAL ANALYSIS STEP 1B — CONSTRUCT AND FREEZE THE 45,360-ROW "
    "CANONICAL CROSS-PROJECT MASTER DATASET"
)

print(
    "\nSTATUS:",
    STEP1A_STATUS,
)

print(
    "=" * 142
)


=== THESIS GLOBAL ANALYSIS — CELL 2 / STEP 1A: CANONICAL PROJECT-RUN SOURCE RESOLUTION ===
Plausible non-delta 1,890-row seed-level candidates: 76

Step 1A validation:


,Check,Expected,Actual,Pass
0,Step-0 checkpoint SHA-256,b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d...,b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d...,True
1,Step-0 status,PASS_GLOBAL_ANALYSIS_STEP0_24_PROJECT_COHORT_A...,PASS_GLOBAL_ANALYSIS_STEP0_24_PROJECT_COHORT_A...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,"Projects without plausible 1,890-row candidate",0,0,True
4,Projects with scientifically disagreeing valid...,0,0,True
5,Resolved project sources,24,24,True
6,Resolved ProjectNumbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
7,Rows per selected source,[1890],[1890],True
8,Noise levels per selected source,[9],[9],True
9,Seeds per selected source,[30],[30],True



Resolved canonical source map:


,ProjectNumber,Project,SelectedSourceGroup,SelectedFileName,Rows,ValidEquivalentCandidates,EquivalentDuplicateCopies
0,1,Angel-ML@angel,AGGREGATED,angel_30seed_project_run_results.csv,1890,1,0
1,2,apache@airavata,AGGREGATED,airavata_30seed_project_run_results.csv,1890,1,0
2,3,b2ihealthcare@snow-owl,AGGREGATED,project_run_metrics_30_seeds.csv,1890,1,0
3,4,eclipse@paho.mqtt.java,AGGREGATED,project_run_metrics_30_seeds.csv,1890,1,0
4,5,thinkaurelius@titan,AGGREGATED,titan_all_project_run_metrics.csv,1890,3,2
5,6,eclipse@jetty.project,AGGREGATED,jetty_all_project_run_metrics.csv,1890,4,3
6,7,CompEvol@beast2,AGGREGATED,project_run_metrics_all.csv,1890,2,1
7,8,optimatika@ojAlgo,AGGREGATED,project_run_metrics_all.csv,1890,3,2
8,9,camunda@camunda-bpm-platform,AGGREGATED,camunda_project_run_all.csv,1890,2,1
9,10,spring-cloud@spring-cloud-dataflow,AGGREGATED,spring_cloud_dataflow_project_run_all.csv,1890,2,1



=== THESIS GLOBAL ANALYSIS — CELL 2 / STEP 1A RESULT ===
Projects resolved: 24 / 24
Rows per project source: 1890
Global rows implied: 45360 / 45360
Valid candidate files: 76
Invalid plausible candidate files: 0
Equivalent duplicate copies: 52
Projects with scientific candidate disagreement: 0
Projects without valid canonical source: 0

Source groups selected:
SelectedSourceGroup
AGGREGATED       10
FINAL_PACKAGE    14

Isolation:
Raw condition outputs accessed: False
Project outputs modified: False
Completion registry modified: False
Models trained: False
Statistical tests executed: False

Validation checks: 18
Failed checks: 0

Step 1A checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/global_analysis_step1a_checkpoint.json
Checkpoint SHA-256: 18a121d4da0dff809db5a18a9c59fadf39af1ca4313b9fc1ddb27abeea6d74f9

Next required step: GLOBAL ANALYSIS STEP 1B — CONSTRUCT AND FREEZE THE 45,360-ROW CANONICAL CROSS-PROJECT MASTER DATASET

STATUS: PASS_GLOBAL_ANALYSIS_STEP1A_CANONICAL_P

In [5]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 3 / STEP 1B
# CONSTRUCT + FREEZE THE CANONICAL 24-PROJECT CROSS-PROJECT MASTER DATASETS
# ==================================================================================================
#
# PURPOSE
# -------
# Global Analysis Step 1A resolved exactly one authoritative 1,890-row project-run source
# for each of the 24 eligible projects.
#
# This cell consumes ONLY that frozen 24-row source map and creates the canonical analysis
# datasets that all later RQ1/RQ2/RQ3 cells will use.
#
# OUTPUTS
# -------
# A. Seed-level master:
#      24 projects × 9 noise levels × 30 seeds × 7 techniques = 45,360 rows
#
# B. Project-level master:
#      24 projects × 9 noise levels × 7 techniques = 1,512 rows
#      Each row summarizes the 30 repeated seeds for one Project×Noise×Technique.
#
# C. Seed-level clean-relative delta master:
#      45,360 rows
#      Delta_MeanAPFDc = MeanAPFDc(noise p, same project/seed/technique)
#                        - MeanAPFDc(noise 0, same project/seed/technique)
#      Delta_MeanAPFD  = analogous APFD delta
#
# D. Project-level clean-relative delta master:
#      1,512 rows
#      Summarizes the 30 seed-level deltas for one Project×Noise×Technique.
#
# IMPORTANT STATISTICAL INTERPRETATION
# ------------------------------------
# The independent empirical unit remains PROJECT (N=24).
# Seeds are repeated stochastic runs within each project; they are NOT treated as 720
# independent datasets.
#
# NO inferential statistical tests are executed here.
# NO p-values are computed here.
# NO project outputs or completion-registry rows are modified.
#
# After this passes, the next step is to freeze the Statistical Analysis Contract BEFORE
# any RQ hypothesis testing.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 144)
print("=== THESIS GLOBAL ANALYSIS — CELL 3 / STEP 1B: CANONICAL CROSS-PROJECT MASTER DATASETS ===")
print("=" * 144)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT CONTRACT
# --------------------------------------------------------------------------------------------------

STEP1A_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1A_CANONICAL_PROJECT_RUN_SOURCES_FROZEN"
)

EXPECTED_STEP1A_CHECKPOINT_SHA256 = (
    "18a121d4da0dff809db5a18a9c59fadf39af1ca4313b9fc1ddb27abeea6d74f9"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

STEP1B_CODE_REVISION = (
    "GLOBAL_ANALYSIS_STEP1B_V1_45360_SEED_ROWS_1512_PROJECT_ROWS_CLEAN_RELATIVE_DELTAS"
)

EXPECTED_PROJECTS = 24
EXPECTED_PROJECT_NUMBERS = list(range(1, 25))

EXPECTED_NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
EXPECTED_SEEDS = list(range(1, 31))

EXPECTED_TECHNIQUES = [
    "LatestFail",
    "LightGBM",
    "NaiveBayes",
    "QTF-Avg",
    "Random",
    "RandomForest",
    "XGBoost",
]

ML_TECHNIQUES = [
    "LightGBM",
    "NaiveBayes",
    "RandomForest",
    "XGBoost",
]

BASELINE_TECHNIQUES = [
    "LatestFail",
    "QTF-Avg",
    "Random",
]

EXPECTED_ROWS_PER_PROJECT = 1_890
EXPECTED_SEED_MASTER_ROWS = 45_360
EXPECTED_PROJECT_MASTER_ROWS = 1_512
EXPECTED_SEED_DELTA_ROWS = 45_360
EXPECTED_PROJECT_DELTA_ROWS = 1_512

EXPECTED_ROWS_PER_PROJECT_NOISE = 30 * 7        # 210
EXPECTED_ROWS_PER_PROJECT_TECHNIQUE = 9 * 30   # 270
EXPECTED_ROWS_PER_NOISE = 24 * 30 * 7          # 5,040
EXPECTED_ROWS_PER_TECHNIQUE = 24 * 9 * 30      # 6,480
EXPECTED_ROWS_PER_PROJECT_NOISE_TECHNIQUE = 30

FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"

STEP1A_CHECKPOINT = (
    NOTES / "global_analysis_step1a_checkpoint.json"
)

STEP1B_ROOT = (
    ANALYSIS_ROOT
    / "Step_1B_Canonical_Cross_Project_Master_Datasets"
)

SEED_MASTER_PATH = (
    STEP1B_ROOT
    / "global_master_seed_level_project_runs.csv"
)

PROJECT_MASTER_PATH = (
    STEP1B_ROOT
    / "global_master_project_noise_technique.csv"
)

SEED_DELTA_PATH = (
    STEP1B_ROOT
    / "global_master_seed_level_clean_deltas.csv"
)

PROJECT_DELTA_PATH = (
    STEP1B_ROOT
    / "global_master_project_noise_technique_clean_deltas.csv"
)

PROJECT_COVERAGE_PATH = (
    STEP1B_ROOT
    / "global_master_project_coverage_audit.csv"
)

TECHNIQUE_COVERAGE_PATH = (
    STEP1B_ROOT
    / "global_master_technique_coverage_audit.csv"
)

BASELINE_DIAGNOSTIC_PATH = (
    STEP1B_ROOT
    / "global_master_baseline_invariance_diagnostic.csv"
)

READBACK_AUDIT_PATH = (
    STEP1B_ROOT
    / "global_master_readback_audit.csv"
)

VALIDATION_PATH = (
    STEP1B_ROOT
    / "global_analysis_step1b_validation.csv"
)

REPORT_PATH = (
    STEP1B_ROOT
    / "global_analysis_step1b_report.json"
)

STATUS_PATH = (
    STEP1B_ROOT
    / "global_analysis_step1b_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP1B_ROOT
    / "global_analysis_step1b_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_step1b_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def build_output_manifest(paths):
    rows = []

    for path in sorted(
        (
            Path(path)
            for path in paths
        ),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"Step-1B output missing: {path}"
            )

        rows.append({
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Path",
            "Bytes",
            "SHA256",
        ],
    )


def read_table(path):
    path = Path(path)
    lower = path.name.lower()

    if lower.endswith(".parquet"):
        return pd.read_parquet(path)

    if lower.endswith(".csv.gz"):
        return pd.read_csv(
            path,
            low_memory=False,
        )

    if lower.endswith(".csv"):
        return pd.read_csv(
            path,
            low_memory=False,
        )

    raise RuntimeError(
        f"Unsupported selected source extension: {path}"
    )


def canonicalize_noise(series):
    numeric = pd.to_numeric(
        series,
        errors="raise",
    ).astype(float)

    if numeric.max() <= 0.500000000001:
        numeric = numeric * 100.0

    rounded = np.rint(
        numeric.to_numpy(dtype=float)
    ).astype(int)

    if not np.allclose(
        numeric.to_numpy(dtype=float),
        rounded.astype(float),
        atol=1e-9,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Noise values are not exact tested levels."
        )

    return pd.Series(
        rounded,
        index=series.index,
        dtype="int64",
    )


def canonicalize_seed(series):
    numeric = pd.to_numeric(
        series,
        errors="raise",
    ).astype(float)

    rounded = np.rint(
        numeric.to_numpy(dtype=float)
    ).astype(int)

    if not np.allclose(
        numeric.to_numpy(dtype=float),
        rounded.astype(float),
        atol=1e-9,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Seed values are not integral."
        )

    return pd.Series(
        rounded,
        index=series.index,
        dtype="int64",
    )


def normalise(value):
    return "".join(
        character.lower()
        for character in str(value)
        if character.isalnum()
    )


TECHNIQUE_ALIASES = {
    "randomforest": "RandomForest",
    "rf": "RandomForest",

    "xgboost": "XGBoost",
    "xgb": "XGBoost",

    "lightgbm": "LightGBM",
    "lgbm": "LightGBM",

    "naivebayes": "NaiveBayes",
    "nb": "NaiveBayes",
    "gaussiannb": "NaiveBayes",

    "random": "Random",
    "rand": "Random",

    "latestfail": "LatestFail",
    "latestfailure": "LatestFail",
    "latestfailed": "LatestFail",
    "lastfail": "LatestFail",

    "qtfavg": "QTF-Avg",
    "qtfaverage": "QTF-Avg",
    "qtf": "QTF-Avg",
}


def canonicalize_technique(series):
    output = []
    unknown = set()

    for value in series.astype(str):
        key = normalise(value)

        if key in TECHNIQUE_ALIASES:
            output.append(
                TECHNIQUE_ALIASES[key]
            )

        else:
            unknown.add(value)
            output.append(None)

    if unknown:
        raise RuntimeError(
            "Unknown technique labels: "
            + repr(
                sorted(unknown)
            )
        )

    return pd.Series(
        output,
        index=series.index,
        dtype="object",
    )


def scientific_hash(
    dataframe,
    columns,
):
    ordered = dataframe.loc[
        :,
        columns
    ]

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False,
        name=None,
    ):
        parts = []

        for value in row:
            if isinstance(
                value,
                (
                    float,
                    np.floating,
                ),
            ):
                parts.append(
                    f"{float(value):.17g}"
                )

            else:
                parts.append(
                    str(value)
                )

        digest.update(
            (
                "\0".join(parts)
                + "\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def max_abs_float_difference(
    left,
    right,
    columns,
):
    if len(left) != len(right):
        return np.inf

    max_difference = 0.0

    for column in columns:
        left_values = pd.to_numeric(
            left[column],
            errors="raise",
        ).to_numpy(
            dtype=float
        )

        right_values = pd.to_numeric(
            right[column],
            errors="raise",
        ).to_numpy(
            dtype=float
        )

        difference = float(
            np.max(
                np.abs(
                    left_values
                    - right_values
                )
            )
        )

        max_difference = max(
            max_difference,
            difference,
        )

    return max_difference


# --------------------------------------------------------------------------------------------------
# 4. PRECONDITIONS / FREEZE GUARD
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "Global Analysis Step 1B is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to the Statistical Analysis Contract step."
    )

if not STEP1A_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-1A checkpoint missing: {STEP1A_CHECKPOINT}"
    )

actual_step1a_checkpoint_sha = sha256_file(
    STEP1A_CHECKPOINT
)

if (
    actual_step1a_checkpoint_sha
    != EXPECTED_STEP1A_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Global Analysis Step-1A checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_STEP1A_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_step1a_checkpoint_sha}"
    )

step1a_checkpoint = load_json(
    STEP1A_CHECKPOINT
)

if step1a_checkpoint.get(
    "Status"
) != STEP1A_STATUS:
    raise RuntimeError(
        "Step-1A checkpoint status is not the frozen PASS state."
    )

if not bool(
    step1a_checkpoint.get(
        "ReadyForGlobalAnalysisStep1B",
        False,
    )
):
    raise RuntimeError(
        "Step-1A checkpoint is not marked ready for Step 1B."
    )

source_map_path = Path(
    step1a_checkpoint[
        "ProjectRunSourceMapPath"
    ]
)

expected_source_map_sha = str(
    step1a_checkpoint[
        "ProjectRunSourceMapSHA256"
    ]
).lower()

if not source_map_path.is_file():
    raise FileNotFoundError(
        f"Frozen Step-1A source map missing: {source_map_path}"
    )

actual_source_map_sha = sha256_file(
    source_map_path
)

if actual_source_map_sha != expected_source_map_sha:
    raise RuntimeError(
        "Frozen Step-1A source-map SHA mismatch."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from the final 24-project freeze."
    )

if STEP1B_ROOT.exists():
    shutil.rmtree(
        STEP1B_ROOT,
        ignore_errors=True,
    )

STEP1B_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. LOAD + VALIDATE THE FROZEN 24-ROW SOURCE MAP
# --------------------------------------------------------------------------------------------------

source_map = pd.read_csv(
    source_map_path,
    low_memory=False,
)

required_source_map_columns = [
    "ProjectNumber",
    "Project",
    "ProjectSlug",
    "SelectedPath",
    "SelectedFileSHA256",
    "Rows",
    "NoiseLevels",
    "Seeds",
    "Techniques",
    "NoiseColumn",
    "SeedColumn",
    "TechniqueColumn",
    "APFDcColumn",
    "APFDColumn",
]

missing_source_map_columns = [
    column
    for column in required_source_map_columns
    if column not in source_map.columns
]

if missing_source_map_columns:
    raise RuntimeError(
        "Step-1A source map is missing columns:\n"
        + "\n".join(
            missing_source_map_columns
        )
    )

source_map[
    "ProjectNumber"
] = pd.to_numeric(
    source_map[
        "ProjectNumber"
    ],
    errors="raise",
).astype(
    int
)

if len(
    source_map
) != 24:
    raise RuntimeError(
        f"Expected 24 source-map rows; found {len(source_map)}."
    )

if sorted(
    source_map[
        "ProjectNumber"
    ].tolist()
) != EXPECTED_PROJECT_NUMBERS:
    raise RuntimeError(
        "Source-map ProjectNumbers are not exactly 1..24."
    )

if source_map[
    "ProjectNumber"
].duplicated().any():
    raise RuntimeError(
        "Duplicate ProjectNumber in frozen source map."
    )


# --------------------------------------------------------------------------------------------------
# 6. BUILD THE 45,360-ROW CANONICAL SEED-LEVEL MASTER
# --------------------------------------------------------------------------------------------------

master_parts = []
per_project_input_audit = []

for ordinal, row in enumerate(
    source_map.sort_values(
        "ProjectNumber",
        kind="mergesort",
    ).to_dict(
        orient="records"
    ),
    start=1,
):
    project_number = int(
        row[
            "ProjectNumber"
        ]
    )

    project_name = str(
        row[
            "Project"
        ]
    )

    project_slug = str(
        row[
            "ProjectSlug"
        ]
    )

    source_path = Path(
        str(
            row[
                "SelectedPath"
            ]
        )
    )

    expected_source_sha = str(
        row[
            "SelectedFileSHA256"
        ]
    ).strip().lower()

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Project {project_number}: selected source disappeared:\n"
            f"{source_path}"
        )

    actual_source_sha = sha256_file(
        source_path
    )

    if actual_source_sha != expected_source_sha:
        raise RuntimeError(
            f"Project {project_number}: selected source SHA differs from Step 1A."
        )

    frame = read_table(
        source_path
    )

    if len(
        frame
    ) != EXPECTED_ROWS_PER_PROJECT:
        raise RuntimeError(
            f"Project {project_number}: selected source rows changed."
        )

    noise_col = str(
        row[
            "NoiseColumn"
        ]
    )

    seed_col = str(
        row[
            "SeedColumn"
        ]
    )

    technique_col = str(
        row[
            "TechniqueColumn"
        ]
    )

    apfdc_col = str(
        row[
            "APFDcColumn"
        ]
    )

    apfd_col = str(
        row[
            "APFDColumn"
        ]
    )

    required_selected_columns = [
        noise_col,
        seed_col,
        technique_col,
        apfdc_col,
        apfd_col,
    ]

    absent_selected_columns = [
        column
        for column in required_selected_columns
        if column not in frame.columns
    ]

    if absent_selected_columns:
        raise RuntimeError(
            f"Project {project_number}: selected source columns disappeared: "
            f"{absent_selected_columns}"
        )

    part = pd.DataFrame({
        "ProjectNumber":
            project_number,

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "NoisePercent":
            canonicalize_noise(
                frame[
                    noise_col
                ]
            ),

        "RepetitionSeed":
            canonicalize_seed(
                frame[
                    seed_col
                ]
            ),

        "Technique":
            canonicalize_technique(
                frame[
                    technique_col
                ]
            ),

        "MeanAPFDc":
            pd.to_numeric(
                frame[
                    apfdc_col
                ],
                errors="raise",
            ).astype(
                float
            ),

        "MeanAPFD":
            pd.to_numeric(
                frame[
                    apfd_col
                ],
                errors="raise",
            ).astype(
                float
            ),
    })

    part[
        "TechniqueFamily"
    ] = np.where(
        part[
            "Technique"
        ].isin(
            ML_TECHNIQUES
        ),
        "ML",
        "Baseline",
    )

    numeric_metrics = part[
        [
            "MeanAPFDc",
            "MeanAPFD",
        ]
    ].to_numpy(
        dtype=float
    )

    if not np.isfinite(
        numeric_metrics
    ).all():
        raise RuntimeError(
            f"Project {project_number}: non-finite APFD/APFDc value."
        )

    if (
        (
            numeric_metrics
            < -FLOAT_TOL
        ).any()
        or (
            numeric_metrics
            > 1.0
            + FLOAT_TOL
        ).any()
    ):
        raise RuntimeError(
            f"Project {project_number}: APFD/APFDc outside [0,1]."
        )

    duplicate_coordinates = int(
        part.duplicated(
            [
                "ProjectNumber",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            keep=False,
        ).sum()
    )

    if duplicate_coordinates != 0:
        raise RuntimeError(
            f"Project {project_number}: duplicate master coordinates."
        )

    observed_noise = sorted(
        part[
            "NoisePercent"
        ].unique().tolist()
    )

    observed_seeds = sorted(
        part[
            "RepetitionSeed"
        ].unique().tolist()
    )

    observed_techniques = sorted(
        part[
            "Technique"
        ].unique().tolist()
    )

    if observed_noise != EXPECTED_NOISE_LEVELS:
        raise RuntimeError(
            f"Project {project_number}: noise grid mismatch."
        )

    if observed_seeds != EXPECTED_SEEDS:
        raise RuntimeError(
            f"Project {project_number}: seed grid mismatch."
        )

    if observed_techniques != EXPECTED_TECHNIQUES:
        raise RuntimeError(
            f"Project {project_number}: technique grid mismatch."
        )

    if len(
        part
    ) != EXPECTED_ROWS_PER_PROJECT:
        raise RuntimeError(
            f"Project {project_number}: canonical row count != 1,890."
        )

    part = (
        part.sort_values(
            [
                "ProjectNumber",
                "NoisePercent",
                "RepetitionSeed",
                "Technique",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    master_parts.append(
        part
    )

    per_project_input_audit.append({
        "ProjectNumber":
            project_number,

        "Project":
            project_name,

        "ProjectSlug":
            project_slug,

        "SourcePath":
            str(
                source_path
            ),

        "SourceSHA256":
            actual_source_sha,

        "Rows":
            len(
                part
            ),

        "NoiseLevels":
            len(
                observed_noise
            ),

        "Seeds":
            len(
                observed_seeds
            ),

        "Techniques":
            len(
                observed_techniques
            ),

        "DuplicateCoordinates":
            duplicate_coordinates,

        "MinMeanAPFDc":
            float(
                part[
                    "MeanAPFDc"
                ].min()
            ),

        "MaxMeanAPFDc":
            float(
                part[
                    "MeanAPFDc"
                ].max()
            ),

        "MinMeanAPFD":
            float(
                part[
                    "MeanAPFD"
                ].min()
            ),

        "MaxMeanAPFD":
            float(
                part[
                    "MeanAPFD"
                ].max()
            ),
    })

    if (
        ordinal % 4 == 0
        or ordinal == 24
    ):
        print(
            f"  Canonical project sources loaded: {ordinal}/24"
        )


seed_master = pd.concat(
    master_parts,
    ignore_index=True,
)

seed_master = (
    seed_master.sort_values(
        [
            "ProjectNumber",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

project_coverage = pd.DataFrame(
    per_project_input_audit
).sort_values(
    "ProjectNumber",
    kind="mergesort",
).reset_index(
    drop=True
)


# --------------------------------------------------------------------------------------------------
# 7. GLOBAL COORDINATE / COVERAGE VALIDATION
# --------------------------------------------------------------------------------------------------

global_duplicate_coordinates = int(
    seed_master.duplicated(
        [
            "ProjectNumber",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        keep=False,
    ).sum()
)

project_row_counts = (
    seed_master.groupby(
        "ProjectNumber",
        sort=True,
    )
    .size()
)

project_noise_counts = (
    seed_master.groupby(
        [
            "ProjectNumber",
            "NoisePercent",
        ],
        sort=True,
    )
    .size()
)

project_technique_counts = (
    seed_master.groupby(
        [
            "ProjectNumber",
            "Technique",
        ],
        sort=True,
    )
    .size()
)

project_noise_technique_counts = (
    seed_master.groupby(
        [
            "ProjectNumber",
            "NoisePercent",
            "Technique",
        ],
        sort=True,
    )
    .size()
)

noise_counts = (
    seed_master.groupby(
        "NoisePercent",
        sort=True,
    )
    .size()
)

technique_counts = (
    seed_master.groupby(
        "Technique",
        sort=True,
    )
    .size()
)

technique_coverage = (
    seed_master.groupby(
        [
            "Technique",
            "TechniqueFamily",
        ],
        sort=True,
    )
    .agg(
        Projects=(
            "ProjectNumber",
            "nunique",
        ),

        NoiseLevels=(
            "NoisePercent",
            "nunique",
        ),

        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        Rows=(
            "Technique",
            "size",
        ),

        MinMeanAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaxMeanAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        MinMeanAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaxMeanAPFD=(
            "MeanAPFD",
            "max",
        ),
    )
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD PROJECT-LEVEL MASTER (30-SEED SUMMARY; PROJECT REMAINS INDEPENDENT UNIT)
# --------------------------------------------------------------------------------------------------

project_master = (
    seed_master.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "NoisePercent",
            "Technique",
            "TechniqueFamily",
        ],
        sort=True,
        observed=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanSeedMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        SDSeedMeanAPFDc=(
            "MeanAPFDc",
            lambda values:
                float(
                    values.std(
                        ddof=1
                    )
                ),
        ),

        MedianSeedMeanAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        MinSeedMeanAPFDc=(
            "MeanAPFDc",
            "min",
        ),

        MaxSeedMeanAPFDc=(
            "MeanAPFDc",
            "max",
        ),

        MeanSeedMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),

        SDSeedMeanAPFD=(
            "MeanAPFD",
            lambda values:
                float(
                    values.std(
                        ddof=1
                    )
                ),
        ),

        MedianSeedMeanAPFD=(
            "MeanAPFD",
            "median",
        ),

        MinSeedMeanAPFD=(
            "MeanAPFD",
            "min",
        ),

        MaxSeedMeanAPFD=(
            "MeanAPFD",
            "max",
        ),
    )
    .reset_index()
)

project_master = (
    project_master.sort_values(
        [
            "ProjectNumber",
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 9. BUILD SEED-LEVEL CLEAN-RELATIVE DELTAS
# --------------------------------------------------------------------------------------------------

clean_reference = (
    seed_master.loc[
        seed_master[
            "NoisePercent"
        ].eq(
            0
        ),
        [
            "ProjectNumber",
            "RepetitionSeed",
            "Technique",
            "MeanAPFDc",
            "MeanAPFD",
        ],
    ]
    .rename(
        columns={
            "MeanAPFDc":
                "CleanMeanAPFDc",

            "MeanAPFD":
                "CleanMeanAPFD",
        }
    )
)

expected_clean_rows = (
    EXPECTED_PROJECTS
    * len(
        EXPECTED_SEEDS
    )
    * len(
        EXPECTED_TECHNIQUES
    )
)  # 5,040

if len(
    clean_reference
) != expected_clean_rows:
    raise RuntimeError(
        f"Clean reference rows != {expected_clean_rows}."
    )

if clean_reference.duplicated(
    [
        "ProjectNumber",
        "RepetitionSeed",
        "Technique",
    ],
    keep=False,
).any():
    raise RuntimeError(
        "Duplicate clean reference coordinate."
    )

seed_delta = seed_master.merge(
    clean_reference,
    on=[
        "ProjectNumber",
        "RepetitionSeed",
        "Technique",
    ],
    how="left",
    validate="many_to_one",
)

if seed_delta[
    [
        "CleanMeanAPFDc",
        "CleanMeanAPFD",
    ]
].isna().any().any():
    raise RuntimeError(
        "Missing clean reference during seed-level delta construction."
    )

seed_delta[
    "DeltaMeanAPFDc"
] = (
    seed_delta[
        "MeanAPFDc"
    ]
    - seed_delta[
        "CleanMeanAPFDc"
    ]
)

seed_delta[
    "DeltaMeanAPFD"
] = (
    seed_delta[
        "MeanAPFD"
    ]
    - seed_delta[
        "CleanMeanAPFD"
    ]
)

# Numerically snap trivial floating-point zero at the clean condition.
for column in [
    "DeltaMeanAPFDc",
    "DeltaMeanAPFD",
]:
    values = seed_delta[
        column
    ].to_numpy(
        dtype=float
    )

    values[
        np.abs(
            values
        )
        <= FLOAT_TOL
    ] = 0.0

    seed_delta[
        column
    ] = values

seed_delta = seed_delta[
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "TechniqueFamily",
        "MeanAPFDc",
        "CleanMeanAPFDc",
        "DeltaMeanAPFDc",
        "MeanAPFD",
        "CleanMeanAPFD",
        "DeltaMeanAPFD",
    ]
]

seed_delta = (
    seed_delta.sort_values(
        [
            "ProjectNumber",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

zero_noise_delta_failures = int(
    (
        seed_delta.loc[
            seed_delta[
                "NoisePercent"
            ].eq(
                0
            ),
            [
                "DeltaMeanAPFDc",
                "DeltaMeanAPFD",
            ],
        ].abs()
        > FLOAT_TOL
    ).sum().sum()
)


# --------------------------------------------------------------------------------------------------
# 10. BUILD PROJECT-LEVEL CLEAN-RELATIVE DELTA MASTER
# --------------------------------------------------------------------------------------------------

project_delta = (
    seed_delta.groupby(
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "NoisePercent",
            "Technique",
            "TechniqueFamily",
        ],
        sort=True,
        observed=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        MeanDeltaMeanAPFDc=(
            "DeltaMeanAPFDc",
            "mean",
        ),

        SDDeltaMeanAPFDc=(
            "DeltaMeanAPFDc",
            lambda values:
                float(
                    values.std(
                        ddof=1
                    )
                ),
        ),

        MedianDeltaMeanAPFDc=(
            "DeltaMeanAPFDc",
            "median",
        ),

        MinDeltaMeanAPFDc=(
            "DeltaMeanAPFDc",
            "min",
        ),

        MaxDeltaMeanAPFDc=(
            "DeltaMeanAPFDc",
            "max",
        ),

        MeanDeltaMeanAPFD=(
            "DeltaMeanAPFD",
            "mean",
        ),

        SDDeltaMeanAPFD=(
            "DeltaMeanAPFD",
            lambda values:
                float(
                    values.std(
                        ddof=1
                    )
                ),
        ),

        MedianDeltaMeanAPFD=(
            "DeltaMeanAPFD",
            "median",
        ),

        MinDeltaMeanAPFD=(
            "DeltaMeanAPFD",
            "min",
        ),

        MaxDeltaMeanAPFD=(
            "DeltaMeanAPFD",
            "max",
        ),
    )
    .reset_index()
)

project_delta = (
    project_delta.sort_values(
        [
            "ProjectNumber",
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. DESCRIPTIVE BASELINE INVARIANCE DIAGNOSTIC (NOT A FREEZE FAILURE)
# --------------------------------------------------------------------------------------------------
#
# We record rather than impose a methodological assumption here.
# The thesis proposal describes baseline behavior differently across baseline types, and
# the historical project protocol evolved. Therefore this is diagnostic evidence only.
# --------------------------------------------------------------------------------------------------

baseline_rows = (
    seed_master.loc[
        seed_master[
            "Technique"
        ].isin(
            BASELINE_TECHNIQUES
        )
    ]
)

baseline_diagnostic = (
    baseline_rows.groupby(
        [
            "ProjectNumber",
            "Project",
            "Technique",
            "RepetitionSeed",
        ],
        sort=True,
    )
    .agg(
        NoiseLevels=(
            "NoisePercent",
            "nunique",
        ),

        APFDcRangeAcrossNoise=(
            "MeanAPFDc",
            lambda values:
                float(
                    values.max()
                    - values.min()
                ),
        ),

        APFDRangeAcrossNoise=(
            "MeanAPFD",
            lambda values:
                float(
                    values.max()
                    - values.min()
                ),
        ),
    )
    .reset_index()
)

baseline_diagnostic[
    "InvariantAcrossNoiseAt1e12"
] = (
    baseline_diagnostic[
        "APFDcRangeAcrossNoise"
    ].le(
        FLOAT_TOL
    )
    & baseline_diagnostic[
        "APFDRangeAcrossNoise"
    ].le(
        FLOAT_TOL
    )
)


# --------------------------------------------------------------------------------------------------
# 12. VALIDATE MASTER DATASETS BEFORE WRITING
# --------------------------------------------------------------------------------------------------

checks = []

registry_sha_after_computation = sha256_file(
    REGISTRY
)

add_check(
    checks,
    "Step-1A checkpoint SHA-256",
    EXPECTED_STEP1A_CHECKPOINT_SHA256,
    actual_step1a_checkpoint_sha,
    actual_step1a_checkpoint_sha
    == EXPECTED_STEP1A_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-1A source-map SHA-256",
    expected_source_map_sha,
    actual_source_map_sha,
    actual_source_map_sha
    == expected_source_map_sha,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after_computation,
    registry_sha_after_computation
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Seed-level master rows",
    EXPECTED_SEED_MASTER_ROWS,
    len(
        seed_master
    ),
    len(
        seed_master
    )
    == EXPECTED_SEED_MASTER_ROWS,
)

add_check(
    checks,
    "Seed-level master projects",
    24,
    seed_master[
        "ProjectNumber"
    ].nunique(),
    seed_master[
        "ProjectNumber"
    ].nunique()
    == 24,
)

add_check(
    checks,
    "Seed-level master noise levels",
    EXPECTED_NOISE_LEVELS,
    sorted(
        seed_master[
            "NoisePercent"
        ].unique().tolist()
    ),
    sorted(
        seed_master[
            "NoisePercent"
        ].unique().tolist()
    )
    == EXPECTED_NOISE_LEVELS,
)

add_check(
    checks,
    "Seed-level master seeds",
    EXPECTED_SEEDS,
    sorted(
        seed_master[
            "RepetitionSeed"
        ].unique().tolist()
    ),
    sorted(
        seed_master[
            "RepetitionSeed"
        ].unique().tolist()
    )
    == EXPECTED_SEEDS,
)

add_check(
    checks,
    "Seed-level master techniques",
    EXPECTED_TECHNIQUES,
    sorted(
        seed_master[
            "Technique"
        ].unique().tolist()
    ),
    sorted(
        seed_master[
            "Technique"
        ].unique().tolist()
    )
    == EXPECTED_TECHNIQUES,
)

add_check(
    checks,
    "Global duplicate coordinates",
    0,
    global_duplicate_coordinates,
    global_duplicate_coordinates
    == 0,
)

add_check(
    checks,
    "Rows per project",
    [EXPECTED_ROWS_PER_PROJECT],
    sorted(
        project_row_counts.unique().tolist()
    ),
    sorted(
        project_row_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_PROJECT
    ],
)

add_check(
    checks,
    "Rows per Project×Noise",
    [EXPECTED_ROWS_PER_PROJECT_NOISE],
    sorted(
        project_noise_counts.unique().tolist()
    ),
    sorted(
        project_noise_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_PROJECT_NOISE
    ],
)

add_check(
    checks,
    "Rows per Project×Technique",
    [EXPECTED_ROWS_PER_PROJECT_TECHNIQUE],
    sorted(
        project_technique_counts.unique().tolist()
    ),
    sorted(
        project_technique_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_PROJECT_TECHNIQUE
    ],
)

add_check(
    checks,
    "Rows per Project×Noise×Technique",
    [EXPECTED_ROWS_PER_PROJECT_NOISE_TECHNIQUE],
    sorted(
        project_noise_technique_counts.unique().tolist()
    ),
    sorted(
        project_noise_technique_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_PROJECT_NOISE_TECHNIQUE
    ],
)

add_check(
    checks,
    "Rows per noise level",
    [EXPECTED_ROWS_PER_NOISE],
    sorted(
        noise_counts.unique().tolist()
    ),
    sorted(
        noise_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_NOISE
    ],
)

add_check(
    checks,
    "Rows per technique",
    [EXPECTED_ROWS_PER_TECHNIQUE],
    sorted(
        technique_counts.unique().tolist()
    ),
    sorted(
        technique_counts.unique().tolist()
    )
    == [
        EXPECTED_ROWS_PER_TECHNIQUE
    ],
)

add_check(
    checks,
    "Project-level master rows",
    EXPECTED_PROJECT_MASTER_ROWS,
    len(
        project_master
    ),
    len(
        project_master
    )
    == EXPECTED_PROJECT_MASTER_ROWS,
)

add_check(
    checks,
    "Project-level master seeds per row",
    [30],
    sorted(
        project_master[
            "Seeds"
        ].unique().tolist()
    ),
    sorted(
        project_master[
            "Seeds"
        ].unique().tolist()
    )
    == [30],
)

add_check(
    checks,
    "Seed-level clean-delta rows",
    EXPECTED_SEED_DELTA_ROWS,
    len(
        seed_delta
    ),
    len(
        seed_delta
    )
    == EXPECTED_SEED_DELTA_ROWS,
)

add_check(
    checks,
    "Zero-noise delta failures",
    0,
    zero_noise_delta_failures,
    zero_noise_delta_failures
    == 0,
)

add_check(
    checks,
    "Project-level clean-delta rows",
    EXPECTED_PROJECT_DELTA_ROWS,
    len(
        project_delta
    ),
    len(
        project_delta
    )
    == EXPECTED_PROJECT_DELTA_ROWS,
)

add_check(
    checks,
    "Project-level clean-delta seeds per row",
    [30],
    sorted(
        project_delta[
            "Seeds"
        ].unique().tolist()
    ),
    sorted(
        project_delta[
            "Seeds"
        ].unique().tolist()
    )
    == [30],
)

add_check(
    checks,
    "Independent empirical unit",
    "Project (N=24)",
    "Project (N=24)",
    True,
)

add_check(
    checks,
    "Repeated stochastic unit",
    "Seed (30 within each project)",
    "Seed (30 within each project)",
    True,
)

add_check(
    checks,
    "Inferential statistical tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after_computation
    != registry_sha_before,
    registry_sha_after_computation
    == registry_sha_before,
)

validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nGlobal Analysis Step 1B pre-write validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 1B PRE-WRITE VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 13. WRITE THE FOUR CANONICAL DATASETS
# --------------------------------------------------------------------------------------------------

atomic_csv(
    SEED_MASTER_PATH,
    seed_master,
)

atomic_csv(
    PROJECT_MASTER_PATH,
    project_master,
)

atomic_csv(
    SEED_DELTA_PATH,
    seed_delta,
)

atomic_csv(
    PROJECT_DELTA_PATH,
    project_delta,
)

atomic_csv(
    PROJECT_COVERAGE_PATH,
    project_coverage,
)

atomic_csv(
    TECHNIQUE_COVERAGE_PATH,
    technique_coverage,
)

atomic_csv(
    BASELINE_DIAGNOSTIC_PATH,
    baseline_diagnostic,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 14. INDEPENDENT READBACK VALIDATION
# --------------------------------------------------------------------------------------------------

seed_master_rb = pd.read_csv(
    SEED_MASTER_PATH,
    low_memory=False,
)

project_master_rb = pd.read_csv(
    PROJECT_MASTER_PATH,
    low_memory=False,
)

seed_delta_rb = pd.read_csv(
    SEED_DELTA_PATH,
    low_memory=False,
)

project_delta_rb = pd.read_csv(
    PROJECT_DELTA_PATH,
    low_memory=False,
)

readback_rows = []


def add_readback(
    name,
    expected_frame,
    actual_frame,
    key_columns,
    float_columns,
):
    row_count_match = (
        len(
            expected_frame
        )
        == len(
            actual_frame
        )
    )

    expected_keys = (
        expected_frame[
            key_columns
        ].astype(
            str
        )
    )

    actual_keys = (
        actual_frame[
            key_columns
        ].astype(
            str
        )
    )

    key_match = bool(
        row_count_match
        and expected_keys.equals(
            actual_keys
        )
    )

    max_abs = (
        max_abs_float_difference(
            expected_frame,
            actual_frame,
            float_columns,
        )
        if row_count_match
        else np.inf
    )

    readback_rows.append({
        "Dataset":
            name,

        "ExpectedRows":
            len(
                expected_frame
            ),

        "ReadbackRows":
            len(
                actual_frame
            ),

        "RowCountMatch":
            row_count_match,

        "KeyOrderMatch":
            key_match,

        "MaxAbsFloatDifference":
            max_abs,

        "FloatReadbackPass":
            bool(
                np.isfinite(
                    max_abs
                )
                and max_abs
                <= FLOAT_TOL
            ),
    })


add_readback(
    "SeedLevelMaster",
    seed_master,
    seed_master_rb,
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "TechniqueFamily",
    ],
    [
        "MeanAPFDc",
        "MeanAPFD",
    ],
)

add_readback(
    "ProjectLevelMaster",
    project_master,
    project_master_rb,
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "Technique",
        "TechniqueFamily",
    ],
    [
        "MeanSeedMeanAPFDc",
        "SDSeedMeanAPFDc",
        "MedianSeedMeanAPFDc",
        "MinSeedMeanAPFDc",
        "MaxSeedMeanAPFDc",
        "MeanSeedMeanAPFD",
        "SDSeedMeanAPFD",
        "MedianSeedMeanAPFD",
        "MinSeedMeanAPFD",
        "MaxSeedMeanAPFD",
    ],
)

add_readback(
    "SeedLevelCleanDeltas",
    seed_delta,
    seed_delta_rb,
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "TechniqueFamily",
    ],
    [
        "MeanAPFDc",
        "CleanMeanAPFDc",
        "DeltaMeanAPFDc",
        "MeanAPFD",
        "CleanMeanAPFD",
        "DeltaMeanAPFD",
    ],
)

add_readback(
    "ProjectLevelCleanDeltas",
    project_delta,
    project_delta_rb,
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "Technique",
        "TechniqueFamily",
    ],
    [
        "MeanDeltaMeanAPFDc",
        "SDDeltaMeanAPFDc",
        "MedianDeltaMeanAPFDc",
        "MinDeltaMeanAPFDc",
        "MaxDeltaMeanAPFDc",
        "MeanDeltaMeanAPFD",
        "SDDeltaMeanAPFD",
        "MedianDeltaMeanAPFD",
        "MinDeltaMeanAPFD",
        "MaxDeltaMeanAPFD",
    ],
)

readback_audit = pd.DataFrame(
    readback_rows
)

readback_failures = int(
    (
        ~readback_audit[
            "RowCountMatch"
        ].astype(
            bool
        )
        |
        ~readback_audit[
            "KeyOrderMatch"
        ].astype(
            bool
        )
        |
        ~readback_audit[
            "FloatReadbackPass"
        ].astype(
            bool
        )
    ).sum()
)

if readback_failures != 0:
    atomic_csv(
        READBACK_AUDIT_PATH,
        readback_audit,
    )

    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 1B READBACK VALIDATION FAILED.\n"
        + readback_audit.to_string(
            index=False
        )
    )

atomic_csv(
    READBACK_AUDIT_PATH,
    readback_audit,
)


# --------------------------------------------------------------------------------------------------
# 15. SCIENTIFIC CONTENT HASHES + REPORT
# --------------------------------------------------------------------------------------------------

seed_master_scientific_sha = scientific_hash(
    seed_master,
    [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "NoisePercent",
        "RepetitionSeed",
        "Technique",
        "TechniqueFamily",
        "MeanAPFDc",
        "MeanAPFD",
    ],
)

project_master_scientific_sha = scientific_hash(
    project_master,
    list(
        project_master.columns
    ),
)

seed_delta_scientific_sha = scientific_hash(
    seed_delta,
    list(
        seed_delta.columns
    ),
)

project_delta_scientific_sha = scientific_hash(
    project_delta,
    list(
        project_delta.columns
    ),
)

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "Step":
        "GLOBAL_ANALYSIS_STEP_1B",

    "Status":
        STEP1B_STATUS,

    "CodeRevision":
        STEP1B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1ACheckpointSHA256":
        EXPECTED_STEP1A_CHECKPOINT_SHA256,

    "SourceMapSHA256":
        actual_source_map_sha,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "SeedsPerProject":
        30,

    "NoiseLevels":
        EXPECTED_NOISE_LEVELS,

    "Techniques":
        EXPECTED_TECHNIQUES,

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "PrimaryMetric":
        "MeanAPFDc",

    "SecondaryMetric":
        "MeanAPFD",

    "SeedLevelMasterRows":
        len(
            seed_master
        ),

    "ProjectLevelMasterRows":
        len(
            project_master
        ),

    "SeedLevelCleanDeltaRows":
        len(
            seed_delta
        ),

    "ProjectLevelCleanDeltaRows":
        len(
            project_delta
        ),

    "SeedLevelMasterPath":
        str(
            SEED_MASTER_PATH
        ),

    "SeedLevelMasterSHA256":
        sha256_file(
            SEED_MASTER_PATH
        ),

    "SeedLevelMasterScientificSHA256":
        seed_master_scientific_sha,

    "ProjectLevelMasterPath":
        str(
            PROJECT_MASTER_PATH
        ),

    "ProjectLevelMasterSHA256":
        sha256_file(
            PROJECT_MASTER_PATH
        ),

    "ProjectLevelMasterScientificSHA256":
        project_master_scientific_sha,

    "SeedLevelCleanDeltaPath":
        str(
            SEED_DELTA_PATH
        ),

    "SeedLevelCleanDeltaSHA256":
        sha256_file(
            SEED_DELTA_PATH
        ),

    "SeedLevelCleanDeltaScientificSHA256":
        seed_delta_scientific_sha,

    "ProjectLevelCleanDeltaPath":
        str(
            PROJECT_DELTA_PATH
        ),

    "ProjectLevelCleanDeltaSHA256":
        sha256_file(
            PROJECT_DELTA_PATH
        ),

    "ProjectLevelCleanDeltaScientificSHA256":
        project_delta_scientific_sha,

    "ZeroNoiseDeltaFailures":
        zero_noise_delta_failures,

    "ReadbackFailures":
        readback_failures,

    "BaselineInvarianceDiagnosticRows":
        len(
            baseline_diagnostic
        ),

    "BaselineInvariantAcrossNoiseRows":
        int(
            baseline_diagnostic[
                "InvariantAcrossNoiseAt1e12"
            ].astype(
                bool
            ).sum()
        ),

    "BaselineNonInvariantAcrossNoiseRows":
        int(
            (
                ~baseline_diagnostic[
                    "InvariantAcrossNoiseAt1e12"
                ].astype(
                    bool
                )
            ).sum()
        ),

    "InferentialStatisticalTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 2 — "
            "FREEZE THE STATISTICAL ANALYSIS CONTRACT BEFORE RQ1/RQ2/RQ3 TESTING"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            STEP1B_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "MasterDatasetsFrozen":
            True,

        "ReadyForStatisticalAnalysisContract":
            True,
    },
)

output_paths = [
    SEED_MASTER_PATH,
    PROJECT_MASTER_PATH,
    SEED_DELTA_PATH,
    PROJECT_DELTA_PATH,
    PROJECT_COVERAGE_PATH,
    TECHNIQUE_COVERAGE_PATH,
    BASELINE_DIAGNOSTIC_PATH,
    READBACK_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

registry_sha_final = sha256_file(
    REGISTRY
)

if registry_sha_final != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during Step 1B."
    )


# --------------------------------------------------------------------------------------------------
# 16. FREEZE STEP-1B CHECKPOINT
# --------------------------------------------------------------------------------------------------

checkpoint = {
    "CheckpointType":
        "GLOBAL_ANALYSIS_CANONICAL_CROSS_PROJECT_MASTER_DATASETS",

    "Status":
        STEP1B_STATUS,

    "CodeRevision":
        STEP1B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1ACheckpointSHA256":
        EXPECTED_STEP1A_CHECKPOINT_SHA256,

    "SourceMapSHA256":
        actual_source_map_sha,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "SeedLevelMasterPath":
        str(
            SEED_MASTER_PATH
        ),

    "SeedLevelMasterSHA256":
        sha256_file(
            SEED_MASTER_PATH
        ),

    "SeedLevelMasterScientificSHA256":
        seed_master_scientific_sha,

    "ProjectLevelMasterPath":
        str(
            PROJECT_MASTER_PATH
        ),

    "ProjectLevelMasterSHA256":
        sha256_file(
            PROJECT_MASTER_PATH
        ),

    "ProjectLevelMasterScientificSHA256":
        project_master_scientific_sha,

    "SeedLevelCleanDeltaPath":
        str(
            SEED_DELTA_PATH
        ),

    "SeedLevelCleanDeltaSHA256":
        sha256_file(
            SEED_DELTA_PATH
        ),

    "SeedLevelCleanDeltaScientificSHA256":
        seed_delta_scientific_sha,

    "ProjectLevelCleanDeltaPath":
        str(
            PROJECT_DELTA_PATH
        ),

    "ProjectLevelCleanDeltaSHA256":
        sha256_file(
            PROJECT_DELTA_PATH
        ),

    "ProjectLevelCleanDeltaScientificSHA256":
        project_delta_scientific_sha,

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "SeedLevelMasterRows":
        EXPECTED_SEED_MASTER_ROWS,

    "ProjectLevelMasterRows":
        EXPECTED_PROJECT_MASTER_ROWS,

    "SeedLevelCleanDeltaRows":
        EXPECTED_SEED_DELTA_ROWS,

    "ProjectLevelCleanDeltaRows":
        EXPECTED_PROJECT_DELTA_ROWS,

    "InferentialStatisticalTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "ReadyForStatisticalAnalysisContract":
        True,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 2 — "
            "FREEZE THE STATISTICAL ANALYSIS CONTRACT BEFORE RQ1/RQ2/RQ3 TESTING"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 144
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 3 / STEP 1B RESULT ==="
)

print(
    "=" * 144
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Repeated stochastic unit: Seed (30 within each project)"
)

print(
    "\nCanonical seed-level master:"
)

print(
    "Rows:",
    len(
        seed_master
    ),
    "/",
    EXPECTED_SEED_MASTER_ROWS,
)

print(
    "Projects:",
    seed_master[
        "ProjectNumber"
    ].nunique(),
)

print(
    "Noise levels:",
    seed_master[
        "NoisePercent"
    ].nunique(),
)

print(
    "Seeds:",
    seed_master[
        "RepetitionSeed"
    ].nunique(),
)

print(
    "Techniques:",
    seed_master[
        "Technique"
    ].nunique(),
)

print(
    "Duplicate coordinates:",
    global_duplicate_coordinates,
)

print(
    "CSV SHA-256:",
    sha256_file(
        SEED_MASTER_PATH
    ),
)

print(
    "Scientific content SHA-256:",
    seed_master_scientific_sha,
)

print(
    "\nCanonical project-level master:"
)

print(
    "Rows:",
    len(
        project_master
    ),
    "/",
    EXPECTED_PROJECT_MASTER_ROWS,
)

print(
    "Seeds summarized per row:",
    sorted(
        project_master[
            "Seeds"
        ].unique().tolist()
    ),
)

print(
    "CSV SHA-256:",
    sha256_file(
        PROJECT_MASTER_PATH
    ),
)

print(
    "\nClean-relative delta masters:"
)

print(
    "Seed-level delta rows:",
    len(
        seed_delta
    ),
    "/",
    EXPECTED_SEED_DELTA_ROWS,
)

print(
    "Project-level delta rows:",
    len(
        project_delta
    ),
    "/",
    EXPECTED_PROJECT_DELTA_ROWS,
)

print(
    "Zero-noise delta failures:",
    zero_noise_delta_failures,
)

print(
    "\nBaseline invariance diagnostic (descriptive only):"
)

print(
    "Rows:",
    len(
        baseline_diagnostic
    ),
)

print(
    "Invariant across noise:",
    int(
        baseline_diagnostic[
            "InvariantAcrossNoiseAt1e12"
        ].astype(
            bool
        ).sum()
    ),
)

print(
    "Non-invariant across noise:",
    int(
        (
            ~baseline_diagnostic[
                "InvariantAcrossNoiseAt1e12"
            ].astype(
                bool
            )
        ).sum()
    ),
)

print(
    "\nReadback:"
)

print(
    "Datasets re-read:",
    len(
        readback_audit
    ),
)

print(
    "Readback failures:",
    readback_failures,
)

print(
    "\nIsolation:"
)

print(
    "Inferential statistical tests executed: False"
)

print(
    "P-values computed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nStep 1B checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "GLOBAL ANALYSIS STEP 2 — FREEZE THE STATISTICAL ANALYSIS CONTRACT "
    "BEFORE RQ1/RQ2/RQ3 TESTING"
)

print(
    "\nSTATUS:",
    STEP1B_STATUS,
)

print(
    "=" * 144
)


=== THESIS GLOBAL ANALYSIS — CELL 3 / STEP 1B: CANONICAL CROSS-PROJECT MASTER DATASETS ===
  Canonical project sources loaded: 4/24
  Canonical project sources loaded: 8/24
  Canonical project sources loaded: 12/24
  Canonical project sources loaded: 16/24
  Canonical project sources loaded: 20/24
  Canonical project sources loaded: 24/24

Global Analysis Step 1B pre-write validation:


,Check,Expected,Actual,Pass
0,Step-1A checkpoint SHA-256,18a121d4da0dff809db5a18a9c59fadf39af1ca4313b9f...,18a121d4da0dff809db5a18a9c59fadf39af1ca4313b9f...,True
1,Step-1A source-map SHA-256,f0fdd1b0b8af179f378aa23f54cea359a84738b1bb28aa...,f0fdd1b0b8af179f378aa23f54cea359a84738b1bb28aa...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,Seed-level master rows,45360,45360,True
4,Seed-level master projects,24,24,True
5,Seed-level master noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
6,Seed-level master seeds,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
7,Seed-level master techniques,"[LatestFail, LightGBM, NaiveBayes, QTF-Avg, Ra...","[LatestFail, LightGBM, NaiveBayes, QTF-Avg, Ra...",True
8,Global duplicate coordinates,0,0,True
9,Rows per project,[1890],[1890],True



=== THESIS GLOBAL ANALYSIS — CELL 3 / STEP 1B RESULT ===
Independent empirical unit: Project (N=24)
Repeated stochastic unit: Seed (30 within each project)

Canonical seed-level master:
Rows: 45360 / 45360
Projects: 24
Noise levels: 9
Seeds: 30
Techniques: 7
Duplicate coordinates: 0
CSV SHA-256: e869119e9b5078e1e58fbed8cc4992afde5cbeb0dc2d12cd2f49bbe584f947ea
Scientific content SHA-256: d801fdb47d663aa9762a9d7daf119272f1526662f611da845a0a8ac4fe4e11b6

Canonical project-level master:
Rows: 1512 / 1512
Seeds summarized per row: [30]
CSV SHA-256: 65f4c09ea38f5952c6af4ca46de017940be761dc8cfd4a6edd6acf92bf03bc78

Clean-relative delta masters:
Seed-level delta rows: 45360 / 45360
Project-level delta rows: 1512 / 1512
Zero-noise delta failures: 0

Baseline invariance diagnostic (descriptive only):
Rows: 2160
Invariant across noise: 1440
Non-invariant across noise: 720

Readback:
Datasets re-read: 4
Readback failures: 0

Isolation:
Inferential statistical tests executed: False
P-values comput

In [6]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 4 / STEP 2
# FREEZE THE STATISTICAL ANALYSIS CONTRACT BEFORE RQ1 / RQ2 / RQ3 TESTING
# ==================================================================================================
#
# PURPOSE
# -------
# The canonical 24-project master datasets are already frozen.
#
# BEFORE computing any inferential statistic or p-value, this cell freezes the exact analysis
# rules that will be used to answer RQ1, RQ2 and RQ3.
#
# This prevents post-hoc test selection or threshold definition after seeing the results.
#
# IMPORTANT:
#   - This cell does NOT compute p-values.
#   - This cell does NOT run Wilcoxon, Friedman or Nemenyi.
#   - This cell does NOT inspect which ML algorithm performs best.
#   - This cell does NOT modify any project result or registry row.
#
# INDEPENDENT EMPIRICAL UNIT
# --------------------------
# Project (N = 24).
#
# The 30 seeds are repeated stochastic runs WITHIN a project. They are used for stability,
# uncertainty and within-project aggregation; they are NOT counted as 720 independent projects.
#
# PRIMARY METRIC
# --------------
# APFDc.
#
# SECONDARY METRIC
# ----------------
# APFD.
#
# RQ1
# ---
# For each of the four ML algorithms, compare each noisy level
# {5,10,15,20,25,30,40,50}% against 0% using a paired, two-sided Wilcoxon signed-rank test
# on the 24 project-level means (each project-level mean already summarizes its 30 seeds).
#
# Bonferroni family:
#   - 8 comparisons within each algorithm for APFDc (primary family)
#   - 8 comparisons within each algorithm for APFD (separate secondary family)
# Therefore adjusted alpha per family = 0.05 / 8 = 0.00625.
#
# RQ2
# ---
# Primary crossover metric = APFDc.
#
# For each ML algorithm and each baseline, define the FIRST OBSERVED crossover as the lowest
# TESTED noise level p where the equally-weighted cross-project mean ML APFDc <= the equally-weighted
# cross-project mean baseline APFDc at the SAME noise level.
#
# No interpolation between tested noise levels is allowed.
#
# The proposal's primary threshold is versus LatestFail.
# Because the RQ names Random, LatestFail and QTF-Avg, pairwise thresholds versus all three are
# also reported. A conservative "strongest baseline" threshold is also reported using the maximum
# baseline APFDc at each tested noise level.
#
# A sustained crossover is supplementary: the first tested level from which the ML technique
# remains <= the comparator at all higher tested levels.
#
# RQ2 is descriptive by the proposal: no new primary hypothesis test is introduced here.
# Project-level paired advantages and win/tie/loss counts are supporting descriptive evidence.
#
# RQ3
# ---
# At each of the nine noise levels, compare RF / XGBoost / LightGBM / NaiveBayes using a Friedman
# test on the 24 project-level mean APFDc values.
#
# The nine APFDc Friedman omnibus tests form one family and are Holm-corrected at alpha = 0.05.
# Only a noise level with a Holm-significant Friedman result proceeds to Nemenyi post-hoc
# comparison of average ranks at alpha = 0.05.
#
# APFD repeats the same RQ3 procedure as a separate SECONDARY sensitivity family.
#
# Operational definition of "most robust":
#   1. Primary practical robustness focuses on the stress-test levels {30,40,50}%.
#   2. For each algorithm, compute equally-weighted cross-project mean APFDc at each of those levels.
#   3. Compute its mean absolute APFDc across {30,40,50}.
#   4. Compute average Friedman rank across {30,40,50}; lower rank is better.
#   5. Clean-relative degradation/retention is reported as supporting evidence.
#   6. A unique "most robust" winner is claimed only when the absolute high-noise performance and
#      rank evidence point to the same leading model and the inferential evidence does not contradict
#      that interpretation. Otherwise the thesis reports a trade-off / no unique robust winner.
#
# DESCRIPTIVE 30-SEED UNCERTAINTY
# -------------------------------
# To preserve the proposal's 30-repetition reporting:
#   - for each Noise×Technique×Seed, first compute the equally-weighted mean across 24 projects;
#   - across the 30 resulting global-seed means, report mean, sample SD (ddof=1), and a two-sided
#     95% Student-t confidence interval with df=29.
#
# Cross-project heterogeneity is also reported from the 24 project-level means using mean, median,
# sample SD and IQR. Inferential tests always use projects as the independent paired blocks.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import scipy


print("=" * 146)
print("=== THESIS GLOBAL ANALYSIS — CELL 4 / STEP 2: STATISTICAL ANALYSIS CONTRACT FREEZE ===")
print("=" * 146)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

EXPECTED_STEP1B_CHECKPOINT_SHA256 = (
    "eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3e0642d1b19f456ad50"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

STEP2_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
)

STEP2_CODE_REVISION = (
    "GLOBAL_ANALYSIS_STEP2_V1_PREINFERENCE_RQ1_RQ2_RQ3_STATISTICAL_CONTRACT"
)

ALPHA = 0.05
RQ1_BONFERRONI_FAMILY_SIZE = 8
RQ1_ADJUSTED_ALPHA = ALPHA / RQ1_BONFERRONI_FAMILY_SIZE

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
NOISY_LEVELS = [5, 10, 15, 20, 25, 30, 40, 50]
STRESS_LEVELS = [30, 40, 50]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

EXPECTED_PROJECTS = 24
EXPECTED_SEEDS = 30
EXPECTED_SEED_MASTER_ROWS = 45_360
EXPECTED_PROJECT_MASTER_ROWS = 1_512
EXPECTED_SEED_DELTA_ROWS = 45_360
EXPECTED_PROJECT_DELTA_ROWS = 1_512


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"

STEP1B_CHECKPOINT = (
    NOTES / "global_analysis_step1b_checkpoint.json"
)

STEP2_ROOT = (
    ANALYSIS_ROOT
    / "Step_2_Statistical_Analysis_Contract"
)

CONTRACT_JSON_PATH = (
    STEP2_ROOT
    / "global_statistical_analysis_contract.json"
)

CONTRACT_TABLE_PATH = (
    STEP2_ROOT
    / "global_statistical_analysis_contract.csv"
)

PLANNED_TESTS_PATH = (
    STEP2_ROOT
    / "global_planned_inferential_tests.csv"
)

RQ2_THRESHOLD_PLAN_PATH = (
    STEP2_ROOT
    / "global_rq2_threshold_definition_plan.csv"
)

VALIDATION_PATH = (
    STEP2_ROOT
    / "global_analysis_step2_validation.csv"
)

REPORT_PATH = (
    STEP2_ROOT
    / "global_analysis_step2_report.json"
)

STATUS_PATH = (
    STEP2_ROOT
    / "global_analysis_step2_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP2_ROOT
    / "global_analysis_step2_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES / "global_analysis_step2_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def atomic_csv(path, dataframe):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def build_output_manifest(paths):
    rows = []

    for path in sorted(
        (
            Path(path)
            for path in paths
        ),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"Step-2 output missing: {path}"
            )

        rows.append({
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Path",
            "Bytes",
            "SHA256",
        ],
    )


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME FREEZE GUARD + UPSTREAM VERIFICATION
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "Global Analysis Step 2 is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ1."
    )

if not STEP1B_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-1B checkpoint missing: {STEP1B_CHECKPOINT}"
    )

actual_step1b_checkpoint_sha = sha256_file(
    STEP1B_CHECKPOINT
)

if (
    actual_step1b_checkpoint_sha
    != EXPECTED_STEP1B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-1B checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_STEP1B_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_step1b_checkpoint_sha}"
    )

step1b = load_json(
    STEP1B_CHECKPOINT
)

if step1b.get(
    "Status"
) != STEP1B_STATUS:
    raise RuntimeError(
        "Step-1B checkpoint is not in the frozen PASS state."
    )

if not bool(
    step1b.get(
        "ReadyForStatisticalAnalysisContract",
        False,
    )
):
    raise RuntimeError(
        "Step-1B checkpoint is not marked ready for the statistical contract."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from the 24-project final freeze."
    )

master_specs = [
    (
        "SeedLevelMaster",
        "SeedLevelMasterPath",
        "SeedLevelMasterSHA256",
        "SeedLevelMasterRows",
        EXPECTED_SEED_MASTER_ROWS,
    ),
    (
        "ProjectLevelMaster",
        "ProjectLevelMasterPath",
        "ProjectLevelMasterSHA256",
        "ProjectLevelMasterRows",
        EXPECTED_PROJECT_MASTER_ROWS,
    ),
    (
        "SeedLevelCleanDelta",
        "SeedLevelCleanDeltaPath",
        "SeedLevelCleanDeltaSHA256",
        "SeedLevelCleanDeltaRows",
        EXPECTED_SEED_DELTA_ROWS,
    ),
    (
        "ProjectLevelCleanDelta",
        "ProjectLevelCleanDeltaPath",
        "ProjectLevelCleanDeltaSHA256",
        "ProjectLevelCleanDeltaRows",
        EXPECTED_PROJECT_DELTA_ROWS,
    ),
]

master_anchor_rows = []

for (
    dataset_name,
    path_key,
    sha_key,
    rows_key,
    expected_rows,
) in master_specs:
    path = Path(
        step1b[
            path_key
        ]
    )

    expected_sha = str(
        step1b[
            sha_key
        ]
    ).lower()

    checkpoint_rows = int(
        step1b[
            rows_key
        ]
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen {dataset_name} file is missing: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{dataset_name} SHA differs from Step-1B checkpoint."
        )

    if checkpoint_rows != expected_rows:
        raise RuntimeError(
            f"{dataset_name} checkpoint row count mismatch: "
            f"{checkpoint_rows} != {expected_rows}"
        )

    # Read only to verify row count and schema; NO result inspection or statistics.
    frame = pd.read_csv(
        path,
        low_memory=False,
    )

    if len(
        frame
    ) != expected_rows:
        raise RuntimeError(
            f"{dataset_name} current row count mismatch."
        )

    master_anchor_rows.append({
        "Dataset": dataset_name,
        "Path": str(path),
        "Rows": len(frame),
        "SHA256": actual_sha,
        "ColumnsJSON": json.dumps(
            frame.columns.tolist(),
            separators=(",", ":"),
        ),
    })

master_anchor_table = pd.DataFrame(
    master_anchor_rows
)

if STEP2_ROOT.exists():
    shutil.rmtree(
        STEP2_ROOT,
        ignore_errors=True,
    )

STEP2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. FREEZE THE HUMAN/SCIENTIFIC ANALYSIS CONTRACT
# --------------------------------------------------------------------------------------------------

contract = {
    "ContractName":
        "THESIS_GLOBAL_STATISTICAL_ANALYSIS_CONTRACT_V1",

    "Status":
        STEP2_STATUS,

    "CodeRevision":
        STEP2_CODE_REVISION,

    "FrozenBeforeInferentialTesting":
        True,

    "UpstreamStep1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "StudySample": {
        "IncludedProjects":
            24,

        "IndependentEmpiricalUnit":
            "Project",

        "IndependentEmpiricalUnitN":
            24,

        "RepeatedStochasticUnit":
            "RepetitionSeed",

        "SeedsPerProject":
            30,

        "NoiseLevelsPercent":
            NOISE_LEVELS,

        "MLTechniques":
            ML_TECHNIQUES,

        "Baselines":
            BASELINES,
    },

    "Metrics": {
        "Primary":
            "APFDc",

        "PrimaryProjectLevelVariable":
            "MeanSeedMeanAPFDc",

        "PrimarySeedLevelVariable":
            "MeanAPFDc",

        "Secondary":
            "APFD",

        "SecondaryProjectLevelVariable":
            "MeanSeedMeanAPFD",

        "SecondarySeedLevelVariable":
            "MeanAPFD",

        "MetricRange":
            "[0,1]",

        "Direction":
            "Higher is better",
    },

    "DescriptiveReporting": {
        "ThirtySeedGlobalCurveProcedure": (
            "For each Noise×Technique×Seed, compute the equally-weighted mean across the "
            "24 projects. Across the resulting 30 global-seed means, report the mean, "
            "sample SD (ddof=1), and two-sided 95% Student-t confidence interval with df=29."
        ),

        "CrossProjectHeterogeneityProcedure": (
            "At each Noise×Technique, summarize the 24 project-level means using mean, "
            "median, sample SD, Q1, Q3 and IQR."
        ),

        "EqualProjectWeighting":
            True,
    },

    "GeneralInference": {
        "Alpha":
            ALPHA,

        "IndependentBlocks":
            "Projects",

        "ProjectBlocksN":
            24,

        "SeedPseudoReplicationForbidden":
            True,

        "SeedsCountedAsIndependentProjects":
            False,

        "PrimaryConclusionsMetric":
            "APFDc",

        "SecondarySensitivityMetric":
            "APFD",
    },

    "RQ1": {
        "Question": (
            "How do increasing levels of simulated flaky test noise (0%-50%) in historical "
            "training data affect supervised ML-based TCP effectiveness as measured by APFD/APFDc?"
        ),

        "Techniques":
            ML_TECHNIQUES,

        "ReferenceNoisePercent":
            0,

        "ComparedNoisePercent":
            NOISY_LEVELS,

        "PrimaryTest":
            "Wilcoxon signed-rank",

        "PairingUnit":
            "Project",

        "InputPerPair":
            (
                "Project-level mean over the 30 seeds at noisy p versus the same project's "
                "project-level mean over the 30 seeds at 0%."
            ),

        "Alternative":
            "two-sided",

        "WilcoxonZeroMethod":
            "wilcox",

        "WilcoxonMethod":
            "auto",

        "PrimaryMetric":
            "APFDc",

        "PrimaryFamilyDefinition":
            "8 noisy-vs-clean comparisons within each ML algorithm",

        "PrimaryCorrection":
            "Bonferroni",

        "FamilySize":
            8,

        "NominalAlpha":
            ALPHA,

        "BonferroniAdjustedAlpha":
            RQ1_ADJUSTED_ALPHA,

        "AdjustedPValueRule":
            "min(raw_p * 8, 1.0)",

        "SecondaryMetric":
            "APFD",

        "SecondaryFamilyDefinition":
            "Separate 8-comparison Bonferroni family within each ML algorithm",

        "EffectSize":
            "paired rank-biserial correlation",

        "EffectSizeSignConvention":
            (
                "Difference = noisy - clean. Negative effect indicates degradation."
            ),

        "RequiredDescriptives": [
            "mean project-level APFDc",
            "median project-level APFDc",
            "sample SD across project means",
            "mean clean-relative delta",
            "median clean-relative delta",
            "percentage degradation where clean value > 0",
            "retention percentage where clean value > 0",
            "30-seed global curve SD and 95% t CI",
        ],
    },

    "RQ2": {
        "Question": (
            "At what noise level do supervised ML-based TCP techniques cease to outperform "
            "Random, LatestFail and QTF-Avg?"
        ),

        "PrimaryMetric":
            "APFDc",

        "PrimaryProposalComparator":
            "LatestFail",

        "AllReportedComparators":
            BASELINES,

        "CrossProjectTechniqueScore":
            (
                "Equally-weighted mean of the 24 project-level 30-seed mean APFDc values "
                "at the same tested noise level."
            ),

        "PairwiseAdvantage":
            "ML cross-project mean APFDc - baseline cross-project mean APFDc",

        "FirstObservedCrossoverDefinition":
            (
                "Lowest TESTED noise level p where pairwise advantage <= 0."
            ),

        "AlreadyNotOutperformingAtCleanRule":
            (
                "If advantage <= 0 at 0%, report that the ML technique does not have a "
                "positive noise-tolerance threshold against that comparator."
            ),

        "NoCrossoverThrough50Rule":
            (
                "If advantage remains > 0 through 50%, report >50% / not observed in tested range."
            ),

        "InterpolationAllowed":
            False,

        "SustainedCrossoverDefinition":
            (
                "Supplementary: lowest tested p for which advantage <= 0 at p and every "
                "higher tested noise level."
            ),

        "StrongestBaselineDefinition":
            (
                "At each noise level, strongest baseline = maximum cross-project mean APFDc "
                "among Random, LatestFail and QTF-Avg."
            ),

        "StrongestBaselineThresholdReported":
            True,

        "PrimaryHypothesisTestIntroduced":
            False,

        "SupportingDescriptives": [
            "paired project-level ML-minus-baseline differences",
            "mean paired advantage",
            "median paired advantage",
            "project win count",
            "project tie count at numerical tolerance 1e-12",
            "project loss count",
            "first observed crossover",
            "sustained crossover",
        ],

        "SecondarySensitivityMetric":
            "APFD",
    },

    "RQ3": {
        "Question": (
            "Which of Random Forest, XGBoost, LightGBM and Naive Bayes is most robust "
            "to flaky test noise in its training data?"
        ),

        "Algorithms":
            ML_TECHNIQUES,

        "PrimaryMetric":
            "APFDc",

        "OmnibusTest":
            "Friedman",

        "Blocks":
            "24 projects",

        "Treatments":
            "4 ML algorithms",

        "NoiseLevelsTestedSeparately":
            NOISE_LEVELS,

        "OmnibusFamilySize":
            9,

        "OmnibusMultipleTestingCorrection":
            "Holm",

        "OmnibusFamilyAlpha":
            ALPHA,

        "PostHocTrigger":
            (
                "Run Nemenyi at a noise level only if its APFDc Friedman omnibus result "
                "is significant after Holm correction across the 9 APFDc omnibus tests."
            ),

        "PostHocTest":
            "Nemenyi average-rank comparison",

        "PostHocAlpha":
            ALPHA,

        "WithinNoisePostHocMultiplicity":
            "Controlled by the Nemenyi procedure",

        "RankingDirection":
            "Rank 1 = highest APFDc / best",

        "StressNoiseLevelsPercent":
            STRESS_LEVELS,

        "PrimaryRobustnessSummary": (
            "For each algorithm, compute its equally-weighted cross-project mean APFDc at "
            "30%, 40% and 50%; average those three values. Also average its Friedman rank "
            "over 30%, 40% and 50%."
        ),

        "SupportingRobustnessEvidence": [
            "clean-relative APFDc degradation",
            "APFDc retention percentage",
            "performance at practical low-noise levels 5%, 10%, 15%",
            "performance at stress levels 30%, 40%, 50%",
        ],

        "UniqueWinnerRule": (
            "Claim a unique most-robust model only if the absolute high-noise APFDc summary "
            "and stress-level rank evidence identify the same leading model and the inferential "
            "results do not contradict that interpretation. Otherwise report no unique universal "
            "winner and describe the observed trade-off."
        ),

        "SecondarySensitivityMetric":
            "APFD",

        "SecondarySensitivityProcedure": (
            "Repeat Friedman/Holm/Nemenyi using APFD as a separate secondary family."
        ),
    },

    "ReportingRules": {
        "NoContinuousThresholdInterpolation":
            True,

        "DoNotTreatSeedsAsIndependentDatasets":
            True,

        "DoNotSelectTestsAfterSeeingPValues":
            True,

        "DoNotDropProjectsBecauseOfUnfavorableResults":
            True,

        "DoNotRemoveOutliersWithoutPredeclaredDataIntegrityReason":
            True,

        "APFDcPrimary":
            True,

        "APFDSecondary":
            True,

        "AllowNoThresholdFinding":
            True,

        "AllowNoUniqueRQ3Winner":
            True,
    },

    "SoftwareContract": {
        "Python":
            platform.python_version(),

        "NumPy":
            np.__version__,

        "Pandas":
            pd.__version__,

        "SciPy":
            scipy.__version__,
    },

    "FrozenMasterDatasets":
        master_anchor_rows,

    "InferentialTestsExecutedInThisCell":
        False,

    "PValuesComputedInThisCell":
        False,
}


# --------------------------------------------------------------------------------------------------
# 6. BUILD FLAT CONTRACT TABLE
# --------------------------------------------------------------------------------------------------

contract_rows = [
    {
        "Section": "Global",
        "Decision": "Independent empirical unit",
        "FrozenValue": "Project (N=24)",
        "Rationale": "Avoid seed pseudoreplication; projects are independent datasets.",
    },
    {
        "Section": "Global",
        "Decision": "Repeated stochastic unit",
        "FrozenValue": "30 seeds within each project",
        "Rationale": "Seeds quantify stochastic variability but are not independent projects.",
    },
    {
        "Section": "Global",
        "Decision": "Primary metric",
        "FrozenValue": "APFDc",
        "Rationale": "Cost-aware metric specified as primary in the thesis proposal.",
    },
    {
        "Section": "Global",
        "Decision": "Secondary metric",
        "FrozenValue": "APFD",
        "Rationale": "Secondary comparability/sensitivity metric from the thesis proposal.",
    },
    {
        "Section": "RQ1",
        "Decision": "Primary inferential test",
        "FrozenValue": "Paired two-sided Wilcoxon signed-rank",
        "Rationale": "Non-parametric paired noisy-vs-clean comparison across 24 projects.",
    },
    {
        "Section": "RQ1",
        "Decision": "Bonferroni family",
        "FrozenValue": "8 noise-vs-clean tests per algorithm per metric",
        "Rationale": "Predeclares multiple-comparison family before p-values are observed.",
    },
    {
        "Section": "RQ1",
        "Decision": "Bonferroni adjusted alpha",
        "FrozenValue": f"{RQ1_ADJUSTED_ALPHA:.17g}",
        "Rationale": "0.05 / 8.",
    },
    {
        "Section": "RQ2",
        "Decision": "Primary threshold comparator",
        "FrozenValue": "LatestFail",
        "Rationale": "Matches the proposal's explicit threshold definition.",
    },
    {
        "Section": "RQ2",
        "Decision": "Full RQ comparator set",
        "FrozenValue": "Random; LatestFail; QTF-Avg",
        "Rationale": "Matches the wording of RQ2.",
    },
    {
        "Section": "RQ2",
        "Decision": "Crossover",
        "FrozenValue": "First tested noise level with ML mean APFDc <= comparator mean APFDc",
        "Rationale": "No interpolation beyond the experimentally tested grid.",
    },
    {
        "Section": "RQ2",
        "Decision": "Formal primary test",
        "FrozenValue": "None",
        "Rationale": "Proposal defines RQ2 primarily as a descriptive crossover analysis.",
    },
    {
        "Section": "RQ3",
        "Decision": "Omnibus test",
        "FrozenValue": "Friedman at each noise level",
        "Rationale": "Matches the proposal's multi-algorithm comparison plan.",
    },
    {
        "Section": "RQ3",
        "Decision": "Omnibus family correction",
        "FrozenValue": "Holm across 9 APFDc Friedman tests",
        "Rationale": "Controls family-wise error across tested noise levels.",
    },
    {
        "Section": "RQ3",
        "Decision": "Post-hoc",
        "FrozenValue": "Nemenyi only after Holm-significant Friedman",
        "Rationale": "Matches proposal and prevents unnecessary pairwise testing.",
    },
    {
        "Section": "RQ3",
        "Decision": "Stress levels",
        "FrozenValue": "30%; 40%; 50%",
        "Rationale": "Proposal identifies these as high/stress-test noise conditions.",
    },
    {
        "Section": "RQ3",
        "Decision": "Unique winner rule",
        "FrozenValue": "Require aligned absolute high-noise APFDc + rank evidence",
        "Rationale": "Prevents forcing a winner when robustness evidence is mixed.",
    },
]


contract_table = pd.DataFrame(
    contract_rows
)


# --------------------------------------------------------------------------------------------------
# 7. PRE-REGISTER EVERY RQ1 AND RQ3 INFERENTIAL TEST
# --------------------------------------------------------------------------------------------------

planned_tests = []

# RQ1 APFDc primary + APFD secondary.
for metric_role, metric in [
    ("PRIMARY", "APFDc"),
    ("SECONDARY", "APFD"),
]:
    for algorithm in ML_TECHNIQUES:
        family_id = (
            f"RQ1_{metric}_{algorithm}_NOISY_VS_CLEAN"
        )

        for noise in NOISY_LEVELS:
            planned_tests.append({
                "RQ": "RQ1",
                "MetricRole": metric_role,
                "Metric": metric,
                "TechniqueOrSet": algorithm,
                "NoisePercent": noise,
                "ReferenceNoisePercent": 0,
                "Test": "Wilcoxon signed-rank",
                "Alternative": "two-sided",
                "IndependentUnit": "Project",
                "N": 24,
                "FamilyID": family_id,
                "FamilySize": 8,
                "Correction": "Bonferroni",
                "Alpha": ALPHA,
                "FamilyAdjustedAlpha": RQ1_ADJUSTED_ALPHA,
                "PostHocConditional": False,
            })

# RQ3 APFDc primary + APFD secondary omnibus.
for metric_role, metric in [
    ("PRIMARY", "APFDc"),
    ("SECONDARY", "APFD"),
]:
    family_id = (
        f"RQ3_{metric}_FRIEDMAN_ACROSS_NOISE"
    )

    for noise in NOISE_LEVELS:
        planned_tests.append({
            "RQ": "RQ3",
            "MetricRole": metric_role,
            "Metric": metric,
            "TechniqueOrSet": "RF|XGBoost|LightGBM|NaiveBayes",
            "NoisePercent": noise,
            "ReferenceNoisePercent": "",
            "Test": "Friedman",
            "Alternative": "not_applicable",
            "IndependentUnit": "Project",
            "N": 24,
            "FamilyID": family_id,
            "FamilySize": 9,
            "Correction": "Holm",
            "Alpha": ALPHA,
            "FamilyAdjustedAlpha": "",
            "PostHocConditional": True,
        })


planned_tests = pd.DataFrame(
    planned_tests
)

expected_planned_tests = (
    4 * 8 * 2
    + 9 * 2
)  # 82

if len(
    planned_tests
) != expected_planned_tests:
    raise RuntimeError(
        "Internal planned-test registry row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 8. FREEZE RQ2 THRESHOLD DEFINITIONS
# --------------------------------------------------------------------------------------------------

rq2_rows = []

for algorithm in ML_TECHNIQUES:
    for comparator in BASELINES:
        rq2_rows.append({
            "Algorithm": algorithm,
            "Comparator": comparator,
            "ComparatorRole": (
                "PRIMARY_PROPOSAL_COMPARATOR"
                if comparator == "LatestFail"
                else "ADDITIONAL_RQ_COMPARATOR"
            ),
            "PrimaryMetric": "APFDc",
            "FirstObservedCrossover": (
                "lowest tested p where mean_ML_APFDc(p) <= mean_baseline_APFDc(p)"
            ),
            "SustainedCrossover": (
                "lowest tested p where ML<=baseline at p and every higher tested p"
            ),
            "Interpolation": "FORBIDDEN",
            "CleanAlreadyNotBetter": (
                "report NO_POSITIVE_TOLERANCE_THRESHOLD"
            ),
            "NoCrossThrough50": (
                "report >50 / NOT_OBSERVED_IN_TESTED_RANGE"
            ),
        })

for algorithm in ML_TECHNIQUES:
    rq2_rows.append({
        "Algorithm": algorithm,
        "Comparator": "StrongestBaselineAtEachNoise",
        "ComparatorRole": "CONSERVATIVE_SUPPLEMENTARY",
        "PrimaryMetric": "APFDc",
        "FirstObservedCrossover": (
            "lowest tested p where mean_ML_APFDc(p) <= max baseline mean APFDc(p)"
        ),
        "SustainedCrossover": (
            "lowest tested p where ML<=strongest baseline at p and every higher tested p"
        ),
        "Interpolation": "FORBIDDEN",
        "CleanAlreadyNotBetter": (
            "report NO_POSITIVE_TOLERANCE_THRESHOLD"
        ),
        "NoCrossThrough50": (
            "report >50 / NOT_OBSERVED_IN_TESTED_RANGE"
        ),
    })


rq2_threshold_plan = pd.DataFrame(
    rq2_rows
)


# --------------------------------------------------------------------------------------------------
# 9. VALIDATION — STILL NO P-VALUES OR INFERENTIAL TESTS
# --------------------------------------------------------------------------------------------------

registry_sha_after = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Step-1B checkpoint SHA-256",
    EXPECTED_STEP1B_CHECKPOINT_SHA256,
    actual_step1b_checkpoint_sha,
    actual_step1b_checkpoint_sha
    == EXPECTED_STEP1B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-1B status",
    STEP1B_STATUS,
    step1b.get(
        "Status"
    ),
    step1b.get(
        "Status"
    )
    == STEP1B_STATUS,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Frozen master datasets verified",
    4,
    len(
        master_anchor_table
    ),
    len(
        master_anchor_table
    )
    == 4,
)

add_check(
    checks,
    "Seed-level master rows",
    EXPECTED_SEED_MASTER_ROWS,
    int(
        master_anchor_table.loc[
            master_anchor_table[
                "Dataset"
            ].eq(
                "SeedLevelMaster"
            ),
            "Rows",
        ].iloc[
            0
        ]
    ),
    int(
        master_anchor_table.loc[
            master_anchor_table[
                "Dataset"
            ].eq(
                "SeedLevelMaster"
            ),
            "Rows",
        ].iloc[
            0
        ]
    )
    == EXPECTED_SEED_MASTER_ROWS,
)

add_check(
    checks,
    "Project-level master rows",
    EXPECTED_PROJECT_MASTER_ROWS,
    int(
        master_anchor_table.loc[
            master_anchor_table[
                "Dataset"
            ].eq(
                "ProjectLevelMaster"
            ),
            "Rows",
        ].iloc[
            0
        ]
    ),
    int(
        master_anchor_table.loc[
            master_anchor_table[
                "Dataset"
            ].eq(
                "ProjectLevelMaster"
            ),
            "Rows",
        ].iloc[
            0
        ]
    )
    == EXPECTED_PROJECT_MASTER_ROWS,
)

add_check(
    checks,
    "Independent empirical unit",
    "Project (N=24)",
    "Project (N=24)",
    True,
)

add_check(
    checks,
    "RQ1 planned inferential tests",
    64,
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ1"
        ).sum()
    ),
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ1"
        ).sum()
    )
    == 64,
)

add_check(
    checks,
    "RQ3 planned omnibus tests",
    18,
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ3"
        ).sum()
    ),
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ3"
        ).sum()
    )
    == 18,
)

add_check(
    checks,
    "Total pre-registered inferential tests",
    82,
    len(
        planned_tests
    ),
    len(
        planned_tests
    )
    == 82,
)

add_check(
    checks,
    "RQ1 Bonferroni family size",
    8,
    RQ1_BONFERRONI_FAMILY_SIZE,
    RQ1_BONFERRONI_FAMILY_SIZE
    == 8,
)

add_check(
    checks,
    "RQ1 Bonferroni adjusted alpha",
    0.00625,
    RQ1_ADJUSTED_ALPHA,
    abs(
        RQ1_ADJUSTED_ALPHA
        - 0.00625
    )
    < 1e-15,
)

add_check(
    checks,
    "RQ2 formal primary hypothesis tests pre-registered",
    0,
    0,
    True,
)

add_check(
    checks,
    "RQ2 pairwise comparator definitions",
    16,
    len(
        rq2_threshold_plan
    ),
    len(
        rq2_threshold_plan
    )
    == 16,
)

add_check(
    checks,
    "RQ3 APFDc Friedman family size",
    9,
    int(
        planned_tests.loc[
            planned_tests[
                "RQ"
            ].eq(
                "RQ3"
            )
            & planned_tests[
                "Metric"
            ].eq(
                "APFDc"
            ),
        ].shape[
            0
        ]
    ),
    int(
        planned_tests.loc[
            planned_tests[
                "RQ"
            ].eq(
                "RQ3"
            )
            & planned_tests[
                "Metric"
            ].eq(
                "APFDc"
            ),
        ].shape[
            0
        ]
    )
    == 9,
)

add_check(
    checks,
    "Inferential statistical tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "P-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)

validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nStatistical Analysis Contract validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 2 VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. WRITE + FREEZE CONTRACT
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

contract[
    "FrozenAtUTC"
] = completed_at_utc

atomic_json(
    CONTRACT_JSON_PATH,
    contract,
)

atomic_csv(
    CONTRACT_TABLE_PATH,
    contract_table,
)

atomic_csv(
    PLANNED_TESTS_PATH,
    planned_tests,
)

atomic_csv(
    RQ2_THRESHOLD_PLAN_PATH,
    rq2_threshold_plan,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

report = {
    "Step":
        "GLOBAL_ANALYSIS_STEP_2",

    "Status":
        STEP2_STATUS,

    "CodeRevision":
        STEP2_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "Seed",

    "SeedsPerProject":
        30,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RQ1PrimaryTest":
        "Paired two-sided Wilcoxon signed-rank",

    "RQ1BonferroniFamilySize":
        8,

    "RQ1AdjustedAlpha":
        RQ1_ADJUSTED_ALPHA,

    "RQ2PrimaryComparator":
        "LatestFail",

    "RQ2FormalPrimaryHypothesisTest":
        None,

    "RQ3Omnibus":
        "Friedman",

    "RQ3OmnibusCorrection":
        "Holm across nine noise levels",

    "RQ3PostHoc":
        "Nemenyi conditional on Holm-significant Friedman",

    "RQ3StressLevels":
        STRESS_LEVELS,

    "PlannedInferentialTestRows":
        len(
            planned_tests
        ),

    "RQ1PlannedTestRows":
        int(
            planned_tests[
                "RQ"
            ].eq(
                "RQ1"
            ).sum()
        ),

    "RQ3PlannedOmnibusRows":
        int(
            planned_tests[
                "RQ"
            ].eq(
                "RQ3"
            ).sum()
        ),

    "InferentialTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "RQ1 STEP 3A — DESCRIPTIVE DEGRADATION ANALYSIS AND FIGURE DATA"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            STEP2_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "ContractFrozenBeforeInference":
            True,

        "ReadyForRQ1":
            True,
    },
)

output_paths = [
    CONTRACT_JSON_PATH,
    CONTRACT_TABLE_PATH,
    PLANNED_TESTS_PATH,
    RQ2_THRESHOLD_PLAN_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

registry_sha_final = sha256_file(
    REGISTRY
)

if registry_sha_final != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during Step 2."
    )

checkpoint = {
    "CheckpointType":
        "GLOBAL_ANALYSIS_STATISTICAL_ANALYSIS_CONTRACT",

    "Status":
        STEP2_STATUS,

    "CodeRevision":
        STEP2_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "ContractJSONPath":
        str(
            CONTRACT_JSON_PATH
        ),

    "ContractJSONSHA256":
        sha256_file(
            CONTRACT_JSON_PATH
        ),

    "ContractTablePath":
        str(
            CONTRACT_TABLE_PATH
        ),

    "ContractTableSHA256":
        sha256_file(
            CONTRACT_TABLE_PATH
        ),

    "PlannedTestsPath":
        str(
            PLANNED_TESTS_PATH
        ),

    "PlannedTestsSHA256":
        sha256_file(
            PLANNED_TESTS_PATH
        ),

    "RQ2ThresholdPlanPath":
        str(
            RQ2_THRESHOLD_PLAN_PATH
        ),

    "RQ2ThresholdPlanSHA256":
        sha256_file(
            RQ2_THRESHOLD_PLAN_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "InferentialTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "ContractFrozenBeforeInference":
        True,

    "ReadyForRQ1":
        True,

    "NextRequiredStep":
        "RQ1 STEP 3A — DESCRIPTIVE DEGRADATION ANALYSIS AND FIGURE DATA",
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 146
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 4 / STEP 2 RESULT ==="
)

print(
    "=" * 146
)

print(
    "Statistical contract frozen before inference: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Repeated stochastic unit: 30 seeds within each project"
)

print(
    "\nMetrics:"
)

print(
    "Primary: APFDc"
)

print(
    "Secondary: APFD"
)

print(
    "\nRQ1 contract:"
)

print(
    "Test: paired two-sided Wilcoxon signed-rank"
)

print(
    "Comparisons per algorithm per metric: 8"
)

print(
    "Correction: Bonferroni"
)

print(
    "Adjusted alpha:",
    RQ1_ADJUSTED_ALPHA,
)

print(
    "Primary inference unit: 24 paired projects"
)

print(
    "\nRQ2 contract:"
)

print(
    "Primary comparator: LatestFail"
)

print(
    "Additional comparators: Random, QTF-Avg, strongest baseline"
)

print(
    "Threshold: first OBSERVED tested crossover"
)

print(
    "Interpolation: forbidden"
)

print(
    "Formal primary hypothesis test introduced: False"
)

print(
    "\nRQ3 contract:"
)

print(
    "Omnibus: Friedman at each of 9 noise levels"
)

print(
    "Omnibus correction: Holm across the 9 APFDc tests"
)

print(
    "Post-hoc: Nemenyi only after Holm-significant Friedman"
)

print(
    "Stress levels for robustness summary:",
    STRESS_LEVELS,
)

print(
    "\nPre-registered inferential-test rows:"
)

print(
    "RQ1:",
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ1"
        ).sum()
    ),
)

print(
    "RQ3:",
    int(
        planned_tests[
            "RQ"
        ].eq(
            "RQ3"
        ).sum()
    ),
)

print(
    "Total:",
    len(
        planned_tests
    ),
)

print(
    "\nSoftware contract:"
)

print(
    "Python:",
    platform.python_version()
)

print(
    "NumPy:",
    np.__version__
)

print(
    "Pandas:",
    pd.__version__
)

print(
    "SciPy:",
    scipy.__version__
)

print(
    "\nIsolation:"
)

print(
    "Inferential statistical tests executed: False"
)

print(
    "P-values computed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nStep 2 checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "RQ1 STEP 3A — DESCRIPTIVE DEGRADATION ANALYSIS AND FIGURE DATA"
)

print(
    "\nSTATUS:",
    STEP2_STATUS,
)

print(
    "=" * 146
)


=== THESIS GLOBAL ANALYSIS — CELL 4 / STEP 2: STATISTICAL ANALYSIS CONTRACT FREEZE ===

Statistical Analysis Contract validation:


,Check,Expected,Actual,Pass
0,Step-1B checkpoint SHA-256,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,True
1,Step-1B status,PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PR...,PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PR...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,Frozen master datasets verified,4,4,True
4,Seed-level master rows,45360,45360,True
5,Project-level master rows,1512,1512,True
6,Independent empirical unit,Project (N=24),Project (N=24),True
7,RQ1 planned inferential tests,64,64,True
8,RQ3 planned omnibus tests,18,18,True
9,Total pre-registered inferential tests,82,82,True



=== THESIS GLOBAL ANALYSIS — CELL 4 / STEP 2 RESULT ===
Statistical contract frozen before inference: True

Study unit:
Independent empirical unit: Project (N=24)
Repeated stochastic unit: 30 seeds within each project

Metrics:
Primary: APFDc
Secondary: APFD

RQ1 contract:
Test: paired two-sided Wilcoxon signed-rank
Comparisons per algorithm per metric: 8
Correction: Bonferroni
Adjusted alpha: 0.00625
Primary inference unit: 24 paired projects

RQ2 contract:
Primary comparator: LatestFail
Additional comparators: Random, QTF-Avg, strongest baseline
Threshold: first OBSERVED tested crossover
Interpolation: forbidden
Formal primary hypothesis test introduced: False

RQ3 contract:
Omnibus: Friedman at each of 9 noise levels
Omnibus correction: Holm across the 9 APFDc tests
Post-hoc: Nemenyi only after Holm-significant Friedman
Stress levels for robustness summary: [30, 40, 50]

Pre-registered inferential-test rows:
RQ1: 64
RQ3: 18
Total: 82

Software contract:
Python: 3.12.13
NumPy: 2.0.2

In [1]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 5 / RQ1 STEP 3A
# DESCRIPTIVE DEGRADATION ANALYSIS + FIGURE DATA
# ==================================================================================================
#
# RQ1
# ---
# How do increasing levels of simulated flaky test noise (0%–50%) in historical training data
# affect the prioritization effectiveness of supervised ML-based TCP techniques, as measured
# by APFD and APFDc?
#
# THIS CELL IS DESCRIPTIVE ONLY.
#
# It DOES:
#   - verify the frozen Statistical Analysis Contract;
#   - verify the frozen canonical master datasets;
#   - restrict analysis to the four supervised ML techniques;
#   - produce the 30-seed global degradation curves required by the proposal;
#   - produce 95% Student-t CIs across the 30 global-seed means;
#   - summarize cross-project heterogeneity over the 24 independent projects;
#   - compute clean-relative APFDc/APFD deltas, retention and degradation;
#   - count projects that degrade / tie / improve at each tested noise level;
#   - freeze figure-ready data for RQ1.
#
# It DOES NOT:
#   - execute Wilcoxon signed-rank tests;
#   - compute p-values;
#   - compute rank-biserial effect sizes;
#   - decide statistical significance;
#   - answer RQ2 or RQ3;
#   - modify any project result or completion-registry row.
#
# STATISTICAL UNIT
# ----------------
# Independent empirical unit: PROJECT (N=24).
#
# Seeds are repeated stochastic runs within projects. For the proposal's 30-run curve:
#   1. For each Noise×Technique×Seed, average equally across the 24 projects.
#   2. Across the resulting 30 global-seed means, calculate mean, sample SD (ddof=1),
#      and a two-sided 95% Student-t confidence interval (df=29).
#
# Because the design is perfectly balanced, the mean across the 30 global-seed means must equal
# the equally-weighted mean of the 24 project-level 30-seed means (within floating tolerance).
# This cell verifies that identity before freezing RQ1 descriptive outputs.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import t as student_t


print("=" * 146)
print("=== THESIS GLOBAL ANALYSIS — CELL 5 / RQ1 STEP 3A: DESCRIPTIVE DEGRADATION + FIGURE DATA ===")
print("=" * 146)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM CONTRACT
# --------------------------------------------------------------------------------------------------

STEP2_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
)

EXPECTED_STEP2_CHECKPOINT_SHA256 = (
    "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"
)

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

EXPECTED_STEP1B_CHECKPOINT_SHA256 = (
    "eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3e0642d1b19f456ad50"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ1_STEP3A_STATUS = (
    "PASS_RQ1_STEP3A_DESCRIPTIVE_DEGRADATION_AND_FIGURE_DATA_FROZEN"
)

RQ1_STEP3A_CODE_REVISION = (
    "RQ1_STEP3A_V1_PROJECT_N24_GLOBAL_SEED_CURVES_CLEAN_RELATIVE_DESCRIPTIVES"
)

EXPECTED_PROJECTS = 24
EXPECTED_SEEDS = 30

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

EXPECTED_ML_SEED_ROWS = (
    EXPECTED_PROJECTS
    * len(NOISE_LEVELS)
    * EXPECTED_SEEDS
    * len(ML_TECHNIQUES)
)  # 25,920

EXPECTED_ML_PROJECT_ROWS = (
    EXPECTED_PROJECTS
    * len(NOISE_LEVELS)
    * len(ML_TECHNIQUES)
)  # 864

EXPECTED_GLOBAL_SEED_REPLICATE_ROWS = (
    len(NOISE_LEVELS)
    * len(ML_TECHNIQUES)
    * EXPECTED_SEEDS
)  # 1,080

EXPECTED_CURVE_ROWS = (
    len(NOISE_LEVELS)
    * len(ML_TECHNIQUES)
)  # 36

FLOAT_TOL = 1e-12

CI_LEVEL = 0.95
CI_DF = EXPECTED_SEEDS - 1
T_CRITICAL_95_DF29 = float(
    student_t.ppf(
        0.975,
        df=CI_DF,
    )
)


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

STEP2_CHECKPOINT = (
    NOTES
    / "global_analysis_step2_checkpoint.json"
)

STEP1B_CHECKPOINT = (
    NOTES
    / "global_analysis_step1b_checkpoint.json"
)

RQ1_ROOT = (
    ANALYSIS_ROOT
    / "RQ1"
)

STEP3A_ROOT = (
    RQ1_ROOT
    / "Step_3A_Descriptive_Degradation_and_Figure_Data"
)

GLOBAL_SEED_REPLICATES_PATH = (
    STEP3A_ROOT
    / "rq1_global_seed_curve_replicates.csv"
)

GLOBAL_SEED_CURVE_SUMMARY_PATH = (
    STEP3A_ROOT
    / "rq1_global_seed_curve_summary.csv"
)

CROSS_PROJECT_SUMMARY_PATH = (
    STEP3A_ROOT
    / "rq1_cross_project_descriptive_summary.csv"
)

PROJECT_CLEAN_RELATIVE_PATH = (
    STEP3A_ROOT
    / "rq1_project_clean_relative_observations.csv"
)

CLEAN_RELATIVE_SUMMARY_PATH = (
    STEP3A_ROOT
    / "rq1_clean_relative_summary.csv"
)

FIGURE_DATA_PATH = (
    STEP3A_ROOT
    / "rq1_figure_data.csv"
)

MONOTONICITY_DIAGNOSTIC_PATH = (
    STEP3A_ROOT
    / "rq1_monotonicity_diagnostic.csv"
)

READBACK_AUDIT_PATH = (
    STEP3A_ROOT
    / "rq1_step3a_readback_audit.csv"
)

VALIDATION_PATH = (
    STEP3A_ROOT
    / "rq1_step3a_validation.csv"
)

REPORT_PATH = (
    STEP3A_ROOT
    / "rq1_step3a_report.json"
)

STATUS_PATH = (
    STEP3A_ROOT
    / "rq1_step3a_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP3A_ROOT
    / "rq1_step3a_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq1_step3a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary,
        path,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def build_output_manifest(
    paths,
):
    rows = []

    for path in sorted(
        (
            Path(
                path
            )
            for path in paths
        ),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"RQ1 Step-3A output missing: {path}"
            )

        rows.append({
            "Path":
                str(
                    path
                ),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Path",
            "Bytes",
            "SHA256",
        ],
    )


def scientific_hash(
    dataframe,
    columns=None,
):
    if columns is None:
        columns = list(
            dataframe.columns
        )

    digest = hashlib.sha256()

    for row in dataframe[
        columns
    ].itertuples(
        index=False,
        name=None,
    ):
        parts = []

        for value in row:
            if isinstance(
                value,
                (
                    float,
                    np.floating,
                ),
            ):
                if math.isnan(
                    float(
                        value
                    )
                ):
                    parts.append(
                        "NaN"
                    )

                else:
                    parts.append(
                        f"{float(value):.17g}"
                    )

            else:
                parts.append(
                    str(
                        value
                    )
                )

        digest.update(
            (
                "\0".join(
                    parts
                )
                + "\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def max_abs_float_difference(
    left,
    right,
    columns,
):
    if len(
        left
    ) != len(
        right
    ):
        return np.inf

    max_difference = 0.0

    for column in columns:
        left_values = pd.to_numeric(
            left[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        right_values = pd.to_numeric(
            right[
                column
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        same_nan = (
            np.isnan(
                left_values
            )
            & np.isnan(
                right_values
            )
        )

        both_finite = (
            np.isfinite(
                left_values
            )
            & np.isfinite(
                right_values
            )
        )

        incompatible = ~(
            same_nan
            | both_finite
        )

        if incompatible.any():
            return np.inf

        if both_finite.any():
            difference = float(
                np.max(
                    np.abs(
                        left_values[
                            both_finite
                        ]
                        - right_values[
                            both_finite
                        ]
                    )
                )
            )

            max_difference = max(
                max_difference,
                difference,
            )

    return max_difference


def sample_summary(
    values,
):
    array = np.asarray(
        values,
        dtype=float,
    )

    if len(
        array
    ) == 0:
        raise RuntimeError(
            "Cannot summarize an empty array."
        )

    return {
        "N":
            int(
                len(
                    array
                )
            ),

        "Mean":
            float(
                np.mean(
                    array
                )
            ),

        "Median":
            float(
                np.median(
                    array
                )
            ),

        "SD":
            (
                float(
                    np.std(
                        array,
                        ddof=1,
                    )
                )
                if len(
                    array
                )
                > 1
                else float(
                    "nan"
                )
            ),

        "Q1":
            float(
                np.quantile(
                    array,
                    0.25,
                    method="linear",
                )
            ),

        "Q3":
            float(
                np.quantile(
                    array,
                    0.75,
                    method="linear",
                )
            ),

        "Min":
            float(
                np.min(
                    array
                )
            ),

        "Max":
            float(
                np.max(
                    array
                )
            ),
    }


def count_direction(
    values,
    tolerance=FLOAT_TOL,
):
    array = np.asarray(
        values,
        dtype=float,
    )

    return {
        "Degraded":
            int(
                (
                    array
                    < -tolerance
                ).sum()
            ),

        "Tied":
            int(
                (
                    np.abs(
                        array
                    )
                    <= tolerance
                ).sum()
            ),

        "Improved":
            int(
                (
                    array
                    > tolerance
                ).sum()
            ),
    }


# --------------------------------------------------------------------------------------------------
# 4. FREEZE GUARD + VERIFY STEP 2
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ1 Step 3A is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ1 Step 3B."
    )

if not STEP2_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-2 checkpoint missing: {STEP2_CHECKPOINT}"
    )

actual_step2_sha = sha256_file(
    STEP2_CHECKPOINT
)

if (
    actual_step2_sha
    != EXPECTED_STEP2_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Statistical Analysis Contract checkpoint SHA mismatch.\n"
        f"Expected: {EXPECTED_STEP2_CHECKPOINT_SHA256}\n"
        f"Actual:   {actual_step2_sha}"
    )

step2 = load_json(
    STEP2_CHECKPOINT
)

if step2.get(
    "Status"
) != STEP2_STATUS:
    raise RuntimeError(
        "Step-2 checkpoint is not in the frozen PASS state."
    )

if not bool(
    step2.get(
        "ContractFrozenBeforeInference",
        False,
    )
):
    raise RuntimeError(
        "Statistical contract was not frozen before inference."
    )

if not bool(
    step2.get(
        "ReadyForRQ1",
        False,
    )
):
    raise RuntimeError(
        "Step-2 checkpoint is not marked ready for RQ1."
    )

contract_json_path = Path(
    step2[
        "ContractJSONPath"
    ]
)

expected_contract_sha = str(
    step2[
        "ContractJSONSHA256"
    ]
).lower()

if not contract_json_path.is_file():
    raise FileNotFoundError(
        f"Frozen statistical contract JSON missing: {contract_json_path}"
    )

actual_contract_sha = sha256_file(
    contract_json_path
)

if actual_contract_sha != expected_contract_sha:
    raise RuntimeError(
        "Frozen statistical contract JSON SHA mismatch."
    )

contract = load_json(
    contract_json_path
)

if not bool(
    contract.get(
        "FrozenBeforeInferentialTesting",
        False,
    )
):
    raise RuntimeError(
        "Contract JSON does not assert pre-inference freeze."
    )

if contract[
    "StudySample"
][
    "IndependentEmpiricalUnit"
] != "Project":
    raise RuntimeError(
        "Contract independent empirical unit is not Project."
    )

if int(
    contract[
        "StudySample"
    ][
        "IndependentEmpiricalUnitN"
    ]
) != 24:
    raise RuntimeError(
        "Contract N is not 24 projects."
    )

if contract[
    "Metrics"
][
    "Primary"
] != "APFDc":
    raise RuntimeError(
        "Contract primary metric is not APFDc."
    )

if contract[
    "Metrics"
][
    "Secondary"
] != "APFD":
    raise RuntimeError(
        "Contract secondary metric is not APFD."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 1B MASTER DATASETS
# --------------------------------------------------------------------------------------------------

if not STEP1B_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-1B checkpoint missing: {STEP1B_CHECKPOINT}"
    )

actual_step1b_sha = sha256_file(
    STEP1B_CHECKPOINT
)

if (
    actual_step1b_sha
    != EXPECTED_STEP1B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-1B checkpoint SHA mismatch."
    )

step1b = load_json(
    STEP1B_CHECKPOINT
)

if step1b.get(
    "Status"
) != STEP1B_STATUS:
    raise RuntimeError(
        "Step-1B checkpoint is not in the frozen PASS state."
    )

seed_master_path = Path(
    step1b[
        "SeedLevelMasterPath"
    ]
)

project_master_path = Path(
    step1b[
        "ProjectLevelMasterPath"
    ]
)

project_delta_path = Path(
    step1b[
        "ProjectLevelCleanDeltaPath"
    ]
)

seed_delta_path = Path(
    step1b[
        "SeedLevelCleanDeltaPath"
    ]
)

master_specs = [
    (
        seed_master_path,
        str(
            step1b[
                "SeedLevelMasterSHA256"
            ]
        ).lower(),
    ),
    (
        project_master_path,
        str(
            step1b[
                "ProjectLevelMasterSHA256"
            ]
        ).lower(),
    ),
    (
        project_delta_path,
        str(
            step1b[
                "ProjectLevelCleanDeltaSHA256"
            ]
        ).lower(),
    ),
    (
        seed_delta_path,
        str(
            step1b[
                "SeedLevelCleanDeltaSHA256"
            ]
        ).lower(),
    ),
]

for path, expected_sha in master_specs:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen master dataset missing: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen master dataset SHA mismatch: {path.name}"
        )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

if STEP3A_ROOT.exists():
    shutil.rmtree(
        STEP3A_ROOT,
        ignore_errors=True,
    )

STEP3A_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 6. LOAD MASTER DATASETS + ISOLATE RQ1 ML DATA
# --------------------------------------------------------------------------------------------------

seed_master = pd.read_csv(
    seed_master_path,
    low_memory=False,
)

project_master = pd.read_csv(
    project_master_path,
    low_memory=False,
)

project_delta_master = pd.read_csv(
    project_delta_path,
    low_memory=False,
)

if len(
    seed_master
) != 45_360:
    raise RuntimeError(
        "Seed-level master row count changed."
    )

if len(
    project_master
) != 1_512:
    raise RuntimeError(
        "Project-level master row count changed."
    )

if len(
    project_delta_master
) != 1_512:
    raise RuntimeError(
        "Project-level delta master row count changed."
    )

seed_ml = seed_master.loc[
    seed_master[
        "Technique"
    ].isin(
        ML_TECHNIQUES
    )
].copy()

project_ml = project_master.loc[
    project_master[
        "Technique"
    ].isin(
        ML_TECHNIQUES
    )
].copy()

project_delta_ml = project_delta_master.loc[
    project_delta_master[
        "Technique"
    ].isin(
        ML_TECHNIQUES
    )
].copy()

for dataframe in [
    seed_ml,
    project_ml,
    project_delta_ml,
]:
    dataframe[
        "Technique"
    ] = pd.Categorical(
        dataframe[
            "Technique"
        ],
        categories=ML_TECHNIQUES,
        ordered=True,
    )

if len(
    seed_ml
) != EXPECTED_ML_SEED_ROWS:
    raise RuntimeError(
        f"RQ1 ML seed rows != {EXPECTED_ML_SEED_ROWS}."
    )

if len(
    project_ml
) != EXPECTED_ML_PROJECT_ROWS:
    raise RuntimeError(
        f"RQ1 ML project rows != {EXPECTED_ML_PROJECT_ROWS}."
    )

if len(
    project_delta_ml
) != EXPECTED_ML_PROJECT_ROWS:
    raise RuntimeError(
        f"RQ1 ML project delta rows != {EXPECTED_ML_PROJECT_ROWS}."
    )


# --------------------------------------------------------------------------------------------------
# 7. PROPOSAL-PRESERVING 30-SEED GLOBAL CURVE REPLICATES
# --------------------------------------------------------------------------------------------------
#
# For each Noise×Technique×Seed:
#     equally weighted mean across the 24 projects.
#
# This gives 9×4×30 = 1,080 global replicate rows.
# --------------------------------------------------------------------------------------------------

global_seed_replicates = (
    seed_ml.groupby(
        [
            "NoisePercent",
            "Technique",
            "RepetitionSeed",
        ],
        sort=True,
        observed=True,
    )
    .agg(
        Projects=(
            "ProjectNumber",
            "nunique",
        ),

        GlobalSeedMeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),

        GlobalSeedMeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
    )
    .reset_index()
)

global_seed_replicates[
    "Technique"
] = global_seed_replicates[
    "Technique"
].astype(
    str
)

global_seed_replicates = (
    global_seed_replicates.sort_values(
        [
            "Technique",
            "NoisePercent",
            "RepetitionSeed",
        ],
        key=lambda series:
            (
                series.map(
                    {
                        technique:
                            index
                        for index, technique
                        in enumerate(
                            ML_TECHNIQUES
                        )
                    }
                )
                if series.name
                == "Technique"
                else series
            ),
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

if len(
    global_seed_replicates
) != EXPECTED_GLOBAL_SEED_REPLICATE_ROWS:
    raise RuntimeError(
        "Global seed replicate row count is not 1,080."
    )

if not global_seed_replicates[
    "Projects"
].eq(
    24
).all():
    raise RuntimeError(
        "A global seed replicate does not contain all 24 projects."
    )


# --------------------------------------------------------------------------------------------------
# 8. GLOBAL 30-SEED CURVE SUMMARY + 95% STUDENT-T CI
# --------------------------------------------------------------------------------------------------

global_curve_rows = []

for technique in ML_TECHNIQUES:
    for noise in NOISE_LEVELS:
        block = global_seed_replicates.loc[
            global_seed_replicates[
                "Technique"
            ].eq(
                technique
            )
            & global_seed_replicates[
                "NoisePercent"
            ].eq(
                noise
            )
        ].sort_values(
            "RepetitionSeed",
            kind="mergesort",
        )

        if len(
            block
        ) != 30:
            raise RuntimeError(
                f"{technique} {noise}% does not have 30 global seed replicates."
            )

        apfdc = block[
            "GlobalSeedMeanAPFDc"
        ].to_numpy(
            dtype=float
        )

        apfd = block[
            "GlobalSeedMeanAPFD"
        ].to_numpy(
            dtype=float
        )

        apfdc_mean = float(
            np.mean(
                apfdc
            )
        )

        apfdc_sd = float(
            np.std(
                apfdc,
                ddof=1,
            )
        )

        apfdc_se = (
            apfdc_sd
            / math.sqrt(
                30
            )
        )

        apfdc_margin = (
            T_CRITICAL_95_DF29
            * apfdc_se
        )

        apfd_mean = float(
            np.mean(
                apfd
            )
        )

        apfd_sd = float(
            np.std(
                apfd,
                ddof=1,
            )
        )

        apfd_se = (
            apfd_sd
            / math.sqrt(
                30
            )
        )

        apfd_margin = (
            T_CRITICAL_95_DF29
            * apfd_se
        )

        global_curve_rows.append({
            "Technique":
                technique,

            "NoisePercent":
                noise,

            "ProjectsPerSeed":
                24,

            "Seeds":
                30,

            "MeanGlobalSeedAPFDc":
                apfdc_mean,

            "SDGlobalSeedAPFDc":
                apfdc_sd,

            "SEGlobalSeedAPFDc":
                apfdc_se,

            "CI95LowGlobalSeedAPFDc":
                apfdc_mean
                - apfdc_margin,

            "CI95HighGlobalSeedAPFDc":
                apfdc_mean
                + apfdc_margin,

            "MeanGlobalSeedAPFD":
                apfd_mean,

            "SDGlobalSeedAPFD":
                apfd_sd,

            "SEGlobalSeedAPFD":
                apfd_se,

            "CI95LowGlobalSeedAPFD":
                apfd_mean
                - apfd_margin,

            "CI95HighGlobalSeedAPFD":
                apfd_mean
                + apfd_margin,
        })


global_seed_curve_summary = pd.DataFrame(
    global_curve_rows
)

if len(
    global_seed_curve_summary
) != EXPECTED_CURVE_ROWS:
    raise RuntimeError(
        "Global seed curve summary must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 9. CROSS-PROJECT HETEROGENEITY SUMMARY (24 INDEPENDENT PROJECT MEANS)
# --------------------------------------------------------------------------------------------------

cross_project_rows = []

for technique in ML_TECHNIQUES:
    for noise in NOISE_LEVELS:
        block = project_ml.loc[
            project_ml[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
            & project_ml[
                "NoisePercent"
            ].eq(
                noise
            )
        ].sort_values(
            "ProjectNumber",
            kind="mergesort",
        )

        if len(
            block
        ) != 24:
            raise RuntimeError(
                f"{technique} {noise}% does not have 24 project-level observations."
            )

        apfdc_summary = sample_summary(
            block[
                "MeanSeedMeanAPFDc"
            ].to_numpy(
                dtype=float
            )
        )

        apfd_summary = sample_summary(
            block[
                "MeanSeedMeanAPFD"
            ].to_numpy(
                dtype=float
            )
        )

        cross_project_rows.append({
            "Technique":
                technique,

            "NoisePercent":
                noise,

            "Projects":
                24,

            "CrossProjectMeanAPFDc":
                apfdc_summary[
                    "Mean"
                ],

            "CrossProjectMedianAPFDc":
                apfdc_summary[
                    "Median"
                ],

            "CrossProjectSDAPFDc":
                apfdc_summary[
                    "SD"
                ],

            "CrossProjectQ1APFDc":
                apfdc_summary[
                    "Q1"
                ],

            "CrossProjectQ3APFDc":
                apfdc_summary[
                    "Q3"
                ],

            "CrossProjectIQRAPFDc":
                (
                    apfdc_summary[
                        "Q3"
                    ]
                    - apfdc_summary[
                        "Q1"
                    ]
                ),

            "CrossProjectMinAPFDc":
                apfdc_summary[
                    "Min"
                ],

            "CrossProjectMaxAPFDc":
                apfdc_summary[
                    "Max"
                ],

            "CrossProjectMeanAPFD":
                apfd_summary[
                    "Mean"
                ],

            "CrossProjectMedianAPFD":
                apfd_summary[
                    "Median"
                ],

            "CrossProjectSDAPFD":
                apfd_summary[
                    "SD"
                ],

            "CrossProjectQ1APFD":
                apfd_summary[
                    "Q1"
                ],

            "CrossProjectQ3APFD":
                apfd_summary[
                    "Q3"
                ],

            "CrossProjectIQRAPFD":
                (
                    apfd_summary[
                        "Q3"
                    ]
                    - apfd_summary[
                        "Q1"
                    ]
                ),

            "CrossProjectMinAPFD":
                apfd_summary[
                    "Min"
                ],

            "CrossProjectMaxAPFD":
                apfd_summary[
                    "Max"
                ],
        })


cross_project_summary = pd.DataFrame(
    cross_project_rows
)

if len(
    cross_project_summary
) != EXPECTED_CURVE_ROWS:
    raise RuntimeError(
        "Cross-project descriptive summary must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 10. BALANCED-DESIGN IDENTITY:
#     mean over 30 seed-global means == mean over 24 project means
# --------------------------------------------------------------------------------------------------

curve_identity = global_seed_curve_summary.merge(
    cross_project_summary,
    on=[
        "Technique",
        "NoisePercent",
    ],
    how="inner",
    validate="one_to_one",
)

curve_identity[
    "APFDcMeanDifference"
] = (
    curve_identity[
        "MeanGlobalSeedAPFDc"
    ]
    - curve_identity[
        "CrossProjectMeanAPFDc"
    ]
)

curve_identity[
    "APFDMeanDifference"
] = (
    curve_identity[
        "MeanGlobalSeedAPFD"
    ]
    - curve_identity[
        "CrossProjectMeanAPFD"
    ]
)

max_balanced_apfdc_difference = float(
    curve_identity[
        "APFDcMeanDifference"
    ].abs().max()
)

max_balanced_apfd_difference = float(
    curve_identity[
        "APFDMeanDifference"
    ].abs().max()
)


# --------------------------------------------------------------------------------------------------
# 11. BUILD PROJECT-LEVEL CLEAN-RELATIVE OBSERVATIONS
# --------------------------------------------------------------------------------------------------

clean_project = (
    project_ml.loc[
        project_ml[
            "NoisePercent"
        ].eq(
            0
        ),
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "Technique",
            "MeanSeedMeanAPFDc",
            "MeanSeedMeanAPFD",
        ],
    ]
    .copy()
    .rename(
        columns={
            "MeanSeedMeanAPFDc":
                "CleanProjectMeanAPFDc",

            "MeanSeedMeanAPFD":
                "CleanProjectMeanAPFD",
        }
    )
)

if len(
    clean_project
) != (
    24
    * 4
):
    raise RuntimeError(
        "Clean project reference must contain 96 Project×ML rows."
    )

project_clean_relative = project_ml.merge(
    clean_project[
        [
            "ProjectNumber",
            "Technique",
            "CleanProjectMeanAPFDc",
            "CleanProjectMeanAPFD",
        ]
    ],
    on=[
        "ProjectNumber",
        "Technique",
    ],
    how="left",
    validate="many_to_one",
)

if project_clean_relative[
    [
        "CleanProjectMeanAPFDc",
        "CleanProjectMeanAPFD",
    ]
].isna().any().any():
    raise RuntimeError(
        "Missing clean project reference."
    )

project_clean_relative[
    "DeltaProjectMeanAPFDc"
] = (
    project_clean_relative[
        "MeanSeedMeanAPFDc"
    ]
    - project_clean_relative[
        "CleanProjectMeanAPFDc"
    ]
)

project_clean_relative[
    "DeltaProjectMeanAPFD"
] = (
    project_clean_relative[
        "MeanSeedMeanAPFD"
    ]
    - project_clean_relative[
        "CleanProjectMeanAPFD"
    ]
)

for column in [
    "DeltaProjectMeanAPFDc",
    "DeltaProjectMeanAPFD",
]:
    values = project_clean_relative[
        column
    ].to_numpy(
        dtype=float
    )

    values[
        np.abs(
            values
        )
        <= FLOAT_TOL
    ] = 0.0

    project_clean_relative[
        column
    ] = values

project_clean_relative[
    "APFDcCleanPositive"
] = (
    project_clean_relative[
        "CleanProjectMeanAPFDc"
    ]
    > 0
)

project_clean_relative[
    "APFDCleanPositive"
] = (
    project_clean_relative[
        "CleanProjectMeanAPFD"
    ]
    > 0
)

project_clean_relative[
    "RetentionPctAPFDc"
] = np.where(
    project_clean_relative[
        "APFDcCleanPositive"
    ],
    100.0
    * project_clean_relative[
        "MeanSeedMeanAPFDc"
    ]
    / project_clean_relative[
        "CleanProjectMeanAPFDc"
    ],
    np.nan,
)

project_clean_relative[
    "DegradationPctAPFDc"
] = np.where(
    project_clean_relative[
        "APFDcCleanPositive"
    ],
    100.0
    * (
        project_clean_relative[
            "CleanProjectMeanAPFDc"
        ]
        - project_clean_relative[
            "MeanSeedMeanAPFDc"
        ]
    )
    / project_clean_relative[
        "CleanProjectMeanAPFDc"
    ],
    np.nan,
)

project_clean_relative[
    "RetentionPctAPFD"
] = np.where(
    project_clean_relative[
        "APFDCleanPositive"
    ],
    100.0
    * project_clean_relative[
        "MeanSeedMeanAPFD"
    ]
    / project_clean_relative[
        "CleanProjectMeanAPFD"
    ],
    np.nan,
)

project_clean_relative[
    "DegradationPctAPFD"
] = np.where(
    project_clean_relative[
        "APFDCleanPositive"
    ],
    100.0
    * (
        project_clean_relative[
            "CleanProjectMeanAPFD"
        ]
        - project_clean_relative[
            "MeanSeedMeanAPFD"
        ]
    )
    / project_clean_relative[
        "CleanProjectMeanAPFD"
    ],
    np.nan,
)

project_clean_relative[
    "Technique"
] = project_clean_relative[
    "Technique"
].astype(
    str
)

project_clean_relative = (
    project_clean_relative[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "Technique",
            "NoisePercent",
            "Seeds",
            "MeanSeedMeanAPFDc",
            "CleanProjectMeanAPFDc",
            "DeltaProjectMeanAPFDc",
            "RetentionPctAPFDc",
            "DegradationPctAPFDc",
            "MeanSeedMeanAPFD",
            "CleanProjectMeanAPFD",
            "DeltaProjectMeanAPFD",
            "RetentionPctAPFD",
            "DegradationPctAPFD",
        ]
    ]
    .sort_values(
        [
            "Technique",
            "NoisePercent",
            "ProjectNumber",
        ],
        key=lambda series:
            (
                series.map(
                    {
                        technique:
                            index
                        for index, technique
                        in enumerate(
                            ML_TECHNIQUES
                        )
                    }
                )
                if series.name
                == "Technique"
                else series
            ),
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

if len(
    project_clean_relative
) != EXPECTED_ML_PROJECT_ROWS:
    raise RuntimeError(
        "Project clean-relative table must contain 864 rows."
    )


# --------------------------------------------------------------------------------------------------
# 12. CROSS-PROJECT CLEAN-RELATIVE SUMMARY
# --------------------------------------------------------------------------------------------------

clean_relative_rows = []

for technique in ML_TECHNIQUES:
    for noise in NOISE_LEVELS:
        block = project_clean_relative.loc[
            project_clean_relative[
                "Technique"
            ].eq(
                technique
            )
            & project_clean_relative[
                "NoisePercent"
            ].eq(
                noise
            )
        ].sort_values(
            "ProjectNumber",
            kind="mergesort",
        )

        if len(
            block
        ) != 24:
            raise RuntimeError(
                f"{technique} {noise}% clean-relative block does not have 24 projects."
            )

        delta_apfdc = block[
            "DeltaProjectMeanAPFDc"
        ].to_numpy(
            dtype=float
        )

        delta_apfd = block[
            "DeltaProjectMeanAPFD"
        ].to_numpy(
            dtype=float
        )

        delta_apfdc_summary = sample_summary(
            delta_apfdc
        )

        delta_apfd_summary = sample_summary(
            delta_apfd
        )

        apfdc_directions = count_direction(
            delta_apfdc
        )

        apfd_directions = count_direction(
            delta_apfd
        )

        retention_apfdc = block[
            "RetentionPctAPFDc"
        ].dropna().to_numpy(
            dtype=float
        )

        degradation_apfdc = block[
            "DegradationPctAPFDc"
        ].dropna().to_numpy(
            dtype=float
        )

        retention_apfd = block[
            "RetentionPctAPFD"
        ].dropna().to_numpy(
            dtype=float
        )

        degradation_apfd = block[
            "DegradationPctAPFD"
        ].dropna().to_numpy(
            dtype=float
        )

        clean_relative_rows.append({
            "Technique":
                technique,

            "NoisePercent":
                noise,

            "Projects":
                24,

            "MeanDeltaAPFDc":
                delta_apfdc_summary[
                    "Mean"
                ],

            "MedianDeltaAPFDc":
                delta_apfdc_summary[
                    "Median"
                ],

            "SDDeltaAPFDc":
                delta_apfdc_summary[
                    "SD"
                ],

            "MinDeltaAPFDc":
                delta_apfdc_summary[
                    "Min"
                ],

            "MaxDeltaAPFDc":
                delta_apfdc_summary[
                    "Max"
                ],

            "APFDcDegradedProjects":
                apfdc_directions[
                    "Degraded"
                ],

            "APFDcTiedProjects":
                apfdc_directions[
                    "Tied"
                ],

            "APFDcImprovedProjects":
                apfdc_directions[
                    "Improved"
                ],

            "APFDcRetentionDefinedProjects":
                len(
                    retention_apfdc
                ),

            "MeanRetentionPctAPFDc":
                (
                    float(
                        np.mean(
                            retention_apfdc
                        )
                    )
                    if len(
                        retention_apfdc
                    )
                    else np.nan
                ),

            "MedianRetentionPctAPFDc":
                (
                    float(
                        np.median(
                            retention_apfdc
                        )
                    )
                    if len(
                        retention_apfdc
                    )
                    else np.nan
                ),

            "MeanDegradationPctAPFDc":
                (
                    float(
                        np.mean(
                            degradation_apfdc
                        )
                    )
                    if len(
                        degradation_apfdc
                    )
                    else np.nan
                ),

            "MedianDegradationPctAPFDc":
                (
                    float(
                        np.median(
                            degradation_apfdc
                        )
                    )
                    if len(
                        degradation_apfdc
                    )
                    else np.nan
                ),

            "MeanDeltaAPFD":
                delta_apfd_summary[
                    "Mean"
                ],

            "MedianDeltaAPFD":
                delta_apfd_summary[
                    "Median"
                ],

            "SDDeltaAPFD":
                delta_apfd_summary[
                    "SD"
                ],

            "MinDeltaAPFD":
                delta_apfd_summary[
                    "Min"
                ],

            "MaxDeltaAPFD":
                delta_apfd_summary[
                    "Max"
                ],

            "APFDDegradedProjects":
                apfd_directions[
                    "Degraded"
                ],

            "APFDTiedProjects":
                apfd_directions[
                    "Tied"
                ],

            "APFDImprovedProjects":
                apfd_directions[
                    "Improved"
                ],

            "APFDRetentionDefinedProjects":
                len(
                    retention_apfd
                ),

            "MeanRetentionPctAPFD":
                (
                    float(
                        np.mean(
                            retention_apfd
                        )
                    )
                    if len(
                        retention_apfd
                    )
                    else np.nan
                ),

            "MedianRetentionPctAPFD":
                (
                    float(
                        np.median(
                            retention_apfd
                        )
                    )
                    if len(
                        retention_apfd
                    )
                    else np.nan
                ),

            "MeanDegradationPctAPFD":
                (
                    float(
                        np.mean(
                            degradation_apfd
                        )
                    )
                    if len(
                        degradation_apfd
                    )
                    else np.nan
                ),

            "MedianDegradationPctAPFD":
                (
                    float(
                        np.median(
                            degradation_apfd
                        )
                    )
                    if len(
                        degradation_apfd
                    )
                    else np.nan
                ),
        })


clean_relative_summary = pd.DataFrame(
    clean_relative_rows
)

if len(
    clean_relative_summary
) != EXPECTED_CURVE_ROWS:
    raise RuntimeError(
        "Clean-relative summary must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 13. VERIFY CLEAN-RELATIVE SUMMARY AGAINST FROZEN STEP-1B DELTA MASTER
# --------------------------------------------------------------------------------------------------

delta_crosscheck = (
    project_delta_ml[
        [
            "ProjectNumber",
            "NoisePercent",
            "Technique",
            "MeanDeltaMeanAPFDc",
            "MeanDeltaMeanAPFD",
        ]
    ]
    .copy()
)

delta_crosscheck[
    "Technique"
] = delta_crosscheck[
    "Technique"
].astype(
    str
)

derived_delta_crosscheck = project_clean_relative[
    [
        "ProjectNumber",
        "NoisePercent",
        "Technique",
        "DeltaProjectMeanAPFDc",
        "DeltaProjectMeanAPFD",
    ]
].copy()

delta_crosscheck = delta_crosscheck.merge(
    derived_delta_crosscheck,
    on=[
        "ProjectNumber",
        "NoisePercent",
        "Technique",
    ],
    how="inner",
    validate="one_to_one",
)

if len(
    delta_crosscheck
) != EXPECTED_ML_PROJECT_ROWS:
    raise RuntimeError(
        "Delta crosscheck does not contain all 864 ML project observations."
    )

max_delta_apfdc_crosscheck_difference = float(
    np.max(
        np.abs(
            delta_crosscheck[
                "MeanDeltaMeanAPFDc"
            ].to_numpy(
                dtype=float
            )
            - delta_crosscheck[
                "DeltaProjectMeanAPFDc"
            ].to_numpy(
                dtype=float
            )
        )
    )
)

max_delta_apfd_crosscheck_difference = float(
    np.max(
        np.abs(
            delta_crosscheck[
                "MeanDeltaMeanAPFD"
            ].to_numpy(
                dtype=float
            )
            - delta_crosscheck[
                "DeltaProjectMeanAPFD"
            ].to_numpy(
                dtype=float
            )
        )
    )
)


# --------------------------------------------------------------------------------------------------
# 14. COMBINE INTO ONE FIGURE-READY RQ1 TABLE
# --------------------------------------------------------------------------------------------------

figure_data = (
    global_seed_curve_summary.merge(
        cross_project_summary,
        on=[
            "Technique",
            "NoisePercent",
        ],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        clean_relative_summary,
        on=[
            "Technique",
            "NoisePercent",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "",
            "_CleanRelative",
        ),
    )
)

if len(
    figure_data
) != EXPECTED_CURVE_ROWS:
    raise RuntimeError(
        "RQ1 figure data must contain 36 rows."
    )

figure_data = (
    figure_data.sort_values(
        [
            "Technique",
            "NoisePercent",
        ],
        key=lambda series:
            (
                series.map(
                    {
                        technique:
                            index
                        for index, technique
                        in enumerate(
                            ML_TECHNIQUES
                        )
                    }
                )
                if series.name
                == "Technique"
                else series
            ),
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 15. MONOTONICITY DIAGNOSTIC — DESCRIPTIVE ONLY
# --------------------------------------------------------------------------------------------------
#
# Noise need not produce a perfectly monotonic empirical curve at every 5% step.
# We record upward/downward/equal adjacent steps; we do NOT impose monotonicity as a requirement.
# --------------------------------------------------------------------------------------------------

monotonicity_rows = []

for technique in ML_TECHNIQUES:
    block = figure_data.loc[
        figure_data[
            "Technique"
        ].eq(
            technique
        )
    ].sort_values(
        "NoisePercent",
        kind="mergesort",
    )

    apfdc_values = block[
        "CrossProjectMeanAPFDc"
    ].to_numpy(
        dtype=float
    )

    apfd_values = block[
        "CrossProjectMeanAPFD"
    ].to_numpy(
        dtype=float
    )

    apfdc_steps = np.diff(
        apfdc_values
    )

    apfd_steps = np.diff(
        apfd_values
    )

    monotonicity_rows.append({
        "Technique":
            technique,

        "APFDcDownwardSteps":
            int(
                (
                    apfdc_steps
                    < -FLOAT_TOL
                ).sum()
            ),

        "APFDcFlatSteps":
            int(
                (
                    np.abs(
                        apfdc_steps
                    )
                    <= FLOAT_TOL
                ).sum()
            ),

        "APFDcUpwardSteps":
            int(
                (
                    apfdc_steps
                    > FLOAT_TOL
                ).sum()
            ),

        "APFDDownwardSteps":
            int(
                (
                    apfd_steps
                    < -FLOAT_TOL
                ).sum()
            ),

        "APFDFlatSteps":
            int(
                (
                    np.abs(
                        apfd_steps
                    )
                    <= FLOAT_TOL
                ).sum()
            ),

        "APFDUpwardSteps":
            int(
                (
                    apfd_steps
                    > FLOAT_TOL
                ).sum()
            ),
    })


monotonicity_diagnostic = pd.DataFrame(
    monotonicity_rows
)


# --------------------------------------------------------------------------------------------------
# 16. VALIDATION BEFORE WRITE
# --------------------------------------------------------------------------------------------------

registry_sha_after_computation = sha256_file(
    REGISTRY
)

zero_noise_rows = clean_relative_summary.loc[
    clean_relative_summary[
        "NoisePercent"
    ].eq(
        0
    )
]

zero_noise_delta_failures = int(
    (
        zero_noise_rows[
            [
                "MeanDeltaAPFDc",
                "MedianDeltaAPFDc",
                "MeanDeltaAPFD",
                "MedianDeltaAPFD",
            ]
        ].abs()
        > FLOAT_TOL
    ).sum().sum()
)

zero_noise_retention_failures = int(
    (
        (
            zero_noise_rows[
                "MeanRetentionPctAPFDc"
            ].sub(
                100.0
            ).abs()
            > FLOAT_TOL
        )
        |
        (
            zero_noise_rows[
                "MeanRetentionPctAPFD"
            ].sub(
                100.0
            ).abs()
            > FLOAT_TOL
        )
    ).sum()
)

direction_count_failures = 0

for _, row in clean_relative_summary.iterrows():
    if (
        int(
            row[
                "APFDcDegradedProjects"
            ]
        )
        + int(
            row[
                "APFDcTiedProjects"
            ]
        )
        + int(
            row[
                "APFDcImprovedProjects"
            ]
        )
        != 24
    ):
        direction_count_failures += 1

    if (
        int(
            row[
                "APFDDegradedProjects"
            ]
        )
        + int(
            row[
                "APFDTiedProjects"
            ]
        )
        + int(
            row[
                "APFDImprovedProjects"
            ]
        )
        != 24
    ):
        direction_count_failures += 1


checks = []

add_check(
    checks,
    "Step-2 checkpoint SHA-256",
    EXPECTED_STEP2_CHECKPOINT_SHA256,
    actual_step2_sha,
    actual_step2_sha
    == EXPECTED_STEP2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-2 status",
    STEP2_STATUS,
    step2.get(
        "Status"
    ),
    step2.get(
        "Status"
    )
    == STEP2_STATUS,
)

add_check(
    checks,
    "Statistical contract JSON SHA-256",
    expected_contract_sha,
    actual_contract_sha,
    actual_contract_sha
    == expected_contract_sha,
)

add_check(
    checks,
    "Step-1B checkpoint SHA-256",
    EXPECTED_STEP1B_CHECKPOINT_SHA256,
    actual_step1b_sha,
    actual_step1b_sha
    == EXPECTED_STEP1B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after_computation,
    registry_sha_after_computation
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "RQ1 ML seed-level rows",
    EXPECTED_ML_SEED_ROWS,
    len(
        seed_ml
    ),
    len(
        seed_ml
    )
    == EXPECTED_ML_SEED_ROWS,
)

add_check(
    checks,
    "RQ1 ML project-level rows",
    EXPECTED_ML_PROJECT_ROWS,
    len(
        project_ml
    ),
    len(
        project_ml
    )
    == EXPECTED_ML_PROJECT_ROWS,
)

add_check(
    checks,
    "Global seed replicate rows",
    EXPECTED_GLOBAL_SEED_REPLICATE_ROWS,
    len(
        global_seed_replicates
    ),
    len(
        global_seed_replicates
    )
    == EXPECTED_GLOBAL_SEED_REPLICATE_ROWS,
)

add_check(
    checks,
    "Projects per global seed replicate",
    [24],
    sorted(
        global_seed_replicates[
            "Projects"
        ].unique().tolist()
    ),
    sorted(
        global_seed_replicates[
            "Projects"
        ].unique().tolist()
    )
    == [24],
)

add_check(
    checks,
    "Global curve rows",
    EXPECTED_CURVE_ROWS,
    len(
        global_seed_curve_summary
    ),
    len(
        global_seed_curve_summary
    )
    == EXPECTED_CURVE_ROWS,
)

add_check(
    checks,
    "Cross-project summary rows",
    EXPECTED_CURVE_ROWS,
    len(
        cross_project_summary
    ),
    len(
        cross_project_summary
    )
    == EXPECTED_CURVE_ROWS,
)

add_check(
    checks,
    "Project clean-relative rows",
    EXPECTED_ML_PROJECT_ROWS,
    len(
        project_clean_relative
    ),
    len(
        project_clean_relative
    )
    == EXPECTED_ML_PROJECT_ROWS,
)

add_check(
    checks,
    "Clean-relative summary rows",
    EXPECTED_CURVE_ROWS,
    len(
        clean_relative_summary
    ),
    len(
        clean_relative_summary
    )
    == EXPECTED_CURVE_ROWS,
)

add_check(
    checks,
    "Figure-data rows",
    EXPECTED_CURVE_ROWS,
    len(
        figure_data
    ),
    len(
        figure_data
    )
    == EXPECTED_CURVE_ROWS,
)

add_check(
    checks,
    "Balanced design APFDc mean identity max difference <=1e-12",
    "<=1e-12",
    max_balanced_apfdc_difference,
    max_balanced_apfdc_difference
    <= FLOAT_TOL,
)

add_check(
    checks,
    "Balanced design APFD mean identity max difference <=1e-12",
    "<=1e-12",
    max_balanced_apfd_difference,
    max_balanced_apfd_difference
    <= FLOAT_TOL,
)

add_check(
    checks,
    "Step-1B delta crosscheck APFDc max difference <=1e-12",
    "<=1e-12",
    max_delta_apfdc_crosscheck_difference,
    max_delta_apfdc_crosscheck_difference
    <= FLOAT_TOL,
)

add_check(
    checks,
    "Step-1B delta crosscheck APFD max difference <=1e-12",
    "<=1e-12",
    max_delta_apfd_crosscheck_difference,
    max_delta_apfd_crosscheck_difference
    <= FLOAT_TOL,
)

add_check(
    checks,
    "Zero-noise delta failures",
    0,
    zero_noise_delta_failures,
    zero_noise_delta_failures
    == 0,
)

add_check(
    checks,
    "Zero-noise mean-retention failures",
    0,
    zero_noise_retention_failures,
    zero_noise_retention_failures
    == 0,
)

add_check(
    checks,
    "Project direction-count failures",
    0,
    direction_count_failures,
    direction_count_failures
    == 0,
)

add_check(
    checks,
    "Student-t CI degrees of freedom",
    29,
    CI_DF,
    CI_DF
    == 29,
)

add_check(
    checks,
    "Student-t 95% critical value",
    "t(0.975,29)",
    T_CRITICAL_95_DF29,
    np.isfinite(
        T_CRITICAL_95_DF29
    )
    and T_CRITICAL_95_DF29
    > 0,
)

add_check(
    checks,
    "Independent empirical unit",
    "Project (N=24)",
    "Project (N=24)",
    True,
)

add_check(
    checks,
    "Wilcoxon tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "P-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Statistical significance decisions made",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after_computation
    != registry_sha_before,
    registry_sha_after_computation
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ1 Step 3A pre-write validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ1 STEP 3A PRE-WRITE VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 17. WRITE DESCRIPTIVE OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_csv(
    GLOBAL_SEED_REPLICATES_PATH,
    global_seed_replicates,
)

atomic_csv(
    GLOBAL_SEED_CURVE_SUMMARY_PATH,
    global_seed_curve_summary,
)

atomic_csv(
    CROSS_PROJECT_SUMMARY_PATH,
    cross_project_summary,
)

atomic_csv(
    PROJECT_CLEAN_RELATIVE_PATH,
    project_clean_relative,
)

atomic_csv(
    CLEAN_RELATIVE_SUMMARY_PATH,
    clean_relative_summary,
)

atomic_csv(
    FIGURE_DATA_PATH,
    figure_data,
)

atomic_csv(
    MONOTONICITY_DIAGNOSTIC_PATH,
    monotonicity_diagnostic,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 18. INDEPENDENT READBACK
# --------------------------------------------------------------------------------------------------

readback_specs = [
    (
        "GlobalSeedReplicates",
        GLOBAL_SEED_REPLICATES_PATH,
        global_seed_replicates,
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        [
            "GlobalSeedMeanAPFDc",
            "GlobalSeedMeanAPFD",
        ],
    ),
    (
        "GlobalSeedCurveSummary",
        GLOBAL_SEED_CURVE_SUMMARY_PATH,
        global_seed_curve_summary,
        [
            "NoisePercent",
        ],
        [
            "MeanGlobalSeedAPFDc",
            "SDGlobalSeedAPFDc",
            "SEGlobalSeedAPFDc",
            "CI95LowGlobalSeedAPFDc",
            "CI95HighGlobalSeedAPFDc",
            "MeanGlobalSeedAPFD",
            "SDGlobalSeedAPFD",
            "SEGlobalSeedAPFD",
            "CI95LowGlobalSeedAPFD",
            "CI95HighGlobalSeedAPFD",
        ],
    ),
    (
        "CrossProjectSummary",
        CROSS_PROJECT_SUMMARY_PATH,
        cross_project_summary,
        [
            "NoisePercent",
        ],
        [
            column
            for column in cross_project_summary.columns
            if column
            not in {
                "Technique",
                "NoisePercent",
                "Projects",
            }
        ],
    ),
    (
        "ProjectCleanRelative",
        PROJECT_CLEAN_RELATIVE_PATH,
        project_clean_relative,
        [
            "ProjectNumber",
            "NoisePercent",
        ],
        [
            "MeanSeedMeanAPFDc",
            "CleanProjectMeanAPFDc",
            "DeltaProjectMeanAPFDc",
            "RetentionPctAPFDc",
            "DegradationPctAPFDc",
            "MeanSeedMeanAPFD",
            "CleanProjectMeanAPFD",
            "DeltaProjectMeanAPFD",
            "RetentionPctAPFD",
            "DegradationPctAPFD",
        ],
    ),
    (
        "CleanRelativeSummary",
        CLEAN_RELATIVE_SUMMARY_PATH,
        clean_relative_summary,
        [
            "NoisePercent",
        ],
        [
            column
            for column in clean_relative_summary.columns
            if column
            not in {
                "Technique",
                "NoisePercent",
                "Projects",
                "APFDcDegradedProjects",
                "APFDcTiedProjects",
                "APFDcImprovedProjects",
                "APFDcRetentionDefinedProjects",
                "APFDDegradedProjects",
                "APFDTiedProjects",
                "APFDImprovedProjects",
                "APFDRetentionDefinedProjects",
            }
        ],
    ),
    (
        "FigureData",
        FIGURE_DATA_PATH,
        figure_data,
        [
            "NoisePercent",
        ],
        [
            column
            for column in figure_data.columns
            if column
            not in {
                "Technique",
                "NoisePercent",
                "ProjectsPerSeed",
                "Seeds",
                "Projects",
                "Projects_CleanRelative",
                "APFDcDegradedProjects",
                "APFDcTiedProjects",
                "APFDcImprovedProjects",
                "APFDcRetentionDefinedProjects",
                "APFDDegradedProjects",
                "APFDTiedProjects",
                "APFDImprovedProjects",
                "APFDRetentionDefinedProjects",
            }
            and pd.api.types.is_numeric_dtype(
                figure_data[
                    column
                ]
            )
        ],
    ),
]

readback_rows = []

for (
    dataset_name,
    path,
    expected_frame,
    key_numeric_columns,
    float_columns,
) in readback_specs:
    actual_frame = pd.read_csv(
        path,
        low_memory=False,
    )

    row_match = (
        len(
            actual_frame
        )
        == len(
            expected_frame
        )
    )

    technique_match = bool(
        row_match
        and actual_frame[
            "Technique"
        ].astype(
            str
        ).tolist()
        == expected_frame[
            "Technique"
        ].astype(
            str
        ).tolist()
    )

    numeric_key_match = True

    for column in key_numeric_columns:
        numeric_key_match = bool(
            numeric_key_match
            and np.array_equal(
                pd.to_numeric(
                    actual_frame[
                        column
                    ],
                    errors="raise",
                ).to_numpy(),
                pd.to_numeric(
                    expected_frame[
                        column
                    ],
                    errors="raise",
                ).to_numpy(),
            )
        )

    max_abs = (
        max_abs_float_difference(
            expected_frame,
            actual_frame,
            float_columns,
        )
        if row_match
        else np.inf
    )

    readback_rows.append({
        "Dataset":
            dataset_name,

        "ExpectedRows":
            len(
                expected_frame
            ),

        "ReadbackRows":
            len(
                actual_frame
            ),

        "RowCountMatch":
            row_match,

        "TechniqueOrderMatch":
            technique_match,

        "NumericKeyOrderMatch":
            numeric_key_match,

        "MaxAbsFloatDifference":
            max_abs,

        "ReadbackPass":
            bool(
                row_match
                and technique_match
                and numeric_key_match
                and np.isfinite(
                    max_abs
                )
                and max_abs
                <= FLOAT_TOL
            ),
    })


readback_audit = pd.DataFrame(
    readback_rows
)

readback_failures = int(
    (
        ~readback_audit[
            "ReadbackPass"
        ].astype(
            bool
        )
    ).sum()
)

atomic_csv(
    READBACK_AUDIT_PATH,
    readback_audit,
)

if readback_failures != 0:
    raise RuntimeError(
        "RQ1 STEP 3A READBACK VALIDATION FAILED.\n"
        + readback_audit.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 19. FREEZE REPORT / MANIFEST / CHECKPOINT
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

figure_data_scientific_sha = scientific_hash(
    figure_data
)

project_clean_relative_scientific_sha = scientific_hash(
    project_clean_relative
)

report = {
    "Step":
        "RQ1_STEP_3A",

    "Status":
        RQ1_STEP3A_STATUS,

    "CodeRevision":
        RQ1_STEP3A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "SeedsPerProject":
        30,

    "MLTechniques":
        ML_TECHNIQUES,

    "NoiseLevels":
        NOISE_LEVELS,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "GlobalSeedReplicateRows":
        len(
            global_seed_replicates
        ),

    "GlobalCurveRows":
        len(
            global_seed_curve_summary
        ),

    "CrossProjectSummaryRows":
        len(
            cross_project_summary
        ),

    "ProjectCleanRelativeRows":
        len(
            project_clean_relative
        ),

    "CleanRelativeSummaryRows":
        len(
            clean_relative_summary
        ),

    "FigureDataRows":
        len(
            figure_data
        ),

    "StudentTCILevel":
        CI_LEVEL,

    "StudentTCIDegreesOfFreedom":
        CI_DF,

    "StudentTCriticalValue":
        T_CRITICAL_95_DF29,

    "MaxBalancedDesignAPFDcMeanDifference":
        max_balanced_apfdc_difference,

    "MaxBalancedDesignAPFDMeanDifference":
        max_balanced_apfd_difference,

    "MaxStep1BDeltaCrosscheckAPFDcDifference":
        max_delta_apfdc_crosscheck_difference,

    "MaxStep1BDeltaCrosscheckAPFDDifference":
        max_delta_apfd_crosscheck_difference,

    "FigureDataPath":
        str(
            FIGURE_DATA_PATH
        ),

    "FigureDataSHA256":
        sha256_file(
            FIGURE_DATA_PATH
        ),

    "FigureDataScientificSHA256":
        figure_data_scientific_sha,

    "ProjectCleanRelativePath":
        str(
            PROJECT_CLEAN_RELATIVE_PATH
        ),

    "ProjectCleanRelativeSHA256":
        sha256_file(
            PROJECT_CLEAN_RELATIVE_PATH
        ),

    "ProjectCleanRelativeScientificSHA256":
        project_clean_relative_scientific_sha,

    "WilcoxonTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "SignificanceDecisionsMade":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "RQ1 STEP 3B — EXECUTE THE PRE-REGISTERED WILCOXON TESTS, "
            "BONFERRONI CORRECTION AND EFFECT SIZES"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ1_STEP3A_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "DescriptiveRQ1Frozen":
            True,

        "WilcoxonTestsExecuted":
            False,

        "PValuesComputed":
            False,

        "ReadyForRQ1Step3B":
            True,
    },
)

output_paths = [
    GLOBAL_SEED_REPLICATES_PATH,
    GLOBAL_SEED_CURVE_SUMMARY_PATH,
    CROSS_PROJECT_SUMMARY_PATH,
    PROJECT_CLEAN_RELATIVE_PATH,
    CLEAN_RELATIVE_SUMMARY_PATH,
    FIGURE_DATA_PATH,
    MONOTONICITY_DIAGNOSTIC_PATH,
    READBACK_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

registry_sha_final = sha256_file(
    REGISTRY
)

if registry_sha_final != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during RQ1 Step 3A."
    )

checkpoint = {
    "CheckpointType":
        "RQ1_DESCRIPTIVE_DEGRADATION_AND_FIGURE_DATA",

    "Status":
        RQ1_STEP3A_STATUS,

    "CodeRevision":
        RQ1_STEP3A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "RepetitionSeed",

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "GlobalSeedReplicatesPath":
        str(
            GLOBAL_SEED_REPLICATES_PATH
        ),

    "GlobalSeedReplicatesSHA256":
        sha256_file(
            GLOBAL_SEED_REPLICATES_PATH
        ),

    "GlobalSeedCurveSummaryPath":
        str(
            GLOBAL_SEED_CURVE_SUMMARY_PATH
        ),

    "GlobalSeedCurveSummarySHA256":
        sha256_file(
            GLOBAL_SEED_CURVE_SUMMARY_PATH
        ),

    "CrossProjectSummaryPath":
        str(
            CROSS_PROJECT_SUMMARY_PATH
        ),

    "CrossProjectSummarySHA256":
        sha256_file(
            CROSS_PROJECT_SUMMARY_PATH
        ),

    "ProjectCleanRelativePath":
        str(
            PROJECT_CLEAN_RELATIVE_PATH
        ),

    "ProjectCleanRelativeSHA256":
        sha256_file(
            PROJECT_CLEAN_RELATIVE_PATH
        ),

    "ProjectCleanRelativeScientificSHA256":
        project_clean_relative_scientific_sha,

    "CleanRelativeSummaryPath":
        str(
            CLEAN_RELATIVE_SUMMARY_PATH
        ),

    "CleanRelativeSummarySHA256":
        sha256_file(
            CLEAN_RELATIVE_SUMMARY_PATH
        ),

    "FigureDataPath":
        str(
            FIGURE_DATA_PATH
        ),

    "FigureDataSHA256":
        sha256_file(
            FIGURE_DATA_PATH
        ),

    "FigureDataScientificSHA256":
        figure_data_scientific_sha,

    "MonotonicityDiagnosticPath":
        str(
            MONOTONICITY_DIAGNOSTIC_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "WilcoxonTestsExecuted":
        False,

    "PValuesComputed":
        False,

    "SignificanceDecisionsMade":
        False,

    "ReadyForRQ1Step3B":
        True,

    "NextRequiredStep":
        (
            "RQ1 STEP 3B — EXECUTE THE PRE-REGISTERED WILCOXON TESTS, "
            "BONFERRONI CORRECTION AND EFFECT SIZES"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 20. USER-VISIBLE DESCRIPTIVE TABLE
# --------------------------------------------------------------------------------------------------

display_table = figure_data[
    [
        "Technique",
        "NoisePercent",
        "MeanGlobalSeedAPFDc",
        "CI95LowGlobalSeedAPFDc",
        "CI95HighGlobalSeedAPFDc",
        "MeanDeltaAPFDc",
        "MeanDegradationPctAPFDc",
        "MeanRetentionPctAPFDc",
        "APFDcDegradedProjects",
        "APFDcTiedProjects",
        "APFDcImprovedProjects",
        "MeanGlobalSeedAPFD",
        "MeanDeltaAPFD",
    ]
].copy()

print(
    "\nRQ1 descriptive APFDc/APFD summary (NO significance tests yet):"
)

try:
    from IPython.display import display

    display(
        display_table
    )

except Exception:
    print(
        display_table.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 21. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 146
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 5 / RQ1 STEP 3A RESULT ==="
)

print(
    "=" * 146
)

print(
    "RQ1 descriptive analysis frozen: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Repeated stochastic unit: Seed (30 within each project)"
)

print(
    "\nRQ1 ML data:"
)

print(
    "Seed-level ML rows:",
    len(
        seed_ml
    ),
    "/",
    EXPECTED_ML_SEED_ROWS,
)

print(
    "Project-level ML rows:",
    len(
        project_ml
    ),
    "/",
    EXPECTED_ML_PROJECT_ROWS,
)

print(
    "\nProposal-preserving global seed curves:"
)

print(
    "Global seed replicate rows:",
    len(
        global_seed_replicates
    ),
    "/",
    EXPECTED_GLOBAL_SEED_REPLICATE_ROWS,
)

print(
    "Curve summary rows:",
    len(
        global_seed_curve_summary
    ),
    "/",
    EXPECTED_CURVE_ROWS,
)

print(
    "95% t-CI df:",
    CI_DF,
)

print(
    "95% t critical:",
    T_CRITICAL_95_DF29,
)

print(
    "\nCross-project / clean-relative:"
)

print(
    "Cross-project summary rows:",
    len(
        cross_project_summary
    ),
    "/",
    EXPECTED_CURVE_ROWS,
)

print(
    "Project clean-relative observations:",
    len(
        project_clean_relative
    ),
    "/",
    EXPECTED_ML_PROJECT_ROWS,
)

print(
    "Clean-relative summary rows:",
    len(
        clean_relative_summary
    ),
    "/",
    EXPECTED_CURVE_ROWS,
)

print(
    "Figure-data rows:",
    len(
        figure_data
    ),
    "/",
    EXPECTED_CURVE_ROWS,
)

print(
    "\nConsistency:"
)

print(
    "Balanced-design max APFDc mean difference:",
    max_balanced_apfdc_difference,
)

print(
    "Balanced-design max APFD mean difference:",
    max_balanced_apfd_difference,
)

print(
    "Step-1B delta crosscheck max APFDc difference:",
    max_delta_apfdc_crosscheck_difference,
)

print(
    "Step-1B delta crosscheck max APFD difference:",
    max_delta_apfd_crosscheck_difference,
)

print(
    "Readback failures:",
    readback_failures,
)

print(
    "\nFrozen descriptive outputs:"
)

print(
    "Figure data SHA-256:",
    sha256_file(
        FIGURE_DATA_PATH
    ),
)

print(
    "Figure data scientific SHA-256:",
    figure_data_scientific_sha,
)

print(
    "Project clean-relative SHA-256:",
    sha256_file(
        PROJECT_CLEAN_RELATIVE_PATH
    ),
)

print(
    "\nInference isolation:"
)

print(
    "Wilcoxon tests executed: False"
)

print(
    "P-values computed: False"
)

print(
    "Statistical significance decisions made: False"
)

print(
    "Completion registry modified: False"
)

print(
    "Project outputs modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)

print(
    "\nRQ1 Step 3A checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "RQ1 STEP 3B — EXECUTE THE PRE-REGISTERED WILCOXON TESTS, "
    "BONFERRONI CORRECTION AND EFFECT SIZES"
)

print(
    "\nSTATUS:",
    RQ1_STEP3A_STATUS,
)

print(
    "=" * 146
)


=== THESIS GLOBAL ANALYSIS — CELL 5 / RQ1 STEP 3A: DESCRIPTIVE DEGRADATION + FIGURE DATA ===

RQ1 Step 3A pre-write validation:


,Check,Expected,Actual,Pass
0,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
1,Step-2 status,PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSI...,PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSI...,True
2,Statistical contract JSON SHA-256,5e95823f1353a835c57b4c93a98737200522375c04fa79...,5e95823f1353a835c57b4c93a98737200522375c04fa79...,True
3,Step-1B checkpoint SHA-256,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,True
4,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
5,RQ1 ML seed-level rows,25920,25920,True
6,RQ1 ML project-level rows,864,864,True
7,Global seed replicate rows,1080,1080,True
8,Projects per global seed replicate,[24],[24],True
9,Global curve rows,36,36,True



RQ1 descriptive APFDc/APFD summary (NO significance tests yet):


,Technique,NoisePercent,MeanGlobalSeedAPFDc,CI95LowGlobalSeedAPFDc,CI95HighGlobalSeedAPFDc,MeanDeltaAPFDc,MeanDegradationPctAPFDc,MeanRetentionPctAPFDc,APFDcDegradedProjects,APFDcTiedProjects,APFDcImprovedProjects,MeanGlobalSeedAPFD,MeanDeltaAPFD
0,RandomForest,0,0.803239,0.801003,0.805474,0.000000,0.000000,100.000000,0,24,0,0.903964,0.000000
1,RandomForest,5,0.674281,0.669142,0.679421,-0.128957,15.710129,84.289871,22,0,2,0.746524,-0.157440
2,RandomForest,10,0.625262,0.618379,0.632146,-0.177976,21.034748,78.965252,23,0,1,0.670719,-0.233245
3,RandomForest,15,0.591190,0.583368,0.599012,-0.212048,24.848051,75.151949,23,0,1,0.634206,-0.269757
4,RandomForest,20,0.567499,0.557763,0.577234,-0.235740,27.645925,72.354075,23,0,1,0.603619,-0.300344
5,RandomForest,25,0.543888,0.533436,0.554340,-0.259351,30.421123,69.578877,23,0,1,0.578378,-0.325586
6,RandomForest,30,0.521425,0.511879,0.530971,-0.281814,33.051305,66.948695,24,0,0,0.549337,-0.354627
7,RandomForest,40,0.479389,0.467519,0.491258,-0.323850,38.036766,61.963234,24,0,0,0.491977,-0.411987
8,RandomForest,50,0.443023,0.432406,0.453641,-0.360215,42.255380,57.744620,24,0,0,0.432852,-0.471112
9,XGBoost,0,0.784217,0.784217,0.784217,0.000000,0.000000,100.000000,0,24,0,0.917151,0.000000



=== THESIS GLOBAL ANALYSIS — CELL 5 / RQ1 STEP 3A RESULT ===
RQ1 descriptive analysis frozen: True

Study unit:
Independent empirical unit: Project (N=24)
Repeated stochastic unit: Seed (30 within each project)

RQ1 ML data:
Seed-level ML rows: 25920 / 25920
Project-level ML rows: 864 / 864

Proposal-preserving global seed curves:
Global seed replicate rows: 1080 / 1080
Curve summary rows: 36 / 36
95% t-CI df: 29
95% t critical: 2.045229642132703

Cross-project / clean-relative:
Cross-project summary rows: 36 / 36
Project clean-relative observations: 864 / 864
Clean-relative summary rows: 36 / 36
Figure-data rows: 36 / 36

Consistency:
Balanced-design max APFDc mean difference: 2.220446049250313e-16
Balanced-design max APFD mean difference: 4.440892098500626e-16
Step-1B delta crosscheck max APFDc difference: 3.608224830031759e-16
Step-1B delta crosscheck max APFD difference: 3.3306690738754696e-16
Readback failures: 0

Frozen descriptive outputs:
Figure data SHA-256: 663ffe08db075e1bc

In [2]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 6 / RQ1 STEP 3B
# PRE-REGISTERED WILCOXON TESTS + BONFERRONI CORRECTION + PAIRED RANK-BISERIAL EFFECT SIZES
# ==================================================================================================
#
# RQ1
# ---
# How do increasing levels of simulated flaky test noise (0%–50%) in historical training data
# affect the prioritization effectiveness of supervised ML-based TCP techniques, as measured
# by APFD and APFDc?
#
# FROZEN CONTRACT
# ---------------
# Independent empirical unit: Project (N=24)
# Repeated stochastic unit: 30 seeds within each project
#
# For each ML algorithm and each noisy level {5,10,15,20,25,30,40,50}%:
#   - compare the project's 30-seed mean at noisy p against the SAME project's 30-seed mean at 0%;
#   - paired, two-sided Wilcoxon signed-rank test;
#   - zero_method='wilcox';
#   - method='auto';
#   - 8-test Bonferroni family WITHIN algorithm and metric;
#   - alpha = 0.05;
#   - Bonferroni adjusted alpha = 0.00625;
#   - APFDc is PRIMARY;
#   - APFD repeats the procedure as a separate SECONDARY family.
#
# EFFECT SIZE
# -----------
# Paired rank-biserial correlation:
#
#       r_rb = (W_plus - W_minus) / (W_plus + W_minus)
#
# where ranks are computed on absolute NONZERO paired differences and
# difference = noisy - clean.
#
# Therefore:
#   negative r_rb -> degradation
#   positive r_rb -> improvement
#
# This cell executes ONLY the inferential tests already pre-registered in Step 2.
# It does NOT answer RQ2/RQ3 and does not modify any project output or completion-registry row.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, wilcoxon


print("=" * 148)
print("=== THESIS GLOBAL ANALYSIS — CELL 6 / RQ1 STEP 3B: WILCOXON + BONFERRONI + EFFECT SIZES ===")
print("=" * 148)

STEP2_STATUS = "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
EXPECTED_STEP2_CHECKPOINT_SHA256 = "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"

RQ1_STEP3A_STATUS = "PASS_RQ1_STEP3A_DESCRIPTIVE_DEGRADATION_AND_FIGURE_DATA_FROZEN"
EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256 = "5f5403a21337144590848659395002fbeb31819427e0def1dfa30d56e65fe074"

EXPECTED_REGISTRY_SHA256 = "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"

RQ1_STEP3B_STATUS = "PASS_RQ1_STEP3B_WILCOXON_BONFERRONI_AND_EFFECT_SIZES_FROZEN"
RQ1_STEP3B_CODE_REVISION = "RQ1_STEP3B_V2_CORRECT_TYPED_TOLERANT_CSV_READBACK_AUDIT"

ALPHA = 0.05
FAMILY_SIZE = 8
ADJUSTED_ALPHA = ALPHA / FAMILY_SIZE

NOISY_LEVELS = [5, 10, 15, 20, 25, 30, 40, 50]

ML_TECHNIQUES = ["RandomForest", "XGBoost", "LightGBM", "NaiveBayes"]

EXPECTED_PROJECTS = 24
EXPECTED_RQ1_TESTS = 64
EXPECTED_PRIMARY_TESTS = 32
EXPECTED_SECONDARY_TESTS = 32
EXPECTED_PROJECT_CLEAN_RELATIVE_ROWS = 864
EXPECTED_FIGURE_ROWS = 36

FLOAT_TOL = 1e-12

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"
ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"

STEP2_CHECKPOINT = NOTES / "global_analysis_step2_checkpoint.json"
RQ1_STEP3A_CHECKPOINT = NOTES / "global_analysis_rq1_step3a_checkpoint.json"

STEP3B_ROOT = ANALYSIS_ROOT / "RQ1" / "Step_3B_Wilcoxon_Bonferroni_and_Effect_Sizes"

TEST_RESULTS_PATH = STEP3B_ROOT / "rq1_wilcoxon_bonferroni_effect_sizes.csv"
APFDC_THESIS_TABLE_PATH = STEP3B_ROOT / "rq1_apfdc_thesis_results_table.csv"
APFD_SECONDARY_TABLE_PATH = STEP3B_ROOT / "rq1_apfd_secondary_results_table.csv"
SIGNIFICANCE_SUMMARY_PATH = STEP3B_ROOT / "rq1_significance_summary.csv"
VALIDATION_PATH = STEP3B_ROOT / "rq1_step3b_validation.csv"
READBACK_AUDIT_PATH = STEP3B_ROOT / "rq1_step3b_readback_audit.csv"
REPORT_PATH = STEP3B_ROOT / "rq1_step3b_report.json"
STATUS_PATH = STEP3B_ROOT / "rq1_step3b_status.json"
OUTPUT_MANIFEST_PATH = STEP3B_ROOT / "rq1_step3b_output_manifest.csv"
CHECKPOINT_PATH = NOTES / "global_analysis_rq1_step3b_checkpoint.json"


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_csv(path, dataframe):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    dataframe.to_csv(temporary, index=False, lineterminator="\n", float_format="%.17g")
    os.replace(temporary, path)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        handle.write("\n")
    os.replace(temporary, path)


def add_check(rows, check, expected, actual, passed):
    rows.append({"Check": check, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


def build_output_manifest(paths):
    rows = []
    for path in sorted((Path(path) for path in paths), key=str):
        if not path.is_file():
            raise FileNotFoundError(f"RQ1 Step-3B output missing: {path}")
        rows.append({"Path": str(path), "Bytes": int(path.stat().st_size), "SHA256": sha256_file(path)})
    return pd.DataFrame(rows, columns=["Path", "Bytes", "SHA256"])


def scientific_hash(dataframe):
    digest = hashlib.sha256()
    for row in dataframe.itertuples(index=False, name=None):
        parts = []
        for value in row:
            if isinstance(value, (float, np.floating)):
                parts.append("NaN" if math.isnan(float(value)) else f"{float(value):.17g}")
            else:
                parts.append(str(value))
        digest.update(("\0".join(parts) + "\n").encode("utf-8"))
    return digest.hexdigest()



def _normalize_object_for_readback(value):
    """
    CSV round-trips can convert an intentionally empty string into NaN.
    Treat both as the same missing textual value for persistence validation.
    """
    if pd.isna(value):
        return ""
    return str(value)


def _normalize_bool_for_readback(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        raise RuntimeError("Boolean readback value is missing.")

    text = str(value).strip().lower()

    if text in {"true", "1"}:
        return True

    if text in {"false", "0"}:
        return False

    raise RuntimeError(
        f"Could not normalize boolean readback value: {value!r}"
    )


def compare_dataframe_after_csv_roundtrip(
    expected_frame,
    actual_frame,
    float_tolerance=FLOAT_TOL,
):
    """
    Scientifically validate a CSV round-trip without requiring byte/type identity.

    Why:
      pandas may deserialize the same persisted value with a slightly different dtype
      (e.g. empty string -> NaN, bool/object normalization, or a tiny floating parse
      difference). Exact Python-object hashing is therefore too strict for a CSV readback
      audit and can reject scientifically identical data.

    Contract:
      - same rows
      - same column names/order
      - integers exact
      - booleans exact after normalization
      - strings exact after treating empty string and NaN as the same CSV-missing text
      - floating values equal within 1e-12, including identical NaN placement
    """

    row_count_match = len(expected_frame) == len(actual_frame)
    column_order_match = (
        expected_frame.columns.tolist()
        == actual_frame.columns.tolist()
    )

    if not row_count_match or not column_order_match:
        return {
            "RowCountMatch": row_count_match,
            "ColumnOrderMatch": column_order_match,
            "DiscreteColumnsMatch": False,
            "FloatColumnsMatch": False,
            "MaxAbsFloatDifference": np.inf,
            "NaNPatternMatch": False,
            "ReadbackPass": False,
        }

    discrete_match = True
    float_match = True
    nan_pattern_match = True
    max_abs_float_difference = 0.0

    for column in expected_frame.columns:
        expected_series = expected_frame[column]
        actual_series = actual_frame[column]

        if pd.api.types.is_bool_dtype(expected_series.dtype):
            expected_values = [
                _normalize_bool_for_readback(value)
                for value in expected_series.tolist()
            ]

            actual_values = [
                _normalize_bool_for_readback(value)
                for value in actual_series.tolist()
            ]

            if expected_values != actual_values:
                discrete_match = False

        elif pd.api.types.is_integer_dtype(expected_series.dtype):
            expected_numeric = pd.to_numeric(
                expected_series,
                errors="raise",
            ).astype("int64").to_numpy()

            actual_numeric = pd.to_numeric(
                actual_series,
                errors="raise",
            ).astype("int64").to_numpy()

            if not np.array_equal(
                expected_numeric,
                actual_numeric,
            ):
                discrete_match = False

        elif pd.api.types.is_float_dtype(expected_series.dtype):
            expected_numeric = pd.to_numeric(
                expected_series,
                errors="coerce",
            ).to_numpy(dtype=float)

            actual_numeric = pd.to_numeric(
                actual_series,
                errors="coerce",
            ).to_numpy(dtype=float)

            expected_nan = np.isnan(expected_numeric)
            actual_nan = np.isnan(actual_numeric)

            if not np.array_equal(
                expected_nan,
                actual_nan,
            ):
                nan_pattern_match = False
                float_match = False
                continue

            finite_mask = ~expected_nan

            if finite_mask.any():
                column_max = float(
                    np.max(
                        np.abs(
                            expected_numeric[finite_mask]
                            - actual_numeric[finite_mask]
                        )
                    )
                )

                max_abs_float_difference = max(
                    max_abs_float_difference,
                    column_max,
                )

                if column_max > float_tolerance:
                    float_match = False

        else:
            expected_values = [
                _normalize_object_for_readback(value)
                for value in expected_series.tolist()
            ]

            actual_values = [
                _normalize_object_for_readback(value)
                for value in actual_series.tolist()
            ]

            if expected_values != actual_values:
                discrete_match = False

    readback_pass = bool(
        row_count_match
        and column_order_match
        and discrete_match
        and float_match
        and nan_pattern_match
        and max_abs_float_difference <= float_tolerance
    )

    return {
        "RowCountMatch": row_count_match,
        "ColumnOrderMatch": column_order_match,
        "DiscreteColumnsMatch": discrete_match,
        "FloatColumnsMatch": float_match,
        "MaxAbsFloatDifference": max_abs_float_difference,
        "NaNPatternMatch": nan_pattern_match,
        "ReadbackPass": readback_pass,
    }


def rank_biserial_from_differences(differences, tolerance=FLOAT_TOL):
    differences = np.asarray(differences, dtype=float)
    nonzero = differences[np.abs(differences) > tolerance]
    zero_pairs = int(len(differences) - len(nonzero))

    if len(nonzero) == 0:
        return {
            "NonZeroPairs": 0,
            "ZeroPairs": zero_pairs,
            "WPlus": 0.0,
            "WMinus": 0.0,
            "RankSumTotal": 0.0,
            "RankBiserial": 0.0,
        }

    ranks = rankdata(np.abs(nonzero), method="average")
    w_plus = float(ranks[nonzero > 0].sum())
    w_minus = float(ranks[nonzero < 0].sum())
    total = w_plus + w_minus
    r_rb = 0.0 if total <= 0 else (w_plus - w_minus) / total

    return {
        "NonZeroPairs": int(len(nonzero)),
        "ZeroPairs": zero_pairs,
        "WPlus": w_plus,
        "WMinus": w_minus,
        "RankSumTotal": total,
        "RankBiserial": float(r_rb),
    }


def run_wilcoxon_test(noisy, clean):
    noisy = np.asarray(noisy, dtype=float)
    clean = np.asarray(clean, dtype=float)

    if len(noisy) != EXPECTED_PROJECTS or len(clean) != EXPECTED_PROJECTS:
        raise RuntimeError("Wilcoxon vectors must both contain 24 projects.")

    if not np.isfinite(noisy).all() or not np.isfinite(clean).all():
        raise RuntimeError("Wilcoxon vectors contain non-finite values.")

    differences = noisy - clean
    effect = rank_biserial_from_differences(differences)

    if effect["NonZeroPairs"] == 0:
        statistic = 0.0
        raw_p = 1.0
    else:
        result = wilcoxon(
            noisy,
            clean,
            zero_method="wilcox",
            correction=False,
            alternative="two-sided",
            method="auto",
        )
        statistic = float(result.statistic)
        raw_p = float(result.pvalue)

    return differences, statistic, raw_p, effect


if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ1 Step 3B is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ1 Step 3C."
    )

if not STEP2_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Step-2 checkpoint missing: {STEP2_CHECKPOINT}")

actual_step2_sha = sha256_file(STEP2_CHECKPOINT)
if actual_step2_sha != EXPECTED_STEP2_CHECKPOINT_SHA256:
    raise RuntimeError("Step-2 checkpoint SHA mismatch.")

step2 = load_json(STEP2_CHECKPOINT)
if step2.get("Status") != STEP2_STATUS:
    raise RuntimeError("Step-2 checkpoint status is not PASS.")

planned_tests_path = Path(step2["PlannedTestsPath"])
expected_planned_tests_sha = str(step2["PlannedTestsSHA256"]).lower()

if not planned_tests_path.is_file():
    raise FileNotFoundError(f"Frozen planned-test registry missing: {planned_tests_path}")

actual_planned_tests_sha = sha256_file(planned_tests_path)
if actual_planned_tests_sha != expected_planned_tests_sha:
    raise RuntimeError("Frozen planned-test registry SHA mismatch.")

planned_tests = pd.read_csv(planned_tests_path, low_memory=False)
rq1_plan = planned_tests.loc[planned_tests["RQ"].astype(str).eq("RQ1")].copy()

if len(rq1_plan) != EXPECTED_RQ1_TESTS:
    raise RuntimeError("Frozen RQ1 planned-test registry does not contain 64 tests.")

if not rq1_plan["Test"].astype(str).eq("Wilcoxon signed-rank").all():
    raise RuntimeError("Frozen RQ1 plan contains a non-Wilcoxon test.")

if not rq1_plan["Alternative"].astype(str).eq("two-sided").all():
    raise RuntimeError("Frozen RQ1 plan alternative is not uniformly two-sided.")

if not rq1_plan["IndependentUnit"].astype(str).eq("Project").all():
    raise RuntimeError("Frozen RQ1 plan independent unit is not Project.")

if not pd.to_numeric(rq1_plan["N"], errors="raise").astype(int).eq(24).all():
    raise RuntimeError("Frozen RQ1 planned N is not 24.")

if not pd.to_numeric(rq1_plan["FamilySize"], errors="raise").astype(int).eq(FAMILY_SIZE).all():
    raise RuntimeError("Frozen RQ1 family size is not 8.")

if not rq1_plan["Correction"].astype(str).eq("Bonferroni").all():
    raise RuntimeError("Frozen RQ1 correction is not Bonferroni.")

if not RQ1_STEP3A_CHECKPOINT.is_file():
    raise FileNotFoundError(f"RQ1 Step-3A checkpoint missing: {RQ1_STEP3A_CHECKPOINT}")

actual_step3a_sha = sha256_file(RQ1_STEP3A_CHECKPOINT)
if actual_step3a_sha != EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256:
    raise RuntimeError("RQ1 Step-3A checkpoint SHA mismatch.")

step3a = load_json(RQ1_STEP3A_CHECKPOINT)
if step3a.get("Status") != RQ1_STEP3A_STATUS:
    raise RuntimeError("RQ1 Step-3A checkpoint status is not PASS.")

project_clean_relative_path = Path(step3a["ProjectCleanRelativePath"])
expected_project_clean_relative_sha = str(step3a["ProjectCleanRelativeSHA256"]).lower()
figure_data_path = Path(step3a["FigureDataPath"])
expected_figure_data_sha = str(step3a["FigureDataSHA256"]).lower()

for path, expected_sha in [
    (project_clean_relative_path, expected_project_clean_relative_sha),
    (figure_data_path, expected_figure_data_sha),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Frozen RQ1 Step-3A input missing: {path}")
    if sha256_file(path) != expected_sha:
        raise RuntimeError(f"Frozen RQ1 Step-3A input SHA mismatch: {path.name}")

project_clean_relative = pd.read_csv(project_clean_relative_path, low_memory=False)
figure_data = pd.read_csv(figure_data_path, low_memory=False)

if len(project_clean_relative) != EXPECTED_PROJECT_CLEAN_RELATIVE_ROWS:
    raise RuntimeError("Frozen project clean-relative input row count changed.")

if len(figure_data) != EXPECTED_FIGURE_ROWS:
    raise RuntimeError("Frozen figure-data row count changed.")

if not REGISTRY.is_file():
    raise FileNotFoundError(f"Completion registry missing: {REGISTRY}")

registry_sha_before = sha256_file(REGISTRY)
if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError("Completion registry differs from final 24-project freeze.")

if STEP3B_ROOT.exists():
    shutil.rmtree(STEP3B_ROOT, ignore_errors=True)

STEP3B_ROOT.mkdir(parents=True, exist_ok=True)

result_rows = []

for metric_role, metric, noisy_column, clean_column, delta_column in [
    ("PRIMARY", "APFDc", "MeanSeedMeanAPFDc", "CleanProjectMeanAPFDc", "DeltaProjectMeanAPFDc"),
    ("SECONDARY", "APFD", "MeanSeedMeanAPFD", "CleanProjectMeanAPFD", "DeltaProjectMeanAPFD"),
]:
    for algorithm in ML_TECHNIQUES:
        algorithm_plan = rq1_plan.loc[
            rq1_plan["Metric"].astype(str).eq(metric)
            & rq1_plan["TechniqueOrSet"].astype(str).eq(algorithm)
        ].copy()

        planned_noise = sorted(
            pd.to_numeric(algorithm_plan["NoisePercent"], errors="raise").astype(int).tolist()
        )
        if planned_noise != NOISY_LEVELS:
            raise RuntimeError(f"{metric} {algorithm}: frozen noise plan mismatch.")

        family_id_values = algorithm_plan["FamilyID"].astype(str).unique().tolist()
        if len(family_id_values) != 1:
            raise RuntimeError(f"{metric} {algorithm}: expected one family ID.")
        family_id = family_id_values[0]

        family_rows = []

        for noise in NOISY_LEVELS:
            block = project_clean_relative.loc[
                project_clean_relative["Technique"].astype(str).eq(algorithm)
                & pd.to_numeric(project_clean_relative["NoisePercent"], errors="raise").astype(int).eq(noise)
            ].copy()

            block["ProjectNumber"] = pd.to_numeric(block["ProjectNumber"], errors="raise").astype(int)
            block = block.sort_values("ProjectNumber", kind="mergesort")

            if len(block) != 24 or block["ProjectNumber"].tolist() != list(range(1, 25)):
                raise RuntimeError(f"{metric} {algorithm} {noise}% project coverage/order mismatch.")

            noisy = pd.to_numeric(block[noisy_column], errors="raise").to_numpy(dtype=float)
            clean = pd.to_numeric(block[clean_column], errors="raise").to_numpy(dtype=float)
            stored_delta = pd.to_numeric(block[delta_column], errors="raise").to_numpy(dtype=float)

            differences, statistic, raw_p, effect = run_wilcoxon_test(noisy, clean)

            max_delta_identity_error = float(np.max(np.abs(differences - stored_delta)))
            if max_delta_identity_error > FLOAT_TOL:
                raise RuntimeError(f"{metric} {algorithm} {noise}% stored delta mismatch.")

            row = {
                "RQ": "RQ1",
                "MetricRole": metric_role,
                "Metric": metric,
                "Algorithm": algorithm,
                "NoisePercent": noise,
                "ReferenceNoisePercent": 0,
                "FamilyID": family_id,
                "FamilySize": FAMILY_SIZE,
                "IndependentUnit": "Project",
                "Projects": 24,
                "MeanNoisy": float(np.mean(noisy)),
                "MeanClean": float(np.mean(clean)),
                "MeanDifferenceNoisyMinusClean": float(np.mean(differences)),
                "MedianDifferenceNoisyMinusClean": float(np.median(differences)),
                "SDDifference": float(np.std(differences, ddof=1)),
                "DegradedProjects": int((differences < -FLOAT_TOL).sum()),
                "TiedProjects": int((np.abs(differences) <= FLOAT_TOL).sum()),
                "ImprovedProjects": int((differences > FLOAT_TOL).sum()),
                "NonZeroPairs": effect["NonZeroPairs"],
                "ZeroPairs": effect["ZeroPairs"],
                "WPlus": effect["WPlus"],
                "WMinus": effect["WMinus"],
                "WilcoxonStatistic": statistic,
                "RawPValue": raw_p,
                "BonferroniAdjustedPValue": np.nan,
                "NominalAlpha": ALPHA,
                "BonferroniAdjustedAlpha": ADJUSTED_ALPHA,
                "SignificantBonferroni": False,
                "PairedRankBiserial": effect["RankBiserial"],
                "EffectDirection": (
                    "Degradation"
                    if effect["RankBiserial"] < -FLOAT_TOL
                    else "Improvement"
                    if effect["RankBiserial"] > FLOAT_TOL
                    else "NoDirectionalEffect"
                ),
                "MaxStoredDeltaIdentityError": max_delta_identity_error,
            }
            family_rows.append(row)

        if len(family_rows) != 8:
            raise RuntimeError(f"{family_id}: family row count != 8.")

        for row in family_rows:
            adjusted_p = min(float(row["RawPValue"]) * FAMILY_SIZE, 1.0)
            row["BonferroniAdjustedPValue"] = adjusted_p
            row["SignificantBonferroni"] = bool(adjusted_p <= ALPHA + 1e-15)
            result_rows.append(row)

test_results = pd.DataFrame(result_rows)

metric_order = {"APFDc": 0, "APFD": 1}
algorithm_order = {algorithm: index for index, algorithm in enumerate(ML_TECHNIQUES)}

test_results["__MetricOrder"] = test_results["Metric"].map(metric_order)
test_results["__AlgorithmOrder"] = test_results["Algorithm"].map(algorithm_order)

test_results = (
    test_results.sort_values(
        ["__MetricOrder", "__AlgorithmOrder", "NoisePercent"],
        kind="mergesort",
    )
    .drop(columns=["__MetricOrder", "__AlgorithmOrder"])
    .reset_index(drop=True)
)

if len(test_results) != EXPECTED_RQ1_TESTS:
    raise RuntimeError("RQ1 inferential results do not contain exactly 64 rows.")

family_audit = (
    test_results.groupby(["Metric", "Algorithm", "FamilyID"], sort=True)
    .agg(
        Tests=("NoisePercent", "size"),
        UniqueNoiseLevels=("NoisePercent", "nunique"),
        SignificantTests=("SignificantBonferroni", "sum"),
    )
    .reset_index()
)

if len(family_audit) != 8 or not family_audit["Tests"].eq(8).all():
    raise RuntimeError("Bonferroni family structure is not 8 families × 8 tests.")

figure_lookup = figure_data[
    [
        "Technique",
        "NoisePercent",
        "MeanGlobalSeedAPFDc",
        "MeanDeltaAPFDc",
        "MeanGlobalSeedAPFD",
        "MeanDeltaAPFD",
    ]
].copy()

apfdc_results = test_results.loc[test_results["Metric"].eq("APFDc")].copy()
apfd_results = test_results.loc[test_results["Metric"].eq("APFD")].copy()

apfdc_check = apfdc_results.merge(
    figure_lookup[["Technique", "NoisePercent", "MeanGlobalSeedAPFDc", "MeanDeltaAPFDc"]],
    left_on=["Algorithm", "NoisePercent"],
    right_on=["Technique", "NoisePercent"],
    how="left",
    validate="one_to_one",
)

apfd_check = apfd_results.merge(
    figure_lookup[["Technique", "NoisePercent", "MeanGlobalSeedAPFD", "MeanDeltaAPFD"]],
    left_on=["Algorithm", "NoisePercent"],
    right_on=["Technique", "NoisePercent"],
    how="left",
    validate="one_to_one",
)

max_apfdc_mean_crosscheck = float(
    np.max(np.abs(apfdc_check["MeanNoisy"].to_numpy(dtype=float) - apfdc_check["MeanGlobalSeedAPFDc"].to_numpy(dtype=float)))
)
max_apfdc_delta_crosscheck = float(
    np.max(np.abs(apfdc_check["MeanDifferenceNoisyMinusClean"].to_numpy(dtype=float) - apfdc_check["MeanDeltaAPFDc"].to_numpy(dtype=float)))
)
max_apfd_mean_crosscheck = float(
    np.max(np.abs(apfd_check["MeanNoisy"].to_numpy(dtype=float) - apfd_check["MeanGlobalSeedAPFD"].to_numpy(dtype=float)))
)
max_apfd_delta_crosscheck = float(
    np.max(np.abs(apfd_check["MeanDifferenceNoisyMinusClean"].to_numpy(dtype=float) - apfd_check["MeanDeltaAPFD"].to_numpy(dtype=float)))
)

apfdc_thesis = figure_data[
    [
        "Technique",
        "NoisePercent",
        "MeanGlobalSeedAPFDc",
        "CI95LowGlobalSeedAPFDc",
        "CI95HighGlobalSeedAPFDc",
        "MeanDeltaAPFDc",
        "MedianDeltaAPFDc",
        "MeanDegradationPctAPFDc",
        "MeanRetentionPctAPFDc",
        "APFDcDegradedProjects",
        "APFDcTiedProjects",
        "APFDcImprovedProjects",
    ]
].copy()

apfdc_thesis = apfdc_thesis.loc[
    apfdc_thesis["NoisePercent"].isin(NOISY_LEVELS)
].merge(
    apfdc_results[
        [
            "Algorithm",
            "NoisePercent",
            "WilcoxonStatistic",
            "RawPValue",
            "BonferroniAdjustedPValue",
            "SignificantBonferroni",
            "PairedRankBiserial",
            "NonZeroPairs",
            "ZeroPairs",
        ]
    ],
    left_on=["Technique", "NoisePercent"],
    right_on=["Algorithm", "NoisePercent"],
    how="inner",
    validate="one_to_one",
).drop(columns=["Algorithm"])

apfd_secondary = figure_data[
    [
        "Technique",
        "NoisePercent",
        "MeanGlobalSeedAPFD",
        "MeanDeltaAPFD",
        "APFDDegradedProjects",
        "APFDTiedProjects",
        "APFDImprovedProjects",
    ]
].copy()

apfd_secondary = apfd_secondary.loc[
    apfd_secondary["NoisePercent"].isin(NOISY_LEVELS)
].merge(
    apfd_results[
        [
            "Algorithm",
            "NoisePercent",
            "WilcoxonStatistic",
            "RawPValue",
            "BonferroniAdjustedPValue",
            "SignificantBonferroni",
            "PairedRankBiserial",
            "NonZeroPairs",
            "ZeroPairs",
        ]
    ],
    left_on=["Technique", "NoisePercent"],
    right_on=["Algorithm", "NoisePercent"],
    how="inner",
    validate="one_to_one",
).drop(columns=["Algorithm"])

significance_rows = []

for metric in ["APFDc", "APFD"]:
    for algorithm in ML_TECHNIQUES:
        block = test_results.loc[
            test_results["Metric"].eq(metric)
            & test_results["Algorithm"].eq(algorithm)
        ].sort_values("NoisePercent", kind="mergesort")

        significant_noise = block.loc[
            block["SignificantBonferroni"].astype(bool),
            "NoisePercent",
        ].astype(int).tolist()

        significance_rows.append({
            "Metric": metric,
            "MetricRole": "PRIMARY" if metric == "APFDc" else "SECONDARY",
            "Algorithm": algorithm,
            "Tests": len(block),
            "BonferroniSignificantTests": int(block["SignificantBonferroni"].astype(bool).sum()),
            "SignificantNoiseLevelsJSON": json.dumps(significant_noise, separators=(",", ":")),
            "FirstBonferroniSignificantNoisePercent": min(significant_noise) if significant_noise else "",
            "AllHigherTestedLevelsSignificantFromFirst": (
                bool(
                    block.loc[
                        block["NoisePercent"].ge(min(significant_noise)),
                        "SignificantBonferroni",
                    ].astype(bool).all()
                )
                if significant_noise
                else False
            ),
            "MostNegativeRankBiserial": float(block["PairedRankBiserial"].min()),
            "LeastNegativeOrMostPositiveRankBiserial": float(block["PairedRankBiserial"].max()),
        })

significance_summary = pd.DataFrame(significance_rows)

registry_sha_after = sha256_file(REGISTRY)

checks = []

add_check(checks, "Step-2 checkpoint SHA-256", EXPECTED_STEP2_CHECKPOINT_SHA256, actual_step2_sha, actual_step2_sha == EXPECTED_STEP2_CHECKPOINT_SHA256)
add_check(checks, "Frozen planned-test registry SHA-256", expected_planned_tests_sha, actual_planned_tests_sha, actual_planned_tests_sha == expected_planned_tests_sha)
add_check(checks, "RQ1 Step-3A checkpoint SHA-256", EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256, actual_step3a_sha, actual_step3a_sha == EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256)
add_check(checks, "Completion registry SHA-256", EXPECTED_REGISTRY_SHA256, registry_sha_after, registry_sha_after == EXPECTED_REGISTRY_SHA256)
add_check(checks, "Pre-registered RQ1 tests", 64, len(rq1_plan), len(rq1_plan) == 64)
add_check(checks, "Executed RQ1 tests", 64, len(test_results), len(test_results) == 64)
add_check(checks, "Primary APFDc tests", 32, int(test_results["Metric"].eq("APFDc").sum()), int(test_results["Metric"].eq("APFDc").sum()) == 32)
add_check(checks, "Secondary APFD tests", 32, int(test_results["Metric"].eq("APFD").sum()), int(test_results["Metric"].eq("APFD").sum()) == 32)
add_check(checks, "Bonferroni families", 8, len(family_audit), len(family_audit) == 8)
add_check(checks, "Tests per family", [8], sorted(family_audit["Tests"].unique().tolist()), sorted(family_audit["Tests"].unique().tolist()) == [8])
add_check(checks, "Adjusted alpha", 0.00625, ADJUSTED_ALPHA, abs(ADJUSTED_ALPHA - 0.00625) <= 1e-15)
add_check(checks, "All raw p-values finite", True, bool(np.isfinite(test_results["RawPValue"].to_numpy(dtype=float)).all()), bool(np.isfinite(test_results["RawPValue"].to_numpy(dtype=float)).all()))
add_check(checks, "All adjusted p-values in [0,1]", True, bool(test_results["BonferroniAdjustedPValue"].between(0, 1, inclusive="both").all()), bool(test_results["BonferroniAdjustedPValue"].between(0, 1, inclusive="both").all()))
add_check(checks, "All rank-biserial values in [-1,1]", True, bool(test_results["PairedRankBiserial"].between(-1, 1, inclusive="both").all()), bool(test_results["PairedRankBiserial"].between(-1, 1, inclusive="both").all()))
add_check(checks, "Direction counts sum to 24", True, bool((test_results["DegradedProjects"] + test_results["TiedProjects"] + test_results["ImprovedProjects"]).eq(24).all()), bool((test_results["DegradedProjects"] + test_results["TiedProjects"] + test_results["ImprovedProjects"]).eq(24).all()))
add_check(checks, "Nonzero+zero pairs sum to 24", True, bool((test_results["NonZeroPairs"] + test_results["ZeroPairs"]).eq(24).all()), bool((test_results["NonZeroPairs"] + test_results["ZeroPairs"]).eq(24).all()))
add_check(checks, "Stored delta identity max <=1e-12", "<=1e-12", float(test_results["MaxStoredDeltaIdentityError"].max()), float(test_results["MaxStoredDeltaIdentityError"].max()) <= FLOAT_TOL)
add_check(checks, "Step-3A APFDc mean crosscheck <=1e-12", "<=1e-12", max_apfdc_mean_crosscheck, max_apfdc_mean_crosscheck <= FLOAT_TOL)
add_check(checks, "Step-3A APFDc delta crosscheck <=1e-12", "<=1e-12", max_apfdc_delta_crosscheck, max_apfdc_delta_crosscheck <= FLOAT_TOL)
add_check(checks, "Step-3A APFD mean crosscheck <=1e-12", "<=1e-12", max_apfd_mean_crosscheck, max_apfd_mean_crosscheck <= FLOAT_TOL)
add_check(checks, "Step-3A APFD delta crosscheck <=1e-12", "<=1e-12", max_apfd_delta_crosscheck, max_apfd_delta_crosscheck <= FLOAT_TOL)
add_check(checks, "APFDc thesis table rows", 32, len(apfdc_thesis), len(apfdc_thesis) == 32)
add_check(checks, "APFD secondary table rows", 32, len(apfd_secondary), len(apfd_secondary) == 32)
add_check(checks, "Significance summary rows", 8, len(significance_summary), len(significance_summary) == 8)
add_check(checks, "Independent empirical unit", "Project (N=24)", "Project (N=24)", True)
add_check(checks, "Completion registry modified", False, registry_sha_after != registry_sha_before, registry_sha_after == registry_sha_before)

validation = pd.DataFrame(checks)
failed_validation = validation.loc[~validation["Pass"].astype(bool)].copy()

print("\nRQ1 Step 3B pre-write validation:")
try:
    from IPython.display import display
    display(validation)
except Exception:
    print(validation.to_string(index=False))

if not failed_validation.empty:
    raise RuntimeError(
        "RQ1 STEP 3B PRE-WRITE VALIDATION FAILED.\n"
        + failed_validation.to_string(index=False)
    )

atomic_csv(TEST_RESULTS_PATH, test_results)
atomic_csv(APFDC_THESIS_TABLE_PATH, apfdc_thesis)
atomic_csv(APFD_SECONDARY_TABLE_PATH, apfd_secondary)
atomic_csv(SIGNIFICANCE_SUMMARY_PATH, significance_summary)
atomic_csv(VALIDATION_PATH, validation)

readback_rows = []

for dataset_name, path, expected_frame in [
    ("TestResults", TEST_RESULTS_PATH, test_results),
    ("APFDcThesisTable", APFDC_THESIS_TABLE_PATH, apfdc_thesis),
    ("APFDSecondaryTable", APFD_SECONDARY_TABLE_PATH, apfd_secondary),
    ("SignificanceSummary", SIGNIFICANCE_SUMMARY_PATH, significance_summary),
]:
    actual_frame = pd.read_csv(
        path,
        low_memory=False,
    )

    # Align columns only if the exact same set is present; column order itself is audited below.
    if set(actual_frame.columns) == set(expected_frame.columns):
        actual_frame = actual_frame[
            expected_frame.columns.tolist()
        ]

    comparison = compare_dataframe_after_csv_roundtrip(
        expected_frame,
        actual_frame,
        float_tolerance=FLOAT_TOL,
    )

    readback_rows.append({
        "Dataset":
            dataset_name,

        "ExpectedRows":
            len(expected_frame),

        "ReadbackRows":
            len(actual_frame),

        "RowCountMatch":
            comparison["RowCountMatch"],

        "ColumnOrderMatch":
            comparison["ColumnOrderMatch"],

        "DiscreteColumnsMatch":
            comparison["DiscreteColumnsMatch"],

        "FloatColumnsMatch":
            comparison["FloatColumnsMatch"],

        "NaNPatternMatch":
            comparison["NaNPatternMatch"],

        "MaxAbsFloatDifference":
            comparison["MaxAbsFloatDifference"],

        "FloatTolerance":
            FLOAT_TOL,

        "ReadbackPass":
            comparison["ReadbackPass"],
    })


readback_audit = pd.DataFrame(
    readback_rows
)

readback_failures = int(
    (
        ~readback_audit[
            "ReadbackPass"
        ].astype(
            bool
        )
    ).sum()
)

atomic_csv(
    READBACK_AUDIT_PATH,
    readback_audit,
)

if readback_failures != 0:
    raise RuntimeError(
        "RQ1 STEP 3B V2 READBACK AUDIT FAILED.\n"
        + readback_audit.to_string(
            index=False
        )
    )

completed_at_utc = pd.Timestamp.now(tz="UTC").isoformat()

test_results_scientific_sha = scientific_hash(test_results)
apfdc_thesis_scientific_sha = scientific_hash(apfdc_thesis)

primary_significant_count = int(
    test_results.loc[test_results["Metric"].eq("APFDc"), "SignificantBonferroni"].astype(bool).sum()
)
secondary_significant_count = int(
    test_results.loc[test_results["Metric"].eq("APFD"), "SignificantBonferroni"].astype(bool).sum()
)

report = {
    "Step": "RQ1_STEP_3B",
    "Status": RQ1_STEP3B_STATUS,
    "CodeRevision": RQ1_STEP3B_CODE_REVISION,
    "CompletedAtUTC": completed_at_utc,
    "Step2CheckpointSHA256": EXPECTED_STEP2_CHECKPOINT_SHA256,
    "RQ1Step3ACheckpointSHA256": EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,
    "CompletionRegistrySHA256": EXPECTED_REGISTRY_SHA256,
    "IndependentEmpiricalUnit": "Project",
    "IndependentEmpiricalUnitN": 24,
    "PrimaryMetric": "APFDc",
    "SecondaryMetric": "APFD",
    "Test": "paired two-sided Wilcoxon signed-rank",
    "WilcoxonZeroMethod": "wilcox",
    "WilcoxonMethod": "auto",
    "BonferroniFamilySize": 8,
    "NominalAlpha": ALPHA,
    "AdjustedAlpha": ADJUSTED_ALPHA,
    "EffectSize": "paired rank-biserial correlation",
    "EffectSizeDifferenceConvention": "noisy - clean",
    "ExecutedTests": len(test_results),
    "PrimaryAPFDcTests": 32,
    "SecondaryAPFDTests": 32,
    "PrimaryAPFDcBonferroniSignificantTests": primary_significant_count,
    "SecondaryAPFDBonferroniSignificantTests": secondary_significant_count,
    "TestResultsPath": str(TEST_RESULTS_PATH),
    "TestResultsSHA256": sha256_file(TEST_RESULTS_PATH),
    "TestResultsScientificSHA256": test_results_scientific_sha,
    "APFDcThesisTablePath": str(APFDC_THESIS_TABLE_PATH),
    "APFDcThesisTableSHA256": sha256_file(APFDC_THESIS_TABLE_PATH),
    "APFDcThesisTableScientificSHA256": apfdc_thesis_scientific_sha,
    "APFDSecondaryTablePath": str(APFD_SECONDARY_TABLE_PATH),
    "SignificanceSummaryPath": str(SIGNIFICANCE_SUMMARY_PATH),
    "ReadbackFailures": readback_failures,
    "ReadbackValidationMethod": (
        "typed CSV round-trip equivalence: exact discrete values, "
        "identical NaN pattern, floating tolerance <=1e-12"
    ),
    "ReadbackMaxAbsFloatDifference": float(
        readback_audit[
            "MaxAbsFloatDifference"
        ].max()
    ),
    "CompletionRegistryModified": False,
    "ProjectOutputsModified": False,
    "NextRequiredStep": (
        "RQ1 STEP 3C — GENERATE FINAL RQ1 FIGURES, SYNTHESIS TABLES, "
        "AND FREEZE THE RQ1 ANSWER PACKAGE"
    ),
}

atomic_json(REPORT_PATH, report)

atomic_json(
    STATUS_PATH,
    {
        "Status": RQ1_STEP3B_STATUS,
        "CompletedAtUTC": completed_at_utc,
        "InferentialRQ1Frozen": True,
        "ReadyForRQ1Step3C": True,
    },
)

output_paths = [
    TEST_RESULTS_PATH,
    APFDC_THESIS_TABLE_PATH,
    APFD_SECONDARY_TABLE_PATH,
    SIGNIFICANCE_SUMMARY_PATH,
    READBACK_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(output_paths)
atomic_csv(OUTPUT_MANIFEST_PATH, output_manifest)
output_manifest_sha = sha256_file(OUTPUT_MANIFEST_PATH)

if sha256_file(REGISTRY) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError("Completion registry changed during RQ1 Step 3B.")

checkpoint = {
    "CheckpointType": "RQ1_WILCOXON_BONFERRONI_AND_EFFECT_SIZES",
    "Status": RQ1_STEP3B_STATUS,
    "CodeRevision": RQ1_STEP3B_CODE_REVISION,
    "CompletedAtUTC": completed_at_utc,
    "Step2CheckpointSHA256": EXPECTED_STEP2_CHECKPOINT_SHA256,
    "RQ1Step3ACheckpointSHA256": EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,
    "CompletionRegistrySHA256": EXPECTED_REGISTRY_SHA256,
    "IndependentEmpiricalUnit": "Project",
    "IndependentEmpiricalUnitN": 24,
    "TestResultsPath": str(TEST_RESULTS_PATH),
    "TestResultsSHA256": sha256_file(TEST_RESULTS_PATH),
    "TestResultsScientificSHA256": test_results_scientific_sha,
    "APFDcThesisTablePath": str(APFDC_THESIS_TABLE_PATH),
    "APFDcThesisTableSHA256": sha256_file(APFDC_THESIS_TABLE_PATH),
    "APFDSecondaryTablePath": str(APFD_SECONDARY_TABLE_PATH),
    "APFDSecondaryTableSHA256": sha256_file(APFD_SECONDARY_TABLE_PATH),
    "SignificanceSummaryPath": str(SIGNIFICANCE_SUMMARY_PATH),
    "SignificanceSummarySHA256": sha256_file(SIGNIFICANCE_SUMMARY_PATH),
    "OutputManifestPath": str(OUTPUT_MANIFEST_PATH),
    "OutputManifestSHA256": output_manifest_sha,
    "ExecutedTests": 64,
    "PrimaryAPFDcTests": 32,
    "SecondaryAPFDTests": 32,
    "BonferroniFamilySize": 8,
    "AdjustedAlpha": ADJUSTED_ALPHA,
    "ReadyForRQ1Step3C": True,
    "NextRequiredStep": (
        "RQ1 STEP 3C — GENERATE FINAL RQ1 FIGURES, SYNTHESIS TABLES, "
        "AND FREEZE THE RQ1 ANSWER PACKAGE"
    ),
}

atomic_json(CHECKPOINT_PATH, checkpoint)
checkpoint_sha = sha256_file(CHECKPOINT_PATH)

print("\nRQ1 PRIMARY APFDc inference table:")
primary_display_columns = [
    "Technique",
    "NoisePercent",
    "MeanGlobalSeedAPFDc",
    "MeanDeltaAPFDc",
    "MeanDegradationPctAPFDc",
    "MeanRetentionPctAPFDc",
    "APFDcDegradedProjects",
    "APFDcImprovedProjects",
    "WilcoxonStatistic",
    "RawPValue",
    "BonferroniAdjustedPValue",
    "SignificantBonferroni",
    "PairedRankBiserial",
]

try:
    from IPython.display import display
    display(apfdc_thesis[primary_display_columns])
    print("\nRQ1 significance summary:")
    display(significance_summary)
except Exception:
    print(apfdc_thesis[primary_display_columns].to_string(index=False))
    print("\nRQ1 significance summary:")
    print(significance_summary.to_string(index=False))

print("\n" + "=" * 148)
print("=== THESIS GLOBAL ANALYSIS — CELL 6 / RQ1 STEP 3B V2 RESULT ===")
print("=" * 148)

print("Pre-registered RQ1 tests executed:", len(test_results), "/ 64")
print("Primary APFDc tests:", EXPECTED_PRIMARY_TESTS)
print("Secondary APFD tests:", EXPECTED_SECONDARY_TESTS)

print("\nStatistical contract:")
print("Independent empirical unit: Project (N=24)")
print("Test: paired two-sided Wilcoxon signed-rank")
print("Bonferroni family size:", FAMILY_SIZE)
print("Adjusted alpha:", ADJUSTED_ALPHA)
print("Effect size: paired rank-biserial correlation (noisy - clean)")

print("\nSignificance counts:")
print("Primary APFDc Bonferroni-significant tests:", primary_significant_count, "/ 32")
print("Secondary APFD Bonferroni-significant tests:", secondary_significant_count, "/ 32")

print("\nConsistency:")
print("Step-3A APFDc mean crosscheck max difference:", max_apfdc_mean_crosscheck)
print("Step-3A APFDc delta crosscheck max difference:", max_apfdc_delta_crosscheck)
print("Step-3A APFD mean crosscheck max difference:", max_apfd_mean_crosscheck)
print("Step-3A APFD delta crosscheck max difference:", max_apfd_delta_crosscheck)
print("Readback failures:", readback_failures)
print(
    "Readback max absolute float difference:",
    float(
        readback_audit[
            "MaxAbsFloatDifference"
        ].max()
    ),
)
print("Readback tolerance:", FLOAT_TOL)

print("\nFrozen results:")
print("Test-results SHA-256:", sha256_file(TEST_RESULTS_PATH))
print("Test-results scientific SHA-256:", test_results_scientific_sha)
print("APFDc thesis-table SHA-256:", sha256_file(APFDC_THESIS_TABLE_PATH))

print("\nIsolation:")
print("RQ2 tests executed: False")
print("RQ3 tests executed: False")
print("Project outputs modified: False")
print("Completion registry modified: False")

print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))

print("\nRQ1 Step 3B checkpoint:")
print(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", checkpoint_sha)

print(
    "\nNext required step: "
    "RQ1 STEP 3C — GENERATE FINAL RQ1 FIGURES, SYNTHESIS TABLES, "
    "AND FREEZE THE RQ1 ANSWER PACKAGE"
)

print("\nSTATUS:", RQ1_STEP3B_STATUS)
print("=" * 148)


=== THESIS GLOBAL ANALYSIS — CELL 6 / RQ1 STEP 3B: WILCOXON + BONFERRONI + EFFECT SIZES ===

RQ1 Step 3B pre-write validation:


,Check,Expected,Actual,Pass
0,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
1,Frozen planned-test registry SHA-256,42a7d2450e19331249707a99092cf6fdd1ff3102ef568e...,42a7d2450e19331249707a99092cf6fdd1ff3102ef568e...,True
2,RQ1 Step-3A checkpoint SHA-256,5f5403a21337144590848659395002fbeb31819427e0de...,5f5403a21337144590848659395002fbeb31819427e0de...,True
3,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
4,Pre-registered RQ1 tests,64,64,True
5,Executed RQ1 tests,64,64,True
6,Primary APFDc tests,32,32,True
7,Secondary APFD tests,32,32,True
8,Bonferroni families,8,8,True
9,Tests per family,[8],[8],True



RQ1 PRIMARY APFDc inference table:


,Technique,NoisePercent,MeanGlobalSeedAPFDc,MeanDeltaAPFDc,MeanDegradationPctAPFDc,MeanRetentionPctAPFDc,APFDcDegradedProjects,APFDcImprovedProjects,WilcoxonStatistic,RawPValue,BonferroniAdjustedPValue,SignificantBonferroni,PairedRankBiserial
0,RandomForest,5,0.674281,-0.128957,15.710129,84.289871,22,2,9.0,3.933907e-06,3.147125e-05,True,-0.940000
1,RandomForest,10,0.625262,-0.177976,21.034748,78.965252,23,1,5.0,1.192093e-06,9.536743e-06,True,-0.966667
2,RandomForest,15,0.591190,-0.212048,24.848051,75.151949,23,1,5.0,1.192093e-06,9.536743e-06,True,-0.966667
3,RandomForest,20,0.567499,-0.235740,27.645925,72.354075,23,1,3.0,5.960464e-07,4.768372e-06,True,-0.980000
4,RandomForest,25,0.543888,-0.259351,30.421123,69.578877,23,1,1.0,2.384186e-07,1.907349e-06,True,-0.993333
5,RandomForest,30,0.521425,-0.281814,33.051305,66.948695,24,0,0.0,1.192093e-07,9.536743e-07,True,-1.000000
6,RandomForest,40,0.479389,-0.323850,38.036766,61.963234,24,0,0.0,1.192093e-07,9.536743e-07,True,-1.000000
7,RandomForest,50,0.443023,-0.360215,42.255380,57.744620,24,0,0.0,1.192093e-07,9.536743e-07,True,-1.000000
8,XGBoost,5,0.684539,-0.099678,12.366411,87.633589,22,2,16.0,2.014637e-05,1.611710e-04,True,-0.893333
9,XGBoost,10,0.641205,-0.143012,17.583378,82.416622,22,2,7.0,2.264977e-06,1.811981e-05,True,-0.953333



RQ1 significance summary:


,Metric,MetricRole,Algorithm,Tests,BonferroniSignificantTests,SignificantNoiseLevelsJSON,FirstBonferroniSignificantNoisePercent,AllHigherTestedLevelsSignificantFromFirst,MostNegativeRankBiserial,LeastNegativeOrMostPositiveRankBiserial
0,APFDc,PRIMARY,RandomForest,8,8,"[5,10,15,20,25,30,40,50]",5,True,-1.000000,-0.940000
1,APFDc,PRIMARY,XGBoost,8,8,"[5,10,15,20,25,30,40,50]",5,True,-1.000000,-0.893333
2,APFDc,PRIMARY,LightGBM,8,8,"[5,10,15,20,25,30,40,50]",5,True,-1.000000,-0.693333
3,APFDc,PRIMARY,NaiveBayes,8,1,[50],50,True,-0.686667,-0.440000
4,APFD,SECONDARY,RandomForest,8,8,"[5,10,15,20,25,30,40,50]",5,True,-1.000000,-0.986667
5,APFD,SECONDARY,XGBoost,8,8,"[5,10,15,20,25,30,40,50]",5,True,-1.000000,-0.993333
6,APFD,SECONDARY,LightGBM,8,8,"[5,10,15,20,25,30,40,50]",5,True,-0.993333,-0.840000
7,APFD,SECONDARY,NaiveBayes,8,4,"[25,30,40,50]",25,True,-0.980000,0.286667



=== THESIS GLOBAL ANALYSIS — CELL 6 / RQ1 STEP 3B V2 RESULT ===
Pre-registered RQ1 tests executed: 64 / 64
Primary APFDc tests: 32
Secondary APFD tests: 32

Statistical contract:
Independent empirical unit: Project (N=24)
Test: paired two-sided Wilcoxon signed-rank
Bonferroni family size: 8
Adjusted alpha: 0.00625
Effect size: paired rank-biserial correlation (noisy - clean)

Significance counts:
Primary APFDc Bonferroni-significant tests: 25 / 32
Secondary APFD Bonferroni-significant tests: 28 / 32

Consistency:
Step-3A APFDc mean crosscheck max difference: 2.220446049250313e-16
Step-3A APFDc delta crosscheck max difference: 1.1102230246251565e-16
Step-3A APFD mean crosscheck max difference: 2.220446049250313e-16
Step-3A APFD delta crosscheck max difference: 1.6653345369377348e-16
Readback failures: 0
Readback max absolute float difference: 1.4210854715202004e-14
Readback tolerance: 1e-12

Frozen results:
Test-results SHA-256: 935ccfe65a34b47c2c2ef01fa342ad0bf8812ca4c19e945f21eaa4b9e

In [3]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 7 / RQ1 STEP 3C
# FINAL RQ1 FIGURES + SYNTHESIS TABLES + ANSWER-PACKAGE FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# RQ1 Step 3A froze the descriptive degradation evidence.
# RQ1 Step 3B executed and froze the 64 pre-registered Wilcoxon tests,
# Bonferroni corrections and paired rank-biserial effect sizes.
#
# This cell performs NO new inferential tests.
#
# It:
#   - verifies the frozen Step 3A and Step 3B checkpoints and result hashes;
#   - constructs thesis-ready numerical synthesis tables for APFDc (primary) and APFD (secondary);
#   - generates figure-ready PNGs from the already frozen RQ1 data;
#   - freezes a compact final RQ1 answer package with an integrity manifest/root hash;
#   - preserves Project (N=24) as the independent empirical unit;
#   - does NOT modify Project 1..24 results or the completion registry;
#   - does NOT execute RQ2 or RQ3.
#
# IMPORTANT
# ---------
# This package contains numerical evidence and figures, not generated thesis prose.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


print("=" * 150)
print("=== THESIS GLOBAL ANALYSIS — CELL 7 / RQ1 STEP 3C: FINAL FIGURES + SYNTHESIS + ANSWER PACKAGE ===")
print("=" * 150)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

RQ1_STEP3A_STATUS = (
    "PASS_RQ1_STEP3A_DESCRIPTIVE_DEGRADATION_AND_FIGURE_DATA_FROZEN"
)

EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256 = (
    "5f5403a21337144590848659395002fbeb31819427e0def1dfa30d56e65fe074"
)

RQ1_STEP3B_STATUS = (
    "PASS_RQ1_STEP3B_WILCOXON_BONFERRONI_AND_EFFECT_SIZES_FROZEN"
)

EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256 = (
    "27e0b52469023791a37ebf1099b6a63e7509ae2513981694ba7e7541277c47c3"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ1_STEP3C_STATUS = (
    "PASS_RQ1_STEP3C_FINAL_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

RQ1_STEP3C_CODE_REVISION = (
    "RQ1_STEP3C_V1_FROZEN_NUMERIC_SYNTHESIS_FIGURES_PACKAGE_NO_NEW_INFERENCE"
)

ALPHA = 0.05
BONFERRONI_ADJUSTED_ALPHA = 0.00625
FLOAT_TOL = 1e-12

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

NOISY_LEVELS = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

EXPECTED_FIGURE_DATA_ROWS = 36
EXPECTED_APFDC_TEST_ROWS = 32
EXPECTED_APFD_TEST_ROWS = 32
EXPECTED_SIGNIFICANCE_SUMMARY_ROWS = 8


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

RQ1_STEP3A_CHECKPOINT = (
    NOTES
    / "global_analysis_rq1_step3a_checkpoint.json"
)

RQ1_STEP3B_CHECKPOINT = (
    NOTES
    / "global_analysis_rq1_step3b_checkpoint.json"
)

STEP3C_ROOT = (
    ANALYSIS_ROOT
    / "RQ1"
    / "Step_3C_Final_RQ1_Answer_Package"
)

FIGURES_ROOT = (
    STEP3C_ROOT
    / "Figures"
)

TABLES_ROOT = (
    STEP3C_ROOT
    / "Tables"
)

PRIMARY_SYNTHESIS_PATH = (
    TABLES_ROOT
    / "rq1_primary_apfdc_algorithm_synthesis.csv"
)

PRIMARY_FULL_RESULTS_PATH = (
    TABLES_ROOT
    / "rq1_primary_apfdc_full_results.csv"
)

SECONDARY_FULL_RESULTS_PATH = (
    TABLES_ROOT
    / "rq1_secondary_apfd_full_results.csv"
)

SELECTED_NOISE_TABLE_PATH = (
    TABLES_ROOT
    / "rq1_selected_noise_level_summary.csv"
)

SIGNIFICANCE_SUMMARY_COPY_PATH = (
    TABLES_ROOT
    / "rq1_significance_summary.csv"
)

FIGURE_DATA_COPY_PATH = (
    TABLES_ROOT
    / "rq1_figure_data.csv"
)

FIGURE_APFDC_ABSOLUTE_PATH = (
    FIGURES_ROOT
    / "rq1_apfdc_absolute_performance.png"
)

FIGURE_APFDC_DEGRADATION_PATH = (
    FIGURES_ROOT
    / "rq1_apfdc_clean_relative_degradation.png"
)

FIGURE_APFDC_RETENTION_PATH = (
    FIGURES_ROOT
    / "rq1_apfdc_retention.png"
)

FIGURE_APFDC_EFFECT_SIZE_PATH = (
    FIGURES_ROOT
    / "rq1_apfdc_rank_biserial_effect_size.png"
)

FIGURE_APFD_SECONDARY_PATH = (
    FIGURES_ROOT
    / "rq1_apfd_secondary_performance.png"
)

VALIDATION_PATH = (
    STEP3C_ROOT
    / "rq1_step3c_validation.csv"
)

REPORT_PATH = (
    STEP3C_ROOT
    / "rq1_step3c_report.json"
)

STATUS_PATH = (
    STEP3C_ROOT
    / "rq1_step3c_status.json"
)

FINAL_PACKAGE_MANIFEST_PATH = (
    STEP3C_ROOT
    / "rq1_final_package_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq1_step3c_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary,
        path,
    )


def atomic_copy(
    source,
    destination,
):
    source = Path(
        source
    )

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        f".{destination.name}.tmp_{os.getpid()}"
    )

    shutil.copy2(
        source,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def package_root_hash(
    manifest,
):
    ordered = (
        manifest[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def build_package_manifest(
    root,
    manifest_path,
):
    root = Path(
        root
    )

    manifest_path = Path(
        manifest_path
    )

    rows = []

    for path in sorted(
        (
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
            and candidate.resolve()
            != manifest_path.resolve()
        ),
        key=lambda candidate:
            candidate.relative_to(
                root
            ).as_posix(),
    ):
        rows.append({
            "RelativePath":
                path.relative_to(
                    root
                ).as_posix(),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def technique_sort_key(
    series,
):
    if series.name == "Technique":
        return series.map({
            technique:
                index
            for index, technique
            in enumerate(
                ML_TECHNIQUES
            )
        })

    return series


def save_figure(
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD + VERIFY FROZEN RQ1 STEP 3A / 3B
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ1 Step 3C is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ2."
    )

if not RQ1_STEP3A_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"RQ1 Step-3A checkpoint missing: {RQ1_STEP3A_CHECKPOINT}"
    )

actual_step3a_sha = sha256_file(
    RQ1_STEP3A_CHECKPOINT
)

if (
    actual_step3a_sha
    != EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ1 Step-3A checkpoint SHA mismatch."
    )

step3a = load_json(
    RQ1_STEP3A_CHECKPOINT
)

if step3a.get(
    "Status"
) != RQ1_STEP3A_STATUS:
    raise RuntimeError(
        "RQ1 Step-3A checkpoint status is not PASS."
    )

if not RQ1_STEP3B_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"RQ1 Step-3B checkpoint missing: {RQ1_STEP3B_CHECKPOINT}"
    )

actual_step3b_sha = sha256_file(
    RQ1_STEP3B_CHECKPOINT
)

if (
    actual_step3b_sha
    != EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ1 Step-3B checkpoint SHA mismatch."
    )

step3b = load_json(
    RQ1_STEP3B_CHECKPOINT
)

if step3b.get(
    "Status"
) != RQ1_STEP3B_STATUS:
    raise RuntimeError(
        "RQ1 Step-3B checkpoint status is not PASS."
    )

if not bool(
    step3b.get(
        "ReadyForRQ1Step3C",
        False,
    )
):
    raise RuntimeError(
        "RQ1 Step-3B is not marked ready for Step 3C."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if (
    registry_sha_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

if STEP3C_ROOT.exists():
    shutil.rmtree(
        STEP3C_ROOT,
        ignore_errors=True,
    )

FIGURES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TABLES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY + LOAD FROZEN RQ1 INPUT FILES
# --------------------------------------------------------------------------------------------------

figure_data_path = Path(
    step3a[
        "FigureDataPath"
    ]
)

expected_figure_data_sha = str(
    step3a[
        "FigureDataSHA256"
    ]
).lower()

apfdc_results_path = Path(
    step3b[
        "APFDcThesisTablePath"
    ]
)

expected_apfdc_results_sha = str(
    step3b[
        "APFDcThesisTableSHA256"
    ]
).lower()

apfd_results_path = Path(
    step3b[
        "APFDSecondaryTablePath"
    ]
)

expected_apfd_results_sha = str(
    step3b[
        "APFDSecondaryTableSHA256"
    ]
).lower()

significance_summary_path = Path(
    step3b[
        "SignificanceSummaryPath"
    ]
)

expected_significance_summary_sha = str(
    step3b[
        "SignificanceSummarySHA256"
    ]
).lower()

for path, expected_sha in [
    (
        figure_data_path,
        expected_figure_data_sha,
    ),
    (
        apfdc_results_path,
        expected_apfdc_results_sha,
    ),
    (
        apfd_results_path,
        expected_apfd_results_sha,
    ),
    (
        significance_summary_path,
        expected_significance_summary_sha,
    ),
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ1 input missing: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            f"Frozen RQ1 input SHA mismatch: {path.name}"
        )


figure_data = pd.read_csv(
    figure_data_path,
    low_memory=False,
)

apfdc_results = pd.read_csv(
    apfdc_results_path,
    low_memory=False,
)

apfd_results = pd.read_csv(
    apfd_results_path,
    low_memory=False,
)

significance_summary = pd.read_csv(
    significance_summary_path,
    low_memory=False,
)

if len(
    figure_data
) != EXPECTED_FIGURE_DATA_ROWS:
    raise RuntimeError(
        "RQ1 figure data row count changed."
    )

if len(
    apfdc_results
) != EXPECTED_APFDC_TEST_ROWS:
    raise RuntimeError(
        "RQ1 APFDc inferential result row count changed."
    )

if len(
    apfd_results
) != EXPECTED_APFD_TEST_ROWS:
    raise RuntimeError(
        "RQ1 APFD result row count changed."
    )

if len(
    significance_summary
) != EXPECTED_SIGNIFICANCE_SUMMARY_ROWS:
    raise RuntimeError(
        "RQ1 significance summary row count changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. BUILD PRIMARY APFDC 4-ALGORITHM SYNTHESIS TABLE
# --------------------------------------------------------------------------------------------------

primary_rows = []

for technique in ML_TECHNIQUES:
    curve = (
        figure_data.loc[
            figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
        .copy()
    )

    tests = (
        apfdc_results.loc[
            apfdc_results[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
        .copy()
    )

    if len(
        curve
    ) != 9:
        raise RuntimeError(
            f"{technique}: expected 9 descriptive APFDc rows."
        )

    if len(
        tests
    ) != 8:
        raise RuntimeError(
            f"{technique}: expected 8 APFDc inferential rows."
        )

    significance_row = significance_summary.loc[
        significance_summary[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFDc"
        )
        & significance_summary[
            "Algorithm"
        ].astype(
            str
        ).eq(
            technique
        )
    ]

    if len(
        significance_row
    ) != 1:
        raise RuntimeError(
            f"{technique}: APFDc significance summary row missing."
        )

    significance_row = significance_row.iloc[
        0
    ]

    def curve_value(
        noise,
        column,
    ):
        row = curve.loc[
            pd.to_numeric(
                curve[
                    "NoisePercent"
                ],
                errors="raise",
            ).astype(
                int
            ).eq(
                noise
            )
        ]

        if len(
            row
        ) != 1:
            raise RuntimeError(
                f"{technique}: missing noise {noise}%."
            )

        return float(
            row.iloc[
                0
            ][
                column
            ]
        )

    primary_rows.append({
        "Technique":
            technique,

        "CleanMeanAPFDc":
            curve_value(
                0,
                "MeanGlobalSeedAPFDc",
            ),

        "MeanAPFDc_5":
            curve_value(
                5,
                "MeanGlobalSeedAPFDc",
            ),

        "MeanAPFDc_15":
            curve_value(
                15,
                "MeanGlobalSeedAPFDc",
            ),

        "MeanAPFDc_30":
            curve_value(
                30,
                "MeanGlobalSeedAPFDc",
            ),

        "MeanAPFDc_50":
            curve_value(
                50,
                "MeanGlobalSeedAPFDc",
            ),

        "MeanDegradationPctAPFDc_5":
            curve_value(
                5,
                "MeanDegradationPctAPFDc",
            ),

        "MeanDegradationPctAPFDc_15":
            curve_value(
                15,
                "MeanDegradationPctAPFDc",
            ),

        "MeanDegradationPctAPFDc_30":
            curve_value(
                30,
                "MeanDegradationPctAPFDc",
            ),

        "MeanDegradationPctAPFDc_50":
            curve_value(
                50,
                "MeanDegradationPctAPFDc",
            ),

        "MeanRetentionPctAPFDc_50":
            curve_value(
                50,
                "MeanRetentionPctAPFDc",
            ),

        "BonferroniSignificantNoiseLevels":
            str(
                significance_row[
                    "SignificantNoiseLevelsJSON"
                ]
            ),

        "FirstBonferroniSignificantNoisePercent":
            (
                ""
                if pd.isna(
                    significance_row[
                        "FirstBonferroniSignificantNoisePercent"
                    ]
                )
                else int(
                    float(
                        significance_row[
                            "FirstBonferroniSignificantNoisePercent"
                        ]
                    )
                )
            ),

        "BonferroniSignificantTests":
            int(
                significance_row[
                    "BonferroniSignificantTests"
                ]
            ),

        "MostNegativePairedRankBiserial":
            float(
                significance_row[
                    "MostNegativeRankBiserial"
                ]
            ),

        "LeastNegativeOrMostPositivePairedRankBiserial":
            float(
                significance_row[
                    "LeastNegativeOrMostPositiveRankBiserial"
                ]
            ),
    })


primary_synthesis = pd.DataFrame(
    primary_rows
)


# --------------------------------------------------------------------------------------------------
# 7. SELECTED-NOISE NUMERICAL TABLE
# --------------------------------------------------------------------------------------------------

selected_noise_levels = [
    0,
    5,
    15,
    30,
    50,
]

selected_noise = figure_data.loc[
    pd.to_numeric(
        figure_data[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(
        int
    ).isin(
        selected_noise_levels
    ),
    [
        "Technique",
        "NoisePercent",
        "MeanGlobalSeedAPFDc",
        "CI95LowGlobalSeedAPFDc",
        "CI95HighGlobalSeedAPFDc",
        "MeanDeltaAPFDc",
        "MeanDegradationPctAPFDc",
        "MeanRetentionPctAPFDc",
        "APFDcDegradedProjects",
        "APFDcTiedProjects",
        "APFDcImprovedProjects",
        "MeanGlobalSeedAPFD",
        "MeanDeltaAPFD",
    ],
].copy()

selected_noise = (
    selected_noise.sort_values(
        [
            "Technique",
            "NoisePercent",
        ],
        key=technique_sort_key,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

if len(
    selected_noise
) != (
    4
    * 5
):
    raise RuntimeError(
        "Selected-noise table does not contain 20 rows."
    )


# --------------------------------------------------------------------------------------------------
# 8. WRITE SYNTHESIS TABLES + COPIES OF FROZEN FULL RESULTS
# --------------------------------------------------------------------------------------------------

atomic_csv(
    PRIMARY_SYNTHESIS_PATH,
    primary_synthesis,
)

atomic_copy(
    apfdc_results_path,
    PRIMARY_FULL_RESULTS_PATH,
)

atomic_copy(
    apfd_results_path,
    SECONDARY_FULL_RESULTS_PATH,
)

atomic_csv(
    SELECTED_NOISE_TABLE_PATH,
    selected_noise,
)

atomic_copy(
    significance_summary_path,
    SIGNIFICANCE_SUMMARY_COPY_PATH,
)

atomic_copy(
    figure_data_path,
    FIGURE_DATA_COPY_PATH,
)


# --------------------------------------------------------------------------------------------------
# 9. FIGURE 1 — PRIMARY APFDC ABSOLUTE PERFORMANCE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.5,
        5.5,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        figure_data.loc[
            figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    x = pd.to_numeric(
        block[
            "NoisePercent"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    y = pd.to_numeric(
        block[
            "MeanGlobalSeedAPFDc"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    low = pd.to_numeric(
        block[
            "CI95LowGlobalSeedAPFDc"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    high = pd.to_numeric(
        block[
            "CI95HighGlobalSeedAPFDc"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    line = plt.plot(
        x,
        y,
        marker="o",
        label=technique,
    )[0]

    plt.fill_between(
        x,
        low,
        high,
        alpha=0.12,
        color=line.get_color(),
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Mean APFDc"
)

plt.title(
    "RQ1 — ML TCP effectiveness under increasing training-label noise"
)

plt.xticks(
    NOISE_LEVELS
)

plt.ylim(
    0.0,
    1.0,
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    FIGURE_APFDC_ABSOLUTE_PATH
)


# --------------------------------------------------------------------------------------------------
# 10. FIGURE 2 — APFDC CLEAN-RELATIVE DEGRADATION
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.5,
        5.5,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        figure_data.loc[
            figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "MeanDegradationPctAPFDc"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Mean APFDc degradation from clean (%)"
)

plt.title(
    "RQ1 — Clean-relative APFDc degradation"
)

plt.xticks(
    NOISE_LEVELS
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    FIGURE_APFDC_DEGRADATION_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. FIGURE 3 — APFDC RETENTION
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.5,
        5.5,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        figure_data.loc[
            figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "MeanRetentionPctAPFDc"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.axhline(
    100.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "APFDc retention relative to clean (%)"
)

plt.title(
    "RQ1 — APFDc retention relative to clean training data"
)

plt.xticks(
    NOISE_LEVELS
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    FIGURE_APFDC_RETENTION_PATH
)


# --------------------------------------------------------------------------------------------------
# 12. FIGURE 4 — APFDC PAIRED RANK-BISERIAL EFFECT SIZE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.5,
        5.5,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        apfdc_results.loc[
            apfdc_results[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "PairedRankBiserial"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Paired rank-biserial correlation"
)

plt.title(
    "RQ1 — Paired effect size for noisy vs. clean APFDc"
)

plt.xticks(
    NOISY_LEVELS
)

plt.ylim(
    -1.05,
    1.05,
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    FIGURE_APFDC_EFFECT_SIZE_PATH
)


# --------------------------------------------------------------------------------------------------
# 13. FIGURE 5 — SECONDARY APFD ABSOLUTE PERFORMANCE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.5,
        5.5,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        figure_data.loc[
            figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "MeanGlobalSeedAPFD"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Mean APFD"
)

plt.title(
    "RQ1 — Secondary APFD sensitivity analysis"
)

plt.xticks(
    NOISE_LEVELS
)

plt.ylim(
    0.0,
    1.0,
)

plt.legend(
    frameon=False
)

plt.grid(
    axis="y",
    alpha=0.25,
)

save_figure(
    FIGURE_APFD_SECONDARY_PATH
)


# --------------------------------------------------------------------------------------------------
# 14. NUMERICAL RQ1 FREEZE FACTS
# --------------------------------------------------------------------------------------------------

primary_significant_count = int(
    pd.to_numeric(
        apfdc_results[
            "SignificantBonferroni"
        ].astype(
            str
        ).str.lower().map({
            "true":
                1,

            "false":
                0,
        }),
        errors="raise",
    ).sum()
)

secondary_significant_count = int(
    pd.to_numeric(
        apfd_results[
            "SignificantBonferroni"
        ].astype(
            str
        ).str.lower().map({
            "true":
                1,

            "false":
                0,
        }),
        errors="raise",
    ).sum()
)

first_significant_apfdc = {}

for technique in ML_TECHNIQUES:
    block = significance_summary.loc[
        significance_summary[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFDc"
        )
        & significance_summary[
            "Algorithm"
        ].astype(
            str
        ).eq(
            technique
        )
    ]

    if len(
        block
    ) != 1:
        raise RuntimeError(
            f"{technique}: primary significance summary missing."
        )

    value = block.iloc[
        0
    ][
        "FirstBonferroniSignificantNoisePercent"
    ]

    first_significant_apfdc[
        technique
    ] = (
        None
        if pd.isna(
            value
        )
        or str(
            value
        ).strip()
        == ""
        else int(
            float(
                value
            )
        )
    )


# --------------------------------------------------------------------------------------------------
# 15. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []

registry_sha_after = sha256_file(
    REGISTRY
)

add_check(
    checks,
    "RQ1 Step-3A checkpoint SHA-256",
    EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,
    actual_step3a_sha,
    actual_step3a_sha
    == EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ1 Step-3B checkpoint SHA-256",
    EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256,
    actual_step3b_sha,
    actual_step3b_sha
    == EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Figure-data rows",
    36,
    len(
        figure_data
    ),
    len(
        figure_data
    )
    == 36,
)

add_check(
    checks,
    "Primary APFDc inferential rows",
    32,
    len(
        apfdc_results
    ),
    len(
        apfdc_results
    )
    == 32,
)

add_check(
    checks,
    "Secondary APFD inferential rows",
    32,
    len(
        apfd_results
    ),
    len(
        apfd_results
    )
    == 32,
)

add_check(
    checks,
    "Primary synthesis rows",
    4,
    len(
        primary_synthesis
    ),
    len(
        primary_synthesis
    )
    == 4,
)

add_check(
    checks,
    "Selected-noise rows",
    20,
    len(
        selected_noise
    ),
    len(
        selected_noise
    )
    == 20,
)

add_check(
    checks,
    "Primary APFDc Bonferroni-significant tests",
    25,
    primary_significant_count,
    primary_significant_count
    == 25,
)

add_check(
    checks,
    "Secondary APFD Bonferroni-significant tests",
    28,
    secondary_significant_count,
    secondary_significant_count
    == 28,
)

add_check(
    checks,
    "RandomForest first APFDc significant noise",
    5,
    first_significant_apfdc[
        "RandomForest"
    ],
    first_significant_apfdc[
        "RandomForest"
    ]
    == 5,
)

add_check(
    checks,
    "XGBoost first APFDc significant noise",
    5,
    first_significant_apfdc[
        "XGBoost"
    ],
    first_significant_apfdc[
        "XGBoost"
    ]
    == 5,
)

add_check(
    checks,
    "LightGBM first APFDc significant noise",
    5,
    first_significant_apfdc[
        "LightGBM"
    ],
    first_significant_apfdc[
        "LightGBM"
    ]
    == 5,
)

add_check(
    checks,
    "NaiveBayes first APFDc significant noise",
    50,
    first_significant_apfdc[
        "NaiveBayes"
    ],
    first_significant_apfdc[
        "NaiveBayes"
    ]
    == 50,
)

for figure_path in [
    FIGURE_APFDC_ABSOLUTE_PATH,
    FIGURE_APFDC_DEGRADATION_PATH,
    FIGURE_APFDC_RETENTION_PATH,
    FIGURE_APFDC_EFFECT_SIZE_PATH,
    FIGURE_APFD_SECONDARY_PATH,
]:
    add_check(
        checks,
        f"Figure exists: {figure_path.name}",
        True,
        figure_path.is_file(),
        figure_path.is_file()
        and figure_path.stat().st_size
        > 0,
    )

add_check(
    checks,
    "New inferential tests executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "RQ2 executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "RQ3 executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ1 Step 3C pre-freeze validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ1 STEP 3C VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 16. REPORT + STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "Step":
        "RQ1_STEP_3C",

    "Status":
        RQ1_STEP3C_STATUS,

    "CodeRevision":
        RQ1_STEP3C_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ1Step3ACheckpointSHA256":
        EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,

    "RQ1Step3BCheckpointSHA256":
        EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "Seed (30 within project)",

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "BonferroniAdjustedAlpha":
        BONFERRONI_ADJUSTED_ALPHA,

    "PrimaryAPFDcSignificantTests":
        primary_significant_count,

    "PrimaryAPFDcTotalTests":
        32,

    "SecondaryAPFDSignificantTests":
        secondary_significant_count,

    "SecondaryAPFDTotalTests":
        32,

    "FirstBonferroniSignificantNoisePercentAPFDc":
        first_significant_apfdc,

    "Figures":
        [
            str(
                FIGURE_APFDC_ABSOLUTE_PATH
            ),
            str(
                FIGURE_APFDC_DEGRADATION_PATH
            ),
            str(
                FIGURE_APFDC_RETENTION_PATH
            ),
            str(
                FIGURE_APFDC_EFFECT_SIZE_PATH
            ),
            str(
                FIGURE_APFD_SECONDARY_PATH
            ),
        ],

    "PrimarySynthesisPath":
        str(
            PRIMARY_SYNTHESIS_PATH
        ),

    "PrimaryFullResultsPath":
        str(
            PRIMARY_FULL_RESULTS_PATH
        ),

    "SecondaryFullResultsPath":
        str(
            SECONDARY_FULL_RESULTS_PATH
        ),

    "SelectedNoiseTablePath":
        str(
            SELECTED_NOISE_TABLE_PATH
        ),

    "NewInferentialTestsExecuted":
        False,

    "RQ2Executed":
        False,

    "RQ3Executed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "RQ2 STEP 4A — BASELINE CURVES, ML-BASELINE ADVANTAGES, "
            "AND OBSERVED CROSSOVER THRESHOLDS"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ1_STEP3C_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RQ1AnswerPackageFrozen":
            True,

        "ReadyForRQ2":
            True,
    },
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL PACKAGE MANIFEST + ROOT HASH
# --------------------------------------------------------------------------------------------------

package_manifest = build_package_manifest(
    STEP3C_ROOT,
    FINAL_PACKAGE_MANIFEST_PATH,
)

atomic_csv(
    FINAL_PACKAGE_MANIFEST_PATH,
    package_manifest,
)

package_root_sha = package_root_hash(
    package_manifest
)

package_files = int(
    len(
        package_manifest
    )
)

package_bytes = int(
    package_manifest[
        "Bytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 18. FINAL CHECKPOINT
# --------------------------------------------------------------------------------------------------

if sha256_file(
    REGISTRY
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during RQ1 Step 3C."
    )

checkpoint = {
    "CheckpointType":
        "RQ1_FINAL_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE",

    "Status":
        RQ1_STEP3C_STATUS,

    "CodeRevision":
        RQ1_STEP3C_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ1Step3ACheckpointSHA256":
        EXPECTED_RQ1_STEP3A_CHECKPOINT_SHA256,

    "RQ1Step3BCheckpointSHA256":
        EXPECTED_RQ1_STEP3B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "PackageRoot":
        str(
            STEP3C_ROOT
        ),

    "PackageManifestPath":
        str(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            FINAL_PACKAGE_MANIFEST_PATH
        ),

    "PackageRootSHA256":
        package_root_sha,

    "PackageFiles":
        package_files,

    "PackageBytes":
        package_bytes,

    "PrimarySynthesisPath":
        str(
            PRIMARY_SYNTHESIS_PATH
        ),

    "PrimarySynthesisSHA256":
        sha256_file(
            PRIMARY_SYNTHESIS_PATH
        ),

    "PrimaryFullResultsPath":
        str(
            PRIMARY_FULL_RESULTS_PATH
        ),

    "PrimaryFullResultsSHA256":
        sha256_file(
            PRIMARY_FULL_RESULTS_PATH
        ),

    "SecondaryFullResultsPath":
        str(
            SECONDARY_FULL_RESULTS_PATH
        ),

    "SecondaryFullResultsSHA256":
        sha256_file(
            SECONDARY_FULL_RESULTS_PATH
        ),

    "FigureDataPath":
        str(
            FIGURE_DATA_COPY_PATH
        ),

    "FigureDataSHA256":
        sha256_file(
            FIGURE_DATA_COPY_PATH
        ),

    "PrimaryAPFDcSignificantTests":
        primary_significant_count,

    "SecondaryAPFDSignificantTests":
        secondary_significant_count,

    "FirstBonferroniSignificantNoisePercentAPFDc":
        first_significant_apfdc,

    "NewInferentialTestsExecuted":
        False,

    "RQ2Executed":
        False,

    "RQ3Executed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "ReadyForRQ2":
        True,

    "NextRequiredStep":
        (
            "RQ2 STEP 4A — BASELINE CURVES, ML-BASELINE ADVANTAGES, "
            "AND OBSERVED CROSSOVER THRESHOLDS"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. USER-VISIBLE SYNTHESIS
# --------------------------------------------------------------------------------------------------

print(
    "\nRQ1 primary APFDc algorithm synthesis:"
)

try:
    from IPython.display import display

    display(
        primary_synthesis
    )

except Exception:
    print(
        primary_synthesis.to_string(
            index=False
        )
    )

print(
    "\nRQ1 selected noise-level evidence:"
)

try:
    from IPython.display import display

    display(
        selected_noise
    )

except Exception:
    print(
        selected_noise.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 20. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 150
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 7 / RQ1 STEP 3C RESULT ==="
)

print(
    "=" * 150
)

print(
    "RQ1 final answer package frozen: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Repeated stochastic unit: 30 seeds within each project"
)

print(
    "\nPrimary APFDc inference:"
)

print(
    "Bonferroni-significant comparisons:",
    primary_significant_count,
    "/ 32",
)

print(
    "RandomForest first significant noisy level:",
    first_significant_apfdc[
        "RandomForest"
    ],
)

print(
    "XGBoost first significant noisy level:",
    first_significant_apfdc[
        "XGBoost"
    ],
)

print(
    "LightGBM first significant noisy level:",
    first_significant_apfdc[
        "LightGBM"
    ],
)

print(
    "NaiveBayes first significant noisy level:",
    first_significant_apfdc[
        "NaiveBayes"
    ],
)

print(
    "\nSecondary APFD inference:"
)

print(
    "Bonferroni-significant comparisons:",
    secondary_significant_count,
    "/ 32",
)

print(
    "\nFrozen outputs:"
)

print(
    "Primary synthesis rows:",
    len(
        primary_synthesis
    ),
)

print(
    "Selected-noise evidence rows:",
    len(
        selected_noise
    ),
)

print(
    "Figures:",
    5,
)

print(
    "Package files:",
    package_files,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Package root SHA-256:",
    package_root_sha,
)

print(
    "\nIsolation:"
)

print(
    "New inferential tests executed: False"
)

print(
    "RQ2 executed: False"
)

print(
    "RQ3 executed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nRQ1 Step 3C checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "RQ2 STEP 4A — BASELINE CURVES, ML-BASELINE ADVANTAGES, "
    "AND OBSERVED CROSSOVER THRESHOLDS"
)

print(
    "\nSTATUS:",
    RQ1_STEP3C_STATUS,
)

print(
    "=" * 150
)


=== THESIS GLOBAL ANALYSIS — CELL 7 / RQ1 STEP 3C: FINAL FIGURES + SYNTHESIS + ANSWER PACKAGE ===

RQ1 Step 3C pre-freeze validation:


,Check,Expected,Actual,Pass
0,RQ1 Step-3A checkpoint SHA-256,5f5403a21337144590848659395002fbeb31819427e0de...,5f5403a21337144590848659395002fbeb31819427e0de...,True
1,RQ1 Step-3B checkpoint SHA-256,27e0b52469023791a37ebf1099b6a63e7509ae25139816...,27e0b52469023791a37ebf1099b6a63e7509ae25139816...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,Figure-data rows,36,36,True
4,Primary APFDc inferential rows,32,32,True
5,Secondary APFD inferential rows,32,32,True
6,Primary synthesis rows,4,4,True
7,Selected-noise rows,20,20,True
8,Primary APFDc Bonferroni-significant tests,25,25,True
9,Secondary APFD Bonferroni-significant tests,28,28,True



RQ1 primary APFDc algorithm synthesis:


,Technique,CleanMeanAPFDc,MeanAPFDc_5,MeanAPFDc_15,MeanAPFDc_30,MeanAPFDc_50,MeanDegradationPctAPFDc_5,MeanDegradationPctAPFDc_15,MeanDegradationPctAPFDc_30,MeanDegradationPctAPFDc_50,MeanRetentionPctAPFDc_50,BonferroniSignificantNoiseLevels,FirstBonferroniSignificantNoisePercent,BonferroniSignificantTests,MostNegativePairedRankBiserial,LeastNegativeOrMostPositivePairedRankBiserial
0,RandomForest,0.803239,0.674281,0.591190,0.521425,0.443023,15.710129,24.848051,33.051305,42.255380,57.744620,"[5,10,15,20,25,30,40,50]",5,8,-1.000000,-0.940000
1,XGBoost,0.784217,0.684539,0.609067,0.528129,0.426752,12.366411,21.358264,31.075530,43.012815,56.987185,"[5,10,15,20,25,30,40,50]",5,8,-1.000000,-0.893333
2,LightGBM,0.762060,0.680160,0.627722,0.547930,0.432476,9.559332,16.256329,25.732400,39.608001,60.391999,"[5,10,15,20,25,30,40,50]",5,8,-1.000000,-0.693333
3,NaiveBayes,0.617603,0.573381,0.544015,0.537718,0.492052,3.927477,7.323693,7.369633,12.761964,87.238036,[50],50,1,-0.686667,-0.440000



RQ1 selected noise-level evidence:


,Technique,NoisePercent,MeanGlobalSeedAPFDc,CI95LowGlobalSeedAPFDc,CI95HighGlobalSeedAPFDc,MeanDeltaAPFDc,MeanDegradationPctAPFDc,MeanRetentionPctAPFDc,APFDcDegradedProjects,APFDcTiedProjects,APFDcImprovedProjects,MeanGlobalSeedAPFD,MeanDeltaAPFD
0,RandomForest,0,0.803239,0.801003,0.805474,0.000000,0.000000,100.000000,0,24,0,0.903964,0.000000
1,RandomForest,5,0.674281,0.669142,0.679421,-0.128957,15.710129,84.289871,22,0,2,0.746524,-0.157440
2,RandomForest,15,0.591190,0.583368,0.599012,-0.212048,24.848051,75.151949,23,0,1,0.634206,-0.269757
3,RandomForest,30,0.521425,0.511879,0.530971,-0.281814,33.051305,66.948695,24,0,0,0.549337,-0.354627
4,RandomForest,50,0.443023,0.432406,0.453641,-0.360215,42.255380,57.744620,24,0,0,0.432852,-0.471112
5,XGBoost,0,0.784217,0.784217,0.784217,0.000000,0.000000,100.000000,0,24,0,0.917151,0.000000
6,XGBoost,5,0.684539,0.675644,0.693433,-0.099678,12.366411,87.633589,22,0,2,0.762095,-0.155057
7,XGBoost,15,0.609067,0.599674,0.618460,-0.175150,21.358264,78.641736,22,0,2,0.673993,-0.243158
8,XGBoost,30,0.528129,0.516981,0.539276,-0.256088,31.075530,68.924470,24,0,0,0.583909,-0.333243
9,XGBoost,50,0.426752,0.414692,0.438812,-0.357465,43.012815,56.987185,24,0,0,0.455526,-0.461626



=== THESIS GLOBAL ANALYSIS — CELL 7 / RQ1 STEP 3C RESULT ===
RQ1 final answer package frozen: True

Study unit:
Independent empirical unit: Project (N=24)
Repeated stochastic unit: 30 seeds within each project

Primary APFDc inference:
Bonferroni-significant comparisons: 25 / 32
RandomForest first significant noisy level: 5
XGBoost first significant noisy level: 5
LightGBM first significant noisy level: 5
NaiveBayes first significant noisy level: 50

Secondary APFD inference:
Bonferroni-significant comparisons: 28 / 32

Frozen outputs:
Primary synthesis rows: 4
Selected-noise evidence rows: 20
Figures: 5
Package files: 14
Package bytes: 1197454
Package root SHA-256: 5b7c49e4096c11edd927052807658e5a196e58b5c7302e7f6fa886258ad37ff4

Isolation:
New inferential tests executed: False
RQ2 executed: False
RQ3 executed: False
Project outputs modified: False
Completion registry modified: False

Validation:
Checks: 23
Failed checks: 0

RQ1 Step 3C checkpoint:
/content/drive/MyDrive/Thesis_Exper

In [5]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 8 / RQ2 STEP 4A
# BASELINE CURVES + ML-BASELINE ADVANTAGES + OBSERVED CROSSOVER THRESHOLDS
# ==================================================================================================
#
# RQ2
# ---
# At what noise level do supervised ML-based TCP techniques cease to outperform
# Random, LatestFail and QTF-Avg?
#
# FROZEN CONTRACT (Step 2)
# ------------------------
# Primary metric: APFDc
# Primary proposal comparator: LatestFail
# Additional reported comparators: Random, QTF-Avg
# Conservative supplementary comparator: strongest baseline at each tested noise level
#
# First observed crossover:
#   lowest TESTED noise p where
#       cross-project mean ML(p) <= cross-project mean baseline(p)
#
# Sustained crossover (supplementary):
#   lowest TESTED p for which ML <= comparator at p and at every higher tested p.
#
# NO interpolation between tested noise levels.
#
# If ML is already not better at 0%:
#   NO_POSITIVE_TOLERANCE_THRESHOLD
#
# If no crossover occurs through 50%:
#   >50 / NOT_OBSERVED_IN_TESTED_RANGE
#
# RQ2 is descriptive by the frozen thesis contract.
# This cell performs NO new hypothesis tests and computes NO p-values.
#
# Independent empirical unit:
#   Project (N=24)
#
# Supporting evidence:
#   - equally weighted cross-project curves for all 7 techniques;
#   - project-level ML-minus-baseline paired advantages;
#   - mean/median paired advantage;
#   - project win/tie/loss counts at tolerance 1e-12;
#   - first observed crossover;
#   - sustained crossover;
#   - secondary APFD sensitivity thresholds.
#
# IMPORTANT:
#   The "strongest baseline" is selected separately at each tested noise level as the baseline
#   with the highest equally weighted cross-project mean metric. For project-level supporting
#   win/tie/loss counts, the project values from THAT selected baseline technique are used.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 152)
print("=== THESIS GLOBAL ANALYSIS — CELL 8 / RQ2 STEP 4A: BASELINES + ADVANTAGES + OBSERVED CROSSOVERS ===")
print("=" * 152)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

EXPECTED_STEP1B_CHECKPOINT_SHA256 = (
    "eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3e0642d1b19f456ad50"
)

STEP2_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
)

EXPECTED_STEP2_CHECKPOINT_SHA256 = (
    "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"
)

RQ1_STEP3C_STATUS = (
    "PASS_RQ1_STEP3C_FINAL_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256 = (
    "b4f3d10b77acb39200496542e8256d6213981fd636325487f190ce2dd07f35af"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ2_STEP4A_STATUS = (
    "PASS_RQ2_STEP4A_BASELINE_CURVES_ADVANTAGES_AND_OBSERVED_CROSSOVERS_FROZEN"
)

RQ2_STEP4A_CODE_REVISION = (
    "RQ2_STEP4A_V2_CORRECT_MIXED_NUMERIC_MISSING_CSV_READBACK_AUDIT"
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = [
    "LatestFail",
    "LightGBM",
    "NaiveBayes",
    "QTF-Avg",
    "Random",
    "RandomForest",
    "XGBoost",
]

PRIMARY_COMPARATOR = "LatestFail"
STRONGEST_BASELINE_LABEL = "StrongestBaselineAtEachNoise"

EXPECTED_PROJECTS = 24
EXPECTED_SEEDS = 30

EXPECTED_PROJECT_MASTER_ROWS = 1_512
EXPECTED_SEED_MASTER_ROWS = 45_360

EXPECTED_CURVE_ROWS = 9 * 7             # 63
EXPECTED_PAIRWISE_PROJECT_ROWS = 4 * 3 * 9 * 24   # 2,592
EXPECTED_PAIRWISE_SUMMARY_ROWS = 4 * 3 * 9        # 108
EXPECTED_STRONGEST_PROJECT_ROWS = 4 * 9 * 24      # 864
EXPECTED_STRONGEST_SUMMARY_ROWS = 4 * 9           # 36
EXPECTED_THRESHOLD_ROWS = 4 * 4                   # 16 per metric

FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = RESULTS / "Analysis" / "Global_24_Project_Analysis"

STEP1B_CHECKPOINT = NOTES / "global_analysis_step1b_checkpoint.json"
STEP2_CHECKPOINT = NOTES / "global_analysis_step2_checkpoint.json"
RQ1_STEP3C_CHECKPOINT = NOTES / "global_analysis_rq1_step3c_checkpoint.json"

STEP4A_ROOT = (
    ANALYSIS_ROOT
    / "RQ2"
    / "Step_4A_Baseline_Curves_Advantages_and_Observed_Crossovers"
)

ALL_TECHNIQUE_CURVES_PATH = (
    STEP4A_ROOT
    / "rq2_all_technique_cross_project_curves.csv"
)

BASELINE_CURVES_PATH = (
    STEP4A_ROOT
    / "rq2_baseline_curves.csv"
)

BASELINE_INVARIANCE_PATH = (
    STEP4A_ROOT
    / "rq2_baseline_invariance_by_technique.csv"
)

PAIRWISE_PROJECT_ADVANTAGES_PATH = (
    STEP4A_ROOT
    / "rq2_pairwise_project_advantages.csv"
)

PAIRWISE_ADVANTAGE_SUMMARY_PATH = (
    STEP4A_ROOT
    / "rq2_pairwise_advantage_summary.csv"
)

STRONGEST_BASELINE_SELECTION_PATH = (
    STEP4A_ROOT
    / "rq2_strongest_baseline_selection.csv"
)

STRONGEST_PROJECT_ADVANTAGES_PATH = (
    STEP4A_ROOT
    / "rq2_strongest_baseline_project_advantages.csv"
)

STRONGEST_ADVANTAGE_SUMMARY_PATH = (
    STEP4A_ROOT
    / "rq2_strongest_baseline_advantage_summary.csv"
)

APFDC_THRESHOLDS_PATH = (
    STEP4A_ROOT
    / "rq2_apfdc_observed_crossover_thresholds.csv"
)

APFD_THRESHOLDS_PATH = (
    STEP4A_ROOT
    / "rq2_apfd_secondary_observed_crossover_thresholds.csv"
)

VALIDATION_PATH = (
    STEP4A_ROOT
    / "rq2_step4a_validation.csv"
)

READBACK_AUDIT_PATH = (
    STEP4A_ROOT
    / "rq2_step4a_readback_audit.csv"
)

REPORT_PATH = (
    STEP4A_ROOT
    / "rq2_step4a_report.json"
)

STATUS_PATH = (
    STEP4A_ROOT
    / "rq2_step4a_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP4A_ROOT
    / "rq2_step4a_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq2_step4a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_csv(path, dataframe):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(temporary, path)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary, path)


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def build_output_manifest(paths):
    rows = []

    for path in sorted((Path(path) for path in paths), key=str):
        if not path.is_file():
            raise FileNotFoundError(
                f"RQ2 Step-4A output missing: {path}"
            )

        rows.append({
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        })

    return pd.DataFrame(
        rows,
        columns=["Path", "Bytes", "SHA256"],
    )


def scientific_hash(dataframe):
    digest = hashlib.sha256()

    for row in dataframe.itertuples(index=False, name=None):
        parts = []

        for value in row:
            if isinstance(value, (float, np.floating)):
                if math.isnan(float(value)):
                    parts.append("NaN")
                else:
                    parts.append(f"{float(value):.17g}")
            else:
                parts.append(str(value))

        digest.update(
            ("\0".join(parts) + "\n").encode("utf-8")
        )

    return digest.hexdigest()


def to_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().lower()

    if text in {"true", "1"}:
        return True

    if text in {"false", "0"}:
        return False

    raise RuntimeError(
        f"Cannot parse boolean value: {value!r}"
    )


def classify_advantage(value, tolerance=FLOAT_TOL):
    value = float(value)

    if value > tolerance:
        return "WIN"

    if value < -tolerance:
        return "LOSS"

    return "TIE"


def summarize_project_advantages(block, advantage_column):
    values = pd.to_numeric(
        block[advantage_column],
        errors="raise",
    ).to_numpy(dtype=float)

    classifications = [
        classify_advantage(value)
        for value in values
    ]

    return {
        "Projects": int(len(values)),
        "MeanAdvantage": float(np.mean(values)),
        "MedianAdvantage": float(np.median(values)),
        "SDAdvantage": float(np.std(values, ddof=1)),
        "MinAdvantage": float(np.min(values)),
        "MaxAdvantage": float(np.max(values)),
        "ProjectWins": int(sum(value == "WIN" for value in classifications)),
        "ProjectTies": int(sum(value == "TIE" for value in classifications)),
        "ProjectLosses": int(sum(value == "LOSS" for value in classifications)),
    }


def threshold_from_advantage_curve(
    block,
    advantage_column,
):
    """
    block must contain all 9 tested noise levels in ascending order.

    Returns:
      first observed crossover
      sustained crossover
      status labels
    """

    block = (
        block.sort_values(
            "NoisePercent",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    observed_noise = pd.to_numeric(
        block["NoisePercent"],
        errors="raise",
    ).astype(int).tolist()

    if observed_noise != NOISE_LEVELS:
        raise RuntimeError(
            f"Threshold curve noise grid mismatch: {observed_noise}"
        )

    advantages = pd.to_numeric(
        block[advantage_column],
        errors="raise",
    ).to_numpy(dtype=float)

    not_better = (
        advantages
        <= FLOAT_TOL
    )

    clean_advantage = float(
        advantages[0]
    )

    if clean_advantage <= FLOAT_TOL:
        first_value = 0
        first_status = "NO_POSITIVE_TOLERANCE_THRESHOLD"
    else:
        indices = np.where(
            not_better
        )[0]

        if len(indices) == 0:
            first_value = None
            first_status = "NOT_OBSERVED_THROUGH_50"
        else:
            first_value = int(
                NOISE_LEVELS[
                    int(indices[0])
                ]
            )
            first_status = "OBSERVED"

    sustained_value = None
    sustained_status = "NOT_OBSERVED_THROUGH_50"

    for index, noise in enumerate(NOISE_LEVELS):
        if bool(
            np.all(
                not_better[index:]
            )
        ):
            sustained_value = int(noise)

            if sustained_value == 0:
                sustained_status = (
                    "NO_POSITIVE_TOLERANCE_THRESHOLD"
                )
            else:
                sustained_status = "OBSERVED"

            break

    return {
        "CleanAdvantage": clean_advantage,
        "FirstObservedCrossoverNoisePercent": (
            ""
            if first_value is None
            else first_value
        ),
        "FirstObservedCrossoverStatus": first_status,
        "SustainedCrossoverNoisePercent": (
            ""
            if sustained_value is None
            else sustained_value
        ),
        "SustainedCrossoverStatus": sustained_status,
    }


def _csv_missing_mask(series):
    """
    Treat true missing values and empty strings as the same CSV-level missing representation.
    pandas commonly reloads an empty CSV field as NaN.
    """
    object_series = series.astype("object")

    return np.array(
        [
            (
                pd.isna(value)
                or (
                    isinstance(value, str)
                    and value.strip() == ""
                )
            )
            for value in object_series.tolist()
        ],
        dtype=bool,
    )


def _normalize_bool_token(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().lower()

    if text in {"true", "1"}:
        return True

    if text in {"false", "0"}:
        return False

    raise ValueError(
        f"Not a boolean token: {value!r}"
    )


def compare_csv_roundtrip(
    expected,
    actual,
    float_tolerance=FLOAT_TOL,
):
    """
    Validate scientific equivalence after CSV round-trip.

    V2 FIX
    ------
    V1 compared object-typed columns literally as strings. That is too strict for mixed columns
    such as threshold noise values containing both integers and intentionally blank entries:

        in memory: [5, 10, "", 50]
        CSV reload: [5.0, 10.0, NaN, 50.0]

    Those are scientifically identical, but string comparison sees "5" != "5.0".

    V2 therefore compares each column semantically:
      - row/column structure exact;
      - missing-value positions exact, treating empty string and NaN as equivalent CSV missingness;
      - boolean-like values exact after normalization;
      - numeric-like values compared numerically within <=1e-12 even if dtype changed;
      - genuinely textual values compared exactly.
    """

    row_count_match = (
        len(expected)
        == len(actual)
    )

    column_order_match = (
        expected.columns.tolist()
        == actual.columns.tolist()
    )

    if (
        not row_count_match
        or not column_order_match
    ):
        return False, np.inf

    max_float_difference = 0.0

    for column in expected.columns:
        expected_series = expected[
            column
        ]

        actual_series = actual[
            column
        ]

        expected_missing = _csv_missing_mask(
            expected_series
        )

        actual_missing = _csv_missing_mask(
            actual_series
        )

        if not np.array_equal(
            expected_missing,
            actual_missing,
        ):
            return False, np.inf

        nonmissing = ~expected_missing

        if not nonmissing.any():
            continue

        expected_values = expected_series.astype(
            "object"
        ).to_numpy()[
            nonmissing
        ]

        actual_values = actual_series.astype(
            "object"
        ).to_numpy()[
            nonmissing
        ]

        # 1. Boolean-like semantic comparison.
        boolean_comparison_possible = True

        try:
            expected_bool = np.array(
                [
                    _normalize_bool_token(
                        value
                    )
                    for value in expected_values
                ],
                dtype=bool,
            )

            actual_bool = np.array(
                [
                    _normalize_bool_token(
                        value
                    )
                    for value in actual_values
                ],
                dtype=bool,
            )

        except Exception:
            boolean_comparison_possible = False

        if boolean_comparison_possible:
            if not np.array_equal(
                expected_bool,
                actual_bool,
            ):
                return False, max_float_difference

            continue

        # 2. Numeric-like semantic comparison, regardless of pandas dtype.
        expected_numeric = pd.to_numeric(
            pd.Series(
                expected_values
            ),
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        actual_numeric = pd.to_numeric(
            pd.Series(
                actual_values
            ),
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        if (
            np.isfinite(
                expected_numeric
            ).all()
            and np.isfinite(
                actual_numeric
            ).all()
        ):
            column_difference = float(
                np.max(
                    np.abs(
                        expected_numeric
                        - actual_numeric
                    )
                )
            )

            max_float_difference = max(
                max_float_difference,
                column_difference,
            )

            if (
                column_difference
                > float_tolerance
            ):
                return False, max_float_difference

            continue

        # 3. Genuine text must remain exact.
        expected_text = [
            str(
                value
            )
            for value in expected_values
        ]

        actual_text = [
            str(
                value
            )
            for value in actual_values
        ]

        if (
            expected_text
            != actual_text
        ):
            return False, max_float_difference

    return True, max_float_difference


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD + VERIFY FROZEN CHAIN
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ2 Step 4A is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ2 Step 4B."
    )

for path in [
    STEP1B_CHECKPOINT,
    STEP2_CHECKPOINT,
    RQ1_STEP3C_CHECKPOINT,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required checkpoint missing: {path}"
        )

actual_step1b_sha = sha256_file(
    STEP1B_CHECKPOINT
)

actual_step2_sha = sha256_file(
    STEP2_CHECKPOINT
)

actual_rq1_step3c_sha = sha256_file(
    RQ1_STEP3C_CHECKPOINT
)

if (
    actual_step1b_sha
    != EXPECTED_STEP1B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-1B checkpoint SHA mismatch."
    )

if (
    actual_step2_sha
    != EXPECTED_STEP2_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-2 checkpoint SHA mismatch."
    )

if (
    actual_rq1_step3c_sha
    != EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ1 Step-3C checkpoint SHA mismatch."
    )

step1b = load_json(
    STEP1B_CHECKPOINT
)

step2 = load_json(
    STEP2_CHECKPOINT
)

rq1_step3c = load_json(
    RQ1_STEP3C_CHECKPOINT
)

if step1b.get("Status") != STEP1B_STATUS:
    raise RuntimeError(
        "Step-1B checkpoint status is not PASS."
    )

if step2.get("Status") != STEP2_STATUS:
    raise RuntimeError(
        "Step-2 checkpoint status is not PASS."
    )

if rq1_step3c.get("Status") != RQ1_STEP3C_STATUS:
    raise RuntimeError(
        "RQ1 Step-3C checkpoint status is not PASS."
    )

if not bool(
    rq1_step3c.get(
        "ReadyForRQ2",
        False,
    )
):
    raise RuntimeError(
        "RQ1 Step-3C is not marked ready for RQ2."
    )

rq2_plan_path = Path(
    step2[
        "RQ2ThresholdPlanPath"
    ]
)

expected_rq2_plan_sha = str(
    step2[
        "RQ2ThresholdPlanSHA256"
    ]
).lower()

if not rq2_plan_path.is_file():
    raise FileNotFoundError(
        f"Frozen RQ2 threshold plan missing: {rq2_plan_path}"
    )

if (
    sha256_file(
        rq2_plan_path
    )
    != expected_rq2_plan_sha
):
    raise RuntimeError(
        "Frozen RQ2 threshold-plan SHA mismatch."
    )

rq2_plan = pd.read_csv(
    rq2_plan_path,
    low_memory=False,
)

if len(
    rq2_plan
) != 16:
    raise RuntimeError(
        "Frozen RQ2 threshold plan does not contain 16 rows."
    )

if not rq2_plan[
    "Interpolation"
].astype(
    str
).eq(
    "FORBIDDEN"
).all():
    raise RuntimeError(
        "Frozen RQ2 threshold plan unexpectedly permits interpolation."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if (
    registry_sha_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry differs from the final 24-project freeze."
    )

if STEP4A_ROOT.exists():
    shutil.rmtree(
        STEP4A_ROOT,
        ignore_errors=True,
    )

STEP4A_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY + LOAD CANONICAL MASTERS
# --------------------------------------------------------------------------------------------------

seed_master_path = Path(
    step1b[
        "SeedLevelMasterPath"
    ]
)

project_master_path = Path(
    step1b[
        "ProjectLevelMasterPath"
    ]
)

for path, sha_key in [
    (
        seed_master_path,
        "SeedLevelMasterSHA256",
    ),
    (
        project_master_path,
        "ProjectLevelMasterSHA256",
    ),
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen master missing: {path}"
        )

    if (
        sha256_file(path)
        != str(
            step1b[
                sha_key
            ]
        ).lower()
    ):
        raise RuntimeError(
            f"Frozen master SHA mismatch: {path.name}"
        )

seed_master = pd.read_csv(
    seed_master_path,
    low_memory=False,
)

project_master = pd.read_csv(
    project_master_path,
    low_memory=False,
)

if len(seed_master) != EXPECTED_SEED_MASTER_ROWS:
    raise RuntimeError(
        "Seed master row count changed."
    )

if len(project_master) != EXPECTED_PROJECT_MASTER_ROWS:
    raise RuntimeError(
        "Project master row count changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. BUILD EQUALLY-WEIGHTED CROSS-PROJECT CURVES FOR ALL 7 TECHNIQUES
# --------------------------------------------------------------------------------------------------

curve_rows = []

for metric, project_column in [
    (
        "APFDc",
        "MeanSeedMeanAPFDc",
    ),
    (
        "APFD",
        "MeanSeedMeanAPFD",
    ),
]:
    for technique in ALL_TECHNIQUES:
        for noise in NOISE_LEVELS:
            block = project_master.loc[
                project_master[
                    "Technique"
                ].astype(
                    str
                ).eq(
                    technique
                )
                & pd.to_numeric(
                    project_master[
                        "NoisePercent"
                    ],
                    errors="raise",
                ).astype(
                    int
                ).eq(
                    noise
                )
            ].sort_values(
                "ProjectNumber",
                kind="mergesort",
            )

            if len(block) != EXPECTED_PROJECTS:
                raise RuntimeError(
                    f"{metric} {technique} {noise}% does not contain 24 projects."
                )

            values = pd.to_numeric(
                block[
                    project_column
                ],
                errors="raise",
            ).to_numpy(dtype=float)

            curve_rows.append({
                "Metric":
                    metric,

                "Technique":
                    technique,

                "TechniqueFamily":
                    (
                        "ML"
                        if technique
                        in ML_TECHNIQUES
                        else "Baseline"
                    ),

                "NoisePercent":
                    noise,

                "Projects":
                    24,

                "CrossProjectMean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "CrossProjectMedian":
                    float(
                        np.median(
                            values
                        )
                    ),

                "CrossProjectSD":
                    float(
                        np.std(
                            values,
                            ddof=1,
                        )
                    ),

                "CrossProjectMin":
                    float(
                        np.min(
                            values
                        )
                    ),

                "CrossProjectMax":
                    float(
                        np.max(
                            values
                        )
                    ),
            })


all_technique_curves = pd.DataFrame(
    curve_rows
)

if len(
    all_technique_curves
) != (
    2
    * EXPECTED_CURVE_ROWS
):
    raise RuntimeError(
        "All-technique curve table does not contain 126 rows."
    )

baseline_curves = all_technique_curves.loc[
    all_technique_curves[
        "Technique"
    ].isin(
        BASELINES
    )
].copy()

if len(
    baseline_curves
) != (
    2
    * 3
    * 9
):
    raise RuntimeError(
        "Baseline curve table does not contain 54 rows."
    )


# --------------------------------------------------------------------------------------------------
# 7. BASELINE INVARIANCE DIAGNOSTIC BY TECHNIQUE
# --------------------------------------------------------------------------------------------------

baseline_invariance_rows = []

for metric in [
    "APFDc",
    "APFD",
]:
    for baseline in BASELINES:
        block = baseline_curves.loc[
            baseline_curves[
                "Metric"
            ].eq(
                metric
            )
            & baseline_curves[
                "Technique"
            ].eq(
                baseline
            )
        ].sort_values(
            "NoisePercent",
            kind="mergesort",
        )

        values = pd.to_numeric(
            block[
                "CrossProjectMean"
            ],
            errors="raise",
        ).to_numpy(dtype=float)

        curve_range = float(
            np.max(values)
            - np.min(values)
        )

        baseline_invariance_rows.append({
            "Metric":
                metric,

            "Baseline":
                baseline,

            "NoiseLevels":
                len(
                    block
                ),

            "CrossProjectMeanRangeAcrossNoise":
                curve_range,

            "InvariantAcrossNoiseAt1e12":
                bool(
                    curve_range
                    <= FLOAT_TOL
                ),
        })


baseline_invariance = pd.DataFrame(
    baseline_invariance_rows
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD PROJECT-LEVEL PAIRWISE ADVANTAGES FOR EACH ML × EACH NAMED BASELINE
# --------------------------------------------------------------------------------------------------

pairwise_project_rows = []

for metric, project_column in [
    (
        "APFDc",
        "MeanSeedMeanAPFDc",
    ),
    (
        "APFD",
        "MeanSeedMeanAPFD",
    ),
]:
    for algorithm in ML_TECHNIQUES:
        for baseline in BASELINES:
            for noise in NOISE_LEVELS:
                ml_block = project_master.loc[
                    project_master[
                        "Technique"
                    ].astype(
                        str
                    ).eq(
                        algorithm
                    )
                    & pd.to_numeric(
                        project_master[
                            "NoisePercent"
                        ],
                        errors="raise",
                    ).astype(
                        int
                    ).eq(
                        noise
                    ),
                    [
                        "ProjectNumber",
                        "Project",
                        project_column,
                    ],
                ].copy()

                baseline_block = project_master.loc[
                    project_master[
                        "Technique"
                    ].astype(
                        str
                    ).eq(
                        baseline
                    )
                    & pd.to_numeric(
                        project_master[
                            "NoisePercent"
                        ],
                        errors="raise",
                    ).astype(
                        int
                    ).eq(
                        noise
                    ),
                    [
                        "ProjectNumber",
                        project_column,
                    ],
                ].copy()

                ml_block = ml_block.rename(
                    columns={
                        project_column:
                            "MLValue",
                    }
                )

                baseline_block = baseline_block.rename(
                    columns={
                        project_column:
                            "BaselineValue",
                    }
                )

                paired = ml_block.merge(
                    baseline_block,
                    on="ProjectNumber",
                    how="inner",
                    validate="one_to_one",
                )

                if len(paired) != 24:
                    raise RuntimeError(
                        f"{metric} {algorithm} vs {baseline} {noise}% pairing !=24."
                    )

                paired[
                    "AdvantageMLMinusBaseline"
                ] = (
                    paired[
                        "MLValue"
                    ]
                    - paired[
                        "BaselineValue"
                    ]
                )

                for row in paired.itertuples(
                    index=False
                ):
                    pairwise_project_rows.append({
                        "Metric":
                            metric,

                        "Algorithm":
                            algorithm,

                        "Comparator":
                            baseline,

                        "NoisePercent":
                            noise,

                        "ProjectNumber":
                            int(
                                row.ProjectNumber
                            ),

                        "Project":
                            row.Project,

                        "MLValue":
                            float(
                                row.MLValue
                            ),

                        "BaselineValue":
                            float(
                                row.BaselineValue
                            ),

                        "AdvantageMLMinusBaseline":
                            float(
                                row.AdvantageMLMinusBaseline
                            ),

                        "Outcome":
                            classify_advantage(
                                row.AdvantageMLMinusBaseline
                            ),
                    })


pairwise_project_advantages = pd.DataFrame(
    pairwise_project_rows
)

if len(
    pairwise_project_advantages
) != (
    2
    * EXPECTED_PAIRWISE_PROJECT_ROWS
):
    raise RuntimeError(
        "Pairwise project advantage table row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 9. SUMMARIZE NAMED-BASELINE ADVANTAGES
# --------------------------------------------------------------------------------------------------

pairwise_summary_rows = []

for metric in [
    "APFDc",
    "APFD",
]:
    for algorithm in ML_TECHNIQUES:
        for baseline in BASELINES:
            for noise in NOISE_LEVELS:
                block = pairwise_project_advantages.loc[
                    pairwise_project_advantages[
                        "Metric"
                    ].eq(
                        metric
                    )
                    & pairwise_project_advantages[
                        "Algorithm"
                    ].eq(
                        algorithm
                    )
                    & pairwise_project_advantages[
                        "Comparator"
                    ].eq(
                        baseline
                    )
                    & pairwise_project_advantages[
                        "NoisePercent"
                    ].eq(
                        noise
                    )
                ].sort_values(
                    "ProjectNumber",
                    kind="mergesort",
                )

                if len(block) != 24:
                    raise RuntimeError(
                        "Pairwise summary block does not contain 24 projects."
                    )

                summary = summarize_project_advantages(
                    block,
                    "AdvantageMLMinusBaseline",
                )

                ml_global_mean = float(
                    np.mean(
                        pd.to_numeric(
                            block[
                                "MLValue"
                            ],
                            errors="raise",
                        )
                    )
                )

                baseline_global_mean = float(
                    np.mean(
                        pd.to_numeric(
                            block[
                                "BaselineValue"
                            ],
                            errors="raise",
                        )
                    )
                )

                global_advantage = (
                    ml_global_mean
                    - baseline_global_mean
                )

                if (
                    abs(
                        global_advantage
                        - summary[
                            "MeanAdvantage"
                        ]
                    )
                    > FLOAT_TOL
                ):
                    raise RuntimeError(
                        "Global advantage != mean paired project advantage."
                    )

                pairwise_summary_rows.append({
                    "Metric":
                        metric,

                    "Algorithm":
                        algorithm,

                    "Comparator":
                        baseline,

                    "ComparatorRole":
                        (
                            "PRIMARY_PROPOSAL_COMPARATOR"
                            if baseline
                            == PRIMARY_COMPARATOR
                            else "ADDITIONAL_RQ_COMPARATOR"
                        ),

                    "NoisePercent":
                        noise,

                    "Projects":
                        24,

                    "MLCrossProjectMean":
                        ml_global_mean,

                    "ComparatorCrossProjectMean":
                        baseline_global_mean,

                    "GlobalAdvantageMLMinusComparator":
                        global_advantage,

                    **summary,
                })


pairwise_advantage_summary = pd.DataFrame(
    pairwise_summary_rows
)

if len(
    pairwise_advantage_summary
) != (
    2
    * EXPECTED_PAIRWISE_SUMMARY_ROWS
):
    raise RuntimeError(
        "Pairwise advantage summary row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 10. STRONGEST BASELINE AT EACH NOISE LEVEL
# --------------------------------------------------------------------------------------------------

strongest_selection_rows = []

for metric in [
    "APFDc",
    "APFD",
]:
    for noise in NOISE_LEVELS:
        block = baseline_curves.loc[
            baseline_curves[
                "Metric"
            ].eq(
                metric
            )
            & baseline_curves[
                "NoisePercent"
            ].eq(
                noise
            )
        ].copy()

        if len(block) != 3:
            raise RuntimeError(
                "Strongest-baseline selection block does not contain 3 baselines."
            )

        maximum = float(
            block[
                "CrossProjectMean"
            ].max()
        )

        winners = block.loc[
            np.isclose(
                block[
                    "CrossProjectMean"
                ].to_numpy(dtype=float),
                maximum,
                atol=FLOAT_TOL,
                rtol=0.0,
            )
        ].copy()

        # Deterministic tie handling for the selected comparator used in supporting project counts.
        # If cross-project means are tied within tolerance, choose lexicographically by frozen BASELINES order.
        baseline_order = {
            baseline:
                index
            for index, baseline
            in enumerate(
                BASELINES
            )
        }

        winners[
            "__order"
        ] = winners[
            "Technique"
        ].map(
            baseline_order
        )

        winners = winners.sort_values(
            "__order",
            kind="mergesort",
        )

        selected_baseline = str(
            winners.iloc[
                0
            ][
                "Technique"
            ]
        )

        strongest_selection_rows.append({
            "Metric":
                metric,

            "NoisePercent":
                noise,

            "StrongestBaseline":
                selected_baseline,

            "StrongestBaselineCrossProjectMean":
                maximum,

            "TiedStrongestBaselines":
                int(
                    len(
                        winners
                    )
                ),

            "TiedStrongestBaselineNamesJSON":
                json.dumps(
                    winners[
                        "Technique"
                    ].astype(
                        str
                    ).tolist(),
                    separators=(",", ":"),
                ),
        })


strongest_baseline_selection = pd.DataFrame(
    strongest_selection_rows
)

if len(
    strongest_baseline_selection
) != 18:
    raise RuntimeError(
        "Strongest baseline selection must contain 18 rows."
    )


# --------------------------------------------------------------------------------------------------
# 11. PROJECT-LEVEL ADVANTAGES VS STRONGEST BASELINE
# --------------------------------------------------------------------------------------------------

strongest_project_rows = []
strongest_summary_rows = []

for metric, project_column in [
    (
        "APFDc",
        "MeanSeedMeanAPFDc",
    ),
    (
        "APFD",
        "MeanSeedMeanAPFD",
    ),
]:
    for algorithm in ML_TECHNIQUES:
        for noise in NOISE_LEVELS:
            selection = strongest_baseline_selection.loc[
                strongest_baseline_selection[
                    "Metric"
                ].eq(
                    metric
                )
                & strongest_baseline_selection[
                    "NoisePercent"
                ].eq(
                    noise
                )
            ]

            if len(selection) != 1:
                raise RuntimeError(
                    "Could not resolve exactly one strongest-baseline selection."
                )

            baseline = str(
                selection.iloc[
                    0
                ][
                    "StrongestBaseline"
                ]
            )

            baseline_global_mean = float(
                selection.iloc[
                    0
                ][
                    "StrongestBaselineCrossProjectMean"
                ]
            )

            ml_block = project_master.loc[
                project_master[
                    "Technique"
                ].astype(
                    str
                ).eq(
                    algorithm
                )
                & project_master[
                    "NoisePercent"
                ].eq(
                    noise
                ),
                [
                    "ProjectNumber",
                    "Project",
                    project_column,
                ],
            ].copy()

            baseline_block = project_master.loc[
                project_master[
                    "Technique"
                ].astype(
                    str
                ).eq(
                    baseline
                )
                & project_master[
                    "NoisePercent"
                ].eq(
                    noise
                ),
                [
                    "ProjectNumber",
                    project_column,
                ],
            ].copy()

            ml_block = ml_block.rename(
                columns={
                    project_column:
                        "MLValue",
                }
            )

            baseline_block = baseline_block.rename(
                columns={
                    project_column:
                        "BaselineValue",
                }
            )

            paired = ml_block.merge(
                baseline_block,
                on="ProjectNumber",
                how="inner",
                validate="one_to_one",
            )

            if len(paired) != 24:
                raise RuntimeError(
                    "Strongest-baseline project pairing does not contain 24 rows."
                )

            paired[
                "AdvantageMLMinusBaseline"
            ] = (
                paired[
                    "MLValue"
                ]
                - paired[
                    "BaselineValue"
                ]
            )

            for row in paired.itertuples(
                index=False
            ):
                strongest_project_rows.append({
                    "Metric":
                        metric,

                    "Algorithm":
                        algorithm,

                    "Comparator":
                        STRONGEST_BASELINE_LABEL,

                    "SelectedBaselineTechnique":
                        baseline,

                    "NoisePercent":
                        noise,

                    "ProjectNumber":
                        int(
                            row.ProjectNumber
                        ),

                    "Project":
                        row.Project,

                    "MLValue":
                        float(
                            row.MLValue
                        ),

                    "BaselineValue":
                        float(
                            row.BaselineValue
                        ),

                    "AdvantageMLMinusBaseline":
                        float(
                            row.AdvantageMLMinusBaseline
                        ),

                    "Outcome":
                        classify_advantage(
                            row.AdvantageMLMinusBaseline
                        ),
                })

            summary = summarize_project_advantages(
                paired.assign(
                    AdvantageMLMinusBaseline=(
                        paired[
                            "MLValue"
                        ]
                        - paired[
                            "BaselineValue"
                        ]
                    )
                ),
                "AdvantageMLMinusBaseline",
            )

            ml_global_mean = float(
                np.mean(
                    pd.to_numeric(
                        paired[
                            "MLValue"
                        ],
                        errors="raise",
                    )
                )
            )

            global_advantage = (
                ml_global_mean
                - baseline_global_mean
            )

            if (
                abs(
                    global_advantage
                    - summary[
                        "MeanAdvantage"
                    ]
                )
                > FLOAT_TOL
            ):
                raise RuntimeError(
                    "Strongest-baseline global advantage identity failed."
                )

            strongest_summary_rows.append({
                "Metric":
                    metric,

                "Algorithm":
                    algorithm,

                "Comparator":
                    STRONGEST_BASELINE_LABEL,

                "ComparatorRole":
                    "CONSERVATIVE_SUPPLEMENTARY",

                "SelectedBaselineTechnique":
                    baseline,

                "NoisePercent":
                    noise,

                "Projects":
                    24,

                "MLCrossProjectMean":
                    ml_global_mean,

                "ComparatorCrossProjectMean":
                    baseline_global_mean,

                "GlobalAdvantageMLMinusComparator":
                    global_advantage,

                **summary,
            })


strongest_project_advantages = pd.DataFrame(
    strongest_project_rows
)

strongest_advantage_summary = pd.DataFrame(
    strongest_summary_rows
)

if len(
    strongest_project_advantages
) != (
    2
    * EXPECTED_STRONGEST_PROJECT_ROWS
):
    raise RuntimeError(
        "Strongest-baseline project advantages row count mismatch."
    )

if len(
    strongest_advantage_summary
) != (
    2
    * EXPECTED_STRONGEST_SUMMARY_ROWS
):
    raise RuntimeError(
        "Strongest-baseline advantage summary row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 12. BUILD FROZEN RQ2 OBSERVED CROSSOVER THRESHOLDS
# --------------------------------------------------------------------------------------------------

def build_threshold_table(metric):
    rows = []

    named_summary = pairwise_advantage_summary.loc[
        pairwise_advantage_summary[
            "Metric"
        ].eq(
            metric
        )
    ]

    strongest_summary = strongest_advantage_summary.loc[
        strongest_advantage_summary[
            "Metric"
        ].eq(
            metric
        )
    ]

    for algorithm in ML_TECHNIQUES:
        for comparator in (
            BASELINES
            + [
                STRONGEST_BASELINE_LABEL
            ]
        ):
            if comparator == STRONGEST_BASELINE_LABEL:
                block = strongest_summary.loc[
                    strongest_summary[
                        "Algorithm"
                    ].eq(
                        algorithm
                    )
                ].copy()

                comparator_role = (
                    "CONSERVATIVE_SUPPLEMENTARY"
                )

            else:
                block = named_summary.loc[
                    named_summary[
                        "Algorithm"
                    ].eq(
                        algorithm
                    )
                    & named_summary[
                        "Comparator"
                    ].eq(
                        comparator
                    )
                ].copy()

                comparator_role = (
                    "PRIMARY_PROPOSAL_COMPARATOR"
                    if comparator
                    == PRIMARY_COMPARATOR
                    else "ADDITIONAL_RQ_COMPARATOR"
                )

            if len(
                block
            ) != 9:
                raise RuntimeError(
                    f"{metric} {algorithm} vs {comparator}: threshold curve !=9 rows."
                )

            threshold = threshold_from_advantage_curve(
                block,
                "GlobalAdvantageMLMinusComparator",
            )

            first_noise = threshold[
                "FirstObservedCrossoverNoisePercent"
            ]

            sustained_noise = threshold[
                "SustainedCrossoverNoisePercent"
            ]

            first_row = None

            if (
                first_noise
                != ""
            ):
                selected = block.loc[
                    block[
                        "NoisePercent"
                    ].eq(
                        int(
                            first_noise
                        )
                    )
                ]

                if len(
                    selected
                ) != 1:
                    raise RuntimeError(
                        "First crossover row resolution failed."
                    )

                first_row = selected.iloc[
                    0
                ]

            sustained_row = None

            if (
                sustained_noise
                != ""
            ):
                selected = block.loc[
                    block[
                        "NoisePercent"
                    ].eq(
                        int(
                            sustained_noise
                        )
                    )
                ]

                if len(
                    selected
                ) != 1:
                    raise RuntimeError(
                        "Sustained crossover row resolution failed."
                    )

                sustained_row = selected.iloc[
                    0
                ]

            rows.append({
                "Metric":
                    metric,

                "MetricRole":
                    (
                        "PRIMARY"
                        if metric
                        == "APFDc"
                        else "SECONDARY"
                    ),

                "Algorithm":
                    algorithm,

                "Comparator":
                    comparator,

                "ComparatorRole":
                    comparator_role,

                "CleanAdvantage":
                    threshold[
                        "CleanAdvantage"
                    ],

                "FirstObservedCrossoverNoisePercent":
                    first_noise,

                "FirstObservedCrossoverStatus":
                    threshold[
                        "FirstObservedCrossoverStatus"
                    ],

                "FirstObservedCrossoverAdvantage":
                    (
                        ""
                        if first_row
                        is None
                        else float(
                            first_row[
                                "GlobalAdvantageMLMinusComparator"
                            ]
                        )
                    ),

                "FirstObservedCrossoverProjectWins":
                    (
                        ""
                        if first_row
                        is None
                        else int(
                            first_row[
                                "ProjectWins"
                            ]
                        )
                    ),

                "FirstObservedCrossoverProjectTies":
                    (
                        ""
                        if first_row
                        is None
                        else int(
                            first_row[
                                "ProjectTies"
                            ]
                        )
                    ),

                "FirstObservedCrossoverProjectLosses":
                    (
                        ""
                        if first_row
                        is None
                        else int(
                            first_row[
                                "ProjectLosses"
                            ]
                        )
                    ),

                "SustainedCrossoverNoisePercent":
                    sustained_noise,

                "SustainedCrossoverStatus":
                    threshold[
                        "SustainedCrossoverStatus"
                    ],

                "SustainedCrossoverAdvantage":
                    (
                        ""
                        if sustained_row
                        is None
                        else float(
                            sustained_row[
                                "GlobalAdvantageMLMinusComparator"
                            ]
                        )
                    ),

                "InterpolationAllowed":
                    False,

                "ThresholdGrid":
                    json.dumps(
                        NOISE_LEVELS,
                        separators=(",", ":"),
                    ),

                "NoCrossoverReportingRule":
                    ">50 / NOT_OBSERVED_IN_TESTED_RANGE",
            })

    return pd.DataFrame(
        rows
    )


apfdc_thresholds = build_threshold_table(
    "APFDc"
)

apfd_thresholds = build_threshold_table(
    "APFD"
)

if len(
    apfdc_thresholds
) != EXPECTED_THRESHOLD_ROWS:
    raise RuntimeError(
        "APFDc threshold table must contain 16 rows."
    )

if len(
    apfd_thresholds
) != EXPECTED_THRESHOLD_ROWS:
    raise RuntimeError(
        "APFD threshold table must contain 16 rows."
    )


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION
# --------------------------------------------------------------------------------------------------

registry_sha_after = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Step-1B checkpoint SHA-256",
    EXPECTED_STEP1B_CHECKPOINT_SHA256,
    actual_step1b_sha,
    actual_step1b_sha
    == EXPECTED_STEP1B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-2 checkpoint SHA-256",
    EXPECTED_STEP2_CHECKPOINT_SHA256,
    actual_step2_sha,
    actual_step2_sha
    == EXPECTED_STEP2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ1 Step-3C checkpoint SHA-256",
    EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,
    actual_rq1_step3c_sha,
    actual_rq1_step3c_sha
    == EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Frozen RQ2 threshold-plan rows",
    16,
    len(
        rq2_plan
    ),
    len(
        rq2_plan
    )
    == 16,
)

add_check(
    checks,
    "All-technique curve rows",
    126,
    len(
        all_technique_curves
    ),
    len(
        all_technique_curves
    )
    == 126,
)

add_check(
    checks,
    "Baseline curve rows",
    54,
    len(
        baseline_curves
    ),
    len(
        baseline_curves
    )
    == 54,
)

add_check(
    checks,
    "Baseline invariance rows",
    6,
    len(
        baseline_invariance
    ),
    len(
        baseline_invariance
    )
    == 6,
)

add_check(
    checks,
    "Named pairwise project-advantage rows",
    2 * EXPECTED_PAIRWISE_PROJECT_ROWS,
    len(
        pairwise_project_advantages
    ),
    len(
        pairwise_project_advantages
    )
    == (
        2
        * EXPECTED_PAIRWISE_PROJECT_ROWS
    ),
)

add_check(
    checks,
    "Named pairwise summary rows",
    2 * EXPECTED_PAIRWISE_SUMMARY_ROWS,
    len(
        pairwise_advantage_summary
    ),
    len(
        pairwise_advantage_summary
    )
    == (
        2
        * EXPECTED_PAIRWISE_SUMMARY_ROWS
    ),
)

add_check(
    checks,
    "Strongest-baseline selection rows",
    18,
    len(
        strongest_baseline_selection
    ),
    len(
        strongest_baseline_selection
    )
    == 18,
)

add_check(
    checks,
    "Strongest-baseline project rows",
    2 * EXPECTED_STRONGEST_PROJECT_ROWS,
    len(
        strongest_project_advantages
    ),
    len(
        strongest_project_advantages
    )
    == (
        2
        * EXPECTED_STRONGEST_PROJECT_ROWS
    ),
)

add_check(
    checks,
    "Strongest-baseline summary rows",
    2 * EXPECTED_STRONGEST_SUMMARY_ROWS,
    len(
        strongest_advantage_summary
    ),
    len(
        strongest_advantage_summary
    )
    == (
        2
        * EXPECTED_STRONGEST_SUMMARY_ROWS
    ),
)

add_check(
    checks,
    "APFDc threshold rows",
    16,
    len(
        apfdc_thresholds
    ),
    len(
        apfdc_thresholds
    )
    == 16,
)

add_check(
    checks,
    "APFD secondary threshold rows",
    16,
    len(
        apfd_thresholds
    ),
    len(
        apfd_thresholds
    )
    == 16,
)

add_check(
    checks,
    "All APFDc interpolation flags false",
    True,
    bool(
        (
            ~apfdc_thresholds[
                "InterpolationAllowed"
            ].astype(
                bool
            )
        ).all()
    ),
    bool(
        (
            ~apfdc_thresholds[
                "InterpolationAllowed"
            ].astype(
                bool
            )
        ).all()
    ),
)

add_check(
    checks,
    "All APFD interpolation flags false",
    True,
    bool(
        (
            ~apfd_thresholds[
                "InterpolationAllowed"
            ].astype(
                bool
            )
        ).all()
    ),
    bool(
        (
            ~apfd_thresholds[
                "InterpolationAllowed"
            ].astype(
                bool
            )
        ).all()
    ),
)

add_check(
    checks,
    "Named pairwise direction counts sum to 24",
    True,
    bool(
        (
            pairwise_advantage_summary[
                "ProjectWins"
            ]
            + pairwise_advantage_summary[
                "ProjectTies"
            ]
            + pairwise_advantage_summary[
                "ProjectLosses"
            ]
        ).eq(
            24
        ).all()
    ),
    bool(
        (
            pairwise_advantage_summary[
                "ProjectWins"
            ]
            + pairwise_advantage_summary[
                "ProjectTies"
            ]
            + pairwise_advantage_summary[
                "ProjectLosses"
            ]
        ).eq(
            24
        ).all()
    ),
)

add_check(
    checks,
    "Strongest-baseline direction counts sum to 24",
    True,
    bool(
        (
            strongest_advantage_summary[
                "ProjectWins"
            ]
            + strongest_advantage_summary[
                "ProjectTies"
            ]
            + strongest_advantage_summary[
                "ProjectLosses"
            ]
        ).eq(
            24
        ).all()
    ),
    bool(
        (
            strongest_advantage_summary[
                "ProjectWins"
            ]
            + strongest_advantage_summary[
                "ProjectTies"
            ]
            + strongest_advantage_summary[
                "ProjectLosses"
            ]
        ).eq(
            24
        ).all()
    ),
)

add_check(
    checks,
    "Formal RQ2 hypothesis tests executed",
    0,
    0,
    True,
)

add_check(
    checks,
    "RQ2 p-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "RQ3 executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ2 Step 4A pre-write validation:"
)

try:
    from IPython.display import display
    display(validation)
except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ2 STEP 4A PRE-WRITE VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_csv(
    ALL_TECHNIQUE_CURVES_PATH,
    all_technique_curves,
)

atomic_csv(
    BASELINE_CURVES_PATH,
    baseline_curves,
)

atomic_csv(
    BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_csv(
    PAIRWISE_PROJECT_ADVANTAGES_PATH,
    pairwise_project_advantages,
)

atomic_csv(
    PAIRWISE_ADVANTAGE_SUMMARY_PATH,
    pairwise_advantage_summary,
)

atomic_csv(
    STRONGEST_BASELINE_SELECTION_PATH,
    strongest_baseline_selection,
)

atomic_csv(
    STRONGEST_PROJECT_ADVANTAGES_PATH,
    strongest_project_advantages,
)

atomic_csv(
    STRONGEST_ADVANTAGE_SUMMARY_PATH,
    strongest_advantage_summary,
)

atomic_csv(
    APFDC_THRESHOLDS_PATH,
    apfdc_thresholds,
)

atomic_csv(
    APFD_THRESHOLDS_PATH,
    apfd_thresholds,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK AUDIT
# --------------------------------------------------------------------------------------------------

readback_rows = []

for dataset_name, path, expected in [
    (
        "AllTechniqueCurves",
        ALL_TECHNIQUE_CURVES_PATH,
        all_technique_curves,
    ),
    (
        "PairwiseAdvantageSummary",
        PAIRWISE_ADVANTAGE_SUMMARY_PATH,
        pairwise_advantage_summary,
    ),
    (
        "StrongestBaselineSelection",
        STRONGEST_BASELINE_SELECTION_PATH,
        strongest_baseline_selection,
    ),
    (
        "APFDcThresholds",
        APFDC_THRESHOLDS_PATH,
        apfdc_thresholds,
    ),
    (
        "APFDThresholds",
        APFD_THRESHOLDS_PATH,
        apfd_thresholds,
    ),
]:
    actual = pd.read_csv(
        path,
        low_memory=False,
    )

    actual = actual[
        expected.columns.tolist()
    ]

    passed, max_difference = compare_csv_roundtrip(
        expected,
        actual,
    )

    readback_rows.append({
        "Dataset":
            dataset_name,

        "ExpectedRows":
            len(expected),

        "ReadbackRows":
            len(actual),

        "ReadbackPass":
            passed,

        "MaxAbsFloatDifference":
            max_difference,

        "Tolerance":
            FLOAT_TOL,
    })


readback_audit = pd.DataFrame(
    readback_rows
)

readback_failures = int(
    (
        ~readback_audit[
            "ReadbackPass"
        ].astype(
            bool
        )
    ).sum()
)

atomic_csv(
    READBACK_AUDIT_PATH,
    readback_audit,
)

if readback_failures != 0:
    raise RuntimeError(
        "RQ2 STEP 4A V2 READBACK AUDIT FAILED.\n"
        + readback_audit.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 16. REPORT / STATUS / MANIFEST / CHECKPOINT
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

apfdc_thresholds_scientific_sha = scientific_hash(
    apfdc_thresholds
)

pairwise_summary_scientific_sha = scientific_hash(
    pairwise_advantage_summary
)

report = {
    "Step":
        "RQ2_STEP_4A",

    "Status":
        RQ2_STEP4A_STATUS,

    "CodeRevision":
        RQ2_STEP4A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "RQ1Step3CCheckpointSHA256":
        EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PrimaryComparator":
        PRIMARY_COMPARATOR,

    "NamedComparators":
        BASELINES,

    "StrongestBaselineComparator":
        STRONGEST_BASELINE_LABEL,

    "InterpolationAllowed":
        False,

    "AllTechniqueCurveRows":
        len(
            all_technique_curves
        ),

    "PairwiseProjectAdvantageRows":
        len(
            pairwise_project_advantages
        ),

    "PairwiseAdvantageSummaryRows":
        len(
            pairwise_advantage_summary
        ),

    "StrongestBaselineSelectionRows":
        len(
            strongest_baseline_selection
        ),

    "APFDcThresholdRows":
        len(
            apfdc_thresholds
        ),

    "APFDThresholdRows":
        len(
            apfd_thresholds
        ),

    "APFDcThresholdsPath":
        str(
            APFDC_THRESHOLDS_PATH
        ),

    "APFDcThresholdsSHA256":
        sha256_file(
            APFDC_THRESHOLDS_PATH
        ),

    "APFDcThresholdsScientificSHA256":
        apfdc_thresholds_scientific_sha,

    "PairwiseAdvantageSummaryPath":
        str(
            PAIRWISE_ADVANTAGE_SUMMARY_PATH
        ),

    "PairwiseAdvantageSummarySHA256":
        sha256_file(
            PAIRWISE_ADVANTAGE_SUMMARY_PATH
        ),

    "PairwiseAdvantageSummaryScientificSHA256":
        pairwise_summary_scientific_sha,

    "StrongestBaselineSelectionPath":
        str(
            STRONGEST_BASELINE_SELECTION_PATH
        ),

    "StrongestBaselineSelectionSHA256":
        sha256_file(
            STRONGEST_BASELINE_SELECTION_PATH
        ),

    "StrongestAdvantageSummaryPath":
        str(
            STRONGEST_ADVANTAGE_SUMMARY_PATH
        ),

    "StrongestAdvantageSummarySHA256":
        sha256_file(
            STRONGEST_ADVANTAGE_SUMMARY_PATH
        ),

    "FormalHypothesisTestsExecuted":
        0,

    "PValuesComputed":
        False,

    "ReadbackValidationMethod":
        (
            "semantic CSV round-trip equivalence: exact structure/missingness/text/booleans; "
            "numeric-like mixed columns compared numerically within <=1e-12"
        ),

    "ReadbackMaxAbsFloatDifference":
        float(
            readback_audit[
                "MaxAbsFloatDifference"
            ].max()
        ),

    "RQ3Executed":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "RQ2 STEP 4B — GENERATE RQ2 CROSSOVER FIGURES, SYNTHESIS TABLES, "
            "AND FREEZE THE RQ2 ANSWER PACKAGE"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ2_STEP4A_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RQ2ObservedCrossoversFrozen":
            True,

        "ReadyForRQ2Step4B":
            True,
    },
)

output_paths = [
    ALL_TECHNIQUE_CURVES_PATH,
    BASELINE_CURVES_PATH,
    BASELINE_INVARIANCE_PATH,
    PAIRWISE_PROJECT_ADVANTAGES_PATH,
    PAIRWISE_ADVANTAGE_SUMMARY_PATH,
    STRONGEST_BASELINE_SELECTION_PATH,
    STRONGEST_PROJECT_ADVANTAGES_PATH,
    STRONGEST_ADVANTAGE_SUMMARY_PATH,
    APFDC_THRESHOLDS_PATH,
    APFD_THRESHOLDS_PATH,
    READBACK_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

if (
    sha256_file(
        REGISTRY
    )
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry changed during RQ2 Step 4A."
    )

checkpoint = {
    "CheckpointType":
        "RQ2_BASELINE_CURVES_ADVANTAGES_AND_OBSERVED_CROSSOVERS",

    "Status":
        RQ2_STEP4A_STATUS,

    "CodeRevision":
        RQ2_STEP4A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "RQ1Step3CCheckpointSHA256":
        EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "PrimaryComparator":
        PRIMARY_COMPARATOR,

    "InterpolationAllowed":
        False,

    "APFDcThresholdsPath":
        str(
            APFDC_THRESHOLDS_PATH
        ),

    "APFDcThresholdsSHA256":
        sha256_file(
            APFDC_THRESHOLDS_PATH
        ),

    "APFDcThresholdsScientificSHA256":
        apfdc_thresholds_scientific_sha,

    "APFDThresholdsPath":
        str(
            APFD_THRESHOLDS_PATH
        ),

    "APFDThresholdsSHA256":
        sha256_file(
            APFD_THRESHOLDS_PATH
        ),

    "AllTechniqueCurvesPath":
        str(
            ALL_TECHNIQUE_CURVES_PATH
        ),

    "AllTechniqueCurvesSHA256":
        sha256_file(
            ALL_TECHNIQUE_CURVES_PATH
        ),

    "PairwiseAdvantageSummaryPath":
        str(
            PAIRWISE_ADVANTAGE_SUMMARY_PATH
        ),

    "PairwiseAdvantageSummarySHA256":
        sha256_file(
            PAIRWISE_ADVANTAGE_SUMMARY_PATH
        ),

    "StrongestBaselineSelectionPath":
        str(
            STRONGEST_BASELINE_SELECTION_PATH
        ),

    "StrongestBaselineSelectionSHA256":
        sha256_file(
            STRONGEST_BASELINE_SELECTION_PATH
        ),

    "StrongestAdvantageSummaryPath":
        str(
            STRONGEST_ADVANTAGE_SUMMARY_PATH
        ),

    "StrongestAdvantageSummarySHA256":
        sha256_file(
            STRONGEST_ADVANTAGE_SUMMARY_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "FormalHypothesisTestsExecuted":
        0,

    "PValuesComputed":
        False,

    "RQ3Executed":
        False,

    "ReadyForRQ2Step4B":
        True,

    "NextRequiredStep":
        (
            "RQ2 STEP 4B — GENERATE RQ2 CROSSOVER FIGURES, SYNTHESIS TABLES, "
            "AND FREEZE THE RQ2 ANSWER PACKAGE"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 17. USER-VISIBLE PRIMARY APFDC TABLES
# --------------------------------------------------------------------------------------------------

print(
    "\nRQ2 PRIMARY APFDc observed crossover thresholds:"
)

primary_display = apfdc_thresholds[
    [
        "Algorithm",
        "Comparator",
        "ComparatorRole",
        "CleanAdvantage",
        "FirstObservedCrossoverNoisePercent",
        "FirstObservedCrossoverStatus",
        "FirstObservedCrossoverAdvantage",
        "FirstObservedCrossoverProjectWins",
        "FirstObservedCrossoverProjectTies",
        "FirstObservedCrossoverProjectLosses",
        "SustainedCrossoverNoisePercent",
        "SustainedCrossoverStatus",
    ]
].copy()

try:
    from IPython.display import display
    display(primary_display)
except Exception:
    print(
        primary_display.to_string(
            index=False
        )
    )

print(
    "\nStrongest APFDc baseline by tested noise level:"
)

strongest_apfdc_display = strongest_baseline_selection.loc[
    strongest_baseline_selection[
        "Metric"
    ].eq(
        "APFDc"
    )
].copy()

try:
    from IPython.display import display
    display(strongest_apfdc_display)
except Exception:
    print(
        strongest_apfdc_display.to_string(
            index=False
        )
    )

print(
    "\nBaseline invariance diagnostic:"
)

try:
    from IPython.display import display
    display(baseline_invariance)
except Exception:
    print(
        baseline_invariance.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 18. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 152
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 8 / RQ2 STEP 4A V2 RESULT ==="
)

print(
    "=" * 152
)

print(
    "RQ2 observed crossover analysis frozen: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "\nContract:"
)

print(
    "Primary metric: APFDc"
)

print(
    "Primary comparator: LatestFail"
)

print(
    "Additional comparators: Random, QTF-Avg"
)

print(
    "Conservative supplementary comparator: strongest baseline at each noise"
)

print(
    "Interpolation allowed: False"
)

print(
    "\nFrozen outputs:"
)

print(
    "All-technique curve rows:",
    len(
        all_technique_curves
    ),
)

print(
    "Named pairwise project advantages:",
    len(
        pairwise_project_advantages
    ),
)

print(
    "Named pairwise advantage summaries:",
    len(
        pairwise_advantage_summary
    ),
)

print(
    "Strongest-baseline selections:",
    len(
        strongest_baseline_selection
    ),
)

print(
    "APFDc threshold rows:",
    len(
        apfdc_thresholds
    ),
)

print(
    "APFD secondary threshold rows:",
    len(
        apfd_thresholds
    ),
)

print(
    "\nReadback:"
)

print(
    "Readback failures:",
    readback_failures
)

print(
    "Max readback float difference:",
    float(
        readback_audit[
            "MaxAbsFloatDifference"
        ].max()
    ),
)

print(
    "\nIsolation:"
)

print(
    "Formal RQ2 hypothesis tests executed: 0"
)

print(
    "RQ2 p-values computed: False"
)

print(
    "RQ3 executed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)

print(
    "\nRQ2 Step 4A checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)

print(
    "\nNext required step: "
    "RQ2 STEP 4B — GENERATE RQ2 CROSSOVER FIGURES, SYNTHESIS TABLES, "
    "AND FREEZE THE RQ2 ANSWER PACKAGE"
)

print(
    "\nSTATUS:",
    RQ2_STEP4A_STATUS,
)

print(
    "=" * 152
)


=== THESIS GLOBAL ANALYSIS — CELL 8 / RQ2 STEP 4A: BASELINES + ADVANTAGES + OBSERVED CROSSOVERS ===

RQ2 Step 4A pre-write validation:


,Check,Expected,Actual,Pass
0,Step-1B checkpoint SHA-256,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,True
1,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
2,RQ1 Step-3C checkpoint SHA-256,b4f3d10b77acb39200496542e8256d6213981fd6363254...,b4f3d10b77acb39200496542e8256d6213981fd6363254...,True
3,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
4,Frozen RQ2 threshold-plan rows,16,16,True
5,All-technique curve rows,126,126,True
6,Baseline curve rows,54,54,True
7,Baseline invariance rows,6,6,True
8,Named pairwise project-advantage rows,5184,5184,True
9,Named pairwise summary rows,216,216,True



RQ2 PRIMARY APFDc observed crossover thresholds:


,Algorithm,Comparator,ComparatorRole,CleanAdvantage,FirstObservedCrossoverNoisePercent,FirstObservedCrossoverStatus,FirstObservedCrossoverAdvantage,FirstObservedCrossoverProjectWins,FirstObservedCrossoverProjectTies,FirstObservedCrossoverProjectLosses,SustainedCrossoverNoisePercent,SustainedCrossoverStatus
0,RandomForest,Random,ADDITIONAL_RQ_COMPARATOR,0.307975,40,OBSERVED,-0.015875,12,0,12,40,OBSERVED
1,RandomForest,LatestFail,PRIMARY_PROPOSAL_COMPARATOR,0.227982,5,OBSERVED,-0.109048,6,0,18,5,OBSERVED
2,RandomForest,QTF-Avg,ADDITIONAL_RQ_COMPARATOR,0.147678,10,OBSERVED,-0.030298,9,0,15,10,OBSERVED
3,RandomForest,StrongestBaselineAtEachNoise,CONSERVATIVE_SUPPLEMENTARY,0.147678,5,OBSERVED,-0.109048,6,0,18,5,OBSERVED
4,XGBoost,Random,ADDITIONAL_RQ_COMPARATOR,0.288953,40,OBSERVED,-0.007997,13,0,11,40,OBSERVED
5,XGBoost,LatestFail,PRIMARY_PROPOSAL_COMPARATOR,0.208960,5,OBSERVED,-0.098791,5,0,19,5,OBSERVED
6,XGBoost,QTF-Avg,ADDITIONAL_RQ_COMPARATOR,0.128656,10,OBSERVED,-0.014356,10,0,14,10,OBSERVED
7,XGBoost,StrongestBaselineAtEachNoise,CONSERVATIVE_SUPPLEMENTARY,0.128656,5,OBSERVED,-0.098791,5,0,19,5,OBSERVED
8,LightGBM,Random,ADDITIONAL_RQ_COMPARATOR,0.266796,50,OBSERVED,-0.062787,3,0,21,50,OBSERVED
9,LightGBM,LatestFail,PRIMARY_PROPOSAL_COMPARATOR,0.186803,5,OBSERVED,-0.103170,6,0,18,5,OBSERVED



Strongest APFDc baseline by tested noise level:


,Metric,NoisePercent,StrongestBaseline,StrongestBaselineCrossProjectMean,TiedStrongestBaselines,TiedStrongestBaselineNamesJSON
0,APFDc,0,QTF-Avg,0.655560,1,"[""QTF-Avg""]"
1,APFDc,5,LatestFail,0.783330,1,"[""LatestFail""]"
2,APFDc,10,LatestFail,0.783453,1,"[""LatestFail""]"
3,APFDc,15,LatestFail,0.783334,1,"[""LatestFail""]"
4,APFDc,20,LatestFail,0.782989,1,"[""LatestFail""]"
5,APFDc,25,LatestFail,0.783038,1,"[""LatestFail""]"
6,APFDc,30,LatestFail,0.782263,1,"[""LatestFail""]"
7,APFDc,40,LatestFail,0.780254,1,"[""LatestFail""]"
8,APFDc,50,LatestFail,0.779787,1,"[""LatestFail""]"



Baseline invariance diagnostic:


,Metric,Baseline,NoiseLevels,CrossProjectMeanRangeAcrossNoise,InvariantAcrossNoiseAt1e12
0,APFDc,Random,9,0.000000,True
1,APFDc,LatestFail,9,0.208197,False
2,APFDc,QTF-Avg,9,0.000000,True
3,APFD,Random,9,0.000000,True
4,APFD,LatestFail,9,0.326643,False
5,APFD,QTF-Avg,9,0.000000,True



=== THESIS GLOBAL ANALYSIS — CELL 8 / RQ2 STEP 4A V2 RESULT ===
RQ2 observed crossover analysis frozen: True

Study unit:
Independent empirical unit: Project (N=24)

Contract:
Primary metric: APFDc
Primary comparator: LatestFail
Additional comparators: Random, QTF-Avg
Conservative supplementary comparator: strongest baseline at each noise
Interpolation allowed: False

Frozen outputs:
All-technique curve rows: 126
Named pairwise project advantages: 5184
Named pairwise advantage summaries: 216
Strongest-baseline selections: 18
APFDc threshold rows: 16
APFD secondary threshold rows: 16

Readback:
Readback failures: 0
Max readback float difference: 2.220446049250313e-16

Isolation:
Formal RQ2 hypothesis tests executed: 0
RQ2 p-values computed: False
RQ3 executed: False
Project outputs modified: False
Completion registry modified: False

Validation:
Checks: 23
Failed checks: 0

RQ2 Step 4A checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/global_analysis_rq2_step4a_checkpoint.json

In [6]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 9 / RQ2 STEP 4B
# FINAL RQ2 CROSSOVER FIGURES + SYNTHESIS TABLES + ANSWER-PACKAGE FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# RQ2 Step 4A already froze:
#   - all seven technique curves;
#   - project-level ML-minus-baseline advantages;
#   - named-baseline and strongest-baseline summaries;
#   - first observed and sustained crossover thresholds;
#   - APFDc primary and APFD secondary threshold tables.
#
# This cell performs NO new hypothesis tests and computes NO p-values.
#
# It:
#   - verifies the frozen Step 4A checkpoint and all upstream hashes;
#   - creates a compact 4-algorithm APFDc crossover matrix;
#   - creates thesis-ready primary/secondary numerical tables;
#   - generates RQ2 crossover figures from already frozen Step-4A data;
#   - freezes an integrity-manifested RQ2 answer package;
#   - preserves Project (N=24) as the independent empirical unit;
#   - does NOT modify Project 1..24 outputs or the completion registry;
#   - does NOT execute RQ3.
#
# IMPORTANT
# ---------
# Threshold values are OBSERVED TESTED grid points only.
# No interpolation is performed.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


print("=" * 154)
print("=== THESIS GLOBAL ANALYSIS — CELL 9 / RQ2 STEP 4B: FINAL CROSSOVER FIGURES + SYNTHESIS PACKAGE ===")
print("=" * 154)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

RQ2_STEP4A_STATUS = (
    "PASS_RQ2_STEP4A_BASELINE_CURVES_ADVANTAGES_AND_OBSERVED_CROSSOVERS_FROZEN"
)

EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256 = (
    "e88976b9c82f952896090bc9617feffe872478bc637f7bd94466b188f3a2bcbc"
)

EXPECTED_STEP2_CHECKPOINT_SHA256 = (
    "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ2_STEP4B_STATUS = (
    "PASS_RQ2_STEP4B_FINAL_CROSSOVER_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

RQ2_STEP4B_CODE_REVISION = (
    "RQ2_STEP4B_V1_FROZEN_CROSSOVER_SYNTHESIS_FIGURES_PACKAGE_NO_NEW_INFERENCE"
)

PRIMARY_METRIC = "APFDc"
SECONDARY_METRIC = "APFD"
PRIMARY_COMPARATOR = "LatestFail"
STRONGEST_BASELINE_LABEL = "StrongestBaselineAtEachNoise"

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = [
    "LatestFail",
    "LightGBM",
    "NaiveBayes",
    "QTF-Avg",
    "Random",
    "RandomForest",
    "XGBoost",
]

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

RQ2_STEP4A_CHECKPOINT = (
    NOTES
    / "global_analysis_rq2_step4a_checkpoint.json"
)

STEP2_CHECKPOINT = (
    NOTES
    / "global_analysis_step2_checkpoint.json"
)

STEP4B_ROOT = (
    ANALYSIS_ROOT
    / "RQ2"
    / "Step_4B_Final_RQ2_Answer_Package"
)

TABLES_ROOT = (
    STEP4B_ROOT
    / "Tables"
)

FIGURES_ROOT = (
    STEP4B_ROOT
    / "Figures"
)

PRIMARY_THRESHOLD_MATRIX_PATH = (
    TABLES_ROOT
    / "rq2_primary_apfdc_threshold_matrix.csv"
)

PRIMARY_THRESHOLDS_COPY_PATH = (
    TABLES_ROOT
    / "rq2_primary_apfdc_observed_crossover_thresholds.csv"
)

SECONDARY_THRESHOLDS_COPY_PATH = (
    TABLES_ROOT
    / "rq2_secondary_apfd_observed_crossover_thresholds.csv"
)

PRIMARY_LATESTFAIL_EVIDENCE_PATH = (
    TABLES_ROOT
    / "rq2_primary_latestfail_advantage_curve.csv"
)

PRIMARY_ALL_COMPARATOR_EVIDENCE_PATH = (
    TABLES_ROOT
    / "rq2_primary_all_named_comparator_advantage_curves.csv"
)

PRIMARY_STRONGEST_EVIDENCE_PATH = (
    TABLES_ROOT
    / "rq2_primary_strongest_baseline_advantage_curve.csv"
)

STRONGEST_BASELINE_SELECTION_COPY_PATH = (
    TABLES_ROOT
    / "rq2_strongest_baseline_selection.csv"
)

BASELINE_BEHAVIOR_PATH = (
    TABLES_ROOT
    / "rq2_baseline_behavior_summary.csv"
)

ALL_TECHNIQUE_CURVES_COPY_PATH = (
    TABLES_ROOT
    / "rq2_all_technique_cross_project_curves.csv"
)

FIGURE_ALL_TECHNIQUES_APFDC_PATH = (
    FIGURES_ROOT
    / "rq2_apfdc_all_technique_curves.png"
)

FIGURE_LATESTFAIL_ADVANTAGE_PATH = (
    FIGURES_ROOT
    / "rq2_apfdc_advantage_vs_latestfail.png"
)

FIGURE_RANDOM_ADVANTAGE_PATH = (
    FIGURES_ROOT
    / "rq2_apfdc_advantage_vs_random.png"
)

FIGURE_QTF_ADVANTAGE_PATH = (
    FIGURES_ROOT
    / "rq2_apfdc_advantage_vs_qtf_avg.png"
)

FIGURE_STRONGEST_ADVANTAGE_PATH = (
    FIGURES_ROOT
    / "rq2_apfdc_advantage_vs_strongest_baseline.png"
)

FIGURE_APFD_LATESTFAIL_PATH = (
    FIGURES_ROOT
    / "rq2_apfd_secondary_advantage_vs_latestfail.png"
)

VALIDATION_PATH = (
    STEP4B_ROOT
    / "rq2_step4b_validation.csv"
)

REPORT_PATH = (
    STEP4B_ROOT
    / "rq2_step4b_report.json"
)

STATUS_PATH = (
    STEP4B_ROOT
    / "rq2_step4b_status.json"
)

PACKAGE_MANIFEST_PATH = (
    STEP4B_ROOT
    / "rq2_final_package_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq2_step4b_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def atomic_copy(
    source,
    destination,
):
    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        f".{destination.name}.tmp_{os.getpid()}"
    )

    shutil.copy2(
        source,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def build_package_manifest(
    root,
    manifest_path,
):
    root = Path(root)
    manifest_path = Path(manifest_path)

    rows = []

    for path in sorted(
        (
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
            and candidate.resolve()
            != manifest_path.resolve()
        ),
        key=lambda candidate:
            candidate.relative_to(
                root
            ).as_posix(),
    ):
        rows.append({
            "RelativePath":
                path.relative_to(
                    root
                ).as_posix(),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def package_root_hash(
    manifest,
):
    ordered = (
        manifest[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def save_figure(
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


def parse_threshold_noise(
    value,
):
    if pd.isna(value):
        return None

    text = str(value).strip()

    if text == "":
        return None

    return int(
        float(
            value
        )
    )


def threshold_label(
    row,
):
    status = str(
        row[
            "FirstObservedCrossoverStatus"
        ]
    )

    value = parse_threshold_noise(
        row[
            "FirstObservedCrossoverNoisePercent"
        ]
    )

    if status == "NO_POSITIVE_TOLERANCE_THRESHOLD":
        return "0 (no positive tolerance)"

    if status == "NOT_OBSERVED_THROUGH_50":
        return ">50 (not observed)"

    if value is None:
        raise RuntimeError(
            "Observed threshold has no numeric noise value."
        )

    return str(
        value
    )


def get_threshold_row(
    thresholds,
    algorithm,
    comparator,
):
    block = thresholds.loc[
        thresholds[
            "Algorithm"
        ].astype(
            str
        ).eq(
            algorithm
        )
        & thresholds[
            "Comparator"
        ].astype(
            str
        ).eq(
            comparator
        )
    ]

    if len(block) != 1:
        raise RuntimeError(
            f"Expected one threshold row: {algorithm} vs {comparator}"
        )

    return block.iloc[
        0
    ]


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD + VERIFY RQ2 STEP 4A
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ2 Step 4B is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ3."
    )

if not RQ2_STEP4A_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"RQ2 Step-4A checkpoint missing: {RQ2_STEP4A_CHECKPOINT}"
    )

actual_step4a_sha = sha256_file(
    RQ2_STEP4A_CHECKPOINT
)

if (
    actual_step4a_sha
    != EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ2 Step-4A checkpoint SHA mismatch."
    )

step4a = load_json(
    RQ2_STEP4A_CHECKPOINT
)

if step4a.get(
    "Status"
) != RQ2_STEP4A_STATUS:
    raise RuntimeError(
        "RQ2 Step-4A checkpoint status is not PASS."
    )

if not bool(
    step4a.get(
        "ReadyForRQ2Step4B",
        False,
    )
):
    raise RuntimeError(
        "RQ2 Step-4A checkpoint is not marked ready for Step 4B."
    )

if not STEP2_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Step-2 checkpoint missing: {STEP2_CHECKPOINT}"
    )

actual_step2_sha = sha256_file(
    STEP2_CHECKPOINT
)

if (
    actual_step2_sha
    != EXPECTED_STEP2_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-2 checkpoint SHA mismatch."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if (
    registry_sha_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

if STEP4B_ROOT.exists():
    shutil.rmtree(
        STEP4B_ROOT,
        ignore_errors=True,
    )

TABLES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY + LOAD ALL FROZEN STEP-4A INPUTS
# --------------------------------------------------------------------------------------------------

apfdc_thresholds_path = Path(
    step4a[
        "APFDcThresholdsPath"
    ]
)

apfd_thresholds_path = Path(
    step4a[
        "APFDThresholdsPath"
    ]
)

all_technique_curves_path = Path(
    step4a[
        "AllTechniqueCurvesPath"
    ]
)

pairwise_summary_path = Path(
    step4a[
        "PairwiseAdvantageSummaryPath"
    ]
)

strongest_selection_path = Path(
    step4a[
        "StrongestBaselineSelectionPath"
    ]
)

strongest_summary_path = Path(
    step4a[
        "StrongestAdvantageSummaryPath"
    ]
)

input_specs = [
    (
        apfdc_thresholds_path,
        str(
            step4a[
                "APFDcThresholdsSHA256"
            ]
        ).lower(),
    ),
    (
        apfd_thresholds_path,
        str(
            step4a[
                "APFDThresholdsSHA256"
            ]
        ).lower(),
    ),
    (
        all_technique_curves_path,
        str(
            step4a[
                "AllTechniqueCurvesSHA256"
            ]
        ).lower(),
    ),
    (
        pairwise_summary_path,
        str(
            step4a[
                "PairwiseAdvantageSummarySHA256"
            ]
        ).lower(),
    ),
    (
        strongest_selection_path,
        str(
            step4a[
                "StrongestBaselineSelectionSHA256"
            ]
        ).lower(),
    ),
    (
        strongest_summary_path,
        str(
            step4a[
                "StrongestAdvantageSummarySHA256"
            ]
        ).lower(),
    ),
]

for path, expected_sha in input_specs:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ2 Step-4A input missing: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen RQ2 Step-4A input SHA mismatch: {path.name}"
        )


apfdc_thresholds = pd.read_csv(
    apfdc_thresholds_path,
    low_memory=False,
)

apfd_thresholds = pd.read_csv(
    apfd_thresholds_path,
    low_memory=False,
)

all_technique_curves = pd.read_csv(
    all_technique_curves_path,
    low_memory=False,
)

pairwise_summary = pd.read_csv(
    pairwise_summary_path,
    low_memory=False,
)

strongest_selection = pd.read_csv(
    strongest_selection_path,
    low_memory=False,
)

strongest_summary = pd.read_csv(
    strongest_summary_path,
    low_memory=False,
)

if len(
    apfdc_thresholds
) != 16:
    raise RuntimeError(
        "Primary APFDc threshold table must contain 16 rows."
    )

if len(
    apfd_thresholds
) != 16:
    raise RuntimeError(
        "Secondary APFD threshold table must contain 16 rows."
    )

if len(
    all_technique_curves
) != 126:
    raise RuntimeError(
        "All-technique curve table must contain 126 rows."
    )

if len(
    pairwise_summary
) != 216:
    raise RuntimeError(
        "Pairwise summary table must contain 216 rows."
    )

if len(
    strongest_selection
) != 18:
    raise RuntimeError(
        "Strongest baseline selection must contain 18 rows."
    )

if len(
    strongest_summary
) != 72:
    raise RuntimeError(
        "Strongest baseline summary must contain 72 rows."
    )


# --------------------------------------------------------------------------------------------------
# 6. BUILD THE PRIMARY 4-ALGORITHM THRESHOLD MATRIX
# --------------------------------------------------------------------------------------------------

matrix_rows = []

for algorithm in ML_TECHNIQUES:
    row = {
        "Algorithm":
            algorithm,
    }

    for comparator, column_prefix in [
        (
            PRIMARY_COMPARATOR,
            "LatestFail",
        ),
        (
            "Random",
            "Random",
        ),
        (
            "QTF-Avg",
            "QTFAvg",
        ),
        (
            STRONGEST_BASELINE_LABEL,
            "StrongestBaseline",
        ),
    ]:
        threshold_row = get_threshold_row(
            apfdc_thresholds,
            algorithm,
            comparator,
        )

        row[
            f"{column_prefix}CleanAdvantage"
        ] = float(
            threshold_row[
                "CleanAdvantage"
            ]
        )

        row[
            f"{column_prefix}FirstObservedCrossover"
        ] = threshold_label(
            threshold_row
        )

        row[
            f"{column_prefix}FirstObservedCrossoverStatus"
        ] = str(
            threshold_row[
                "FirstObservedCrossoverStatus"
            ]
        )

        row[
            f"{column_prefix}SustainedCrossover"
        ] = (
            str(
                parse_threshold_noise(
                    threshold_row[
                        "SustainedCrossoverNoisePercent"
                    ]
                )
            )
            if parse_threshold_noise(
                threshold_row[
                    "SustainedCrossoverNoisePercent"
                ]
            )
            is not None
            else ">50 (not observed)"
        )

        row[
            f"{column_prefix}SustainedCrossoverStatus"
        ] = str(
            threshold_row[
                "SustainedCrossoverStatus"
            ]
        )

    matrix_rows.append(
        row
    )


primary_threshold_matrix = pd.DataFrame(
    matrix_rows
)

if len(
    primary_threshold_matrix
) != 4:
    raise RuntimeError(
        "Primary threshold matrix must contain 4 rows."
    )


# --------------------------------------------------------------------------------------------------
# 7. PRIMARY LATESTFAIL / NAMED-BASELINE / STRONGEST-BASELINE EVIDENCE TABLES
# --------------------------------------------------------------------------------------------------

primary_named = pairwise_summary.loc[
    pairwise_summary[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
].copy()

if len(
    primary_named
) != 108:
    raise RuntimeError(
        "Primary named-baseline summary must contain 108 rows."
    )

primary_latestfail = primary_named.loc[
    primary_named[
        "Comparator"
    ].astype(
        str
    ).eq(
        "LatestFail"
    )
].copy()

if len(
    primary_latestfail
) != 36:
    raise RuntimeError(
        "Primary LatestFail evidence must contain 36 rows."
    )

primary_strongest = strongest_summary.loc[
    strongest_summary[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
].copy()

if len(
    primary_strongest
) != 36:
    raise RuntimeError(
        "Primary strongest-baseline evidence must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 8. BASELINE BEHAVIOR SUMMARY FROM FROZEN CURVES
# --------------------------------------------------------------------------------------------------

baseline_behavior_rows = []

for metric in [
    "APFDc",
    "APFD",
]:
    for baseline in BASELINES:
        block = (
            all_technique_curves.loc[
                all_technique_curves[
                    "Metric"
                ].astype(
                    str
                ).eq(
                    metric
                )
                & all_technique_curves[
                    "Technique"
                ].astype(
                    str
                ).eq(
                    baseline
                )
            ]
            .sort_values(
                "NoisePercent",
                kind="mergesort",
            )
        )

        if len(
            block
        ) != 9:
            raise RuntimeError(
                f"{metric} {baseline}: expected 9 curve rows."
            )

        means = pd.to_numeric(
            block[
                "CrossProjectMean"
            ],
            errors="raise",
        ).to_numpy(
            dtype=float
        )

        baseline_behavior_rows.append({
            "Metric":
                metric,

            "Baseline":
                baseline,

            "MeanAt0":
                float(
                    means[
                        0
                    ]
                ),

            "MeanAt5":
                float(
                    means[
                        1
                    ]
                ),

            "MeanAt50":
                float(
                    means[
                        -1
                    ]
                ),

            "RangeAcrossNoise":
                float(
                    np.max(
                        means
                    )
                    - np.min(
                        means
                    )
                ),

            "InvariantAcrossNoiseAt1e12":
                bool(
                    (
                        np.max(
                            means
                        )
                        - np.min(
                            means
                        )
                    )
                    <= FLOAT_TOL
                ),
        })


baseline_behavior = pd.DataFrame(
    baseline_behavior_rows
)


# --------------------------------------------------------------------------------------------------
# 9. WRITE TABLES / FROZEN COPIES
# --------------------------------------------------------------------------------------------------

atomic_csv(
    PRIMARY_THRESHOLD_MATRIX_PATH,
    primary_threshold_matrix,
)

atomic_copy(
    apfdc_thresholds_path,
    PRIMARY_THRESHOLDS_COPY_PATH,
)

atomic_copy(
    apfd_thresholds_path,
    SECONDARY_THRESHOLDS_COPY_PATH,
)

atomic_csv(
    PRIMARY_LATESTFAIL_EVIDENCE_PATH,
    primary_latestfail,
)

atomic_csv(
    PRIMARY_ALL_COMPARATOR_EVIDENCE_PATH,
    primary_named,
)

atomic_csv(
    PRIMARY_STRONGEST_EVIDENCE_PATH,
    primary_strongest,
)

atomic_copy(
    strongest_selection_path,
    STRONGEST_BASELINE_SELECTION_COPY_PATH,
)

atomic_csv(
    BASELINE_BEHAVIOR_PATH,
    baseline_behavior,
)

atomic_copy(
    all_technique_curves_path,
    ALL_TECHNIQUE_CURVES_COPY_PATH,
)


# --------------------------------------------------------------------------------------------------
# 10. FIGURE 1 — ALL SEVEN APFDC CURVES
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        9.0,
        5.8,
    )
)

for technique in ALL_TECHNIQUES:
    block = (
        all_technique_curves.loc[
            all_technique_curves[
                "Metric"
            ].astype(
                str
            ).eq(
                "APFDc"
            )
            & all_technique_curves[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "CrossProjectMean"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Cross-project mean APFDc"
)

plt.title(
    "RQ2 — ML and baseline APFDc curves"
)

plt.xticks(
    NOISE_LEVELS
)

plt.ylim(
    0.0,
    1.0,
)

plt.axhline(
    0.0,
    linewidth=0.8,
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
    ncol=2,
)

save_figure(
    FIGURE_ALL_TECHNIQUES_APFDC_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. FIGURE HELPERS FOR ML ADVANTAGE CURVES
# --------------------------------------------------------------------------------------------------

def plot_named_baseline_advantage(
    comparator,
    title,
    output_path,
):
    plt.figure(
        figsize=(
            8.8,
            5.6,
        )
    )

    for algorithm in ML_TECHNIQUES:
        block = (
            primary_named.loc[
                primary_named[
                    "Algorithm"
                ].astype(
                    str
                ).eq(
                    algorithm
                )
                & primary_named[
                    "Comparator"
                ].astype(
                    str
                ).eq(
                    comparator
                )
            ]
            .sort_values(
                "NoisePercent",
                kind="mergesort",
            )
        )

        if len(
            block
        ) != 9:
            raise RuntimeError(
                f"{algorithm} vs {comparator}: expected 9 rows."
            )

        plt.plot(
            pd.to_numeric(
                block[
                    "NoisePercent"
                ],
                errors="raise",
            ),
            pd.to_numeric(
                block[
                    "GlobalAdvantageMLMinusComparator"
                ],
                errors="raise",
            ),
            marker="o",
            label=algorithm,
        )

    plt.axhline(
        0.0,
        linewidth=1.0,
    )

    plt.xlabel(
        "Training-label noise (%)"
    )

    plt.ylabel(
        "APFDc advantage (ML − baseline)"
    )

    plt.title(
        title
    )

    plt.xticks(
        NOISE_LEVELS
    )

    plt.grid(
        axis="y",
        alpha=0.25,
    )

    plt.legend(
        frameon=False,
    )

    save_figure(
        output_path
    )


plot_named_baseline_advantage(
    "LatestFail",
    "RQ2 — APFDc advantage over LatestFail",
    FIGURE_LATESTFAIL_ADVANTAGE_PATH,
)

plot_named_baseline_advantage(
    "Random",
    "RQ2 — APFDc advantage over Random",
    FIGURE_RANDOM_ADVANTAGE_PATH,
)

plot_named_baseline_advantage(
    "QTF-Avg",
    "RQ2 — APFDc advantage over QTF-Avg",
    FIGURE_QTF_ADVANTAGE_PATH,
)


# --------------------------------------------------------------------------------------------------
# 12. FIGURE 5 — ADVANTAGE VS STRONGEST BASELINE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for algorithm in ML_TECHNIQUES:
    block = (
        primary_strongest.loc[
            primary_strongest[
                "Algorithm"
            ].astype(
                str
            ).eq(
                algorithm
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    if len(
        block
    ) != 9:
        raise RuntimeError(
            f"{algorithm} vs strongest baseline: expected 9 rows."
        )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "GlobalAdvantageMLMinusComparator"
            ],
            errors="raise",
        ),
        marker="o",
        label=algorithm,
    )

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "APFDc advantage (ML − strongest baseline)"
)

plt.title(
    "RQ2 — APFDc advantage over the strongest baseline at each tested noise level"
)

plt.xticks(
    NOISE_LEVELS
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_STRONGEST_ADVANTAGE_PATH
)


# --------------------------------------------------------------------------------------------------
# 13. FIGURE 6 — SECONDARY APFD ADVANTAGE VS LATESTFAIL
# --------------------------------------------------------------------------------------------------

secondary_latestfail = pairwise_summary.loc[
    pairwise_summary[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFD"
    )
    & pairwise_summary[
        "Comparator"
    ].astype(
        str
    ).eq(
        "LatestFail"
    )
].copy()

if len(
    secondary_latestfail
) != 36:
    raise RuntimeError(
        "Secondary APFD LatestFail evidence must contain 36 rows."
    )

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for algorithm in ML_TECHNIQUES:
    block = (
        secondary_latestfail.loc[
            secondary_latestfail[
                "Algorithm"
            ].astype(
                str
            ).eq(
                algorithm
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "GlobalAdvantageMLMinusComparator"
            ],
            errors="raise",
        ),
        marker="o",
        label=algorithm,
    )

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "APFD advantage (ML − LatestFail)"
)

plt.title(
    "RQ2 — Secondary APFD advantage over LatestFail"
)

plt.xticks(
    NOISE_LEVELS
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFD_LATESTFAIL_PATH
)


# --------------------------------------------------------------------------------------------------
# 14. FROZEN OBSERVED APFDC FACTS FOR VALIDATION
# --------------------------------------------------------------------------------------------------

expected_primary_thresholds = {
    ("RandomForest", "LatestFail"):
        (5, "OBSERVED"),

    ("XGBoost", "LatestFail"):
        (5, "OBSERVED"),

    ("LightGBM", "LatestFail"):
        (5, "OBSERVED"),

    ("NaiveBayes", "LatestFail"):
        (5, "OBSERVED"),

    ("RandomForest", "Random"):
        (40, "OBSERVED"),

    ("XGBoost", "Random"):
        (40, "OBSERVED"),

    ("LightGBM", "Random"):
        (50, "OBSERVED"),

    ("NaiveBayes", "Random"):
        (50, "OBSERVED"),

    ("RandomForest", "QTF-Avg"):
        (10, "OBSERVED"),

    ("XGBoost", "QTF-Avg"):
        (10, "OBSERVED"),

    ("LightGBM", "QTF-Avg"):
        (10, "OBSERVED"),

    ("NaiveBayes", "QTF-Avg"):
        (0, "NO_POSITIVE_TOLERANCE_THRESHOLD"),

    ("RandomForest", STRONGEST_BASELINE_LABEL):
        (5, "OBSERVED"),

    ("XGBoost", STRONGEST_BASELINE_LABEL):
        (5, "OBSERVED"),

    ("LightGBM", STRONGEST_BASELINE_LABEL):
        (5, "OBSERVED"),

    ("NaiveBayes", STRONGEST_BASELINE_LABEL):
        (0, "NO_POSITIVE_TOLERANCE_THRESHOLD"),
}


# --------------------------------------------------------------------------------------------------
# 15. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []

registry_sha_after = sha256_file(
    REGISTRY
)

add_check(
    checks,
    "RQ2 Step-4A checkpoint SHA-256",
    EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256,
    actual_step4a_sha,
    actual_step4a_sha
    == EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-2 checkpoint SHA-256",
    EXPECTED_STEP2_CHECKPOINT_SHA256,
    actual_step2_sha,
    actual_step2_sha
    == EXPECTED_STEP2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Primary threshold rows",
    16,
    len(
        apfdc_thresholds
    ),
    len(
        apfdc_thresholds
    )
    == 16,
)

add_check(
    checks,
    "Secondary threshold rows",
    16,
    len(
        apfd_thresholds
    ),
    len(
        apfd_thresholds
    )
    == 16,
)

add_check(
    checks,
    "Primary threshold matrix rows",
    4,
    len(
        primary_threshold_matrix
    ),
    len(
        primary_threshold_matrix
    )
    == 4,
)

add_check(
    checks,
    "Primary LatestFail evidence rows",
    36,
    len(
        primary_latestfail
    ),
    len(
        primary_latestfail
    )
    == 36,
)

add_check(
    checks,
    "Primary named-comparator evidence rows",
    108,
    len(
        primary_named
    ),
    len(
        primary_named
    )
    == 108,
)

add_check(
    checks,
    "Primary strongest-baseline evidence rows",
    36,
    len(
        primary_strongest
    ),
    len(
        primary_strongest
    )
    == 36,
)

add_check(
    checks,
    "Baseline behavior rows",
    6,
    len(
        baseline_behavior
    ),
    len(
        baseline_behavior
    )
    == 6,
)

threshold_fact_failures = 0

for (
    algorithm,
    comparator
), (
    expected_noise,
    expected_status
) in expected_primary_thresholds.items():
    row = get_threshold_row(
        apfdc_thresholds,
        algorithm,
        comparator,
    )

    actual_noise = parse_threshold_noise(
        row[
            "FirstObservedCrossoverNoisePercent"
        ]
    )

    actual_status = str(
        row[
            "FirstObservedCrossoverStatus"
        ]
    )

    if (
        actual_noise
        != expected_noise
        or actual_status
        != expected_status
    ):
        threshold_fact_failures += 1

add_check(
    checks,
    "Frozen APFDc threshold-fact mismatches",
    0,
    threshold_fact_failures,
    threshold_fact_failures
    == 0,
)

# Strongest APFDc baseline: QTF-Avg at 0%, LatestFail at every positive tested noise.
strongest_apfdc = strongest_selection.loc[
    strongest_selection[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
].sort_values(
    "NoisePercent",
    kind="mergesort",
)

expected_strongest_names = [
    "QTF-Avg",
    "LatestFail",
    "LatestFail",
    "LatestFail",
    "LatestFail",
    "LatestFail",
    "LatestFail",
    "LatestFail",
    "LatestFail",
]

actual_strongest_names = strongest_apfdc[
    "StrongestBaseline"
].astype(
    str
).tolist()

add_check(
    checks,
    "Strongest APFDc baseline sequence",
    expected_strongest_names,
    actual_strongest_names,
    actual_strongest_names
    == expected_strongest_names,
)

# Baseline invariance from frozen curves.
for metric in [
    "APFDc",
    "APFD",
]:
    for baseline, expected_invariant in [
        (
            "Random",
            True,
        ),
        (
            "QTF-Avg",
            True,
        ),
        (
            "LatestFail",
            False,
        ),
    ]:
        row = baseline_behavior.loc[
            baseline_behavior[
                "Metric"
            ].eq(
                metric
            )
            & baseline_behavior[
                "Baseline"
            ].eq(
                baseline
            )
        ]

        if len(
            row
        ) != 1:
            raise RuntimeError(
                "Baseline behavior row missing."
            )

        actual_invariant = bool(
            row.iloc[
                0
            ][
                "InvariantAcrossNoiseAt1e12"
            ]
        )

        add_check(
            checks,
            f"{metric} {baseline} invariance",
            expected_invariant,
            actual_invariant,
            actual_invariant
            == expected_invariant,
        )

for figure_path in [
    FIGURE_ALL_TECHNIQUES_APFDC_PATH,
    FIGURE_LATESTFAIL_ADVANTAGE_PATH,
    FIGURE_RANDOM_ADVANTAGE_PATH,
    FIGURE_QTF_ADVANTAGE_PATH,
    FIGURE_STRONGEST_ADVANTAGE_PATH,
    FIGURE_APFD_LATESTFAIL_PATH,
]:
    add_check(
        checks,
        f"Figure exists: {figure_path.name}",
        True,
        figure_path.is_file(),
        figure_path.is_file()
        and figure_path.stat().st_size
        > 0,
    )

add_check(
    checks,
    "New RQ2 hypothesis tests executed",
    0,
    0,
    True,
)

add_check(
    checks,
    "New RQ2 p-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "RQ3 executed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ2 Step 4B pre-freeze validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ2 STEP 4B VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 16. REPORT / STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "Step":
        "RQ2_STEP_4B",

    "Status":
        RQ2_STEP4B_STATUS,

    "CodeRevision":
        RQ2_STEP4B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ2Step4ACheckpointSHA256":
        EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        PRIMARY_METRIC,

    "SecondaryMetric":
        SECONDARY_METRIC,

    "PrimaryComparator":
        PRIMARY_COMPARATOR,

    "ThresholdGrid":
        NOISE_LEVELS,

    "InterpolationAllowed":
        False,

    "PrimaryLatestFailThresholds":
        {
            algorithm:
                threshold_label(
                    get_threshold_row(
                        apfdc_thresholds,
                        algorithm,
                        "LatestFail",
                    )
                )
            for algorithm
            in ML_TECHNIQUES
        },

    "PrimaryRandomThresholds":
        {
            algorithm:
                threshold_label(
                    get_threshold_row(
                        apfdc_thresholds,
                        algorithm,
                        "Random",
                    )
                )
            for algorithm
            in ML_TECHNIQUES
        },

    "PrimaryQTFAvgThresholds":
        {
            algorithm:
                threshold_label(
                    get_threshold_row(
                        apfdc_thresholds,
                        algorithm,
                        "QTF-Avg",
                    )
                )
            for algorithm
            in ML_TECHNIQUES
        },

    "PrimaryStrongestBaselineThresholds":
        {
            algorithm:
                threshold_label(
                    get_threshold_row(
                        apfdc_thresholds,
                        algorithm,
                        STRONGEST_BASELINE_LABEL,
                    )
                )
            for algorithm
            in ML_TECHNIQUES
        },

    "StrongestAPFDcBaselineSequence":
        actual_strongest_names,

    "PrimaryThresholdMatrixPath":
        str(
            PRIMARY_THRESHOLD_MATRIX_PATH
        ),

    "PrimaryThresholdsPath":
        str(
            PRIMARY_THRESHOLDS_COPY_PATH
        ),

    "SecondaryThresholdsPath":
        str(
            SECONDARY_THRESHOLDS_COPY_PATH
        ),

    "Figures":
        [
            str(
                FIGURE_ALL_TECHNIQUES_APFDC_PATH
            ),
            str(
                FIGURE_LATESTFAIL_ADVANTAGE_PATH
            ),
            str(
                FIGURE_RANDOM_ADVANTAGE_PATH
            ),
            str(
                FIGURE_QTF_ADVANTAGE_PATH
            ),
            str(
                FIGURE_STRONGEST_ADVANTAGE_PATH
            ),
            str(
                FIGURE_APFD_LATESTFAIL_PATH
            ),
        ],

    "NewHypothesisTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "RQ3Executed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "NextRequiredStep":
        (
            "RQ3 STEP 5A — FRIEDMAN OMNIBUS TESTS, HOLM CORRECTION, "
            "AVERAGE RANKS, AND CONDITIONAL NEMENYI POST-HOC"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ2_STEP4B_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RQ2AnswerPackageFrozen":
            True,

        "ReadyForRQ3":
            True,
    },
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL PACKAGE MANIFEST + ROOT HASH
# --------------------------------------------------------------------------------------------------

package_manifest = build_package_manifest(
    STEP4B_ROOT,
    PACKAGE_MANIFEST_PATH,
)

atomic_csv(
    PACKAGE_MANIFEST_PATH,
    package_manifest,
)

package_root_sha = package_root_hash(
    package_manifest
)

package_files = int(
    len(
        package_manifest
    )
)

package_bytes = int(
    package_manifest[
        "Bytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 18. CHECKPOINT
# --------------------------------------------------------------------------------------------------

if (
    sha256_file(
        REGISTRY
    )
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry changed during RQ2 Step 4B."
    )

checkpoint = {
    "CheckpointType":
        "RQ2_FINAL_CROSSOVER_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE",

    "Status":
        RQ2_STEP4B_STATUS,

    "CodeRevision":
        RQ2_STEP4B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ2Step4ACheckpointSHA256":
        EXPECTED_RQ2_STEP4A_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        PRIMARY_METRIC,

    "PrimaryComparator":
        PRIMARY_COMPARATOR,

    "InterpolationAllowed":
        False,

    "PackageRoot":
        str(
            STEP4B_ROOT
        ),

    "PackageManifestPath":
        str(
            PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            PACKAGE_MANIFEST_PATH
        ),

    "PackageRootSHA256":
        package_root_sha,

    "PackageFiles":
        package_files,

    "PackageBytes":
        package_bytes,

    "PrimaryThresholdMatrixPath":
        str(
            PRIMARY_THRESHOLD_MATRIX_PATH
        ),

    "PrimaryThresholdMatrixSHA256":
        sha256_file(
            PRIMARY_THRESHOLD_MATRIX_PATH
        ),

    "PrimaryThresholdsPath":
        str(
            PRIMARY_THRESHOLDS_COPY_PATH
        ),

    "PrimaryThresholdsSHA256":
        sha256_file(
            PRIMARY_THRESHOLDS_COPY_PATH
        ),

    "SecondaryThresholdsPath":
        str(
            SECONDARY_THRESHOLDS_COPY_PATH
        ),

    "SecondaryThresholdsSHA256":
        sha256_file(
            SECONDARY_THRESHOLDS_COPY_PATH
        ),

    "PrimaryLatestFailEvidencePath":
        str(
            PRIMARY_LATESTFAIL_EVIDENCE_PATH
        ),

    "PrimaryLatestFailEvidenceSHA256":
        sha256_file(
            PRIMARY_LATESTFAIL_EVIDENCE_PATH
        ),

    "StrongestBaselineSelectionPath":
        str(
            STRONGEST_BASELINE_SELECTION_COPY_PATH
        ),

    "StrongestBaselineSelectionSHA256":
        sha256_file(
            STRONGEST_BASELINE_SELECTION_COPY_PATH
        ),

    "NewHypothesisTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "RQ3Executed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "ReadyForRQ3":
        True,

    "NextRequiredStep":
        (
            "RQ3 STEP 5A — FRIEDMAN OMNIBUS TESTS, HOLM CORRECTION, "
            "AVERAGE RANKS, AND CONDITIONAL NEMENYI POST-HOC"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 19. USER-VISIBLE TABLES
# --------------------------------------------------------------------------------------------------

print(
    "\nRQ2 PRIMARY APFDc threshold matrix:"
)

try:
    from IPython.display import display

    display(
        primary_threshold_matrix
    )

except Exception:
    print(
        primary_threshold_matrix.to_string(
            index=False
        )
    )

print(
    "\nRQ2 APFDc strongest baseline sequence:"
)

try:
    from IPython.display import display

    display(
        strongest_apfdc[
            [
                "NoisePercent",
                "StrongestBaseline",
                "StrongestBaselineCrossProjectMean",
            ]
        ]
    )

except Exception:
    print(
        strongest_apfdc[
            [
                "NoisePercent",
                "StrongestBaseline",
                "StrongestBaselineCrossProjectMean",
            ]
        ].to_string(
            index=False
        )
    )

print(
    "\nRQ2 baseline behavior summary:"
)

try:
    from IPython.display import display

    display(
        baseline_behavior
    )

except Exception:
    print(
        baseline_behavior.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 20. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 154
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 9 / RQ2 STEP 4B RESULT ==="
)

print(
    "=" * 154
)

print(
    "RQ2 final answer package frozen: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "\nPrimary RQ2 comparator — LatestFail:"
)

for algorithm in ML_TECHNIQUES:
    row = get_threshold_row(
        apfdc_thresholds,
        algorithm,
        "LatestFail",
    )

    print(
        f"{algorithm}:",
        threshold_label(
            row
        ),
    )

print(
    "\nAdditional APFDc comparator thresholds:"
)

for comparator in [
    "Random",
    "QTF-Avg",
    STRONGEST_BASELINE_LABEL,
]:
    print(
        comparator + ":"
    )

    for algorithm in ML_TECHNIQUES:
        print(
            f"  {algorithm}:",
            threshold_label(
                get_threshold_row(
                    apfdc_thresholds,
                    algorithm,
                    comparator,
                )
            ),
        )

print(
    "\nStrongest APFDc baseline sequence:"
)

for noise, baseline in zip(
    NOISE_LEVELS,
    actual_strongest_names,
):
    print(
        f"{noise}% -> {baseline}"
    )

print(
    "\nFrozen outputs:"
)

print(
    "Primary threshold matrix rows:",
    len(
        primary_threshold_matrix
    )
)

print(
    "Figures:",
    6
)

print(
    "Package files:",
    package_files
)

print(
    "Package bytes:",
    package_bytes
)

print(
    "Package root SHA-256:",
    package_root_sha
)

print(
    "\nIsolation:"
)

print(
    "New RQ2 hypothesis tests executed: 0"
)

print(
    "New RQ2 p-values computed: False"
)

print(
    "RQ3 executed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nRQ2 Step 4B checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha
)

print(
    "\nNext required step: "
    "RQ3 STEP 5A — FRIEDMAN OMNIBUS TESTS, HOLM CORRECTION, "
    "AVERAGE RANKS, AND CONDITIONAL NEMENYI POST-HOC"
)

print(
    "\nSTATUS:",
    RQ2_STEP4B_STATUS
)

print(
    "=" * 154
)


=== THESIS GLOBAL ANALYSIS — CELL 9 / RQ2 STEP 4B: FINAL CROSSOVER FIGURES + SYNTHESIS PACKAGE ===

RQ2 Step 4B pre-freeze validation:


,Check,Expected,Actual,Pass
0,RQ2 Step-4A checkpoint SHA-256,e88976b9c82f952896090bc9617feffe872478bc637f7b...,e88976b9c82f952896090bc9617feffe872478bc637f7b...,True
1,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,Primary threshold rows,16,16,True
4,Secondary threshold rows,16,16,True
5,Primary threshold matrix rows,4,4,True
6,Primary LatestFail evidence rows,36,36,True
7,Primary named-comparator evidence rows,108,108,True
8,Primary strongest-baseline evidence rows,36,36,True
9,Baseline behavior rows,6,6,True



RQ2 PRIMARY APFDc threshold matrix:


,Algorithm,LatestFailCleanAdvantage,LatestFailFirstObservedCrossover,LatestFailFirstObservedCrossoverStatus,LatestFailSustainedCrossover,LatestFailSustainedCrossoverStatus,RandomCleanAdvantage,RandomFirstObservedCrossover,RandomFirstObservedCrossoverStatus,RandomSustainedCrossover,...,QTFAvgCleanAdvantage,QTFAvgFirstObservedCrossover,QTFAvgFirstObservedCrossoverStatus,QTFAvgSustainedCrossover,QTFAvgSustainedCrossoverStatus,StrongestBaselineCleanAdvantage,StrongestBaselineFirstObservedCrossover,StrongestBaselineFirstObservedCrossoverStatus,StrongestBaselineSustainedCrossover,StrongestBaselineSustainedCrossoverStatus
0,RandomForest,0.227982,5,OBSERVED,5,OBSERVED,0.307975,40,OBSERVED,40,...,0.147678,10,OBSERVED,10,OBSERVED,0.147678,5,OBSERVED,5,OBSERVED
1,XGBoost,0.208960,5,OBSERVED,5,OBSERVED,0.288953,40,OBSERVED,40,...,0.128656,10,OBSERVED,10,OBSERVED,0.128656,5,OBSERVED,5,OBSERVED
2,LightGBM,0.186803,5,OBSERVED,5,OBSERVED,0.266796,50,OBSERVED,50,...,0.106500,10,OBSERVED,10,OBSERVED,0.106500,5,OBSERVED,5,OBSERVED
3,NaiveBayes,0.042347,5,OBSERVED,5,OBSERVED,0.122340,50,OBSERVED,50,...,-0.037957,0 (no positive tolerance),NO_POSITIVE_TOLERANCE_THRESHOLD,0,NO_POSITIVE_TOLERANCE_THRESHOLD,-0.037957,0 (no positive tolerance),NO_POSITIVE_TOLERANCE_THRESHOLD,0,NO_POSITIVE_TOLERANCE_THRESHOLD



RQ2 APFDc strongest baseline sequence:


,NoisePercent,StrongestBaseline,StrongestBaselineCrossProjectMean
0,0,QTF-Avg,0.655560
1,5,LatestFail,0.783330
2,10,LatestFail,0.783453
3,15,LatestFail,0.783334
4,20,LatestFail,0.782989
5,25,LatestFail,0.783038
6,30,LatestFail,0.782263
7,40,LatestFail,0.780254
8,50,LatestFail,0.779787



RQ2 baseline behavior summary:


,Metric,Baseline,MeanAt0,MeanAt5,MeanAt50,RangeAcrossNoise,InvariantAcrossNoiseAt1e12
0,APFDc,Random,0.495264,0.495264,0.495264,0.000000,True
1,APFDc,LatestFail,0.575256,0.783330,0.779787,0.208197,False
2,APFDc,QTF-Avg,0.655560,0.655560,0.655560,0.000000,True
3,APFD,Random,0.495242,0.495242,0.495242,0.000000,True
4,APFD,LatestFail,0.509353,0.833622,0.835995,0.326643,False
5,APFD,QTF-Avg,0.214580,0.214580,0.214580,0.000000,True



=== THESIS GLOBAL ANALYSIS — CELL 9 / RQ2 STEP 4B RESULT ===
RQ2 final answer package frozen: True

Study unit:
Independent empirical unit: Project (N=24)

Primary RQ2 comparator — LatestFail:
RandomForest: 5
XGBoost: 5
LightGBM: 5
NaiveBayes: 5

Additional APFDc comparator thresholds:
Random:
  RandomForest: 40
  XGBoost: 40
  LightGBM: 50
  NaiveBayes: 50
QTF-Avg:
  RandomForest: 10
  XGBoost: 10
  LightGBM: 10
  NaiveBayes: 0 (no positive tolerance)
StrongestBaselineAtEachNoise:
  RandomForest: 5
  XGBoost: 5
  LightGBM: 5
  NaiveBayes: 0 (no positive tolerance)

Strongest APFDc baseline sequence:
0% -> QTF-Avg
5% -> LatestFail
10% -> LatestFail
15% -> LatestFail
20% -> LatestFail
25% -> LatestFail
30% -> LatestFail
40% -> LatestFail
50% -> LatestFail

Frozen outputs:
Primary threshold matrix rows: 4
Figures: 6
Package files: 18
Package bytes: 1546867
Package root SHA-256: cec947adcb77b912dc217ad7012f6b97a760932906c147cfcec2837125552b1b

Isolation:
New RQ2 hypothesis tests executed

In [7]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 10 / RQ3 STEP 5A
# FRIEDMAN OMNIBUS TESTS + HOLM CORRECTION + AVERAGE RANKS + CONDITIONAL NEMENYI POST-HOC
# ==================================================================================================
#
# RQ3
# ---
# Which of Random Forest, XGBoost, LightGBM and Naive Bayes is most robust to flaky test noise
# in its training data?
#
# FROZEN STATISTICAL CONTRACT (Step 2)
# ------------------------------------
# Independent empirical unit:
#   Project (N=24)
#
# Repeated stochastic unit:
#   Seed (30 within each project); seeds are already summarized to one mean per project.
#
# Primary metric:
#   APFDc
#
# Secondary sensitivity metric:
#   APFD
#
# At EACH of the 9 tested noise levels:
#   - treatments = RandomForest, XGBoost, LightGBM, NaiveBayes
#   - blocks = 24 projects
#   - omnibus = Friedman test
#
# Multiple testing:
#   - 9 APFDc Friedman p-values form ONE family
#   - Holm correction across those 9 tests at alpha=0.05
#   - APFD repeats the same procedure as a SEPARATE secondary 9-test family
#
# Conditional post-hoc:
#   - Nemenyi is executed at a noise level ONLY if the corresponding Friedman omnibus
#     is Holm-significant in that metric family.
#   - Nemenyi uses average project-wise ranks (rank 1 = best / highest metric).
#   - Six pairwise comparisons are produced for each triggered noise level.
#   - Nemenyi p-values use the Studentized-range distribution and control the pairwise
#     family at that noise level by the Nemenyi procedure.
#
# IMPORTANT:
#   - This cell DOES execute the 18 pre-registered RQ3 Friedman tests.
#   - It does NOT force a "most robust" winner.
#   - It does NOT yet create the final RQ3 answer package or figures.
#   - Final robustness synthesis (absolute high-noise APFDc + ranks + degradation/retention)
#     is deferred to RQ3 Step 5B.
#   - No project result or completion-registry row is modified.
# ==================================================================================================

from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import (
    friedmanchisquare,
    rankdata,
    studentized_range,
)


print("=" * 156)
print("=== THESIS GLOBAL ANALYSIS — CELL 10 / RQ3 STEP 5A: FRIEDMAN + HOLM + RANKS + CONDITIONAL NEMENYI ===")
print("=" * 156)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

EXPECTED_STEP1B_CHECKPOINT_SHA256 = (
    "eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3e0642d1b19f456ad50"
)

STEP2_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
)

EXPECTED_STEP2_CHECKPOINT_SHA256 = (
    "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"
)

RQ2_STEP4B_STATUS = (
    "PASS_RQ2_STEP4B_FINAL_CROSSOVER_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256 = (
    "8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d352460330974c93feb40"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ3_STEP5A_STATUS = (
    "PASS_RQ3_STEP5A_FRIEDMAN_HOLM_AVERAGE_RANKS_AND_CONDITIONAL_NEMENYI_FROZEN"
)

RQ3_STEP5A_CODE_REVISION = (
    "RQ3_STEP5A_V1_PREREGISTERED_PROJECT_N24_FRIEDMAN_HOLM_CONDITIONAL_NEMENYI"
)

ALPHA = 0.05

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

STRESS_LEVELS = [
    30,
    40,
    50,
]

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

EXPECTED_PROJECTS = 24
EXPECTED_TREATMENTS = 4
EXPECTED_RQ3_OMNIBUS_TESTS = 18
EXPECTED_OMNIBUS_PER_METRIC = 9
EXPECTED_RANK_ROWS = 2 * 9 * 4
EXPECTED_ML_PROJECT_ROWS = 24 * 9 * 4

FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

STEP1B_CHECKPOINT = (
    NOTES
    / "global_analysis_step1b_checkpoint.json"
)

STEP2_CHECKPOINT = (
    NOTES
    / "global_analysis_step2_checkpoint.json"
)

RQ2_STEP4B_CHECKPOINT = (
    NOTES
    / "global_analysis_rq2_step4b_checkpoint.json"
)

STEP5A_ROOT = (
    ANALYSIS_ROOT
    / "RQ3"
    / "Step_5A_Friedman_Holm_Average_Ranks_and_Conditional_Nemenyi"
)

OMNIBUS_RESULTS_PATH = (
    STEP5A_ROOT
    / "rq3_friedman_holm_omnibus_results.csv"
)

AVERAGE_RANKS_PATH = (
    STEP5A_ROOT
    / "rq3_average_ranks_by_noise.csv"
)

PROJECT_RANKS_PATH = (
    STEP5A_ROOT
    / "rq3_project_ranks_by_noise.csv"
)

ABSOLUTE_PERFORMANCE_PATH = (
    STEP5A_ROOT
    / "rq3_absolute_performance_by_noise.csv"
)

NEMENYI_RESULTS_PATH = (
    STEP5A_ROOT
    / "rq3_conditional_nemenyi_posthoc.csv"
)

STRESS_SUMMARY_PREVIEW_PATH = (
    STEP5A_ROOT
    / "rq3_stress_level_absolute_and_rank_preview.csv"
)

VALIDATION_PATH = (
    STEP5A_ROOT
    / "rq3_step5a_validation.csv"
)

READBACK_AUDIT_PATH = (
    STEP5A_ROOT
    / "rq3_step5a_readback_audit.csv"
)

REPORT_PATH = (
    STEP5A_ROOT
    / "rq3_step5a_report.json"
)

STATUS_PATH = (
    STEP5A_ROOT
    / "rq3_step5a_status.json"
)

OUTPUT_MANIFEST_PATH = (
    STEP5A_ROOT
    / "rq3_step5a_output_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq3_step5a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def build_output_manifest(
    paths,
):
    rows = []

    for path in sorted(
        (
            Path(path)
            for path in paths
        ),
        key=str,
    ):
        if not path.is_file():
            raise FileNotFoundError(
                f"RQ3 Step-5A output missing: {path}"
            )

        rows.append({
            "Path":
                str(path),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "Path",
            "Bytes",
            "SHA256",
        ],
    )


def scientific_hash(
    dataframe,
):
    digest = hashlib.sha256()

    for row in dataframe.itertuples(
        index=False,
        name=None,
    ):
        parts = []

        for value in row:
            if isinstance(
                value,
                (
                    float,
                    np.floating,
                ),
            ):
                if math.isnan(
                    float(value)
                ):
                    parts.append(
                        "NaN"
                    )

                else:
                    parts.append(
                        f"{float(value):.17g}"
                    )

            else:
                parts.append(
                    str(value)
                )

        digest.update(
            (
                "\0".join(
                    parts
                )
                + "\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def _csv_missing_mask(
    series,
):
    return np.array(
        [
            (
                pd.isna(value)
                or (
                    isinstance(
                        value,
                        str,
                    )
                    and value.strip()
                    == ""
                )
            )
            for value in series.astype(
                "object"
            ).tolist()
        ],
        dtype=bool,
    )


def _normalize_bool_token(
    value,
):
    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(value)

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
    }:
        return True

    if text in {
        "false",
        "0",
    }:
        return False

    raise ValueError(
        f"Not a boolean token: {value!r}"
    )


def compare_csv_roundtrip(
    expected,
    actual,
    float_tolerance=FLOAT_TOL,
):
    if (
        len(expected)
        != len(actual)
        or expected.columns.tolist()
        != actual.columns.tolist()
    ):
        return False, np.inf

    maximum_float_difference = 0.0

    for column in expected.columns:
        expected_series = expected[
            column
        ]

        actual_series = actual[
            column
        ]

        expected_missing = _csv_missing_mask(
            expected_series
        )

        actual_missing = _csv_missing_mask(
            actual_series
        )

        if not np.array_equal(
            expected_missing,
            actual_missing,
        ):
            return False, np.inf

        nonmissing = ~expected_missing

        if not nonmissing.any():
            continue

        expected_values = expected_series.astype(
            "object"
        ).to_numpy()[
            nonmissing
        ]

        actual_values = actual_series.astype(
            "object"
        ).to_numpy()[
            nonmissing
        ]

        bool_possible = True

        try:
            expected_bool = np.array(
                [
                    _normalize_bool_token(
                        value
                    )
                    for value in expected_values
                ],
                dtype=bool,
            )

            actual_bool = np.array(
                [
                    _normalize_bool_token(
                        value
                    )
                    for value in actual_values
                ],
                dtype=bool,
            )

        except Exception:
            bool_possible = False

        if bool_possible:
            if not np.array_equal(
                expected_bool,
                actual_bool,
            ):
                return False, maximum_float_difference

            continue

        expected_numeric = pd.to_numeric(
            pd.Series(
                expected_values
            ),
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        actual_numeric = pd.to_numeric(
            pd.Series(
                actual_values
            ),
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        if (
            np.isfinite(
                expected_numeric
            ).all()
            and np.isfinite(
                actual_numeric
            ).all()
        ):
            difference = float(
                np.max(
                    np.abs(
                        expected_numeric
                        - actual_numeric
                    )
                )
            )

            maximum_float_difference = max(
                maximum_float_difference,
                difference,
            )

            if difference > float_tolerance:
                return False, maximum_float_difference

            continue

        expected_text = [
            str(value)
            for value in expected_values
        ]

        actual_text = [
            str(value)
            for value in actual_values
        ]

        if expected_text != actual_text:
            return False, maximum_float_difference

    return True, maximum_float_difference


def holm_adjust(
    p_values,
    labels,
):
    """
    Holm step-down adjusted p-values.

    Deterministic tie ordering follows labels, which here are numeric noise levels.
    """

    if len(p_values) != len(labels):
        raise RuntimeError(
            "Holm p-value/label lengths differ."
        )

    m = len(
        p_values
    )

    records = sorted(
        [
            (
                float(p_value),
                int(label),
                original_index,
            )
            for original_index, (
                p_value,
                label,
            )
            in enumerate(
                zip(
                    p_values,
                    labels,
                )
            )
        ],
        key=lambda item:
            (
                item[0],
                item[1],
            ),
    )

    adjusted_sorted = []

    running_max = 0.0

    for sorted_index, (
        p_value,
        label,
        original_index,
    ) in enumerate(
        records
    ):
        multiplier = (
            m
            - sorted_index
        )

        candidate = min(
            1.0,
            multiplier
            * p_value,
        )

        running_max = max(
            running_max,
            candidate,
        )

        adjusted_sorted.append({
            "OriginalIndex":
                original_index,

            "RawPValue":
                p_value,

            "Label":
                label,

            "HolmMultiplier":
                multiplier,

            "HolmAdjustedPValue":
                min(
                    running_max,
                    1.0,
                ),
        })

    adjusted = [
        None
    ] * m

    for record in adjusted_sorted:
        adjusted[
            record[
                "OriginalIndex"
            ]
        ] = record

    return adjusted


def kendalls_w_from_friedman(
    statistic,
    blocks,
    treatments,
):
    denominator = (
        blocks
        * (
            treatments
            - 1
        )
    )

    if denominator <= 0:
        raise RuntimeError(
            "Invalid Kendall W denominator."
        )

    return float(
        statistic
        / denominator
    )


def nemenyi_pairwise_p_value(
    average_rank_a,
    average_rank_b,
    blocks,
    treatments,
):
    """
    Nemenyi p-value from the Studentized-range distribution.

    Standard error:
        sqrt(k(k+1)/(6N))

    Demsar-style q statistic:
        |R_i - R_j| / SE

    scipy.stats.studentized_range uses the unscaled Studentized-range statistic,
    so q is multiplied by sqrt(2) when evaluating the survival function.
    """

    se = math.sqrt(
        treatments
        * (
            treatments
            + 1
        )
        / (
            6.0
            * blocks
        )
    )

    rank_difference = abs(
        float(
            average_rank_a
        )
        - float(
            average_rank_b
        )
    )

    q_demsar = (
        rank_difference
        / se
    )

    q_studentized_range = (
        q_demsar
        * math.sqrt(
            2.0
        )
    )

    p_value = float(
        studentized_range.sf(
            q_studentized_range,
            treatments,
            np.inf,
        )
    )

    return {
        "RankDifferenceAbs":
            rank_difference,

        "NemenyiSE":
            se,

        "QDemšar":
            q_demsar,

        "QStudentizedRange":
            q_studentized_range,

        "NemenyiPValue":
            p_value,
    }


def nemenyi_critical_difference(
    alpha,
    blocks,
    treatments,
):
    se = math.sqrt(
        treatments
        * (
            treatments
            + 1
        )
        / (
            6.0
            * blocks
        )
    )

    studentized_critical = float(
        studentized_range.ppf(
            1.0
            - alpha,
            treatments,
            np.inf,
        )
    )

    demsar_critical = (
        studentized_critical
        / math.sqrt(
            2.0
        )
    )

    critical_difference = (
        demsar_critical
        * se
    )

    return {
        "SE":
            se,

        "StudentizedRangeCritical":
            studentized_critical,

        "DemšarCritical":
            demsar_critical,

        "CriticalDifference":
            critical_difference,
    }


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD + VERIFY FROZEN CHAIN
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ3 Step 5A is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to RQ3 Step 5B."
    )

for path in [
    STEP1B_CHECKPOINT,
    STEP2_CHECKPOINT,
    RQ2_STEP4B_CHECKPOINT,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required checkpoint missing: {path}"
        )

actual_step1b_sha = sha256_file(
    STEP1B_CHECKPOINT
)

actual_step2_sha = sha256_file(
    STEP2_CHECKPOINT
)

actual_rq2_step4b_sha = sha256_file(
    RQ2_STEP4B_CHECKPOINT
)

if (
    actual_step1b_sha
    != EXPECTED_STEP1B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-1B checkpoint SHA mismatch."
    )

if (
    actual_step2_sha
    != EXPECTED_STEP2_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Step-2 checkpoint SHA mismatch."
    )

if (
    actual_rq2_step4b_sha
    != EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ2 Step-4B checkpoint SHA mismatch."
    )

step1b = load_json(
    STEP1B_CHECKPOINT
)

step2 = load_json(
    STEP2_CHECKPOINT
)

rq2_step4b = load_json(
    RQ2_STEP4B_CHECKPOINT
)

if step1b.get(
    "Status"
) != STEP1B_STATUS:
    raise RuntimeError(
        "Step-1B checkpoint status is not PASS."
    )

if step2.get(
    "Status"
) != STEP2_STATUS:
    raise RuntimeError(
        "Step-2 checkpoint status is not PASS."
    )

if rq2_step4b.get(
    "Status"
) != RQ2_STEP4B_STATUS:
    raise RuntimeError(
        "RQ2 Step-4B checkpoint status is not PASS."
    )

if not bool(
    rq2_step4b.get(
        "ReadyForRQ3",
        False,
    )
):
    raise RuntimeError(
        "RQ2 Step-4B is not marked ready for RQ3."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if (
    registry_sha_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

if STEP5A_ROOT.exists():
    shutil.rmtree(
        STEP5A_ROOT,
        ignore_errors=True,
    )

STEP5A_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY PRE-REGISTERED RQ3 TEST PLAN
# --------------------------------------------------------------------------------------------------

planned_tests_path = Path(
    step2[
        "PlannedTestsPath"
    ]
)

expected_planned_tests_sha = str(
    step2[
        "PlannedTestsSHA256"
    ]
).lower()

if not planned_tests_path.is_file():
    raise FileNotFoundError(
        f"Frozen planned-test registry missing: {planned_tests_path}"
    )

actual_planned_tests_sha = sha256_file(
    planned_tests_path
)

if (
    actual_planned_tests_sha
    != expected_planned_tests_sha
):
    raise RuntimeError(
        "Frozen planned-test registry SHA mismatch."
    )

planned_tests = pd.read_csv(
    planned_tests_path,
    low_memory=False,
)

rq3_plan = planned_tests.loc[
    planned_tests[
        "RQ"
    ].astype(
        str
    ).eq(
        "RQ3"
    )
].copy()

if len(
    rq3_plan
) != EXPECTED_RQ3_OMNIBUS_TESTS:
    raise RuntimeError(
        "Frozen RQ3 plan does not contain exactly 18 omnibus tests."
    )

if not rq3_plan[
    "Test"
].astype(
    str
).eq(
        "Friedman"
    ).all():
    raise RuntimeError(
        "Frozen RQ3 plan contains a non-Friedman omnibus test."
    )

if not rq3_plan[
    "IndependentUnit"
].astype(
    str
).eq(
        "Project"
    ).all():
    raise RuntimeError(
        "Frozen RQ3 independent unit is not Project."
    )

if not pd.to_numeric(
    rq3_plan[
        "N"
    ],
    errors="raise",
).astype(
    int
).eq(
    24
).all():
    raise RuntimeError(
        "Frozen RQ3 N is not 24."
    )

if not pd.to_numeric(
    rq3_plan[
        "FamilySize"
    ],
    errors="raise",
).astype(
    int
).eq(
    9
).all():
    raise RuntimeError(
        "Frozen RQ3 Holm family size is not 9."
    )

if not rq3_plan[
    "Correction"
].astype(
    str
).eq(
        "Holm"
    ).all():
    raise RuntimeError(
        "Frozen RQ3 correction is not Holm."
    )

posthoc_values = rq3_plan[
    "PostHocConditional"
].astype(
    str
).str.lower()

if not posthoc_values.isin(
    [
        "true",
        "1",
    ]
).all():
    raise RuntimeError(
        "Frozen RQ3 plan does not require conditional post-hoc testing."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY + LOAD FROZEN PROJECT-LEVEL MASTER
# --------------------------------------------------------------------------------------------------

project_master_path = Path(
    step1b[
        "ProjectLevelMasterPath"
    ]
)

expected_project_master_sha = str(
    step1b[
        "ProjectLevelMasterSHA256"
    ]
).lower()

if not project_master_path.is_file():
    raise FileNotFoundError(
        f"Frozen project master missing: {project_master_path}"
    )

actual_project_master_sha = sha256_file(
    project_master_path
)

if (
    actual_project_master_sha
    != expected_project_master_sha
):
    raise RuntimeError(
        "Frozen project master SHA mismatch."
    )

project_master = pd.read_csv(
    project_master_path,
    low_memory=False,
)

if len(
    project_master
) != 1_512:
    raise RuntimeError(
        "Project master row count changed."
    )

ml_project = project_master.loc[
    project_master[
        "Technique"
    ].astype(
        str
    ).isin(
        ML_TECHNIQUES
    )
].copy()

if len(
    ml_project
) != EXPECTED_ML_PROJECT_ROWS:
    raise RuntimeError(
        "RQ3 ML project-level rows != 864."
    )


# --------------------------------------------------------------------------------------------------
# 7. EXECUTE 18 PRE-REGISTERED FRIEDMAN TESTS + PROJECT RANKS
# --------------------------------------------------------------------------------------------------

omnibus_rows = []
project_rank_rows = []
average_rank_rows = []
absolute_performance_rows = []

metric_specs = [
    (
        "PRIMARY",
        "APFDc",
        "MeanSeedMeanAPFDc",
    ),
    (
        "SECONDARY",
        "APFD",
        "MeanSeedMeanAPFD",
    ),
]

for metric_role, metric, value_column in metric_specs:
    metric_plan = rq3_plan.loc[
        rq3_plan[
            "Metric"
        ].astype(
            str
        ).eq(
            metric
        )
    ].copy()

    planned_noise = sorted(
        pd.to_numeric(
            metric_plan[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(
            int
        ).tolist()
    )

    if planned_noise != NOISE_LEVELS:
        raise RuntimeError(
            f"{metric}: frozen RQ3 noise plan mismatch."
        )

    family_ids = metric_plan[
        "FamilyID"
    ].astype(
        str
    ).unique().tolist()

    if len(
        family_ids
    ) != 1:
        raise RuntimeError(
            f"{metric}: expected one Holm family ID."
        )

    family_id = family_ids[
        0
    ]

    metric_omnibus_indices = []

    for noise in NOISE_LEVELS:
        block = ml_project.loc[
            pd.to_numeric(
                ml_project[
                    "NoisePercent"
                ],
                errors="raise",
            ).astype(
                int
            ).eq(
                noise
            ),
            [
                "ProjectNumber",
                "Project",
                "Technique",
                value_column,
            ],
        ].copy()

        if len(
            block
        ) != (
            24
            * 4
        ):
            raise RuntimeError(
                f"{metric} {noise}% does not contain 96 project-technique rows."
            )

        pivot = block.pivot(
            index=[
                "ProjectNumber",
                "Project",
            ],
            columns="Technique",
            values=value_column,
        )

        missing_columns = [
            technique
            for technique in ML_TECHNIQUES
            if technique
            not in pivot.columns
        ]

        if missing_columns:
            raise RuntimeError(
                f"{metric} {noise}% missing ML techniques: {missing_columns}"
            )

        pivot = (
            pivot[
                ML_TECHNIQUES
            ]
            .reset_index()
            .sort_values(
                "ProjectNumber",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            pivot
        ) != 24:
            raise RuntimeError(
                f"{metric} {noise}% pivot does not contain 24 projects."
            )

        if pivot[
            "ProjectNumber"
        ].astype(
            int
        ).tolist() != list(
            range(
                1,
                25,
            )
        ):
            raise RuntimeError(
                f"{metric} {noise}% project coverage/order mismatch."
            )

        matrices = {
            technique:
                pd.to_numeric(
                    pivot[
                        technique
                    ],
                    errors="raise",
                ).to_numpy(
                    dtype=float
                )
            for technique in ML_TECHNIQUES
        }

        if not all(
            np.isfinite(
                values
            ).all()
            for values in matrices.values()
        ):
            raise RuntimeError(
                f"{metric} {noise}% contains non-finite values."
            )

        friedman_result = friedmanchisquare(
            *[
                matrices[
                    technique
                ]
                for technique in ML_TECHNIQUES
            ]
        )

        statistic = float(
            friedman_result.statistic
        )

        raw_p = float(
            friedman_result.pvalue
        )

        if not np.isfinite(
            statistic
        ):
            raise RuntimeError(
                f"{metric} {noise}% Friedman statistic is non-finite."
            )

        if not np.isfinite(
            raw_p
        ):
            raise RuntimeError(
                f"{metric} {noise}% Friedman p-value is non-finite."
            )

        kendalls_w = kendalls_w_from_friedman(
            statistic,
            EXPECTED_PROJECTS,
            EXPECTED_TREATMENTS,
        )

        omnibus_rows.append({
            "RQ":
                "RQ3",

            "MetricRole":
                metric_role,

            "Metric":
                metric,

            "NoisePercent":
                noise,

            "FamilyID":
                family_id,

            "FamilySize":
                9,

            "IndependentUnit":
                "Project",

            "Projects":
                24,

            "Treatments":
                4,

            "FriedmanStatistic":
                statistic,

            "RawPValue":
                raw_p,

            "HolmMultiplier":
                np.nan,

            "HolmAdjustedPValue":
                np.nan,

            "SignificantHolm":
                False,

            "KendallsW":
                kendalls_w,

            "ConditionalNemenyiTriggered":
                False,
        })

        omnibus_index = (
            len(
                omnibus_rows
            )
            - 1
        )

        metric_omnibus_indices.append(
            omnibus_index
        )

        # ------------------------------------------------------------------------------------------
        # Rank 1 = best / highest metric, project by project.
        # ------------------------------------------------------------------------------------------

        project_value_matrix = np.column_stack(
            [
                matrices[
                    technique
                ]
                for technique in ML_TECHNIQUES
            ]
        )

        rank_matrix = np.vstack(
            [
                rankdata(
                    -row,
                    method="average",
                )
                for row in project_value_matrix
            ]
        )

        for project_index in range(
            24
        ):
            project_number = int(
                pivot.iloc[
                    project_index
                ][
                    "ProjectNumber"
                ]
            )

            project_name = str(
                pivot.iloc[
                    project_index
                ][
                    "Project"
                ]
            )

            for technique_index, technique in enumerate(
                ML_TECHNIQUES
            ):
                project_rank_rows.append({
                    "Metric":
                        metric,

                    "NoisePercent":
                        noise,

                    "ProjectNumber":
                        project_number,

                    "Project":
                        project_name,

                    "Technique":
                        technique,

                    "Value":
                        float(
                            project_value_matrix[
                                project_index,
                                technique_index,
                            ]
                        ),

                    "Rank":
                        float(
                            rank_matrix[
                                project_index,
                                technique_index,
                            ]
                        ),
                })

        average_ranks = np.mean(
            rank_matrix,
            axis=0,
        )

        for technique_index, technique in enumerate(
            ML_TECHNIQUES
        ):
            values = matrices[
                technique
            ]

            average_rank_rows.append({
                "MetricRole":
                    metric_role,

                "Metric":
                    metric,

                "NoisePercent":
                    noise,

                "Technique":
                    technique,

                "Projects":
                    24,

                "AverageRank":
                    float(
                        average_ranks[
                            technique_index
                        ]
                    ),

                "CrossProjectMean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "CrossProjectMedian":
                    float(
                        np.median(
                            values
                        )
                    ),

                "CrossProjectSD":
                    float(
                        np.std(
                            values,
                            ddof=1,
                        )
                    ),
            })

            absolute_performance_rows.append({
                "MetricRole":
                    metric_role,

                "Metric":
                    metric,

                "NoisePercent":
                    noise,

                "Technique":
                    technique,

                "Projects":
                    24,

                "CrossProjectMean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "CrossProjectMedian":
                    float(
                        np.median(
                            values
                        )
                    ),

                "CrossProjectSD":
                    float(
                        np.std(
                            values,
                            ddof=1,
                        )
                    ),

                "CrossProjectMin":
                    float(
                        np.min(
                            values
                        )
                    ),

                "CrossProjectMax":
                    float(
                        np.max(
                            values
                        )
                    ),
            })

    # ----------------------------------------------------------------------------------------------
    # Holm correction across the exact 9 omnibus tests for this metric.
    # ----------------------------------------------------------------------------------------------

    raw_p_values = [
        omnibus_rows[
            index
        ][
            "RawPValue"
        ]
        for index in metric_omnibus_indices
    ]

    noise_labels = [
        omnibus_rows[
            index
        ][
            "NoisePercent"
        ]
        for index in metric_omnibus_indices
    ]

    holm_records = holm_adjust(
        raw_p_values,
        noise_labels,
    )

    for local_index, omnibus_index in enumerate(
        metric_omnibus_indices
    ):
        record = holm_records[
            local_index
        ]

        adjusted_p = float(
            record[
                "HolmAdjustedPValue"
            ]
        )

        omnibus_rows[
            omnibus_index
        ][
            "HolmMultiplier"
        ] = int(
            record[
                "HolmMultiplier"
            ]
        )

        omnibus_rows[
            omnibus_index
        ][
            "HolmAdjustedPValue"
        ] = adjusted_p

        omnibus_rows[
            omnibus_index
        ][
            "SignificantHolm"
        ] = bool(
            adjusted_p
            <= ALPHA
            + 1e-15
        )

        omnibus_rows[
            omnibus_index
        ][
            "ConditionalNemenyiTriggered"
        ] = bool(
            adjusted_p
            <= ALPHA
            + 1e-15
        )


omnibus_results = pd.DataFrame(
    omnibus_rows
)

project_ranks = pd.DataFrame(
    project_rank_rows
)

average_ranks = pd.DataFrame(
    average_rank_rows
)

absolute_performance = pd.DataFrame(
    absolute_performance_rows
)

if len(
    omnibus_results
) != 18:
    raise RuntimeError(
        "RQ3 omnibus result row count != 18."
    )

if len(
    project_ranks
) != (
    2
    * 9
    * 24
    * 4
):
    raise RuntimeError(
        "RQ3 project-rank row count mismatch."
    )

if len(
    average_ranks
) != EXPECTED_RANK_ROWS:
    raise RuntimeError(
        "RQ3 average-rank row count !=72."
    )

if len(
    absolute_performance
) != EXPECTED_RANK_ROWS:
    raise RuntimeError(
        "RQ3 absolute-performance row count !=72."
    )


# --------------------------------------------------------------------------------------------------
# 8. CONDITIONAL NEMENYI POST-HOC
# --------------------------------------------------------------------------------------------------

nemenyi_rows = []

critical = nemenyi_critical_difference(
    ALPHA,
    EXPECTED_PROJECTS,
    EXPECTED_TREATMENTS,
)

for omnibus_row in omnibus_results.itertuples(
    index=False
):
    if not bool(
        omnibus_row.SignificantHolm
    ):
        continue

    metric = str(
        omnibus_row.Metric
    )

    noise = int(
        omnibus_row.NoisePercent
    )

    metric_role = str(
        omnibus_row.MetricRole
    )

    rank_block = average_ranks.loc[
        average_ranks[
            "Metric"
        ].eq(
            metric
        )
        & average_ranks[
            "NoisePercent"
        ].eq(
            noise
        )
    ].copy()

    if len(
        rank_block
    ) != 4:
        raise RuntimeError(
            f"{metric} {noise}% average-rank block !=4."
        )

    rank_lookup = {
        str(
            row.Technique
        ):
            float(
                row.AverageRank
            )
        for row in rank_block.itertuples(
            index=False
        )
    }

    performance_lookup = {
        str(
            row.Technique
        ):
            float(
                row.CrossProjectMean
            )
        for row in rank_block.itertuples(
            index=False
        )
    }

    for technique_a, technique_b in itertools.combinations(
        ML_TECHNIQUES,
        2,
    ):
        pair = nemenyi_pairwise_p_value(
            rank_lookup[
                technique_a
            ],
            rank_lookup[
                technique_b
            ],
            EXPECTED_PROJECTS,
            EXPECTED_TREATMENTS,
        )

        mean_difference = (
            performance_lookup[
                technique_a
            ]
            - performance_lookup[
                technique_b
            ]
        )

        if (
            rank_lookup[
                technique_a
            ]
            < rank_lookup[
                technique_b
            ]
            - FLOAT_TOL
        ):
            better_by_rank = (
                technique_a
            )

        elif (
            rank_lookup[
                technique_b
            ]
            < rank_lookup[
                technique_a
            ]
            - FLOAT_TOL
        ):
            better_by_rank = (
                technique_b
            )

        else:
            better_by_rank = (
                "TIE"
            )

        nemenyi_rows.append({
            "MetricRole":
                metric_role,

            "Metric":
                metric,

            "NoisePercent":
                noise,

            "TechniqueA":
                technique_a,

            "TechniqueB":
                technique_b,

            "AverageRankA":
                rank_lookup[
                    technique_a
                ],

            "AverageRankB":
                rank_lookup[
                    technique_b
                ],

            "RankDifferenceAbs":
                pair[
                    "RankDifferenceAbs"
                ],

            "CrossProjectMeanA":
                performance_lookup[
                    technique_a
                ],

            "CrossProjectMeanB":
                performance_lookup[
                    technique_b
                ],

            "MeanDifferenceAminusB":
                mean_difference,

            "BetterByAverageRank":
                better_by_rank,

            "NemenyiSE":
                pair[
                    "NemenyiSE"
                ],

            "QDemšar":
                pair[
                    "QDemšar"
                ],

            "QStudentizedRange":
                pair[
                    "QStudentizedRange"
                ],

            "NemenyiPValue":
                pair[
                    "NemenyiPValue"
                ],

            "NemenyiAlpha":
                ALPHA,

            "CriticalDifference":
                critical[
                    "CriticalDifference"
                ],

            "SignificantNemenyi":
                bool(
                    pair[
                        "NemenyiPValue"
                    ]
                    <= ALPHA
                    + 1e-15
                ),
        })


nemenyi_columns = [
    "MetricRole",
    "Metric",
    "NoisePercent",
    "TechniqueA",
    "TechniqueB",
    "AverageRankA",
    "AverageRankB",
    "RankDifferenceAbs",
    "CrossProjectMeanA",
    "CrossProjectMeanB",
    "MeanDifferenceAminusB",
    "BetterByAverageRank",
    "NemenyiSE",
    "QDemšar",
    "QStudentizedRange",
    "NemenyiPValue",
    "NemenyiAlpha",
    "CriticalDifference",
    "SignificantNemenyi",
]

nemenyi_results = pd.DataFrame(
    nemenyi_rows,
    columns=nemenyi_columns,
)

significant_omnibus_count = int(
    omnibus_results[
        "SignificantHolm"
    ].astype(
        bool
    ).sum()
)

expected_nemenyi_rows = (
    significant_omnibus_count
    * math.comb(
        4,
        2,
    )
)

if len(
    nemenyi_results
) != expected_nemenyi_rows:
    raise RuntimeError(
        "Conditional Nemenyi row count does not match 6 rows per Holm-significant omnibus."
    )


# --------------------------------------------------------------------------------------------------
# 9. STRESS-LEVEL ABSOLUTE + RANK PREVIEW
# --------------------------------------------------------------------------------------------------
#
# This is descriptive only. Final robustness synthesis/claim is deferred to Step 5B.
# --------------------------------------------------------------------------------------------------

stress_rows = []

for metric_role, metric in [
    (
        "PRIMARY",
        "APFDc",
    ),
    (
        "SECONDARY",
        "APFD",
    ),
]:
    for technique in ML_TECHNIQUES:
        block = average_ranks.loc[
            average_ranks[
                "Metric"
            ].eq(
                metric
            )
            & average_ranks[
                "Technique"
            ].eq(
                technique
            )
            & average_ranks[
                "NoisePercent"
            ].isin(
                STRESS_LEVELS
            )
        ].sort_values(
            "NoisePercent",
            kind="mergesort",
        )

        if len(
            block
        ) != 3:
            raise RuntimeError(
                f"{metric} {technique}: stress-level preview !=3 rows."
            )

        stress_rows.append({
            "MetricRole":
                metric_role,

            "Metric":
                metric,

            "Technique":
                technique,

            "StressNoiseLevels":
                json.dumps(
                    STRESS_LEVELS,
                    separators=(
                        ",",
                        ":",
                    ),
                ),

            "MeanStressAPFDValue":
                float(
                    block[
                        "CrossProjectMean"
                    ].mean()
                ),

            "MeanStressAverageRank":
                float(
                    block[
                        "AverageRank"
                    ].mean()
                ),

            "ValueAt30":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            30
                        ),
                        "CrossProjectMean",
                    ].iloc[
                        0
                    ]
                ),

            "ValueAt40":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            40
                        ),
                        "CrossProjectMean",
                    ].iloc[
                        0
                    ]
                ),

            "ValueAt50":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            50
                        ),
                        "CrossProjectMean",
                    ].iloc[
                        0
                    ]
                ),

            "AverageRankAt30":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            30
                        ),
                        "AverageRank",
                    ].iloc[
                        0
                    ]
                ),

            "AverageRankAt40":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            40
                        ),
                        "AverageRank",
                    ].iloc[
                        0
                    ]
                ),

            "AverageRankAt50":
                float(
                    block.loc[
                        block[
                            "NoisePercent"
                        ].eq(
                            50
                        ),
                        "AverageRank",
                    ].iloc[
                        0
                    ]
                ),
        })


stress_summary_preview = pd.DataFrame(
    stress_rows
)


# --------------------------------------------------------------------------------------------------
# 10. VALIDATION
# --------------------------------------------------------------------------------------------------

registry_sha_after = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Step-1B checkpoint SHA-256",
    EXPECTED_STEP1B_CHECKPOINT_SHA256,
    actual_step1b_sha,
    actual_step1b_sha
    == EXPECTED_STEP1B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-2 checkpoint SHA-256",
    EXPECTED_STEP2_CHECKPOINT_SHA256,
    actual_step2_sha,
    actual_step2_sha
    == EXPECTED_STEP2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ2 Step-4B checkpoint SHA-256",
    EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256,
    actual_rq2_step4b_sha,
    actual_rq2_step4b_sha
    == EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Frozen RQ3 planned omnibus tests",
    18,
    len(
        rq3_plan
    ),
    len(
        rq3_plan
    )
    == 18,
)

add_check(
    checks,
    "Executed RQ3 Friedman tests",
    18,
    len(
        omnibus_results
    ),
    len(
        omnibus_results
    )
    == 18,
)

add_check(
    checks,
    "Primary APFDc Friedman tests",
    9,
    int(
        omnibus_results[
            "Metric"
        ].eq(
            "APFDc"
        ).sum()
    ),
    int(
        omnibus_results[
            "Metric"
        ].eq(
            "APFDc"
        ).sum()
    )
    == 9,
)

add_check(
    checks,
    "Secondary APFD Friedman tests",
    9,
    int(
        omnibus_results[
            "Metric"
        ].eq(
            "APFD"
        ).sum()
    ),
    int(
        omnibus_results[
            "Metric"
        ].eq(
            "APFD"
        ).sum()
    )
    == 9,
)

add_check(
    checks,
    "Average-rank rows",
    EXPECTED_RANK_ROWS,
    len(
        average_ranks
    ),
    len(
        average_ranks
    )
    == EXPECTED_RANK_ROWS,
)

add_check(
    checks,
    "Project-rank rows",
    2 * 9 * 24 * 4,
    len(
        project_ranks
    ),
    len(
        project_ranks
    )
    == (
        2
        * 9
        * 24
        * 4
    ),
)

add_check(
    checks,
    "Absolute-performance rows",
    EXPECTED_RANK_ROWS,
    len(
        absolute_performance
    ),
    len(
        absolute_performance
    )
    == EXPECTED_RANK_ROWS,
)

add_check(
    checks,
    "Conditional Nemenyi rows",
    expected_nemenyi_rows,
    len(
        nemenyi_results
    ),
    len(
        nemenyi_results
    )
    == expected_nemenyi_rows,
)

add_check(
    checks,
    "Nemenyi rows per significant omnibus",
    6,
    (
        0
        if significant_omnibus_count
        == 0
        else len(
            nemenyi_results
        )
        / significant_omnibus_count
    ),
    (
        (
            significant_omnibus_count
            == 0
            and len(
                nemenyi_results
            )
            == 0
        )
        or math.isclose(
            len(
                nemenyi_results
            )
            / significant_omnibus_count,
            6.0,
            rel_tol=0.0,
            abs_tol=0.0,
        )
    ),
)

add_check(
    checks,
    "All Friedman raw p-values finite",
    True,
    bool(
        np.isfinite(
            omnibus_results[
                "RawPValue"
            ].to_numpy(
                dtype=float
            )
        ).all()
    ),
    bool(
        np.isfinite(
            omnibus_results[
                "RawPValue"
            ].to_numpy(
                dtype=float
            )
        ).all()
    ),
)

add_check(
    checks,
    "All Holm p-values in [0,1]",
    True,
    bool(
        omnibus_results[
            "HolmAdjustedPValue"
        ].between(
            0,
            1,
            inclusive="both",
        ).all()
    ),
    bool(
        omnibus_results[
            "HolmAdjustedPValue"
        ].between(
            0,
            1,
            inclusive="both",
        ).all()
    ),
)

add_check(
    checks,
    "All Kendall W values in [0,1] within tolerance",
    True,
    bool(
        (
            omnibus_results[
                "KendallsW"
            ]
            >= -FLOAT_TOL
        ).all()
        and (
            omnibus_results[
                "KendallsW"
            ]
            <= 1.0
            + FLOAT_TOL
        ).all()
    ),
    bool(
        (
            omnibus_results[
                "KendallsW"
            ]
            >= -FLOAT_TOL
        ).all()
        and (
            omnibus_results[
                "KendallsW"
            ]
            <= 1.0
            + FLOAT_TOL
        ).all()
    ),
)

add_check(
    checks,
    "Average ranks per Metric×Noise sum to 10",
    True,
    bool(
        np.allclose(
            average_ranks.groupby(
                [
                    "Metric",
                    "NoisePercent",
                ],
                sort=True,
            )[
                "AverageRank"
            ].sum().to_numpy(
                dtype=float
            ),
            10.0,
            atol=FLOAT_TOL,
            rtol=0.0,
        )
    ),
    bool(
        np.allclose(
            average_ranks.groupby(
                [
                    "Metric",
                    "NoisePercent",
                ],
                sort=True,
            )[
                "AverageRank"
            ].sum().to_numpy(
                dtype=float
            ),
            10.0,
            atol=FLOAT_TOL,
            rtol=0.0,
        )
    ),
)

add_check(
    checks,
    "Every project rank in [1,4]",
    True,
    bool(
        project_ranks[
            "Rank"
        ].between(
            1,
            4,
            inclusive="both",
        ).all()
    ),
    bool(
        project_ranks[
            "Rank"
        ].between(
            1,
            4,
            inclusive="both",
        ).all()
    ),
)

add_check(
    checks,
    "Stress preview rows",
    8,
    len(
        stress_summary_preview
    ),
    len(
        stress_summary_preview
    )
    == 8,
)

add_check(
    checks,
    "Nemenyi critical difference finite positive",
    True,
    critical[
        "CriticalDifference"
    ],
    np.isfinite(
        critical[
            "CriticalDifference"
        ]
    )
    and critical[
        "CriticalDifference"
    ]
    > 0,
)

if len(
    nemenyi_results
) > 0:
    add_check(
        checks,
        "All Nemenyi p-values in [0,1]",
        True,
        bool(
            nemenyi_results[
                "NemenyiPValue"
            ].between(
                0,
                1,
                inclusive="both",
            ).all()
        ),
        bool(
            nemenyi_results[
                "NemenyiPValue"
            ].between(
                0,
                1,
                inclusive="both",
            ).all()
        ),
    )

else:
    add_check(
        checks,
        "All Nemenyi p-values in [0,1]",
        "not applicable because no Holm-significant omnibus",
        "not applicable",
        True,
    )

add_check(
    checks,
    "Final unique robust winner declared",
    False,
    False,
    True,
)

add_check(
    checks,
    "RQ3 final answer package created",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ3 Step 5A pre-write validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ3 STEP 5A PRE-WRITE VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_csv(
    OMNIBUS_RESULTS_PATH,
    omnibus_results,
)

atomic_csv(
    AVERAGE_RANKS_PATH,
    average_ranks,
)

atomic_csv(
    PROJECT_RANKS_PATH,
    project_ranks,
)

atomic_csv(
    ABSOLUTE_PERFORMANCE_PATH,
    absolute_performance,
)

atomic_csv(
    NEMENYI_RESULTS_PATH,
    nemenyi_results,
)

atomic_csv(
    STRESS_SUMMARY_PREVIEW_PATH,
    stress_summary_preview,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 12. READBACK AUDIT
# --------------------------------------------------------------------------------------------------

readback_rows = []

for dataset_name, path, expected in [
    (
        "OmnibusResults",
        OMNIBUS_RESULTS_PATH,
        omnibus_results,
    ),
    (
        "AverageRanks",
        AVERAGE_RANKS_PATH,
        average_ranks,
    ),
    (
        "ProjectRanks",
        PROJECT_RANKS_PATH,
        project_ranks,
    ),
    (
        "AbsolutePerformance",
        ABSOLUTE_PERFORMANCE_PATH,
        absolute_performance,
    ),
    (
        "NemenyiResults",
        NEMENYI_RESULTS_PATH,
        nemenyi_results,
    ),
    (
        "StressSummaryPreview",
        STRESS_SUMMARY_PREVIEW_PATH,
        stress_summary_preview,
    ),
]:
    actual = pd.read_csv(
        path,
        low_memory=False,
    )

    actual = actual[
        expected.columns.tolist()
    ]

    passed, max_difference = compare_csv_roundtrip(
        expected,
        actual,
        float_tolerance=FLOAT_TOL,
    )

    readback_rows.append({
        "Dataset":
            dataset_name,

        "ExpectedRows":
            len(expected),

        "ReadbackRows":
            len(actual),

        "ReadbackPass":
            passed,

        "MaxAbsFloatDifference":
            max_difference,

        "Tolerance":
            FLOAT_TOL,
    })


readback_audit = pd.DataFrame(
    readback_rows
)

readback_failures = int(
    (
        ~readback_audit[
            "ReadbackPass"
        ].astype(
            bool
        )
    ).sum()
)

atomic_csv(
    READBACK_AUDIT_PATH,
    readback_audit,
)

if readback_failures != 0:
    raise RuntimeError(
        "RQ3 STEP 5A READBACK AUDIT FAILED.\n"
        + readback_audit.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 13. REPORT / STATUS / MANIFEST / CHECKPOINT
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

omnibus_scientific_sha = scientific_hash(
    omnibus_results
)

average_ranks_scientific_sha = scientific_hash(
    average_ranks
)

nemenyi_scientific_sha = scientific_hash(
    nemenyi_results
)

primary_significant_omnibus = int(
    omnibus_results.loc[
        omnibus_results[
            "Metric"
        ].eq(
            "APFDc"
        ),
        "SignificantHolm",
    ].astype(
        bool
    ).sum()
)

secondary_significant_omnibus = int(
    omnibus_results.loc[
        omnibus_results[
            "Metric"
        ].eq(
            "APFD"
        ),
        "SignificantHolm",
    ].astype(
        bool
    ).sum()
)

primary_nemenyi_rows = int(
    nemenyi_results[
        "Metric"
    ].eq(
        "APFDc"
    ).sum()
)

secondary_nemenyi_rows = int(
    nemenyi_results[
        "Metric"
    ].eq(
        "APFD"
    ).sum()
)

report = {
    "Step":
        "RQ3_STEP_5A",

    "Status":
        RQ3_STEP5A_STATUS,

    "CodeRevision":
        RQ3_STEP5A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "RQ2Step4BCheckpointSHA256":
        EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "Treatments":
        ML_TECHNIQUES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "OmnibusTest":
        "Friedman",

    "HolmFamilySizePerMetric":
        9,

    "HolmAlpha":
        ALPHA,

    "PostHoc":
        "Nemenyi conditional on Holm-significant Friedman",

    "NemenyiAlpha":
        ALPHA,

    "NemenyiCriticalDifference":
        critical[
            "CriticalDifference"
        ],

    "ExecutedFriedmanTests":
        len(
            omnibus_results
        ),

    "PrimaryAPFDcHolmSignificantOmnibus":
        primary_significant_omnibus,

    "SecondaryAPFDHolmSignificantOmnibus":
        secondary_significant_omnibus,

    "ConditionalNemenyiRows":
        len(
            nemenyi_results
        ),

    "PrimaryAPFDcNemenyiRows":
        primary_nemenyi_rows,

    "SecondaryAPFDNemenyiRows":
        secondary_nemenyi_rows,

    "OmnibusResultsPath":
        str(
            OMNIBUS_RESULTS_PATH
        ),

    "OmnibusResultsSHA256":
        sha256_file(
            OMNIBUS_RESULTS_PATH
        ),

    "OmnibusResultsScientificSHA256":
        omnibus_scientific_sha,

    "AverageRanksPath":
        str(
            AVERAGE_RANKS_PATH
        ),

    "AverageRanksSHA256":
        sha256_file(
            AVERAGE_RANKS_PATH
        ),

    "AverageRanksScientificSHA256":
        average_ranks_scientific_sha,

    "ProjectRanksPath":
        str(
            PROJECT_RANKS_PATH
        ),

    "ProjectRanksSHA256":
        sha256_file(
            PROJECT_RANKS_PATH
        ),

    "AbsolutePerformancePath":
        str(
            ABSOLUTE_PERFORMANCE_PATH
        ),

    "AbsolutePerformanceSHA256":
        sha256_file(
            ABSOLUTE_PERFORMANCE_PATH
        ),

    "NemenyiResultsPath":
        str(
            NEMENYI_RESULTS_PATH
        ),

    "NemenyiResultsSHA256":
        sha256_file(
            NEMENYI_RESULTS_PATH
        ),

    "NemenyiResultsScientificSHA256":
        nemenyi_scientific_sha,

    "StressSummaryPreviewPath":
        str(
            STRESS_SUMMARY_PREVIEW_PATH
        ),

    "StressSummaryPreviewSHA256":
        sha256_file(
            STRESS_SUMMARY_PREVIEW_PATH
        ),

    "ReadbackFailures":
        readback_failures,

    "UniqueRobustWinnerDeclared":
        False,

    "FinalRQ3AnswerPackageCreated":
        False,

    "CompletionRegistryModified":
        False,

    "ProjectOutputsModified":
        False,

    "NextRequiredStep":
        (
            "RQ3 STEP 5B — FINAL ROBUSTNESS SYNTHESIS, DEGRADATION/RETENTION INTEGRATION, "
            "FIGURES, AND RQ3 ANSWER-PACKAGE FREEZE"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ3_STEP5A_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RQ3InferenceFrozen":
            True,

        "UniqueRobustWinnerDeclared":
            False,

        "ReadyForRQ3Step5B":
            True,
    },
)

output_paths = [
    OMNIBUS_RESULTS_PATH,
    AVERAGE_RANKS_PATH,
    PROJECT_RANKS_PATH,
    ABSOLUTE_PERFORMANCE_PATH,
    NEMENYI_RESULTS_PATH,
    STRESS_SUMMARY_PREVIEW_PATH,
    READBACK_AUDIT_PATH,
    VALIDATION_PATH,
    REPORT_PATH,
    STATUS_PATH,
]

output_manifest = build_output_manifest(
    output_paths
)

atomic_csv(
    OUTPUT_MANIFEST_PATH,
    output_manifest,
)

output_manifest_sha = sha256_file(
    OUTPUT_MANIFEST_PATH
)

if (
    sha256_file(
        REGISTRY
    )
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry changed during RQ3 Step 5A."
    )

checkpoint = {
    "CheckpointType":
        "RQ3_FRIEDMAN_HOLM_AVERAGE_RANKS_AND_CONDITIONAL_NEMENYI",

    "Status":
        RQ3_STEP5A_STATUS,

    "CodeRevision":
        RQ3_STEP5A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "RQ2Step4BCheckpointSHA256":
        EXPECTED_RQ2_STEP4B_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "OmnibusResultsPath":
        str(
            OMNIBUS_RESULTS_PATH
        ),

    "OmnibusResultsSHA256":
        sha256_file(
            OMNIBUS_RESULTS_PATH
        ),

    "OmnibusResultsScientificSHA256":
        omnibus_scientific_sha,

    "AverageRanksPath":
        str(
            AVERAGE_RANKS_PATH
        ),

    "AverageRanksSHA256":
        sha256_file(
            AVERAGE_RANKS_PATH
        ),

    "AverageRanksScientificSHA256":
        average_ranks_scientific_sha,

    "ProjectRanksPath":
        str(
            PROJECT_RANKS_PATH
        ),

    "ProjectRanksSHA256":
        sha256_file(
            PROJECT_RANKS_PATH
        ),

    "AbsolutePerformancePath":
        str(
            ABSOLUTE_PERFORMANCE_PATH
        ),

    "AbsolutePerformanceSHA256":
        sha256_file(
            ABSOLUTE_PERFORMANCE_PATH
        ),

    "NemenyiResultsPath":
        str(
            NEMENYI_RESULTS_PATH
        ),

    "NemenyiResultsSHA256":
        sha256_file(
            NEMENYI_RESULTS_PATH
        ),

    "NemenyiResultsScientificSHA256":
        nemenyi_scientific_sha,

    "StressSummaryPreviewPath":
        str(
            STRESS_SUMMARY_PREVIEW_PATH
        ),

    "StressSummaryPreviewSHA256":
        sha256_file(
            STRESS_SUMMARY_PREVIEW_PATH
        ),

    "OutputManifestPath":
        str(
            OUTPUT_MANIFEST_PATH
        ),

    "OutputManifestSHA256":
        output_manifest_sha,

    "ExecutedFriedmanTests":
        18,

    "PrimaryAPFDcHolmSignificantOmnibus":
        primary_significant_omnibus,

    "SecondaryAPFDHolmSignificantOmnibus":
        secondary_significant_omnibus,

    "ConditionalNemenyiRows":
        len(
            nemenyi_results
        ),

    "NemenyiCriticalDifference":
        critical[
            "CriticalDifference"
        ],

    "UniqueRobustWinnerDeclared":
        False,

    "FinalRQ3AnswerPackageCreated":
        False,

    "ReadyForRQ3Step5B":
        True,

    "NextRequiredStep":
        (
            "RQ3 STEP 5B — FINAL ROBUSTNESS SYNTHESIS, DEGRADATION/RETENTION INTEGRATION, "
            "FIGURES, AND RQ3 ANSWER-PACKAGE FREEZE"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 14. USER-VISIBLE TABLES
# --------------------------------------------------------------------------------------------------

print(
    "\nRQ3 Friedman + Holm omnibus results:"
)

try:
    from IPython.display import display

    display(
        omnibus_results[
            [
                "Metric",
                "NoisePercent",
                "FriedmanStatistic",
                "RawPValue",
                "HolmAdjustedPValue",
                "SignificantHolm",
                "KendallsW",
                "ConditionalNemenyiTriggered",
            ]
        ]
    )

except Exception:
    print(
        omnibus_results[
            [
                "Metric",
                "NoisePercent",
                "FriedmanStatistic",
                "RawPValue",
                "HolmAdjustedPValue",
                "SignificantHolm",
                "KendallsW",
                "ConditionalNemenyiTriggered",
            ]
        ].to_string(
            index=False
        )
    )

print(
    "\nRQ3 average ranks (rank 1 = best):"
)

rank_display = (
    average_ranks.pivot(
        index=[
            "Metric",
            "NoisePercent",
        ],
        columns="Technique",
        values="AverageRank",
    )
    .reset_index()
)

rank_display = rank_display[
    [
        "Metric",
        "NoisePercent",
        "RandomForest",
        "XGBoost",
        "LightGBM",
        "NaiveBayes",
    ]
]

try:
    from IPython.display import display

    display(
        rank_display
    )

except Exception:
    print(
        rank_display.to_string(
            index=False
        )
    )

print(
    "\nRQ3 stress-level preview (NOT the final robustness conclusion yet):"
)

try:
    from IPython.display import display

    display(
        stress_summary_preview
    )

except Exception:
    print(
        stress_summary_preview.to_string(
            index=False
        )
    )

print(
    "\nConditional Nemenyi post-hoc rows:"
)

print(
    len(
        nemenyi_results
    )
)

if len(
    nemenyi_results
) > 0:
    try:
        from IPython.display import display

        display(
            nemenyi_results
        )

    except Exception:
        print(
            nemenyi_results.to_string(
                index=False
            )
        )


# --------------------------------------------------------------------------------------------------
# 15. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 156
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 10 / RQ3 STEP 5A RESULT ==="
)

print(
    "=" * 156
)

print(
    "RQ3 inferential analysis frozen: True"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Treatments: RandomForest, XGBoost, LightGBM, NaiveBayes"
)

print(
    "\nInference:"
)

print(
    "Executed pre-registered Friedman tests:",
    len(
        omnibus_results
    ),
    "/ 18",
)

print(
    "Primary APFDc Holm-significant omnibus tests:",
    primary_significant_omnibus,
    "/ 9",
)

print(
    "Secondary APFD Holm-significant omnibus tests:",
    secondary_significant_omnibus,
    "/ 9",
)

print(
    "Conditional Nemenyi rows:",
    len(
        nemenyi_results
    ),
)

print(
    "Nemenyi critical difference:",
    critical[
        "CriticalDifference"
    ],
)

print(
    "\nRobustness conclusion isolation:"
)

print(
    "Unique robust winner declared: False"
)

print(
    "Final RQ3 answer package created: False"
)

print(
    "Final robustness synthesis deferred to Step 5B: True"
)

print(
    "\nReadback:"
)

print(
    "Readback failures:",
    readback_failures
)

print(
    "Max readback float difference:",
    float(
        readback_audit[
            "MaxAbsFloatDifference"
        ].max()
    ),
)

print(
    "\nIsolation:"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nRQ3 Step 5A checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha
)

print(
    "\nNext required step: "
    "RQ3 STEP 5B — FINAL ROBUSTNESS SYNTHESIS, DEGRADATION/RETENTION INTEGRATION, "
    "FIGURES, AND RQ3 ANSWER-PACKAGE FREEZE"
)

print(
    "\nSTATUS:",
    RQ3_STEP5A_STATUS
)

print(
    "=" * 156
)


=== THESIS GLOBAL ANALYSIS — CELL 10 / RQ3 STEP 5A: FRIEDMAN + HOLM + RANKS + CONDITIONAL NEMENYI ===

RQ3 Step 5A pre-write validation:


,Check,Expected,Actual,Pass
0,Step-1B checkpoint SHA-256,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,True
1,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
2,RQ2 Step-4B checkpoint SHA-256,8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d35...,8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d35...,True
3,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
4,Frozen RQ3 planned omnibus tests,18,18,True
5,Executed RQ3 Friedman tests,18,18,True
6,Primary APFDc Friedman tests,9,9,True
7,Secondary APFD Friedman tests,9,9,True
8,Average-rank rows,72,72,True
9,Project-rank rows,1728,1728,True



RQ3 Friedman + Holm omnibus results:


,Metric,NoisePercent,FriedmanStatistic,RawPValue,HolmAdjustedPValue,SignificantHolm,KendallsW,ConditionalNemenyiTriggered
0,APFDc,0,25.90,1.000844e-05,0.000090,True,0.359722,True
1,APFDc,5,16.15,1.056445e-03,0.008452,True,0.224306,True
2,APFDc,10,7.70,5.263627e-02,0.295294,False,0.106944,False
3,APFDc,15,7.85,4.921564e-02,0.295294,False,0.109028,False
4,APFDc,20,7.65,5.382702e-02,0.295294,False,0.106250,False
5,APFDc,25,2.85,4.153350e-01,0.830670,False,0.039583,False
6,APFDc,30,6.65,8.393092e-02,0.295294,False,0.092361,False
7,APFDc,40,2.05,5.620941e-01,0.830670,False,0.028472,False
8,APFDc,50,15.05,1.774415e-03,0.012421,True,0.209028,True
9,APFD,0,33.00,3.220673e-07,0.000003,True,0.458333,True



RQ3 average ranks (rank 1 = best):


...


RQ3 stress-level preview (NOT the final robustness conclusion yet):


,MetricRole,Metric,Technique,StressNoiseLevels,MeanStressAPFDValue,MeanStressAverageRank,ValueAt30,ValueAt40,ValueAt50,AverageRankAt30,AverageRankAt40,AverageRankAt50
0,PRIMARY,APFDc,RandomForest,"[30,40,50]",0.481279,2.680556,0.521425,0.479389,0.443023,2.875000,2.666667,2.500000
1,PRIMARY,APFDc,XGBoost,"[30,40,50]",0.480716,2.625000,0.528129,0.487266,0.426752,2.250000,2.666667,2.958333
2,PRIMARY,APFDc,LightGBM,"[30,40,50]",0.492925,2.472222,0.547930,0.498369,0.432476,2.083333,2.458333,2.875000
3,PRIMARY,APFDc,NaiveBayes,"[30,40,50]",0.517922,2.222222,0.537718,0.523996,0.492052,2.791667,2.208333,1.666667
4,SECONDARY,APFD,RandomForest,"[30,40,50]",0.491389,3.416667,0.549337,0.491977,0.432852,3.500000,3.625000,3.125000
5,SECONDARY,APFD,XGBoost,"[30,40,50]",0.523055,2.527778,0.583909,0.529730,0.455526,2.541667,2.583333,2.458333
6,SECONDARY,APFD,LightGBM,"[30,40,50]",0.532783,2.388889,0.598944,0.542784,0.456621,2.416667,2.333333,2.416667
7,SECONDARY,APFD,NaiveBayes,"[30,40,50]",0.588123,1.666667,0.668810,0.606167,0.489391,1.541667,1.458333,2.000000



Conditional Nemenyi post-hoc rows:
72


,MetricRole,Metric,NoisePercent,TechniqueA,TechniqueB,AverageRankA,AverageRankB,RankDifferenceAbs,CrossProjectMeanA,CrossProjectMeanB,MeanDifferenceAminusB,BetterByAverageRank,NemenyiSE,QDemšar,QStudentizedRange,NemenyiPValue,NemenyiAlpha,CriticalDifference,SignificantNemenyi
0,PRIMARY,APFDc,0,RandomForest,XGBoost,1.708333,2.208333,0.500000,0.803239,0.784217,0.019022,RandomForest,0.372678,1.341641,1.897367,0.536287,0.05,0.957422,False
1,PRIMARY,APFDc,0,RandomForest,LightGBM,1.708333,2.541667,0.833333,0.803239,0.762060,0.041179,RandomForest,0.372678,2.236068,3.162278,0.113581,0.05,0.957422,False
2,PRIMARY,APFDc,0,RandomForest,NaiveBayes,1.708333,3.541667,1.833333,0.803239,0.617603,0.185635,RandomForest,0.372678,4.919350,6.957011,0.000005,0.05,0.957422,True
3,PRIMARY,APFDc,0,XGBoost,LightGBM,2.208333,2.541667,0.333333,0.784217,0.762060,0.022157,XGBoost,0.372678,0.894427,1.264911,0.807757,0.05,0.957422,False
4,PRIMARY,APFDc,0,XGBoost,NaiveBayes,2.208333,3.541667,1.333333,0.784217,0.617603,0.166613,XGBoost,0.372678,3.577709,5.059644,0.001966,0.05,0.957422,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,SECONDARY,APFD,50,RandomForest,LightGBM,3.125000,2.416667,0.708333,0.432852,0.456621,-0.023769,LightGBM,0.372678,1.900658,2.687936,0.227697,0.05,0.957422,False
68,SECONDARY,APFD,50,RandomForest,NaiveBayes,3.125000,2.000000,1.125000,0.432852,0.489391,-0.056539,NaiveBayes,0.372678,3.018692,4.269075,0.013539,0.05,0.957422,True
69,SECONDARY,APFD,50,XGBoost,LightGBM,2.458333,2.416667,0.041667,0.455526,0.456621,-0.001095,LightGBM,0.372678,0.111803,0.158114,0.999500,0.05,0.957422,False
70,SECONDARY,APFD,50,XGBoost,NaiveBayes,2.458333,2.000000,0.458333,0.455526,0.489391,-0.033865,NaiveBayes,0.372678,1.229837,1.739253,0.607809,0.05,0.957422,False



=== THESIS GLOBAL ANALYSIS — CELL 10 / RQ3 STEP 5A RESULT ===
RQ3 inferential analysis frozen: True

Study unit:
Independent empirical unit: Project (N=24)
Treatments: RandomForest, XGBoost, LightGBM, NaiveBayes

Inference:
Executed pre-registered Friedman tests: 18 / 18
Primary APFDc Holm-significant omnibus tests: 3 / 9
Secondary APFD Holm-significant omnibus tests: 9 / 9
Conditional Nemenyi rows: 72
Nemenyi critical difference: 0.9574216132951187

Robustness conclusion isolation:
Unique robust winner declared: False
Final RQ3 answer package created: False
Final robustness synthesis deferred to Step 5B: True

Readback:
Readback failures: 0
Max readback float difference: 3.552713678800501e-15

Isolation:
Project outputs modified: False
Completion registry modified: False

Validation:
Checks: 24
Failed checks: 0

RQ3 Step 5A checkpoint:
/content/drive/MyDrive/Thesis_Experiment/Notes/global_analysis_rq3_step5a_checkpoint.json
Checkpoint SHA-256: f444b91715de86b10e1ec401e96723aa29ecb5c9

In [8]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 11 / RQ3 STEP 5B
# FINAL ROBUSTNESS SYNTHESIS + DEGRADATION/RETENTION INTEGRATION + FIGURES + ANSWER-PACKAGE FREEZE
# ==================================================================================================
#
# PURPOSE
# -------
# RQ3 Step 5A already froze:
#   - 18 pre-registered Friedman omnibus tests;
#   - Holm correction within the 9-test APFDc and APFD families;
#   - project-wise ranks and average ranks;
#   - conditional Nemenyi post-hoc comparisons;
#   - absolute performance summaries.
#
# RQ1 Step 3C already froze:
#   - clean-relative APFDc/APFD degradation and retention evidence.
#
# This Step 5B performs NO NEW INFERENTIAL TESTS.
#
# It implements the pre-declared robustness interpretation:
#
# PRIMARY RQ3 ROBUSTNESS CRITERION
# --------------------------------
#   Highest equally weighted cross-project mean APFDc at the HIGHEST tested noise level (50%).
#
# SUPPORTING / CONTEXTUAL EVIDENCE
# --------------------------------
#   - APFDc average rank at 50%;
#   - APFDc Nemenyi pairwise context at 50%;
#   - mean APFDc across stress levels 30/40/50%;
#   - mean rank across stress levels 30/40/50%;
#   - clean-relative APFDc degradation and retention at 30/40/50%;
#   - secondary APFD stress-level performance/ranks.
#
# IMPORTANT INTERPRETATION RULE
# -----------------------------
# A model may be the PRE-DECLARED primary robustness winner without being statistically
# superior to every other model in all Nemenyi comparisons. These are separate claims.
#
# This cell therefore records BOTH:
#   1. Primary robustness winner by the pre-declared 50%-noise absolute APFDc criterion.
#   2. Whether that winner is Nemenyi-significantly better than EACH of the other three
#      models at 50%.
#
# No winner is selected by post-hoc significance fishing.
# No project output or completion-registry row is modified.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


print("=" * 158)
print("=== THESIS GLOBAL ANALYSIS — CELL 11 / RQ3 STEP 5B: FINAL ROBUSTNESS SYNTHESIS + ANSWER PACKAGE ===")
print("=" * 158)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN UPSTREAM ANCHORS
# --------------------------------------------------------------------------------------------------

RQ3_STEP5A_STATUS = (
    "PASS_RQ3_STEP5A_FRIEDMAN_HOLM_AVERAGE_RANKS_AND_CONDITIONAL_NEMENYI_FROZEN"
)

EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256 = (
    "f444b91715de86b10e1ec401e96723aa29ecb5c920cd36f18122253a9c8123ac"
)

RQ1_STEP3C_STATUS = (
    "PASS_RQ1_STEP3C_FINAL_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256 = (
    "b4f3d10b77acb39200496542e8256d6213981fd636325487f190ce2dd07f35af"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

RQ3_STEP5B_STATUS = (
    "PASS_RQ3_STEP5B_FINAL_ROBUSTNESS_SYNTHESIS_FIGURES_AND_ANSWER_PACKAGE_FROZEN"
)

RQ3_STEP5B_CODE_REVISION = (
    "RQ3_STEP5B_V1_PREDECLARED_50PCT_APFDC_WINNER_WITH_NEMENYI_AND_RETENTION_CONTEXT"
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

STRESS_LEVELS = [
    30,
    40,
    50,
]

PRIMARY_METRIC = "APFDc"
SECONDARY_METRIC = "APFD"
PRIMARY_ROBUSTNESS_NOISE = 50
FLOAT_TOL = 1e-12


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

RQ3_STEP5A_CHECKPOINT = (
    NOTES
    / "global_analysis_rq3_step5a_checkpoint.json"
)

RQ1_STEP3C_CHECKPOINT = (
    NOTES
    / "global_analysis_rq1_step3c_checkpoint.json"
)

STEP5B_ROOT = (
    ANALYSIS_ROOT
    / "RQ3"
    / "Step_5B_Final_Robustness_Synthesis_and_Answer_Package"
)

TABLES_ROOT = (
    STEP5B_ROOT
    / "Tables"
)

FIGURES_ROOT = (
    STEP5B_ROOT
    / "Figures"
)

PRIMARY_ROBUSTNESS_SYNTHESIS_PATH = (
    TABLES_ROOT
    / "rq3_primary_robustness_synthesis.csv"
)

BEST_BY_NOISE_PATH = (
    TABLES_ROOT
    / "rq3_primary_apfdc_best_by_noise.csv"
)

WINNER_NEMENYI_50_PATH = (
    TABLES_ROOT
    / "rq3_primary_winner_nemenyi_at_50pct.csv"
)

OMNIBUS_COPY_PATH = (
    TABLES_ROOT
    / "rq3_friedman_holm_omnibus_results.csv"
)

AVERAGE_RANKS_COPY_PATH = (
    TABLES_ROOT
    / "rq3_average_ranks_by_noise.csv"
)

NEMENYI_COPY_PATH = (
    TABLES_ROOT
    / "rq3_conditional_nemenyi_posthoc.csv"
)

RQ1_FIGURE_DATA_COPY_PATH = (
    TABLES_ROOT
    / "rq3_clean_relative_evidence_from_rq1.csv"
)

FIGURE_APFDC_ABSOLUTE_PATH = (
    FIGURES_ROOT
    / "rq3_apfdc_absolute_performance.png"
)

FIGURE_APFDC_RANKS_PATH = (
    FIGURES_ROOT
    / "rq3_apfdc_average_ranks.png"
)

FIGURE_APFDC_DEGRADATION_PATH = (
    FIGURES_ROOT
    / "rq3_apfdc_clean_relative_degradation.png"
)

FIGURE_APFDC_RETENTION_PATH = (
    FIGURES_ROOT
    / "rq3_apfdc_retention.png"
)

FIGURE_APFD_SECONDARY_STRESS_PATH = (
    FIGURES_ROOT
    / "rq3_apfd_secondary_stress_performance.png"
)

VALIDATION_PATH = (
    STEP5B_ROOT
    / "rq3_step5b_validation.csv"
)

REPORT_PATH = (
    STEP5B_ROOT
    / "rq3_step5b_report.json"
)

STATUS_PATH = (
    STEP5B_ROOT
    / "rq3_step5b_status.json"
)

PACKAGE_MANIFEST_PATH = (
    STEP5B_ROOT
    / "rq3_final_package_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_rq3_step5b_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write("\n")

    os.replace(
        temporary,
        path,
    )


def atomic_copy(
    source,
    destination,
):
    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        f".{destination.name}.tmp_{os.getpid()}"
    )

    shutil.copy2(
        source,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def build_package_manifest(
    root,
    manifest_path,
):
    root = Path(root)
    manifest_path = Path(manifest_path)

    rows = []

    for path in sorted(
        (
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
            and candidate.resolve()
            != manifest_path.resolve()
        ),
        key=lambda candidate:
            candidate.relative_to(
                root
            ).as_posix(),
    ):
        rows.append({
            "RelativePath":
                path.relative_to(
                    root
                ).as_posix(),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def package_root_hash(
    manifest,
):
    ordered = (
        manifest[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def save_figure(
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()


def technique_order_key(
    series,
):
    if series.name == "Technique":
        return series.map({
            technique:
                index
            for index, technique
            in enumerate(
                ML_TECHNIQUES
            )
        })

    return series


def unique_best_technique(
    block,
    value_column,
    maximize=True,
    tolerance=FLOAT_TOL,
):
    values = pd.to_numeric(
        block[
            value_column
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    target = (
        float(
            np.max(
                values
            )
        )
        if maximize
        else float(
            np.min(
                values
            )
        )
    )

    if maximize:
        mask = np.isclose(
            values,
            target,
            atol=tolerance,
            rtol=0.0,
        )
    else:
        mask = np.isclose(
            values,
            target,
            atol=tolerance,
            rtol=0.0,
        )

    winners = block.loc[
        mask
    ].copy()

    winners = winners.sort_values(
        "Technique",
        key=technique_order_key,
        kind="mergesort",
    )

    return {
        "TargetValue":
            target,

        "WinnerCount":
            int(
                len(
                    winners
                )
            ),

        "Winners":
            winners[
                "Technique"
            ].astype(
                str
            ).tolist(),

        "PrimaryWinner":
            str(
                winners.iloc[
                    0
                ][
                    "Technique"
                ]
            ),
    }


def canonical_pair(
    row,
    winner,
):
    technique_a = str(
        row[
            "TechniqueA"
        ]
    )

    technique_b = str(
        row[
            "TechniqueB"
        ]
    )

    if winner not in {
        technique_a,
        technique_b,
    }:
        return None

    opponent = (
        technique_b
        if technique_a
        == winner
        else technique_a
    )

    if technique_a == winner:
        winner_rank = float(
            row[
                "AverageRankA"
            ]
        )

        opponent_rank = float(
            row[
                "AverageRankB"
            ]
        )

        winner_mean = float(
            row[
                "CrossProjectMeanA"
            ]
        )

        opponent_mean = float(
            row[
                "CrossProjectMeanB"
            ]
        )

    else:
        winner_rank = float(
            row[
                "AverageRankB"
            ]
        )

        opponent_rank = float(
            row[
                "AverageRankA"
            ]
        )

        winner_mean = float(
            row[
                "CrossProjectMeanB"
            ]
        )

        opponent_mean = float(
            row[
                "CrossProjectMeanA"
            ]
        )

    return {
        "Winner":
            winner,

        "Opponent":
            opponent,

        "WinnerAverageRank":
            winner_rank,

        "OpponentAverageRank":
            opponent_rank,

        "RankDifferenceAbs":
            float(
                row[
                    "RankDifferenceAbs"
                ]
            ),

        "WinnerCrossProjectMean":
            winner_mean,

        "OpponentCrossProjectMean":
            opponent_mean,

        "WinnerMinusOpponentMean":
            (
                winner_mean
                - opponent_mean
            ),

        "NemenyiPValue":
            float(
                row[
                    "NemenyiPValue"
                ]
            ),

        "CriticalDifference":
            float(
                row[
                    "CriticalDifference"
                ]
            ),

        "SignificantNemenyi":
            bool(
                row[
                    "SignificantNemenyi"
                ]
            ),
    }


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD + VERIFY FROZEN CHAIN
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "RQ3 Step 5B is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. Continue to the final global consolidation step."
    )

for path in [
    RQ3_STEP5A_CHECKPOINT,
    RQ1_STEP3C_CHECKPOINT,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required checkpoint missing: {path}"
        )

actual_step5a_sha = sha256_file(
    RQ3_STEP5A_CHECKPOINT
)

actual_rq1_step3c_sha = sha256_file(
    RQ1_STEP3C_CHECKPOINT
)

if (
    actual_step5a_sha
    != EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ3 Step-5A checkpoint SHA mismatch."
    )

if (
    actual_rq1_step3c_sha
    != EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "RQ1 Step-3C checkpoint SHA mismatch."
    )

step5a = load_json(
    RQ3_STEP5A_CHECKPOINT
)

rq1_step3c = load_json(
    RQ1_STEP3C_CHECKPOINT
)

if step5a.get(
    "Status"
) != RQ3_STEP5A_STATUS:
    raise RuntimeError(
        "RQ3 Step-5A checkpoint status is not PASS."
    )

if rq1_step3c.get(
    "Status"
) != RQ1_STEP3C_STATUS:
    raise RuntimeError(
        "RQ1 Step-3C checkpoint status is not PASS."
    )

if not bool(
    step5a.get(
        "ReadyForRQ3Step5B",
        False,
    )
):
    raise RuntimeError(
        "RQ3 Step-5A is not marked ready for Step 5B."
    )

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if (
    registry_sha_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry differs from final 24-project freeze."
    )

if STEP5B_ROOT.exists():
    shutil.rmtree(
        STEP5B_ROOT,
        ignore_errors=True,
    )

TABLES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 5. VERIFY + LOAD FROZEN RQ3 STEP-5A INPUTS
# --------------------------------------------------------------------------------------------------

omnibus_path = Path(
    step5a[
        "OmnibusResultsPath"
    ]
)

average_ranks_path = Path(
    step5a[
        "AverageRanksPath"
    ]
)

absolute_performance_path = Path(
    step5a[
        "AbsolutePerformancePath"
    ]
)

nemenyi_path = Path(
    step5a[
        "NemenyiResultsPath"
    ]
)

stress_preview_path = Path(
    step5a[
        "StressSummaryPreviewPath"
    ]
)

step5a_inputs = [
    (
        omnibus_path,
        str(
            step5a[
                "OmnibusResultsSHA256"
            ]
        ).lower(),
    ),
    (
        average_ranks_path,
        str(
            step5a[
                "AverageRanksSHA256"
            ]
        ).lower(),
    ),
    (
        absolute_performance_path,
        str(
            step5a[
                "AbsolutePerformanceSHA256"
            ]
        ).lower(),
    ),
    (
        nemenyi_path,
        str(
            step5a[
                "NemenyiResultsSHA256"
            ]
        ).lower(),
    ),
    (
        stress_preview_path,
        str(
            step5a[
                "StressSummaryPreviewSHA256"
            ]
        ).lower(),
    ),
]

for path, expected_sha in step5a_inputs:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ3 Step-5A input missing: {path}"
        )

    if sha256_file(
        path
    ) != expected_sha:
        raise RuntimeError(
            f"Frozen RQ3 Step-5A input SHA mismatch: {path.name}"
        )


omnibus = pd.read_csv(
    omnibus_path,
    low_memory=False,
)

average_ranks = pd.read_csv(
    average_ranks_path,
    low_memory=False,
)

absolute_performance = pd.read_csv(
    absolute_performance_path,
    low_memory=False,
)

nemenyi = pd.read_csv(
    nemenyi_path,
    low_memory=False,
)

stress_preview = pd.read_csv(
    stress_preview_path,
    low_memory=False,
)

if len(
    omnibus
) != 18:
    raise RuntimeError(
        "Frozen RQ3 omnibus table must contain 18 rows."
    )

if len(
    average_ranks
) != 72:
    raise RuntimeError(
        "Frozen RQ3 average-rank table must contain 72 rows."
    )

if len(
    absolute_performance
) != 72:
    raise RuntimeError(
        "Frozen RQ3 absolute-performance table must contain 72 rows."
    )

if len(
    nemenyi
) != 72:
    raise RuntimeError(
        "Frozen RQ3 Nemenyi table must contain 72 rows."
    )

if len(
    stress_preview
) != 8:
    raise RuntimeError(
        "Frozen RQ3 stress preview must contain 8 rows."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY + LOAD FROZEN RQ1 CLEAN-RELATIVE DATA
# --------------------------------------------------------------------------------------------------

rq1_figure_data_path = Path(
    rq1_step3c[
        "FigureDataPath"
    ]
)

expected_rq1_figure_data_sha = str(
    rq1_step3c[
        "FigureDataSHA256"
    ]
).lower()

if not rq1_figure_data_path.is_file():
    raise FileNotFoundError(
        f"Frozen RQ1 figure data missing: {rq1_figure_data_path}"
    )

if sha256_file(
    rq1_figure_data_path
) != expected_rq1_figure_data_sha:
    raise RuntimeError(
        "Frozen RQ1 figure-data SHA mismatch."
    )

rq1_figure_data = pd.read_csv(
    rq1_figure_data_path,
    low_memory=False,
)

if len(
    rq1_figure_data
) != 36:
    raise RuntimeError(
        "Frozen RQ1 figure data must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 7. DETERMINE THE PRE-DECLARED PRIMARY ROBUSTNESS WINNER AT 50% APFDC
# --------------------------------------------------------------------------------------------------

apfdc_50 = average_ranks.loc[
    average_ranks[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
    & pd.to_numeric(
        average_ranks[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        PRIMARY_ROBUSTNESS_NOISE
    )
].copy()

if len(
    apfdc_50
) != 4:
    raise RuntimeError(
        "50% APFDc block must contain 4 techniques."
    )

primary_winner_result = unique_best_technique(
    apfdc_50,
    "CrossProjectMean",
    maximize=True,
)

primary_winner = primary_winner_result[
    "PrimaryWinner"
]

primary_winner_count = primary_winner_result[
    "WinnerCount"
]

primary_winner_value = primary_winner_result[
    "TargetValue"
]

if primary_winner_count != 1:
    raise RuntimeError(
        "Pre-declared primary robustness criterion produced a tie at 50% APFDc."
    )


# --------------------------------------------------------------------------------------------------
# 8. BUILD PRIMARY ROBUSTNESS SYNTHESIS TABLE
# --------------------------------------------------------------------------------------------------

synthesis_rows = []

for technique in ML_TECHNIQUES:
    rank_block = average_ranks.loc[
        average_ranks[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFDc"
        )
        & average_ranks[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
    ].sort_values(
        "NoisePercent",
        kind="mergesort",
    )

    rq1_block = rq1_figure_data.loc[
        rq1_figure_data[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
    ].sort_values(
        "NoisePercent",
        kind="mergesort",
    )

    secondary_stress = stress_preview.loc[
        stress_preview[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFD"
        )
        & stress_preview[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
    ]

    if len(
        rank_block
    ) != 9:
        raise RuntimeError(
            f"{technique}: APFDc rank block !=9."
        )

    if len(
        rq1_block
    ) != 9:
        raise RuntimeError(
            f"{technique}: RQ1 clean-relative block !=9."
        )

    if len(
        secondary_stress
    ) != 1:
        raise RuntimeError(
            f"{technique}: APFD stress preview !=1."
        )

    def rank_value(
        noise,
        column,
    ):
        row = rank_block.loc[
            pd.to_numeric(
                rank_block[
                    "NoisePercent"
                ],
                errors="raise",
            ).astype(
                int
            ).eq(
                noise
            )
        ]

        if len(
            row
        ) != 1:
            raise RuntimeError(
                f"{technique}: missing APFDc rank/performance row at {noise}%."
            )

        return float(
            row.iloc[
                0
            ][
                column
            ]
        )

    def rq1_value(
        noise,
        column,
    ):
        row = rq1_block.loc[
            pd.to_numeric(
                rq1_block[
                    "NoisePercent"
                ],
                errors="raise",
            ).astype(
                int
            ).eq(
                noise
            )
        ]

        if len(
            row
        ) != 1:
            raise RuntimeError(
                f"{technique}: missing RQ1 row at {noise}%."
            )

        return float(
            row.iloc[
                0
            ][
                column
            ]
        )

    stress_apfdc_values = np.array(
        [
            rank_value(
                noise,
                "CrossProjectMean",
            )
            for noise in STRESS_LEVELS
        ],
        dtype=float,
    )

    stress_rank_values = np.array(
        [
            rank_value(
                noise,
                "AverageRank",
            )
            for noise in STRESS_LEVELS
        ],
        dtype=float,
    )

    stress_degradation_values = np.array(
        [
            rq1_value(
                noise,
                "MeanDegradationPctAPFDc",
            )
            for noise in STRESS_LEVELS
        ],
        dtype=float,
    )

    stress_retention_values = np.array(
        [
            rq1_value(
                noise,
                "MeanRetentionPctAPFDc",
            )
            for noise in STRESS_LEVELS
        ],
        dtype=float,
    )

    secondary_row = secondary_stress.iloc[
        0
    ]

    synthesis_rows.append({
        "Technique":
            technique,

        "PrimaryRobustnessWinner":
            bool(
                technique
                == primary_winner
            ),

        "CleanMeanAPFDc":
            rank_value(
                0,
                "CrossProjectMean",
            ),

        "APFDcAt30":
            rank_value(
                30,
                "CrossProjectMean",
            ),

        "APFDcAt40":
            rank_value(
                40,
                "CrossProjectMean",
            ),

        "APFDcAt50":
            rank_value(
                50,
                "CrossProjectMean",
            ),

        "MeanStressAPFDc_30_40_50":
            float(
                np.mean(
                    stress_apfdc_values
                )
            ),

        "AverageRankAt30":
            rank_value(
                30,
                "AverageRank",
            ),

        "AverageRankAt40":
            rank_value(
                40,
                "AverageRank",
            ),

        "AverageRankAt50":
            rank_value(
                50,
                "AverageRank",
            ),

        "MeanStressAverageRank_30_40_50":
            float(
                np.mean(
                    stress_rank_values
                )
            ),

        "MeanDegradationPctAPFDcAt30":
            rq1_value(
                30,
                "MeanDegradationPctAPFDc",
            ),

        "MeanDegradationPctAPFDcAt40":
            rq1_value(
                40,
                "MeanDegradationPctAPFDc",
            ),

        "MeanDegradationPctAPFDcAt50":
            rq1_value(
                50,
                "MeanDegradationPctAPFDc",
            ),

        "MeanStressDegradationPctAPFDc_30_40_50":
            float(
                np.mean(
                    stress_degradation_values
                )
            ),

        "MeanRetentionPctAPFDcAt30":
            rq1_value(
                30,
                "MeanRetentionPctAPFDc",
            ),

        "MeanRetentionPctAPFDcAt40":
            rq1_value(
                40,
                "MeanRetentionPctAPFDc",
            ),

        "MeanRetentionPctAPFDcAt50":
            rq1_value(
                50,
                "MeanRetentionPctAPFDc",
            ),

        "MeanStressRetentionPctAPFDc_30_40_50":
            float(
                np.mean(
                    stress_retention_values
                )
            ),

        "SecondaryMeanStressAPFD_30_40_50":
            float(
                secondary_row[
                    "MeanStressAPFDValue"
                ]
            ),

        "SecondaryMeanStressAverageRank_30_40_50":
            float(
                secondary_row[
                    "MeanStressAverageRank"
                ]
            ),
    })


primary_robustness_synthesis = pd.DataFrame(
    synthesis_rows
)

if len(
    primary_robustness_synthesis
) != 4:
    raise RuntimeError(
        "Primary robustness synthesis must contain 4 rows."
    )


# --------------------------------------------------------------------------------------------------
# 9. BEST MODEL BY ABSOLUTE APFDC AND AVERAGE RANK AT EACH TESTED NOISE
# --------------------------------------------------------------------------------------------------

best_by_noise_rows = []

for noise in NOISE_LEVELS:
    block = average_ranks.loc[
        average_ranks[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFDc"
        )
        & pd.to_numeric(
            average_ranks[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            noise
        )
    ].copy()

    if len(
        block
    ) != 4:
        raise RuntimeError(
            f"APFDc {noise}% best-by-noise block !=4."
        )

    best_absolute = unique_best_technique(
        block,
        "CrossProjectMean",
        maximize=True,
    )

    best_rank = unique_best_technique(
        block,
        "AverageRank",
        maximize=False,
    )

    omnibus_row = omnibus.loc[
        omnibus[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFDc"
        )
        & pd.to_numeric(
            omnibus[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            noise
        )
    ]

    if len(
        omnibus_row
    ) != 1:
        raise RuntimeError(
            f"APFDc {noise}% omnibus row missing."
        )

    omnibus_row = omnibus_row.iloc[
        0
    ]

    best_by_noise_rows.append({
        "NoisePercent":
            noise,

        "BestAbsoluteAPFDcTechnique":
            best_absolute[
                "PrimaryWinner"
            ],

        "BestAbsoluteAPFDc":
            best_absolute[
                "TargetValue"
            ],

        "BestAbsoluteTieCount":
            best_absolute[
                "WinnerCount"
            ],

        "BestAverageRankTechnique":
            best_rank[
                "PrimaryWinner"
            ],

        "BestAverageRank":
            best_rank[
                "TargetValue"
            ],

        "BestAverageRankTieCount":
            best_rank[
                "WinnerCount"
            ],

        "FriedmanHolmSignificant":
            bool(
                omnibus_row[
                    "SignificantHolm"
                ]
            ),

        "FriedmanHolmAdjustedPValue":
            float(
                omnibus_row[
                    "HolmAdjustedPValue"
                ]
            ),

        "KendallsW":
            float(
                omnibus_row[
                    "KendallsW"
                ]
            ),
    })


best_by_noise = pd.DataFrame(
    best_by_noise_rows
)


# --------------------------------------------------------------------------------------------------
# 10. PRIMARY WINNER'S 50% NEMENYI CONTEXT
# --------------------------------------------------------------------------------------------------

nemenyi_50 = nemenyi.loc[
    nemenyi[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
    & pd.to_numeric(
        nemenyi[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        50
    )
].copy()

if len(
    nemenyi_50
) != 6:
    raise RuntimeError(
        "APFDc 50% Nemenyi block must contain 6 pairwise rows."
    )

winner_nemenyi_rows = []

for _, row in nemenyi_50.iterrows():
    pair = canonical_pair(
        row,
        primary_winner,
    )

    if pair is not None:
        winner_nemenyi_rows.append(
            pair
        )

winner_nemenyi_50 = pd.DataFrame(
    winner_nemenyi_rows
)

if len(
    winner_nemenyi_50
) != 3:
    raise RuntimeError(
        "Primary winner must have exactly 3 Nemenyi comparisons at 50%."
    )

winner_nemenyi_50 = (
    winner_nemenyi_50.sort_values(
        "Opponent",
        key=lambda series:
            series.map({
                technique:
                    index
                for index, technique
                in enumerate(
                    ML_TECHNIQUES
                )
            }),
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

winner_significant_vs_all = bool(
    winner_nemenyi_50[
        "SignificantNemenyi"
    ].astype(
        bool
    ).all()
)

winner_significant_opponents = (
    winner_nemenyi_50.loc[
        winner_nemenyi_50[
            "SignificantNemenyi"
        ].astype(
            bool
        ),
        "Opponent",
    ]
    .astype(
        str
    )
    .tolist()
)

winner_nonsignificant_opponents = (
    winner_nemenyi_50.loc[
        ~winner_nemenyi_50[
            "SignificantNemenyi"
        ].astype(
            bool
        ),
        "Opponent",
    ]
    .astype(
        str
    )
    .tolist()
)


# --------------------------------------------------------------------------------------------------
# 11. SUPPORTING WINNER IDENTITIES
# --------------------------------------------------------------------------------------------------

clean_block = primary_robustness_synthesis.copy()

clean_best = unique_best_technique(
    clean_block.rename(
        columns={
            "CleanMeanAPFDc":
                "CrossProjectMean",
        }
    ),
    "CrossProjectMean",
    maximize=True,
)

stress_absolute_best = unique_best_technique(
    primary_robustness_synthesis.rename(
        columns={
            "MeanStressAPFDc_30_40_50":
                "CrossProjectMean",
        }
    ),
    "CrossProjectMean",
    maximize=True,
)

stress_rank_best = unique_best_technique(
    primary_robustness_synthesis.rename(
        columns={
            "MeanStressAverageRank_30_40_50":
                "AverageRank",
        }
    ),
    "AverageRank",
    maximize=False,
)

retention_50_best = unique_best_technique(
    primary_robustness_synthesis.rename(
        columns={
            "MeanRetentionPctAPFDcAt50":
                "Retention",
        }
    ),
    "Retention",
    maximize=True,
)

degradation_50_best = unique_best_technique(
    primary_robustness_synthesis.rename(
        columns={
            "MeanDegradationPctAPFDcAt50":
                "Degradation",
        }
    ),
    "Degradation",
    maximize=False,
)

secondary_stress_best = unique_best_technique(
    primary_robustness_synthesis.rename(
        columns={
            "SecondaryMeanStressAPFD_30_40_50":
                "CrossProjectMean",
        }
    ),
    "CrossProjectMean",
    maximize=True,
)


# --------------------------------------------------------------------------------------------------
# 12. WRITE FROZEN SYNTHESIS TABLES / COPIES
# --------------------------------------------------------------------------------------------------

atomic_csv(
    PRIMARY_ROBUSTNESS_SYNTHESIS_PATH,
    primary_robustness_synthesis,
)

atomic_csv(
    BEST_BY_NOISE_PATH,
    best_by_noise,
)

atomic_csv(
    WINNER_NEMENYI_50_PATH,
    winner_nemenyi_50,
)

atomic_copy(
    omnibus_path,
    OMNIBUS_COPY_PATH,
)

atomic_copy(
    average_ranks_path,
    AVERAGE_RANKS_COPY_PATH,
)

atomic_copy(
    nemenyi_path,
    NEMENYI_COPY_PATH,
)

atomic_copy(
    rq1_figure_data_path,
    RQ1_FIGURE_DATA_COPY_PATH,
)


# --------------------------------------------------------------------------------------------------
# 13. FIGURE 1 — ABSOLUTE APFDC PERFORMANCE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        average_ranks.loc[
            average_ranks[
                "Metric"
            ].astype(
                str
            ).eq(
                "APFDc"
            )
            & average_ranks[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "CrossProjectMean"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Cross-project mean APFDc"
)

plt.title(
    "RQ3 — Absolute APFDc performance across noise levels"
)

plt.xticks(
    NOISE_LEVELS
)

plt.ylim(
    0.0,
    1.0,
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFDC_ABSOLUTE_PATH
)


# --------------------------------------------------------------------------------------------------
# 14. FIGURE 2 — AVERAGE APFDC RANKS
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        average_ranks.loc[
            average_ranks[
                "Metric"
            ].astype(
                str
            ).eq(
                "APFDc"
            )
            & average_ranks[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "AverageRank"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Average project rank (1 = best)"
)

plt.title(
    "RQ3 — Average APFDc ranks"
)

plt.xticks(
    NOISE_LEVELS
)

plt.ylim(
    4.1,
    0.9,
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFDC_RANKS_PATH
)


# --------------------------------------------------------------------------------------------------
# 15. FIGURE 3 — APFDC CLEAN-RELATIVE DEGRADATION
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        rq1_figure_data.loc[
            rq1_figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "MeanDegradationPctAPFDc"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.axhline(
    0.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Mean APFDc degradation from clean (%)"
)

plt.title(
    "RQ3 — Clean-relative APFDc degradation"
)

plt.xticks(
    NOISE_LEVELS
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFDC_DEGRADATION_PATH
)


# --------------------------------------------------------------------------------------------------
# 16. FIGURE 4 — APFDC RETENTION
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for technique in ML_TECHNIQUES:
    block = (
        rq1_figure_data.loc[
            rq1_figure_data[
                "Technique"
            ].astype(
                str
            ).eq(
                technique
            )
        ]
        .sort_values(
            "NoisePercent",
            kind="mergesort",
        )
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "MeanRetentionPctAPFDc"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.axhline(
    100.0,
    linewidth=1.0,
)

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "APFDc retention relative to clean (%)"
)

plt.title(
    "RQ3 — APFDc retention under training-label noise"
)

plt.xticks(
    NOISE_LEVELS
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFDC_RETENTION_PATH
)


# --------------------------------------------------------------------------------------------------
# 17. FIGURE 5 — SECONDARY APFD STRESS-LEVEL PERFORMANCE
# --------------------------------------------------------------------------------------------------

plt.figure(
    figsize=(
        8.8,
        5.6,
    )
)

for technique in ML_TECHNIQUES:
    block = average_ranks.loc[
        average_ranks[
            "Metric"
        ].astype(
            str
        ).eq(
            "APFD"
        )
        & average_ranks[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
        & average_ranks[
            "NoisePercent"
        ].isin(
            STRESS_LEVELS
        )
    ].sort_values(
        "NoisePercent",
        kind="mergesort",
    )

    plt.plot(
        pd.to_numeric(
            block[
                "NoisePercent"
            ],
            errors="raise",
        ),
        pd.to_numeric(
            block[
                "CrossProjectMean"
            ],
            errors="raise",
        ),
        marker="o",
        label=technique,
    )

plt.xlabel(
    "Training-label noise (%)"
)

plt.ylabel(
    "Cross-project mean APFD"
)

plt.title(
    "RQ3 — Secondary APFD performance in the stress region"
)

plt.xticks(
    STRESS_LEVELS
)

plt.ylim(
    0.0,
    1.0,
)

plt.grid(
    axis="y",
    alpha=0.25,
)

plt.legend(
    frameon=False,
)

save_figure(
    FIGURE_APFD_SECONDARY_STRESS_PATH
)


# --------------------------------------------------------------------------------------------------
# 18. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []

registry_sha_after = sha256_file(
    REGISTRY
)

primary_50_omnibus = omnibus.loc[
    omnibus[
        "Metric"
    ].astype(
        str
    ).eq(
        "APFDc"
    )
    & pd.to_numeric(
        omnibus[
            "NoisePercent"
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        50
    )
]

if len(
    primary_50_omnibus
) != 1:
    raise RuntimeError(
        "APFDc 50% omnibus row missing."
    )

primary_50_omnibus = primary_50_omnibus.iloc[
    0
]

add_check(
    checks,
    "RQ3 Step-5A checkpoint SHA-256",
    EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256,
    actual_step5a_sha,
    actual_step5a_sha
    == EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ1 Step-3C checkpoint SHA-256",
    EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,
    actual_rq1_step3c_sha,
    actual_rq1_step3c_sha
    == EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "RQ3 frozen omnibus rows",
    18,
    len(
        omnibus
    ),
    len(
        omnibus
    )
    == 18,
)

add_check(
    checks,
    "RQ3 frozen average-rank rows",
    72,
    len(
        average_ranks
    ),
    len(
        average_ranks
    )
    == 72,
)

add_check(
    checks,
    "RQ3 frozen Nemenyi rows",
    72,
    len(
        nemenyi
    ),
    len(
        nemenyi
    )
    == 72,
)

add_check(
    checks,
    "Primary robustness synthesis rows",
    4,
    len(
        primary_robustness_synthesis
    ),
    len(
        primary_robustness_synthesis
    )
    == 4,
)

add_check(
    checks,
    "Best-by-noise rows",
    9,
    len(
        best_by_noise
    ),
    len(
        best_by_noise
    )
    == 9,
)

add_check(
    checks,
    "Primary-winner 50% Nemenyi rows",
    3,
    len(
        winner_nemenyi_50
    ),
    len(
        winner_nemenyi_50
    )
    == 3,
)

add_check(
    checks,
    "Pre-declared primary robustness winner",
    "NaiveBayes",
    primary_winner,
    primary_winner
    == "NaiveBayes",
)

add_check(
    checks,
    "Primary robustness winner count",
    1,
    primary_winner_count,
    primary_winner_count
    == 1,
)

add_check(
    checks,
    "APFDc 50% Friedman Holm-significant",
    True,
    bool(
        primary_50_omnibus[
            "SignificantHolm"
        ]
    ),
    bool(
        primary_50_omnibus[
            "SignificantHolm"
        ]
    ),
)

add_check(
    checks,
    "Clean APFDc winner",
    "RandomForest",
    clean_best[
        "PrimaryWinner"
    ],
    clean_best[
        "PrimaryWinner"
    ]
    == "RandomForest",
)

add_check(
    checks,
    "Stress-region mean APFDc winner",
    "NaiveBayes",
    stress_absolute_best[
        "PrimaryWinner"
    ],
    stress_absolute_best[
        "PrimaryWinner"
    ]
    == "NaiveBayes",
)

add_check(
    checks,
    "Stress-region mean APFDc-rank winner",
    "NaiveBayes",
    stress_rank_best[
        "PrimaryWinner"
    ],
    stress_rank_best[
        "PrimaryWinner"
    ]
    == "NaiveBayes",
)

add_check(
    checks,
    "50% APFDc retention winner",
    "NaiveBayes",
    retention_50_best[
        "PrimaryWinner"
    ],
    retention_50_best[
        "PrimaryWinner"
    ]
    == "NaiveBayes",
)

add_check(
    checks,
    "50% APFDc lowest degradation",
    "NaiveBayes",
    degradation_50_best[
        "PrimaryWinner"
    ],
    degradation_50_best[
        "PrimaryWinner"
    ]
    == "NaiveBayes",
)

add_check(
    checks,
    "Secondary APFD stress-region winner",
    "NaiveBayes",
    secondary_stress_best[
        "PrimaryWinner"
    ],
    secondary_stress_best[
        "PrimaryWinner"
    ]
    == "NaiveBayes",
)

expected_significant_opponents = sorted(
    [
        "XGBoost",
        "LightGBM",
    ]
)

actual_significant_opponents = sorted(
    winner_significant_opponents
)

add_check(
    checks,
    "NaiveBayes significant Nemenyi opponents at 50% APFDc",
    expected_significant_opponents,
    actual_significant_opponents,
    actual_significant_opponents
    == expected_significant_opponents,
)

add_check(
    checks,
    "NaiveBayes non-significant Nemenyi opponents at 50% APFDc",
    [
        "RandomForest"
    ],
    winner_nonsignificant_opponents,
    winner_nonsignificant_opponents
    == [
        "RandomForest"
    ],
)

add_check(
    checks,
    "Winner Nemenyi-significant vs all three at 50%",
    False,
    winner_significant_vs_all,
    winner_significant_vs_all
    is False,
)

for figure_path in [
    FIGURE_APFDC_ABSOLUTE_PATH,
    FIGURE_APFDC_RANKS_PATH,
    FIGURE_APFDC_DEGRADATION_PATH,
    FIGURE_APFDC_RETENTION_PATH,
    FIGURE_APFD_SECONDARY_STRESS_PATH,
]:
    add_check(
        checks,
        f"Figure exists: {figure_path.name}",
        True,
        figure_path.is_file(),
        figure_path.is_file()
        and figure_path.stat().st_size
        > 0,
    )

add_check(
    checks,
    "New inferential tests executed",
    0,
    0,
    True,
)

add_check(
    checks,
    "New p-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nRQ3 Step 5B pre-freeze validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "RQ3 STEP 5B VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 19. REPORT / STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

primary_winner_row = primary_robustness_synthesis.loc[
    primary_robustness_synthesis[
        "Technique"
    ].eq(
        primary_winner
    )
].iloc[
    0
]

report = {
    "Step":
        "RQ3_STEP_5B",

    "Status":
        RQ3_STEP5B_STATUS,

    "CodeRevision":
        RQ3_STEP5B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ3Step5ACheckpointSHA256":
        EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256,

    "RQ1Step3CCheckpointSHA256":
        EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "PrimaryRobustnessCriterion":
        (
            "highest equally weighted cross-project mean APFDc "
            "at highest tested noise level (50%)"
        ),

    "PrimaryRobustnessWinner":
        primary_winner,

    "PrimaryRobustnessWinnerUnique":
        True,

    "PrimaryRobustnessNoisePercent":
        50,

    "PrimaryWinnerAPFDcAt50":
        float(
            primary_winner_row[
                "APFDcAt50"
            ]
        ),

    "PrimaryWinnerAverageRankAt50":
        float(
            primary_winner_row[
                "AverageRankAt50"
            ]
        ),

    "PrimaryWinnerMeanStressAPFDc":
        float(
            primary_winner_row[
                "MeanStressAPFDc_30_40_50"
            ]
        ),

    "PrimaryWinnerMeanStressAverageRank":
        float(
            primary_winner_row[
                "MeanStressAverageRank_30_40_50"
            ]
        ),

    "PrimaryWinnerMeanDegradationPctAPFDcAt50":
        float(
            primary_winner_row[
                "MeanDegradationPctAPFDcAt50"
            ]
        ),

    "PrimaryWinnerMeanRetentionPctAPFDcAt50":
        float(
            primary_winner_row[
                "MeanRetentionPctAPFDcAt50"
            ]
        ),

    "APFDc50FriedmanHolmSignificant":
        bool(
            primary_50_omnibus[
                "SignificantHolm"
            ]
        ),

    "APFDc50HolmAdjustedPValue":
        float(
            primary_50_omnibus[
                "HolmAdjustedPValue"
            ]
        ),

    "PrimaryWinnerNemenyiSignificantOpponentsAt50":
        winner_significant_opponents,

    "PrimaryWinnerNemenyiNonSignificantOpponentsAt50":
        winner_nonsignificant_opponents,

    "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50":
        winner_significant_vs_all,

    "CleanAPFDcWinner":
        clean_best[
            "PrimaryWinner"
        ],

    "StressMeanAPFDcWinner":
        stress_absolute_best[
            "PrimaryWinner"
        ],

    "StressMeanRankWinner":
        stress_rank_best[
            "PrimaryWinner"
        ],

    "RetentionAt50Winner":
        retention_50_best[
            "PrimaryWinner"
        ],

    "LowestDegradationAt50Winner":
        degradation_50_best[
            "PrimaryWinner"
        ],

    "SecondaryAPFDStressWinner":
        secondary_stress_best[
            "PrimaryWinner"
        ],

    "NewInferentialTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 6A — FINAL CROSS-RQ CONSOLIDATION, "
            "SENSITIVITY/AUDIT TABLES, AND THESIS-READY GLOBAL RESULTS PACKAGE"
        ),
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            RQ3_STEP5B_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "RQ3AnswerPackageFrozen":
            True,

        "PrimaryRobustnessWinner":
            primary_winner,

        "ReadyForFinalGlobalConsolidation":
            True,
    },
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 20. FINAL PACKAGE MANIFEST + ROOT HASH
# --------------------------------------------------------------------------------------------------

package_manifest = build_package_manifest(
    STEP5B_ROOT,
    PACKAGE_MANIFEST_PATH,
)

atomic_csv(
    PACKAGE_MANIFEST_PATH,
    package_manifest,
)

package_root_sha = package_root_hash(
    package_manifest
)

package_files = int(
    len(
        package_manifest
    )
)

package_bytes = int(
    package_manifest[
        "Bytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 21. CHECKPOINT
# --------------------------------------------------------------------------------------------------

if sha256_file(
    REGISTRY
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during RQ3 Step 5B."
    )

checkpoint = {
    "CheckpointType":
        "RQ3_FINAL_ROBUSTNESS_SYNTHESIS_FIGURES_AND_ANSWER_PACKAGE",

    "Status":
        RQ3_STEP5B_STATUS,

    "CodeRevision":
        RQ3_STEP5B_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "RQ3Step5ACheckpointSHA256":
        EXPECTED_RQ3_STEP5A_CHECKPOINT_SHA256,

    "RQ1Step3CCheckpointSHA256":
        EXPECTED_RQ1_STEP3C_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "PrimaryRobustnessCriterion":
        (
            "highest equally weighted cross-project mean APFDc "
            "at highest tested noise level (50%)"
        ),

    "PrimaryRobustnessWinner":
        primary_winner,

    "PrimaryRobustnessWinnerUnique":
        True,

    "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50":
        winner_significant_vs_all,

    "PackageRoot":
        str(
            STEP5B_ROOT
        ),

    "PackageManifestPath":
        str(
            PACKAGE_MANIFEST_PATH
        ),

    "PackageManifestSHA256":
        sha256_file(
            PACKAGE_MANIFEST_PATH
        ),

    "PackageRootSHA256":
        package_root_sha,

    "PackageFiles":
        package_files,

    "PackageBytes":
        package_bytes,

    "PrimaryRobustnessSynthesisPath":
        str(
            PRIMARY_ROBUSTNESS_SYNTHESIS_PATH
        ),

    "PrimaryRobustnessSynthesisSHA256":
        sha256_file(
            PRIMARY_ROBUSTNESS_SYNTHESIS_PATH
        ),

    "BestByNoisePath":
        str(
            BEST_BY_NOISE_PATH
        ),

    "BestByNoiseSHA256":
        sha256_file(
            BEST_BY_NOISE_PATH
        ),

    "WinnerNemenyi50Path":
        str(
            WINNER_NEMENYI_50_PATH
        ),

    "WinnerNemenyi50SHA256":
        sha256_file(
            WINNER_NEMENYI_50_PATH
        ),

    "OmnibusResultsPath":
        str(
            OMNIBUS_COPY_PATH
        ),

    "OmnibusResultsSHA256":
        sha256_file(
            OMNIBUS_COPY_PATH
        ),

    "AverageRanksPath":
        str(
            AVERAGE_RANKS_COPY_PATH
        ),

    "AverageRanksSHA256":
        sha256_file(
            AVERAGE_RANKS_COPY_PATH
        ),

    "NemenyiResultsPath":
        str(
            NEMENYI_COPY_PATH
        ),

    "NemenyiResultsSHA256":
        sha256_file(
            NEMENYI_COPY_PATH
        ),

    "RQ1CleanRelativeEvidencePath":
        str(
            RQ1_FIGURE_DATA_COPY_PATH
        ),

    "RQ1CleanRelativeEvidenceSHA256":
        sha256_file(
            RQ1_FIGURE_DATA_COPY_PATH
        ),

    "NewInferentialTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "ReadyForFinalGlobalConsolidation":
        True,

    "NextRequiredStep":
        (
            "GLOBAL ANALYSIS STEP 6A — FINAL CROSS-RQ CONSOLIDATION, "
            "SENSITIVITY/AUDIT TABLES, AND THESIS-READY GLOBAL RESULTS PACKAGE"
        ),
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 22. USER-VISIBLE TABLES
# --------------------------------------------------------------------------------------------------

print(
    "\nRQ3 FINAL primary robustness synthesis:"
)

try:
    from IPython.display import display

    display(
        primary_robustness_synthesis
    )

except Exception:
    print(
        primary_robustness_synthesis.to_string(
            index=False
        )
    )

print(
    "\nRQ3 APFDc best technique by tested noise:"
)

try:
    from IPython.display import display

    display(
        best_by_noise
    )

except Exception:
    print(
        best_by_noise.to_string(
            index=False
        )
    )

print(
    "\nPrimary winner's 50% APFDc Nemenyi context:"
)

try:
    from IPython.display import display

    display(
        winner_nemenyi_50
    )

except Exception:
    print(
        winner_nemenyi_50.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 23. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 158
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 11 / RQ3 STEP 5B RESULT ==="
)

print(
    "=" * 158
)

print(
    "RQ3 final answer package frozen: True"
)

print(
    "\nPrimary robustness criterion:"
)

print(
    "Highest equally weighted cross-project mean APFDc at 50% noise"
)

print(
    "\nPrimary robustness winner:",
    primary_winner
)

print(
    "Winner unique by primary criterion:",
    primary_winner_count
    == 1
)

print(
    "Winner APFDc at 50%:",
    float(
        primary_winner_row[
            "APFDcAt50"
        ]
    ),
)

print(
    "Winner average rank at 50%:",
    float(
        primary_winner_row[
            "AverageRankAt50"
        ]
    ),
)

print(
    "Winner APFDc retention at 50%:",
    float(
        primary_winner_row[
            "MeanRetentionPctAPFDcAt50"
        ]
    ),
)

print(
    "Winner APFDc degradation at 50%:",
    float(
        primary_winner_row[
            "MeanDegradationPctAPFDcAt50"
        ]
    ),
)

print(
    "\n50% inferential context:"
)

print(
    "Friedman Holm-significant:",
    bool(
        primary_50_omnibus[
            "SignificantHolm"
        ]
    ),
)

print(
    "Holm-adjusted p-value:",
    float(
        primary_50_omnibus[
            "HolmAdjustedPValue"
        ]
    ),
)

print(
    "Nemenyi-significant opponents:",
    winner_significant_opponents
)

print(
    "Nemenyi-non-significant opponents:",
    winner_nonsignificant_opponents
)

print(
    "Winner significantly better than all three competitors:",
    winner_significant_vs_all
)

print(
    "\nSupporting robustness checks:"
)

print(
    "Clean APFDc winner:",
    clean_best[
        "PrimaryWinner"
    ]
)

print(
    "30/40/50 mean APFDc winner:",
    stress_absolute_best[
        "PrimaryWinner"
    ]
)

print(
    "30/40/50 mean APFDc rank winner:",
    stress_rank_best[
        "PrimaryWinner"
    ]
)

print(
    "50% retention winner:",
    retention_50_best[
        "PrimaryWinner"
    ]
)

print(
    "50% lowest-degradation winner:",
    degradation_50_best[
        "PrimaryWinner"
    ]
)

print(
    "Secondary APFD stress-region winner:",
    secondary_stress_best[
        "PrimaryWinner"
    ]
)

print(
    "\nFrozen outputs:"
)

print(
    "Primary synthesis rows:",
    len(
        primary_robustness_synthesis
    ),
)

print(
    "Best-by-noise rows:",
    len(
        best_by_noise
    ),
)

print(
    "Winner Nemenyi rows:",
    len(
        winner_nemenyi_50
    ),
)

print(
    "Figures:",
    5
)

print(
    "Package files:",
    package_files
)

print(
    "Package bytes:",
    package_bytes
)

print(
    "Package root SHA-256:",
    package_root_sha
)

print(
    "\nIsolation:"
)

print(
    "New inferential tests executed: 0"
)

print(
    "New p-values computed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nRQ3 Step 5B checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha
)

print(
    "\nNext required step: "
    "GLOBAL ANALYSIS STEP 6A — FINAL CROSS-RQ CONSOLIDATION, "
    "SENSITIVITY/AUDIT TABLES, AND THESIS-READY GLOBAL RESULTS PACKAGE"
)

print(
    "\nSTATUS:",
    RQ3_STEP5B_STATUS
)

print(
    "=" * 158
)


=== THESIS GLOBAL ANALYSIS — CELL 11 / RQ3 STEP 5B: FINAL ROBUSTNESS SYNTHESIS + ANSWER PACKAGE ===

RQ3 Step 5B pre-freeze validation:


,Check,Expected,Actual,Pass
0,RQ3 Step-5A checkpoint SHA-256,f444b91715de86b10e1ec401e96723aa29ecb5c920cd36...,f444b91715de86b10e1ec401e96723aa29ecb5c920cd36...,True
1,RQ1 Step-3C checkpoint SHA-256,b4f3d10b77acb39200496542e8256d6213981fd6363254...,b4f3d10b77acb39200496542e8256d6213981fd6363254...,True
2,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
3,RQ3 frozen omnibus rows,18,18,True
4,RQ3 frozen average-rank rows,72,72,True
5,RQ3 frozen Nemenyi rows,72,72,True
6,Primary robustness synthesis rows,4,4,True
7,Best-by-noise rows,9,9,True
8,Primary-winner 50% Nemenyi rows,3,3,True
9,Pre-declared primary robustness winner,NaiveBayes,NaiveBayes,True



RQ3 FINAL primary robustness synthesis:


,Technique,PrimaryRobustnessWinner,CleanMeanAPFDc,APFDcAt30,APFDcAt40,APFDcAt50,MeanStressAPFDc_30_40_50,AverageRankAt30,AverageRankAt40,AverageRankAt50,...,MeanDegradationPctAPFDcAt30,MeanDegradationPctAPFDcAt40,MeanDegradationPctAPFDcAt50,MeanStressDegradationPctAPFDc_30_40_50,MeanRetentionPctAPFDcAt30,MeanRetentionPctAPFDcAt40,MeanRetentionPctAPFDcAt50,MeanStressRetentionPctAPFDc_30_40_50,SecondaryMeanStressAPFD_30_40_50,SecondaryMeanStressAverageRank_30_40_50
0,RandomForest,False,0.803239,0.521425,0.479389,0.443023,0.481279,2.875000,2.666667,2.500000,...,33.051305,38.036766,42.255380,37.781150,66.948695,61.963234,57.744620,62.218850,0.491389,3.416667
1,XGBoost,False,0.784217,0.528129,0.487266,0.426752,0.480716,2.250000,2.666667,2.958333,...,31.075530,35.851995,43.012815,36.646780,68.924470,64.148005,56.987185,63.353220,0.523055,2.527778
2,LightGBM,False,0.762060,0.547930,0.498369,0.432476,0.492925,2.083333,2.458333,2.875000,...,25.732400,31.739501,39.608001,32.359967,74.267600,68.260499,60.391999,67.640033,0.532783,2.388889
3,NaiveBayes,True,0.617603,0.537718,0.523996,0.492052,0.517922,2.791667,2.208333,1.666667,...,7.369633,8.023051,12.761964,9.384883,92.630367,91.976949,87.238036,90.615117,0.588123,1.666667



RQ3 APFDc best technique by tested noise:


,NoisePercent,BestAbsoluteAPFDcTechnique,BestAbsoluteAPFDc,BestAbsoluteTieCount,BestAverageRankTechnique,BestAverageRank,BestAverageRankTieCount,FriedmanHolmSignificant,FriedmanHolmAdjustedPValue,KendallsW
0,0,RandomForest,0.803239,1,RandomForest,1.708333,1,True,0.000090,0.359722
1,5,XGBoost,0.684539,1,XGBoost,2.166667,1,True,0.008452,0.224306
2,10,LightGBM,0.647043,1,XGBoost,2.208333,1,False,0.295294,0.106944
3,15,LightGBM,0.627722,1,LightGBM,1.958333,1,False,0.295294,0.109028
4,20,LightGBM,0.598752,1,LightGBM,2.000000,1,False,0.295294,0.106250
5,25,LightGBM,0.572690,1,LightGBM,2.125000,1,False,0.830670,0.039583
6,30,LightGBM,0.547930,1,LightGBM,2.083333,1,False,0.295294,0.092361
7,40,NaiveBayes,0.523996,1,NaiveBayes,2.208333,1,False,0.830670,0.028472
8,50,NaiveBayes,0.492052,1,NaiveBayes,1.666667,1,True,0.012421,0.209028



Primary winner's 50% APFDc Nemenyi context:


,Winner,Opponent,WinnerAverageRank,OpponentAverageRank,RankDifferenceAbs,WinnerCrossProjectMean,OpponentCrossProjectMean,WinnerMinusOpponentMean,NemenyiPValue,CriticalDifference,SignificantNemenyi
0,NaiveBayes,RandomForest,1.666667,2.500000,0.833333,0.492052,0.443023,0.049028,0.113581,0.957422,False
1,NaiveBayes,XGBoost,1.666667,2.958333,1.291667,0.492052,0.426752,0.065300,0.002969,0.957422,True
2,NaiveBayes,LightGBM,1.666667,2.875000,1.208333,0.492052,0.432476,0.059576,0.006512,0.957422,True



=== THESIS GLOBAL ANALYSIS — CELL 11 / RQ3 STEP 5B RESULT ===
RQ3 final answer package frozen: True

Primary robustness criterion:
Highest equally weighted cross-project mean APFDc at 50% noise

Primary robustness winner: NaiveBayes
Winner unique by primary criterion: True
Winner APFDc at 50%: 0.4920519147301996
Winner average rank at 50%: 1.6666666666666667
Winner APFDc retention at 50%: 87.23803558481858
Winner APFDc degradation at 50%: 12.761964415181424

50% inferential context:
Friedman Holm-significant: True
Holm-adjusted p-value: 0.012420906431236
Nemenyi-significant opponents: ['XGBoost', 'LightGBM']
Nemenyi-non-significant opponents: ['RandomForest']
Winner significantly better than all three competitors: False

Supporting robustness checks:
Clean APFDc winner: RandomForest
30/40/50 mean APFDc winner: NaiveBayes
30/40/50 mean APFDc rank winner: NaiveBayes
50% retention winner: NaiveBayes
50% lowest-degradation winner: NaiveBayes
Secondary APFD stress-region winner: NaiveBayes

In [9]:
# ==================================================================================================
# THESIS GLOBAL ANALYSIS — CELL 12 / STEP 6A
# FINAL CROSS-RQ CONSOLIDATION + SENSITIVITY/AUDIT TABLES + THESIS-READY GLOBAL RESULTS PACKAGE
# ==================================================================================================
#
# PURPOSE
# -------
# RQ1, RQ2 and RQ3 are already complete and frozen.
#
# This final global-analysis cell performs NO new inferential tests and computes NO new p-values.
# It:
#
#   1. verifies the final frozen RQ1, RQ2 and RQ3 checkpoints;
#   2. verifies every file covered by each frozen RQ package manifest and recomputes each package root hash;
#   3. verifies the final 24-project completion registry remains unchanged;
#   4. copies the three already-frozen RQ packages into one final global results package;
#   5. creates compact cross-RQ primary-result, metric-sensitivity, study-design,
#      reproducibility-anchor and integrity-audit tables;
#   6. freezes a final integrity manifest/root hash for the complete global-analysis package.
#
# IMPORTANT
# ---------
# This is a CONSOLIDATION / AUDIT step only.
# No post-hoc tests are added.
# No project is removed.
# No threshold is interpolated.
# No RQ result is re-estimated.
# No generated thesis prose is created.
# No project result or completion-registry row is modified.
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd


print("=" * 162)
print("=== THESIS GLOBAL ANALYSIS — CELL 12 / STEP 6A: FINAL CROSS-RQ CONSOLIDATION + GLOBAL RESULTS PACKAGE ===")
print("=" * 162)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN GLOBAL / RQ ANCHORS
# --------------------------------------------------------------------------------------------------

STEP0_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP0_24_PROJECT_COHORT_AND_ANALYSIS_SOURCE_DISCOVERY_FROZEN"
)

EXPECTED_STEP0_CHECKPOINT_SHA256 = (
    "b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d5436cbcb4c97ca00fb"
)

STEP1B_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP1B_CANONICAL_CROSS_PROJECT_MASTER_DATASETS_FROZEN"
)

EXPECTED_STEP1B_CHECKPOINT_SHA256 = (
    "eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3e0642d1b19f456ad50"
)

STEP2_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP2_STATISTICAL_ANALYSIS_CONTRACT_FROZEN"
)

EXPECTED_STEP2_CHECKPOINT_SHA256 = (
    "f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128cf5f23b311a143d04e"
)

RQ1_STATUS = (
    "PASS_RQ1_STEP3C_FINAL_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ1_CHECKPOINT_SHA256 = (
    "b4f3d10b77acb39200496542e8256d6213981fd636325487f190ce2dd07f35af"
)

EXPECTED_RQ1_PACKAGE_ROOT_SHA256 = (
    "5b7c49e4096c11edd927052807658e5a196e58b5c7302e7f6fa886258ad37ff4"
)

RQ2_STATUS = (
    "PASS_RQ2_STEP4B_FINAL_CROSSOVER_FIGURES_SYNTHESIS_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ2_CHECKPOINT_SHA256 = (
    "8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d352460330974c93feb40"
)

EXPECTED_RQ2_PACKAGE_ROOT_SHA256 = (
    "cec947adcb77b912dc217ad7012f6b97a760932906c147cfcec2837125552b1b"
)

RQ3_STATUS = (
    "PASS_RQ3_STEP5B_FINAL_ROBUSTNESS_SYNTHESIS_FIGURES_AND_ANSWER_PACKAGE_FROZEN"
)

EXPECTED_RQ3_CHECKPOINT_SHA256 = (
    "17ceb2c1e2b50ac5aacad8539d40f41ed7b913ee23659431dde919b40854c6ed"
)

EXPECTED_RQ3_PACKAGE_ROOT_SHA256 = (
    "77f25e4a0755a437e44ffcba5a86a360cd9736b63ab02cd5bbd402ddea2993f6"
)

EXPECTED_REGISTRY_SHA256 = (
    "dc5cdc752d89661c0b41adc5680509034ded1c64f2f41934de774f1621ab2596"
)

FINAL_STATUS = (
    "PASS_GLOBAL_ANALYSIS_STEP6A_FINAL_CROSS_RQ_CONSOLIDATION_AND_GLOBAL_RESULTS_PACKAGE_FROZEN"
)

CODE_REVISION = (
    "GLOBAL_ANALYSIS_STEP6A_V1_FINAL_CROSS_RQ_CONSOLIDATION_AUDIT_NO_NEW_INFERENCE"
)

FLOAT_TOL = 1e-12

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

STRESS_LEVELS = [
    30,
    40,
    50,
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

ANALYSIS_ROOT = (
    RESULTS
    / "Analysis"
    / "Global_24_Project_Analysis"
)

STEP0_CHECKPOINT = (
    NOTES
    / "global_analysis_step0_checkpoint.json"
)

STEP1B_CHECKPOINT = (
    NOTES
    / "global_analysis_step1b_checkpoint.json"
)

STEP2_CHECKPOINT = (
    NOTES
    / "global_analysis_step2_checkpoint.json"
)

RQ1_CHECKPOINT = (
    NOTES
    / "global_analysis_rq1_step3c_checkpoint.json"
)

RQ2_CHECKPOINT = (
    NOTES
    / "global_analysis_rq2_step4b_checkpoint.json"
)

RQ3_CHECKPOINT = (
    NOTES
    / "global_analysis_rq3_step5b_checkpoint.json"
)

STEP6A_ROOT = (
    ANALYSIS_ROOT
    / "Step_6A_Final_Cross_RQ_Global_Results_Package"
)

CONSOLIDATED_TABLES_ROOT = (
    STEP6A_ROOT
    / "Consolidated_Tables"
)

FROZEN_RQ_ROOT = (
    STEP6A_ROOT
    / "Frozen_RQ_Packages"
)

FROZEN_RQ1_DEST = (
    FROZEN_RQ_ROOT
    / "RQ1"
)

FROZEN_RQ2_DEST = (
    FROZEN_RQ_ROOT
    / "RQ2"
)

FROZEN_RQ3_DEST = (
    FROZEN_RQ_ROOT
    / "RQ3"
)

STUDY_DESIGN_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_study_population_and_analysis_contract.csv"
)

CROSS_RQ_PRIMARY_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_cross_rq_primary_results.csv"
)

RQ1_SENSITIVITY_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_rq1_apfdc_vs_apfd_sensitivity.csv"
)

RQ2_SENSITIVITY_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_rq2_apfdc_vs_apfd_threshold_sensitivity.csv"
)

RQ3_SENSITIVITY_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_rq3_apfdc_vs_apfd_sensitivity.csv"
)

REPRODUCIBILITY_ANCHORS_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_reproducibility_anchors.csv"
)

INTEGRITY_AUDIT_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "global_integrity_audit.csv"
)

REGISTRY_COPY_PATH = (
    CONSOLIDATED_TABLES_ROOT
    / "completed_project_registry_frozen_copy.csv"
)

VALIDATION_PATH = (
    STEP6A_ROOT
    / "global_step6a_validation.csv"
)

REPORT_PATH = (
    STEP6A_ROOT
    / "global_step6a_report.json"
)

STATUS_PATH = (
    STEP6A_ROOT
    / "global_step6a_status.json"
)

PACKAGE_MANIFEST_PATH = (
    STEP6A_ROOT
    / "global_final_results_package_manifest.csv"
)

CHECKPOINT_PATH = (
    NOTES
    / "global_analysis_step6a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
        float_format="%.17g",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary,
        path,
    )


def atomic_copy(
    source,
    destination,
):
    source = Path(
        source
    )

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        f".{destination.name}.tmp_{os.getpid()}"
    )

    shutil.copy2(
        source,
        temporary,
    )

    os.replace(
        temporary,
        destination,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def package_root_hash(
    manifest,
):
    ordered = (
        manifest[
            [
                "RelativePath",
                "Bytes",
                "SHA256",
            ]
        ]
        .copy()
        .sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    digest = hashlib.sha256()

    for row in ordered.itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def build_package_manifest(
    root,
    manifest_path,
):
    root = Path(
        root
    )

    manifest_path = Path(
        manifest_path
    )

    rows = []

    for path in sorted(
        (
            candidate
            for candidate in root.rglob("*")
            if candidate.is_file()
            and candidate.resolve()
            != manifest_path.resolve()
        ),
        key=lambda candidate:
            candidate.relative_to(
                root
            ).as_posix(),
    ):
        rows.append({
            "RelativePath":
                path.relative_to(
                    root
                ).as_posix(),

            "Bytes":
                int(
                    path.stat().st_size
                ),

            "SHA256":
                sha256_file(
                    path
                ),
        })

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def verify_frozen_package(
    label,
    checkpoint,
    expected_root_sha,
):
    package_root = Path(
        checkpoint[
            "PackageRoot"
        ]
    )

    manifest_path = Path(
        checkpoint[
            "PackageManifestPath"
        ]
    )

    expected_manifest_sha = str(
        checkpoint[
            "PackageManifestSHA256"
        ]
    ).lower()

    checkpoint_root_sha = str(
        checkpoint[
            "PackageRootSHA256"
        ]
    ).lower()

    if checkpoint_root_sha != expected_root_sha:
        raise RuntimeError(
            f"{label}: checkpoint package-root SHA differs from frozen expected anchor."
        )

    if not package_root.is_dir():
        raise FileNotFoundError(
            f"{label}: package root missing: {package_root}"
        )

    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"{label}: package manifest missing: {manifest_path}"
        )

    actual_manifest_sha = sha256_file(
        manifest_path
    )

    if actual_manifest_sha != expected_manifest_sha:
        raise RuntimeError(
            f"{label}: package manifest SHA mismatch."
        )

    manifest = pd.read_csv(
        manifest_path,
        low_memory=False,
    )

    required_columns = [
        "RelativePath",
        "Bytes",
        "SHA256",
    ]

    if manifest.columns.tolist() != required_columns:
        raise RuntimeError(
            f"{label}: package manifest schema changed."
        )

    missing_files = 0
    size_mismatches = 0
    sha_mismatches = 0

    for row in manifest.itertuples(
        index=False
    ):
        path = (
            package_root
            / str(
                row.RelativePath
            )
        )

        if not path.is_file():
            missing_files += 1
            continue

        if int(
            path.stat().st_size
        ) != int(
            row.Bytes
        ):
            size_mismatches += 1

        if sha256_file(
            path
        ) != str(
            row.SHA256
        ).lower():
            sha_mismatches += 1

    recomputed_root_sha = package_root_hash(
        manifest
    )

    if recomputed_root_sha != expected_root_sha:
        raise RuntimeError(
            f"{label}: recomputed package-root SHA mismatch."
        )

    if (
        missing_files
        != 0
        or size_mismatches
        != 0
        or sha_mismatches
        != 0
    ):
        raise RuntimeError(
            f"{label}: package integrity verification failed."
        )

    return {
        "Label":
            label,

        "PackageRoot":
            package_root,

        "ManifestPath":
            manifest_path,

        "Manifest":
            manifest,

        "ManifestSHA256":
            actual_manifest_sha,

        "PackageRootSHA256":
            recomputed_root_sha,

        "PackageFiles":
            int(
                len(
                    manifest
                )
            ),

        "PackageBytes":
            int(
                manifest[
                    "Bytes"
                ].sum()
            ),

        "MissingFiles":
            missing_files,

        "SizeMismatches":
            size_mismatches,

        "SHAMismatches":
            sha_mismatches,
    }


def copy_verified_package(
    verification,
    destination_root,
):
    source_root = Path(
        verification[
            "PackageRoot"
        ]
    )

    source_manifest = verification[
        "Manifest"
    ]

    destination_root = Path(
        destination_root
    )

    if destination_root.exists():
        shutil.rmtree(
            destination_root,
            ignore_errors=True,
        )

    destination_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Copy all files in the frozen package directory, including its own package manifest.
    for source_path in sorted(
        (
            path
            for path in source_root.rglob("*")
            if path.is_file()
        ),
        key=lambda path:
            path.relative_to(
                source_root
            ).as_posix(),
    ):
        relative = source_path.relative_to(
            source_root
        )

        destination = (
            destination_root
            / relative
        )

        atomic_copy(
            source_path,
            destination,
        )

    # Re-verify every file named by the original frozen package manifest.
    copied_missing = 0
    copied_size_mismatches = 0
    copied_sha_mismatches = 0

    for row in source_manifest.itertuples(
        index=False
    ):
        copied_path = (
            destination_root
            / str(
                row.RelativePath
            )
        )

        if not copied_path.is_file():
            copied_missing += 1
            continue

        if int(
            copied_path.stat().st_size
        ) != int(
            row.Bytes
        ):
            copied_size_mismatches += 1

        if sha256_file(
            copied_path
        ) != str(
            row.SHA256
        ).lower():
            copied_sha_mismatches += 1

    copied_root_sha = package_root_hash(
        source_manifest
    )

    return {
        "CopiedMissingFiles":
            copied_missing,

        "CopiedSizeMismatches":
            copied_size_mismatches,

        "CopiedSHAMismatches":
            copied_sha_mismatches,

        "CopiedRootSHA256":
            copied_root_sha,

        "CopiedDirectoryFileCountIncludingSourceManifest":
            sum(
                1
                for path in destination_root.rglob("*")
                if path.is_file()
            ),
    }


def parse_bool(
    value,
):
    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(
            value
        )

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
    }:
        return True

    if text in {
        "false",
        "0",
    }:
        return False

    raise RuntimeError(
        f"Cannot parse boolean value: {value!r}"
    )


def first_significant_noise(
    block,
):
    significant = block.loc[
        block[
            "SignificantBonferroni"
        ].map(
            parse_bool
        )
    ].copy()

    if significant.empty:
        return None

    return int(
        pd.to_numeric(
            significant[
                "NoisePercent"
            ],
            errors="raise",
        ).min()
    )


def threshold_value_or_label(
    row,
    prefix,
):
    status = str(
        row[
            f"{prefix}Status"
        ]
    )

    value = row[
        prefix
    ]

    if (
        pd.isna(
            value
        )
        or str(
            value
        ).strip()
        == ""
    ):
        if status == "NOT_OBSERVED_THROUGH_50":
            return ">50"

        return ""

    numeric_value = int(
        float(
            value
        )
    )

    if status == "NO_POSITIVE_TOLERANCE_THRESHOLD":
        return "0 (no positive tolerance)"

    return str(
        numeric_value
    )


# --------------------------------------------------------------------------------------------------
# 4. ONE-TIME GUARD
# --------------------------------------------------------------------------------------------------

if CHECKPOINT_PATH.exists():
    raise RuntimeError(
        "Global Analysis Step 6A is already frozen.\n"
        f"Checkpoint: {CHECKPOINT_PATH}\n"
        "Do not rerun. The global analysis is complete."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY CHECKPOINT CHAIN
# --------------------------------------------------------------------------------------------------

checkpoint_specs = [
    (
        "Step0",
        STEP0_CHECKPOINT,
        EXPECTED_STEP0_CHECKPOINT_SHA256,
        STEP0_STATUS,
    ),
    (
        "Step1B",
        STEP1B_CHECKPOINT,
        EXPECTED_STEP1B_CHECKPOINT_SHA256,
        STEP1B_STATUS,
    ),
    (
        "Step2",
        STEP2_CHECKPOINT,
        EXPECTED_STEP2_CHECKPOINT_SHA256,
        STEP2_STATUS,
    ),
    (
        "RQ1",
        RQ1_CHECKPOINT,
        EXPECTED_RQ1_CHECKPOINT_SHA256,
        RQ1_STATUS,
    ),
    (
        "RQ2",
        RQ2_CHECKPOINT,
        EXPECTED_RQ2_CHECKPOINT_SHA256,
        RQ2_STATUS,
    ),
    (
        "RQ3",
        RQ3_CHECKPOINT,
        EXPECTED_RQ3_CHECKPOINT_SHA256,
        RQ3_STATUS,
    ),
]

checkpoint_objects = {}
checkpoint_audit_rows = []

for (
    label,
    path,
    expected_sha,
    expected_status,
) in checkpoint_specs:
    if not path.is_file():
        raise FileNotFoundError(
            f"{label} checkpoint missing: {path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{label} checkpoint SHA mismatch."
        )

    payload = load_json(
        path
    )

    if payload.get(
        "Status"
    ) != expected_status:
        raise RuntimeError(
            f"{label} checkpoint status mismatch."
        )

    checkpoint_objects[
        label
    ] = payload

    checkpoint_audit_rows.append({
        "Anchor":
            label,

        "CheckpointPath":
            str(
                path
            ),

        "CheckpointSHA256":
            actual_sha,

        "Status":
            str(
                payload.get(
                    "Status"
                )
            ),

        "CheckpointVerified":
            True,
    })


step0 = checkpoint_objects[
    "Step0"
]

step1b = checkpoint_objects[
    "Step1B"
]

step2 = checkpoint_objects[
    "Step2"
]

rq1 = checkpoint_objects[
    "RQ1"
]

rq2 = checkpoint_objects[
    "RQ2"
]

rq3 = checkpoint_objects[
    "RQ3"
]


# --------------------------------------------------------------------------------------------------
# 6. VERIFY REGISTRY + BASIC FROZEN STUDY UNIT
# --------------------------------------------------------------------------------------------------

if not REGISTRY.is_file():
    raise FileNotFoundError(
        f"Completion registry missing: {REGISTRY}"
    )

registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from the final 24-project freeze."
    )

registry = pd.read_csv(
    REGISTRY,
    low_memory=False,
)

if len(
    registry
) != 24:
    raise RuntimeError(
        "Final completion registry must contain exactly 24 project rows."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY ALL THREE FINAL FROZEN RQ PACKAGES
# --------------------------------------------------------------------------------------------------

rq1_verification = verify_frozen_package(
    "RQ1",
    rq1,
    EXPECTED_RQ1_PACKAGE_ROOT_SHA256,
)

rq2_verification = verify_frozen_package(
    "RQ2",
    rq2,
    EXPECTED_RQ2_PACKAGE_ROOT_SHA256,
)

rq3_verification = verify_frozen_package(
    "RQ3",
    rq3,
    EXPECTED_RQ3_PACKAGE_ROOT_SHA256,
)

rq_verifications = [
    rq1_verification,
    rq2_verification,
    rq3_verification,
]


# --------------------------------------------------------------------------------------------------
# 8. RECREATE ONLY THE FINAL STEP-6A OUTPUT DIRECTORY
# --------------------------------------------------------------------------------------------------

if STEP6A_ROOT.exists():
    shutil.rmtree(
        STEP6A_ROOT,
        ignore_errors=True,
    )

CONSOLIDATED_TABLES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FROZEN_RQ_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------------------
# 9. COPY THE THREE VERIFIED FROZEN RQ PACKAGES
# --------------------------------------------------------------------------------------------------

rq1_copy_audit = copy_verified_package(
    rq1_verification,
    FROZEN_RQ1_DEST,
)

rq2_copy_audit = copy_verified_package(
    rq2_verification,
    FROZEN_RQ2_DEST,
)

rq3_copy_audit = copy_verified_package(
    rq3_verification,
    FROZEN_RQ3_DEST,
)


# --------------------------------------------------------------------------------------------------
# 10. VERIFY + LOAD FROZEN RQ1 TABLES FOR CONSOLIDATION
# --------------------------------------------------------------------------------------------------

rq1_primary_path = Path(
    rq1[
        "PrimaryFullResultsPath"
    ]
)

rq1_secondary_path = Path(
    rq1[
        "SecondaryFullResultsPath"
    ]
)

rq1_synthesis_path = Path(
    rq1[
        "PrimarySynthesisPath"
    ]
)

rq1_figure_data_path = Path(
    rq1[
        "FigureDataPath"
    ]
)

for path, sha_key in [
    (
        rq1_primary_path,
        "PrimaryFullResultsSHA256",
    ),
    (
        rq1_secondary_path,
        "SecondaryFullResultsSHA256",
    ),
    (
        rq1_synthesis_path,
        "PrimarySynthesisSHA256",
    ),
    (
        rq1_figure_data_path,
        "FigureDataSHA256",
    ),
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ1 source missing: {path}"
        )

    if sha256_file(
        path
    ) != str(
        rq1[
            sha_key
        ]
    ).lower():
        raise RuntimeError(
            f"Frozen RQ1 source SHA mismatch: {path.name}"
        )


rq1_primary = pd.read_csv(
    rq1_primary_path,
    low_memory=False,
)

rq1_secondary = pd.read_csv(
    rq1_secondary_path,
    low_memory=False,
)

rq1_synthesis = pd.read_csv(
    rq1_synthesis_path,
    low_memory=False,
)

rq1_figure_data = pd.read_csv(
    rq1_figure_data_path,
    low_memory=False,
)

if len(
    rq1_primary
) != 32:
    raise RuntimeError(
        "RQ1 primary table must contain 32 rows."
    )

if len(
    rq1_secondary
) != 32:
    raise RuntimeError(
        "RQ1 secondary table must contain 32 rows."
    )

if len(
    rq1_synthesis
) != 4:
    raise RuntimeError(
        "RQ1 primary synthesis must contain 4 rows."
    )

if len(
    rq1_figure_data
) != 36:
    raise RuntimeError(
        "RQ1 figure-data table must contain 36 rows."
    )


# --------------------------------------------------------------------------------------------------
# 11. VERIFY + LOAD FROZEN RQ2 TABLES
# --------------------------------------------------------------------------------------------------

rq2_primary_thresholds_path = Path(
    rq2[
        "PrimaryThresholdsPath"
    ]
)

rq2_secondary_thresholds_path = Path(
    rq2[
        "SecondaryThresholdsPath"
    ]
)

rq2_threshold_matrix_path = Path(
    rq2[
        "PrimaryThresholdMatrixPath"
    ]
)

for path, sha_key in [
    (
        rq2_primary_thresholds_path,
        "PrimaryThresholdsSHA256",
    ),
    (
        rq2_secondary_thresholds_path,
        "SecondaryThresholdsSHA256",
    ),
    (
        rq2_threshold_matrix_path,
        "PrimaryThresholdMatrixSHA256",
    ),
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ2 source missing: {path}"
        )

    if sha256_file(
        path
    ) != str(
        rq2[
            sha_key
        ]
    ).lower():
        raise RuntimeError(
            f"Frozen RQ2 source SHA mismatch: {path.name}"
        )


rq2_primary_thresholds = pd.read_csv(
    rq2_primary_thresholds_path,
    low_memory=False,
)

rq2_secondary_thresholds = pd.read_csv(
    rq2_secondary_thresholds_path,
    low_memory=False,
)

rq2_threshold_matrix = pd.read_csv(
    rq2_threshold_matrix_path,
    low_memory=False,
)

if len(
    rq2_primary_thresholds
) != 16:
    raise RuntimeError(
        "RQ2 APFDc threshold table must contain 16 rows."
    )

if len(
    rq2_secondary_thresholds
) != 16:
    raise RuntimeError(
        "RQ2 APFD threshold table must contain 16 rows."
    )

if len(
    rq2_threshold_matrix
) != 4:
    raise RuntimeError(
        "RQ2 threshold matrix must contain 4 rows."
    )


# --------------------------------------------------------------------------------------------------
# 12. VERIFY + LOAD FROZEN RQ3 TABLES
# --------------------------------------------------------------------------------------------------

rq3_synthesis_path = Path(
    rq3[
        "PrimaryRobustnessSynthesisPath"
    ]
)

rq3_best_by_noise_path = Path(
    rq3[
        "BestByNoisePath"
    ]
)

rq3_winner_nemenyi_path = Path(
    rq3[
        "WinnerNemenyi50Path"
    ]
)

rq3_omnibus_path = Path(
    rq3[
        "OmnibusResultsPath"
    ]
)

rq3_average_ranks_path = Path(
    rq3[
        "AverageRanksPath"
    ]
)

for path, sha_key in [
    (
        rq3_synthesis_path,
        "PrimaryRobustnessSynthesisSHA256",
    ),
    (
        rq3_best_by_noise_path,
        "BestByNoiseSHA256",
    ),
    (
        rq3_winner_nemenyi_path,
        "WinnerNemenyi50SHA256",
    ),
    (
        rq3_omnibus_path,
        "OmnibusResultsSHA256",
    ),
    (
        rq3_average_ranks_path,
        "AverageRanksSHA256",
    ),
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen RQ3 source missing: {path}"
        )

    if sha256_file(
        path
    ) != str(
        rq3[
            sha_key
        ]
    ).lower():
        raise RuntimeError(
            f"Frozen RQ3 source SHA mismatch: {path.name}"
        )


rq3_synthesis = pd.read_csv(
    rq3_synthesis_path,
    low_memory=False,
)

rq3_best_by_noise = pd.read_csv(
    rq3_best_by_noise_path,
    low_memory=False,
)

rq3_winner_nemenyi = pd.read_csv(
    rq3_winner_nemenyi_path,
    low_memory=False,
)

rq3_omnibus = pd.read_csv(
    rq3_omnibus_path,
    low_memory=False,
)

rq3_average_ranks = pd.read_csv(
    rq3_average_ranks_path,
    low_memory=False,
)

if len(
    rq3_synthesis
) != 4:
    raise RuntimeError(
        "RQ3 primary robustness synthesis must contain 4 rows."
    )

if len(
    rq3_best_by_noise
) != 9:
    raise RuntimeError(
        "RQ3 best-by-noise table must contain 9 rows."
    )

if len(
    rq3_winner_nemenyi
) != 3:
    raise RuntimeError(
        "RQ3 winner Nemenyi table must contain 3 rows."
    )

if len(
    rq3_omnibus
) != 18:
    raise RuntimeError(
        "RQ3 omnibus table must contain 18 rows."
    )

if len(
    rq3_average_ranks
) != 72:
    raise RuntimeError(
        "RQ3 average-ranks table must contain 72 rows."
    )


# --------------------------------------------------------------------------------------------------
# 13. STUDY POPULATION + ANALYSIS CONTRACT TABLE
# --------------------------------------------------------------------------------------------------

study_design_rows = [
    {
        "Category":
            "Population",

        "Item":
            "Candidate projects screened",

        "Value":
            "25",
    },
    {
        "Category":
            "Population",

        "Item":
            "Eligible projects included",

        "Value":
            "24",
    },
    {
        "Category":
            "Population",

        "Item":
            "Excluded projects",

        "Value":
            "1",
    },
    {
        "Category":
            "Population",

        "Item":
            "Excluded project",

        "Value":
            "Graylog2@graylog2-server",
    },
    {
        "Category":
            "Population",

        "Item":
            "Exclusion reason",

        "Value":
            "0 clean evaluation failures in chronological evaluation split",
    },
    {
        "Category":
            "Design",

        "Item":
            "Chronological split",

        "Value":
            "75% training / 25% evaluation",
    },
    {
        "Category":
            "Design",

        "Item":
            "Noise levels (%)",

        "Value":
            json.dumps(
                NOISE_LEVELS,
                separators=(",", ":"),
            ),
    },
    {
        "Category":
            "Design",

        "Item":
            "Seeds per project-noise condition",

        "Value":
            "30",
    },
    {
        "Category":
            "Design",

        "Item":
            "Supervised ML techniques",

        "Value":
            json.dumps(
                ML_TECHNIQUES,
                separators=(",", ":"),
            ),
    },
    {
        "Category":
            "Design",

        "Item":
            "Baselines",

        "Value":
            json.dumps(
                BASELINES,
                separators=(",", ":"),
            ),
    },
    {
        "Category":
            "Inference",

        "Item":
            "Independent empirical unit",

        "Value":
            "Project (N=24)",
    },
    {
        "Category":
            "Inference",

        "Item":
            "Repeated stochastic unit",

        "Value":
            "30 seeds within each project",
    },
    {
        "Category":
            "Metric",

        "Item":
            "Primary metric",

        "Value":
            "APFDc",
    },
    {
        "Category":
            "Metric",

        "Item":
            "Secondary metric",

        "Value":
            "APFD",
    },
    {
        "Category":
            "RQ1",

        "Item":
            "Inferential procedure",

        "Value":
            "paired two-sided Wilcoxon; Bonferroni family=8 within algorithm×metric",
    },
    {
        "Category":
            "RQ2",

        "Item":
            "Threshold rule",

        "Value":
            "first observed tested crossover; no interpolation",
    },
    {
        "Category":
            "RQ3",

        "Item":
            "Inferential procedure",

        "Value":
            "Friedman at each noise; Holm across 9 per metric; conditional Nemenyi",
    },
    {
        "Category":
            "RQ3",

        "Item":
            "Primary robustness criterion",

        "Value":
            "highest equally weighted cross-project mean APFDc at 50% noise",
    },
]

study_design = pd.DataFrame(
    study_design_rows
)


# --------------------------------------------------------------------------------------------------
# 14. RQ1 APFDC-vs-APFD SENSITIVITY TABLE
# --------------------------------------------------------------------------------------------------

rq1_sensitivity_rows = []

for technique in ML_TECHNIQUES:
    primary_block = rq1_primary.loc[
        rq1_primary[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
    ].copy()

    secondary_block = rq1_secondary.loc[
        rq1_secondary[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
    ].copy()

    if len(
        primary_block
    ) != 8:
        raise RuntimeError(
            f"RQ1 APFDc {technique}: expected 8 inferential rows."
        )

    if len(
        secondary_block
    ) != 8:
        raise RuntimeError(
            f"RQ1 APFD {technique}: expected 8 inferential rows."
        )

    figure_row_50 = rq1_figure_data.loc[
        rq1_figure_data[
            "Technique"
        ].astype(
            str
        ).eq(
            technique
        )
        & pd.to_numeric(
            rq1_figure_data[
                "NoisePercent"
            ],
            errors="raise",
        ).astype(
            int
        ).eq(
            50
        )
    ]

    if len(
        figure_row_50
    ) != 1:
        raise RuntimeError(
            f"RQ1 {technique}: missing 50% figure-data row."
        )

    figure_row_50 = figure_row_50.iloc[
        0
    ]

    primary_significant_count = int(
        primary_block[
            "SignificantBonferroni"
        ].map(
            parse_bool
        ).sum()
    )

    secondary_significant_count = int(
        secondary_block[
            "SignificantBonferroni"
        ].map(
            parse_bool
        ).sum()
    )

    rq1_sensitivity_rows.append({
        "Technique":
            technique,

        "APFDcSignificantComparisonsOf8":
            primary_significant_count,

        "APFDcFirstBonferroniSignificantNoisePercent":
            first_significant_noise(
                primary_block
            ),

        "APFDSignificantComparisonsOf8":
            secondary_significant_count,

        "APFDFirstBonferroniSignificantNoisePercent":
            first_significant_noise(
                secondary_block
            ),

        "APFDcAt50":
            float(
                figure_row_50[
                    "MeanGlobalSeedAPFDc"
                ]
            ),

        "APFDcMeanDegradationPctAt50":
            float(
                figure_row_50[
                    "MeanDegradationPctAPFDc"
                ]
            ),

        "APFDcMeanRetentionPctAt50":
            float(
                figure_row_50[
                    "MeanRetentionPctAPFDc"
                ]
            ),

        "APFDAt50":
            float(
                figure_row_50[
                    "MeanGlobalSeedAPFD"
                ]
            ),

        "APFDMeanDeltaAt50":
            float(
                figure_row_50[
                    "MeanDeltaAPFD"
                ]
            ),
    })


rq1_sensitivity = pd.DataFrame(
    rq1_sensitivity_rows
)


# --------------------------------------------------------------------------------------------------
# 15. RQ2 APFDC-vs-APFD THRESHOLD SENSITIVITY TABLE
# --------------------------------------------------------------------------------------------------

primary_threshold_subset = rq2_primary_thresholds[
    [
        "Algorithm",
        "Comparator",
        "ComparatorRole",
        "CleanAdvantage",
        "FirstObservedCrossoverNoisePercent",
        "FirstObservedCrossoverStatus",
        "SustainedCrossoverNoisePercent",
        "SustainedCrossoverStatus",
    ]
].copy()

primary_threshold_subset = primary_threshold_subset.rename(
    columns={
        "CleanAdvantage":
            "APFDcCleanAdvantage",

        "FirstObservedCrossoverNoisePercent":
            "APFDcFirstObservedCrossoverNoisePercent",

        "FirstObservedCrossoverStatus":
            "APFDcFirstObservedCrossoverStatus",

        "SustainedCrossoverNoisePercent":
            "APFDcSustainedCrossoverNoisePercent",

        "SustainedCrossoverStatus":
            "APFDcSustainedCrossoverStatus",
    }
)

secondary_threshold_subset = rq2_secondary_thresholds[
    [
        "Algorithm",
        "Comparator",
        "CleanAdvantage",
        "FirstObservedCrossoverNoisePercent",
        "FirstObservedCrossoverStatus",
        "SustainedCrossoverNoisePercent",
        "SustainedCrossoverStatus",
    ]
].copy()

secondary_threshold_subset = secondary_threshold_subset.rename(
    columns={
        "CleanAdvantage":
            "APFDCleanAdvantage",

        "FirstObservedCrossoverNoisePercent":
            "APFDFirstObservedCrossoverNoisePercent",

        "FirstObservedCrossoverStatus":
            "APFDFirstObservedCrossoverStatus",

        "SustainedCrossoverNoisePercent":
            "APFDSustainedCrossoverNoisePercent",

        "SustainedCrossoverStatus":
            "APFDSustainedCrossoverStatus",
    }
)

rq2_sensitivity = primary_threshold_subset.merge(
    secondary_threshold_subset,
    on=[
        "Algorithm",
        "Comparator",
    ],
    how="inner",
    validate="one_to_one",
)

if len(
    rq2_sensitivity
) != 16:
    raise RuntimeError(
        "RQ2 metric-sensitivity threshold table must contain 16 rows."
    )


# --------------------------------------------------------------------------------------------------
# 16. RQ3 APFDC-vs-APFD SENSITIVITY TABLE
# --------------------------------------------------------------------------------------------------

rq3_sensitivity_rows = []

for metric in [
    "APFDc",
    "APFD",
]:
    omnibus_block = rq3_omnibus.loc[
        rq3_omnibus[
            "Metric"
        ].astype(
            str
        ).eq(
            metric
        )
    ].copy()

    if len(
        omnibus_block
    ) != 9:
        raise RuntimeError(
            f"RQ3 {metric}: expected 9 omnibus rows."
        )

    significant_count = int(
        omnibus_block[
            "SignificantHolm"
        ].map(
            parse_bool
        ).sum()
    )

    rank_block = rq3_average_ranks.loc[
        rq3_average_ranks[
            "Metric"
        ].astype(
            str
        ).eq(
            metric
        )
        & rq3_average_ranks[
            "NoisePercent"
        ].isin(
            STRESS_LEVELS
        )
    ].copy()

    if len(
        rank_block
    ) != 12:
        raise RuntimeError(
            f"RQ3 {metric}: stress rank block must contain 12 rows."
        )

    stress_summary = (
        rank_block.groupby(
            "Technique",
            sort=False,
        )
        .agg(
            MeanStressValue=(
                "CrossProjectMean",
                "mean",
            ),

            MeanStressAverageRank=(
                "AverageRank",
                "mean",
            ),
        )
        .reset_index()
    )

    best_value = stress_summary.sort_values(
        [
            "MeanStressValue",
            "Technique",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    ).iloc[
        0
    ]

    best_rank = stress_summary.sort_values(
        [
            "MeanStressAverageRank",
            "Technique",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    ).iloc[
        0
    ]

    rq3_sensitivity_rows.append({
        "Metric":
            metric,

        "HolmSignificantFriedmanTestsOf9":
            significant_count,

        "StressRegionBestAbsoluteTechnique":
            str(
                best_value[
                    "Technique"
                ]
            ),

        "StressRegionBestAbsoluteMean":
            float(
                best_value[
                    "MeanStressValue"
                ]
            ),

        "StressRegionBestAverageRankTechnique":
            str(
                best_rank[
                    "Technique"
                ]
            ),

        "StressRegionBestAverageRank":
            float(
                best_rank[
                    "MeanStressAverageRank"
                ]
            ),
    })


rq3_sensitivity = pd.DataFrame(
    rq3_sensitivity_rows
)


# --------------------------------------------------------------------------------------------------
# 17. CROSS-RQ PRIMARY RESULTS TABLE
# --------------------------------------------------------------------------------------------------

rq1_primary_significant_total = int(
    rq1_primary[
        "SignificantBonferroni"
    ].map(
        parse_bool
    ).sum()
)

rq1_secondary_significant_total = int(
    rq1_secondary[
        "SignificantBonferroni"
    ].map(
        parse_bool
    ).sum()
)

rq3_primary_significant_omnibus = int(
    rq3_omnibus.loc[
        rq3_omnibus[
            "Metric"
        ].eq(
            "APFDc"
        ),
        "SignificantHolm",
    ].map(
        parse_bool
    ).sum()
)

rq3_secondary_significant_omnibus = int(
    rq3_omnibus.loc[
        rq3_omnibus[
            "Metric"
        ].eq(
            "APFD"
        ),
        "SignificantHolm",
    ].map(
        parse_bool
    ).sum()
)

rq3_winner_row = rq3_synthesis.loc[
    rq3_synthesis[
        "PrimaryRobustnessWinner"
    ].map(
        parse_bool
    )
]

if len(
    rq3_winner_row
) != 1:
    raise RuntimeError(
        "RQ3 primary robustness winner row resolution failed."
    )

rq3_winner_row = rq3_winner_row.iloc[
    0
]

latestfail_thresholds = rq2_primary_thresholds.loc[
    rq2_primary_thresholds[
        "Comparator"
    ].astype(
        str
    ).eq(
        "LatestFail"
    )
].copy()

if len(
    latestfail_thresholds
) != 4:
    raise RuntimeError(
        "RQ2 primary LatestFail threshold block must contain 4 rows."
    )

latestfail_threshold_json = {
    str(
        row.Algorithm
    ):
        (
            "0 (no positive tolerance)"
            if str(
                row.FirstObservedCrossoverStatus
            )
            == "NO_POSITIVE_TOLERANCE_THRESHOLD"
            else (
                ">50"
                if str(
                    row.FirstObservedCrossoverStatus
                )
                == "NOT_OBSERVED_THROUGH_50"
                else str(
                    int(
                        float(
                            row.FirstObservedCrossoverNoisePercent
                        )
                    )
                )
            )
        )
    for row in latestfail_thresholds.itertuples(
        index=False
    )
}

rq1_first_sig_json = {
    str(
        row.Technique
    ):
        int(
            row.APFDcFirstBonferroniSignificantNoisePercent
        )
    for row in rq1_sensitivity.itertuples(
        index=False
    )
}

cross_rq_primary = pd.DataFrame(
    [
        {
            "RQ":
                "RQ1",

            "PrimaryMetric":
                "APFDc",

            "PrimaryResultType":
                "noisy-vs-clean degradation onset",

            "PrimaryResult":
                json.dumps(
                    rq1_first_sig_json,
                    sort_keys=True,
                    separators=(",", ":"),
                ),

            "PrimaryInferentialContext":
                (
                    f"{rq1_primary_significant_total}/32 Bonferroni-significant APFDc comparisons; "
                    f"{rq1_secondary_significant_total}/32 secondary APFD comparisons"
                ),

            "InterpretationBoundary":
                "paired project-level inference; seeds not treated as independent",
        },
        {
            "RQ":
                "RQ2",

            "PrimaryMetric":
                "APFDc",

            "PrimaryResultType":
                "first observed crossover vs LatestFail",

            "PrimaryResult":
                json.dumps(
                    latestfail_threshold_json,
                    sort_keys=True,
                    separators=(",", ":"),
                ),

            "PrimaryInferentialContext":
                "descriptive threshold analysis; 0 formal RQ2 hypothesis tests",

            "InterpretationBoundary":
                "observed tested grid only; interpolation forbidden",
        },
        {
            "RQ":
                "RQ3",

            "PrimaryMetric":
                "APFDc",

            "PrimaryResultType":
                "robustness winner at highest tested noise",

            "PrimaryResult":
                str(
                    rq3[
                        "PrimaryRobustnessWinner"
                    ]
                ),

            "PrimaryInferentialContext":
                (
                    f"50% Friedman Holm-significant; "
                    f"{rq3_primary_significant_omnibus}/9 APFDc omnibus tests significant; "
                    f"{rq3_secondary_significant_omnibus}/9 secondary APFD omnibus tests significant"
                ),

            "InterpretationBoundary":
                (
                    "winner defined by pre-declared 50% absolute APFDc criterion; "
                    "not Nemenyi-significant versus all competitors"
                ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 18. REPRODUCIBILITY ANCHORS TABLE
# --------------------------------------------------------------------------------------------------

reproducibility_rows = checkpoint_audit_rows.copy()

for verification in rq_verifications:
    reproducibility_rows.append({
        "Anchor":
            f"{verification['Label']}_Package",

        "CheckpointPath":
            str(
                verification[
                    "ManifestPath"
                ]
            ),

        "CheckpointSHA256":
            verification[
                "ManifestSHA256"
            ],

        "Status":
            (
                f"PackageRootSHA256={verification['PackageRootSHA256']}; "
                f"Files={verification['PackageFiles']}; Bytes={verification['PackageBytes']}"
            ),

        "CheckpointVerified":
            True,
    })

reproducibility_rows.append({
    "Anchor":
        "CompletionRegistry",

    "CheckpointPath":
        str(
            REGISTRY
        ),

    "CheckpointSHA256":
        registry_sha_before,

    "Status":
        f"Rows={len(registry)}",

    "CheckpointVerified":
        True,
})

reproducibility_anchors = pd.DataFrame(
    reproducibility_rows
)


# --------------------------------------------------------------------------------------------------
# 19. GLOBAL INTEGRITY AUDIT TABLE
# --------------------------------------------------------------------------------------------------

integrity_rows = []

for verification, copy_audit, destination in [
    (
        rq1_verification,
        rq1_copy_audit,
        FROZEN_RQ1_DEST,
    ),
    (
        rq2_verification,
        rq2_copy_audit,
        FROZEN_RQ2_DEST,
    ),
    (
        rq3_verification,
        rq3_copy_audit,
        FROZEN_RQ3_DEST,
    ),
]:
    integrity_rows.append({
        "Artifact":
            verification[
                "Label"
            ],

        "SourcePackageRoot":
            str(
                verification[
                    "PackageRoot"
                ]
            ),

        "CopiedPackageRoot":
            str(
                destination
            ),

        "ExpectedPackageRootSHA256":
            verification[
                "PackageRootSHA256"
            ],

        "RecomputedSourcePackageRootSHA256":
            verification[
                "PackageRootSHA256"
            ],

        "CopiedPackageRootSHA256":
            copy_audit[
                "CopiedRootSHA256"
            ],

        "SourceManifestFiles":
            verification[
                "PackageFiles"
            ],

        "SourceManifestBytes":
            verification[
                "PackageBytes"
            ],

        "SourceMissingFiles":
            verification[
                "MissingFiles"
            ],

        "SourceSizeMismatches":
            verification[
                "SizeMismatches"
            ],

        "SourceSHAMismatches":
            verification[
                "SHAMismatches"
            ],

        "CopiedMissingFiles":
            copy_audit[
                "CopiedMissingFiles"
            ],

        "CopiedSizeMismatches":
            copy_audit[
                "CopiedSizeMismatches"
            ],

        "CopiedSHAMismatches":
            copy_audit[
                "CopiedSHAMismatches"
            ],

        "IntegrityPass":
            bool(
                verification[
                    "MissingFiles"
                ]
                == 0
                and verification[
                    "SizeMismatches"
                ]
                == 0
                and verification[
                    "SHAMismatches"
                ]
                == 0
                and copy_audit[
                    "CopiedMissingFiles"
                ]
                == 0
                and copy_audit[
                    "CopiedSizeMismatches"
                ]
                == 0
                and copy_audit[
                    "CopiedSHAMismatches"
                ]
                == 0
                and copy_audit[
                    "CopiedRootSHA256"
                ]
                == verification[
                    "PackageRootSHA256"
                ]
            ),
    })


integrity_audit = pd.DataFrame(
    integrity_rows
)


# --------------------------------------------------------------------------------------------------
# 20. WRITE CONSOLIDATED TABLES + FROZEN REGISTRY COPY
# --------------------------------------------------------------------------------------------------

atomic_csv(
    STUDY_DESIGN_PATH,
    study_design,
)

atomic_csv(
    CROSS_RQ_PRIMARY_PATH,
    cross_rq_primary,
)

atomic_csv(
    RQ1_SENSITIVITY_PATH,
    rq1_sensitivity,
)

atomic_csv(
    RQ2_SENSITIVITY_PATH,
    rq2_sensitivity,
)

atomic_csv(
    RQ3_SENSITIVITY_PATH,
    rq3_sensitivity,
)

atomic_csv(
    REPRODUCIBILITY_ANCHORS_PATH,
    reproducibility_anchors,
)

atomic_csv(
    INTEGRITY_AUDIT_PATH,
    integrity_audit,
)

atomic_copy(
    REGISTRY,
    REGISTRY_COPY_PATH,
)


# --------------------------------------------------------------------------------------------------
# 21. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

registry_sha_after = sha256_file(
    REGISTRY
)

checks = []

add_check(
    checks,
    "Step-0 checkpoint SHA-256",
    EXPECTED_STEP0_CHECKPOINT_SHA256,
    sha256_file(
        STEP0_CHECKPOINT
    ),
    sha256_file(
        STEP0_CHECKPOINT
    )
    == EXPECTED_STEP0_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-1B checkpoint SHA-256",
    EXPECTED_STEP1B_CHECKPOINT_SHA256,
    sha256_file(
        STEP1B_CHECKPOINT
    ),
    sha256_file(
        STEP1B_CHECKPOINT
    )
    == EXPECTED_STEP1B_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step-2 checkpoint SHA-256",
    EXPECTED_STEP2_CHECKPOINT_SHA256,
    sha256_file(
        STEP2_CHECKPOINT
    ),
    sha256_file(
        STEP2_CHECKPOINT
    )
    == EXPECTED_STEP2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ1 checkpoint SHA-256",
    EXPECTED_RQ1_CHECKPOINT_SHA256,
    sha256_file(
        RQ1_CHECKPOINT
    ),
    sha256_file(
        RQ1_CHECKPOINT
    )
    == EXPECTED_RQ1_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ2 checkpoint SHA-256",
    EXPECTED_RQ2_CHECKPOINT_SHA256,
    sha256_file(
        RQ2_CHECKPOINT
    ),
    sha256_file(
        RQ2_CHECKPOINT
    )
    == EXPECTED_RQ2_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "RQ3 checkpoint SHA-256",
    EXPECTED_RQ3_CHECKPOINT_SHA256,
    sha256_file(
        RQ3_CHECKPOINT
    ),
    sha256_file(
        RQ3_CHECKPOINT
    )
    == EXPECTED_RQ3_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha_after,
    registry_sha_after
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    checks,
    "Completion registry rows",
    24,
    len(
        registry
    ),
    len(
        registry
    )
    == 24,
)

add_check(
    checks,
    "RQ1 package root SHA-256",
    EXPECTED_RQ1_PACKAGE_ROOT_SHA256,
    rq1_verification[
        "PackageRootSHA256"
    ],
    rq1_verification[
        "PackageRootSHA256"
    ]
    == EXPECTED_RQ1_PACKAGE_ROOT_SHA256,
)

add_check(
    checks,
    "RQ2 package root SHA-256",
    EXPECTED_RQ2_PACKAGE_ROOT_SHA256,
    rq2_verification[
        "PackageRootSHA256"
    ],
    rq2_verification[
        "PackageRootSHA256"
    ]
    == EXPECTED_RQ2_PACKAGE_ROOT_SHA256,
)

add_check(
    checks,
    "RQ3 package root SHA-256",
    EXPECTED_RQ3_PACKAGE_ROOT_SHA256,
    rq3_verification[
        "PackageRootSHA256"
    ],
    rq3_verification[
        "PackageRootSHA256"
    ]
    == EXPECTED_RQ3_PACKAGE_ROOT_SHA256,
)

add_check(
    checks,
    "RQ package integrity failures",
    0,
    int(
        (
            ~integrity_audit[
                "IntegrityPass"
            ].astype(
                bool
            )
        ).sum()
    ),
    bool(
        integrity_audit[
            "IntegrityPass"
        ].astype(
            bool
        ).all()
    ),
)

add_check(
    checks,
    "Study design rows",
    18,
    len(
        study_design
    ),
    len(
        study_design
    )
    == 18,
)

add_check(
    checks,
    "Cross-RQ primary result rows",
    3,
    len(
        cross_rq_primary
    ),
    len(
        cross_rq_primary
    )
    == 3,
)

add_check(
    checks,
    "RQ1 metric-sensitivity rows",
    4,
    len(
        rq1_sensitivity
    ),
    len(
        rq1_sensitivity
    )
    == 4,
)

add_check(
    checks,
    "RQ2 metric-sensitivity rows",
    16,
    len(
        rq2_sensitivity
    ),
    len(
        rq2_sensitivity
    )
    == 16,
)

add_check(
    checks,
    "RQ3 metric-sensitivity rows",
    2,
    len(
        rq3_sensitivity
    ),
    len(
        rq3_sensitivity
    )
    == 2,
)

add_check(
    checks,
    "RQ1 APFDc significant total",
    25,
    rq1_primary_significant_total,
    rq1_primary_significant_total
    == 25,
)

add_check(
    checks,
    "RQ1 APFD significant total",
    28,
    rq1_secondary_significant_total,
    rq1_secondary_significant_total
    == 28,
)

expected_rq1_first_sig = {
    "RandomForest":
        5,

    "XGBoost":
        5,

    "LightGBM":
        5,

    "NaiveBayes":
        50,
}

actual_rq1_first_sig = {
    str(
        row.Technique
    ):
        int(
            row.APFDcFirstBonferroniSignificantNoisePercent
        )
    for row in rq1_sensitivity.itertuples(
        index=False
    )
}

add_check(
    checks,
    "RQ1 APFDc first-significant noise map",
    expected_rq1_first_sig,
    actual_rq1_first_sig,
    actual_rq1_first_sig
    == expected_rq1_first_sig,
)

expected_latestfail_thresholds = {
    "RandomForest":
        "5",

    "XGBoost":
        "5",

    "LightGBM":
        "5",

    "NaiveBayes":
        "5",
}

add_check(
    checks,
    "RQ2 APFDc LatestFail crossover map",
    expected_latestfail_thresholds,
    latestfail_threshold_json,
    latestfail_threshold_json
    == expected_latestfail_thresholds,
)

add_check(
    checks,
    "RQ3 primary robustness winner",
    "NaiveBayes",
    str(
        rq3[
            "PrimaryRobustnessWinner"
        ]
    ),
    str(
        rq3[
            "PrimaryRobustnessWinner"
        ]
    )
    == "NaiveBayes",
)

add_check(
    checks,
    "RQ3 winner APFDc at 50%",
    0.4920519147301996,
    float(
        rq3_winner_row[
            "APFDcAt50"
        ]
    ),
    abs(
        float(
            rq3_winner_row[
                "APFDcAt50"
            ]
        )
        - 0.4920519147301996
    )
    <= FLOAT_TOL,
)

add_check(
    checks,
    "RQ3 winner retention at 50%",
    87.23803558481858,
    float(
        rq3_winner_row[
            "MeanRetentionPctAPFDcAt50"
        ]
    ),
    abs(
        float(
            rq3_winner_row[
                "MeanRetentionPctAPFDcAt50"
            ]
        )
        - 87.23803558481858
    )
    <= FLOAT_TOL,
)

add_check(
    checks,
    "RQ3 winner Nemenyi-significant vs all competitors",
    False,
    bool(
        rq3[
            "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50"
        ]
    ),
    bool(
        rq3[
            "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50"
        ]
    )
    is False,
)

add_check(
    checks,
    "New inferential tests executed",
    0,
    0,
    True,
)

add_check(
    checks,
    "New p-values computed",
    False,
    False,
    True,
)

add_check(
    checks,
    "Project outputs modified",
    False,
    False,
    True,
)

add_check(
    checks,
    "Completion registry modified",
    False,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    == registry_sha_before,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ].astype(
        bool
    )
].copy()

print(
    "\nGlobal Analysis Step 6A pre-freeze validation:"
)

try:
    from IPython.display import display

    display(
        validation
    )

except Exception:
    print(
        validation.to_string(
            index=False
        )
    )

if not failed_validation.empty:
    raise RuntimeError(
        "GLOBAL ANALYSIS STEP 6A VALIDATION FAILED.\n"
        + failed_validation.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 22. REPORT / STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = pd.Timestamp.now(
    tz="UTC"
).isoformat()

report = {
    "Step":
        "GLOBAL_ANALYSIS_STEP_6A",

    "Status":
        FINAL_STATUS,

    "CodeRevision":
        CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidateProjectsScreened":
        25,

    "EligibleIncludedProjects":
        24,

    "ExcludedProjects":
        1,

    "ExcludedProject":
        "Graylog2@graylog2-server",

    "ExcludedReason":
        "0 clean evaluation failures in chronological evaluation split",

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "RepeatedStochasticUnit":
        "30 seeds within project",

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RQ1PrimaryAPFDcSignificantComparisons":
        rq1_primary_significant_total,

    "RQ1PrimaryAPFDcTotalComparisons":
        32,

    "RQ1FirstSignificantNoisePercentByTechnique":
        actual_rq1_first_sig,

    "RQ2PrimaryComparator":
        "LatestFail",

    "RQ2LatestFailFirstObservedCrossoverByTechnique":
        latestfail_threshold_json,

    "RQ2InterpolationAllowed":
        False,

    "RQ3PrimaryRobustnessCriterion":
        (
            "highest equally weighted cross-project mean APFDc "
            "at highest tested noise level (50%)"
        ),

    "RQ3PrimaryRobustnessWinner":
        str(
            rq3[
                "PrimaryRobustnessWinner"
            ]
        ),

    "RQ3WinnerAPFDcAt50":
        float(
            rq3_winner_row[
                "APFDcAt50"
            ]
        ),

    "RQ3WinnerAverageRankAt50":
        float(
            rq3_winner_row[
                "AverageRankAt50"
            ]
        ),

    "RQ3WinnerRetentionPctAt50":
        float(
            rq3_winner_row[
                "MeanRetentionPctAPFDcAt50"
            ]
        ),

    "RQ3WinnerDegradationPctAt50":
        float(
            rq3_winner_row[
                "MeanDegradationPctAPFDcAt50"
            ]
        ),

    "RQ3WinnerNemenyiSignificantAgainstAllAt50":
        bool(
            rq3[
                "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50"
            ]
        ),

    "RQ1PackageRootSHA256":
        EXPECTED_RQ1_PACKAGE_ROOT_SHA256,

    "RQ2PackageRootSHA256":
        EXPECTED_RQ2_PACKAGE_ROOT_SHA256,

    "RQ3PackageRootSHA256":
        EXPECTED_RQ3_PACKAGE_ROOT_SHA256,

    "NewInferentialTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "GlobalAnalysisComplete":
        True,
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    STATUS_PATH,
    {
        "Status":
            FINAL_STATUS,

        "CompletedAtUTC":
            completed_at_utc,

        "GlobalAnalysisComplete":
            True,

        "RQ1Complete":
            True,

        "RQ2Complete":
            True,

        "RQ3Complete":
            True,

        "ReadyForThesisResultsUse":
            True,
    },
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 23. FINAL GLOBAL PACKAGE MANIFEST + ROOT HASH
# --------------------------------------------------------------------------------------------------

package_manifest = build_package_manifest(
    STEP6A_ROOT,
    PACKAGE_MANIFEST_PATH,
)

atomic_csv(
    PACKAGE_MANIFEST_PATH,
    package_manifest,
)

package_root_sha = package_root_hash(
    package_manifest
)

package_files = int(
    len(
        package_manifest
    )
)

package_bytes = int(
    package_manifest[
        "Bytes"
    ].sum()
)


# --------------------------------------------------------------------------------------------------
# 24. FINAL CHECKPOINT
# --------------------------------------------------------------------------------------------------

if sha256_file(
    REGISTRY
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during final global consolidation."
    )

checkpoint = {
    "CheckpointType":
        "FINAL_CROSS_RQ_GLOBAL_RESULTS_PACKAGE",

    "Status":
        FINAL_STATUS,

    "CodeRevision":
        CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "Step0CheckpointSHA256":
        EXPECTED_STEP0_CHECKPOINT_SHA256,

    "Step1BCheckpointSHA256":
        EXPECTED_STEP1B_CHECKPOINT_SHA256,

    "Step2CheckpointSHA256":
        EXPECTED_STEP2_CHECKPOINT_SHA256,

    "RQ1CheckpointSHA256":
        EXPECTED_RQ1_CHECKPOINT_SHA256,

    "RQ2CheckpointSHA256":
        EXPECTED_RQ2_CHECKPOINT_SHA256,

    "RQ3CheckpointSHA256":
        EXPECTED_RQ3_CHECKPOINT_SHA256,

    "CompletionRegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "IndependentEmpiricalUnit":
        "Project",

    "IndependentEmpiricalUnitN":
        24,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RQ1PackageRootSHA256":
        EXPECTED_RQ1_PACKAGE_ROOT_SHA256,

    "RQ2PackageRootSHA256":
        EXPECTED_RQ2_PACKAGE_ROOT_SHA256,

    "RQ3PackageRootSHA256":
        EXPECTED_RQ3_PACKAGE_ROOT_SHA256,

    "GlobalPackageRoot":
        str(
            STEP6A_ROOT
        ),

    "GlobalPackageManifestPath":
        str(
            PACKAGE_MANIFEST_PATH
        ),

    "GlobalPackageManifestSHA256":
        sha256_file(
            PACKAGE_MANIFEST_PATH
        ),

    "GlobalPackageRootSHA256":
        package_root_sha,

    "GlobalPackageFiles":
        package_files,

    "GlobalPackageBytes":
        package_bytes,

    "StudyDesignPath":
        str(
            STUDY_DESIGN_PATH
        ),

    "StudyDesignSHA256":
        sha256_file(
            STUDY_DESIGN_PATH
        ),

    "CrossRQPrimaryResultsPath":
        str(
            CROSS_RQ_PRIMARY_PATH
        ),

    "CrossRQPrimaryResultsSHA256":
        sha256_file(
            CROSS_RQ_PRIMARY_PATH
        ),

    "RQ1SensitivityPath":
        str(
            RQ1_SENSITIVITY_PATH
        ),

    "RQ1SensitivitySHA256":
        sha256_file(
            RQ1_SENSITIVITY_PATH
        ),

    "RQ2SensitivityPath":
        str(
            RQ2_SENSITIVITY_PATH
        ),

    "RQ2SensitivitySHA256":
        sha256_file(
            RQ2_SENSITIVITY_PATH
        ),

    "RQ3SensitivityPath":
        str(
            RQ3_SENSITIVITY_PATH
        ),

    "RQ3SensitivitySHA256":
        sha256_file(
            RQ3_SENSITIVITY_PATH
        ),

    "ReproducibilityAnchorsPath":
        str(
            REPRODUCIBILITY_ANCHORS_PATH
        ),

    "ReproducibilityAnchorsSHA256":
        sha256_file(
            REPRODUCIBILITY_ANCHORS_PATH
        ),

    "IntegrityAuditPath":
        str(
            INTEGRITY_AUDIT_PATH
        ),

    "IntegrityAuditSHA256":
        sha256_file(
            INTEGRITY_AUDIT_PATH
        ),

    "NewInferentialTestsExecuted":
        0,

    "NewPValuesComputed":
        False,

    "ProjectOutputsModified":
        False,

    "CompletionRegistryModified":
        False,

    "GlobalAnalysisComplete":
        True,

    "ReadyForThesisResultsUse":
        True,

    "NextRequiredStep":
        "NONE — GLOBAL ANALYSIS COMPLETE; USE FROZEN RESULTS FOR THESIS RESULTS/DISCUSSION REVIEW",
}

atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)

checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


# --------------------------------------------------------------------------------------------------
# 25. USER-VISIBLE CONSOLIDATED TABLES
# --------------------------------------------------------------------------------------------------

print(
    "\nFINAL cross-RQ primary-results table:"
)

try:
    from IPython.display import display

    display(
        cross_rq_primary
    )

except Exception:
    print(
        cross_rq_primary.to_string(
            index=False
        )
    )

print(
    "\nRQ1 APFDc-vs-APFD sensitivity:"
)

try:
    from IPython.display import display

    display(
        rq1_sensitivity
    )

except Exception:
    print(
        rq1_sensitivity.to_string(
            index=False
        )
    )

print(
    "\nRQ2 APFDc-vs-APFD threshold sensitivity:"
)

try:
    from IPython.display import display

    display(
        rq2_sensitivity
    )

except Exception:
    print(
        rq2_sensitivity.to_string(
            index=False
        )
    )

print(
    "\nRQ3 APFDc-vs-APFD sensitivity:"
)

try:
    from IPython.display import display

    display(
        rq3_sensitivity
    )

except Exception:
    print(
        rq3_sensitivity.to_string(
            index=False
        )
    )

print(
    "\nGlobal integrity audit:"
)

try:
    from IPython.display import display

    display(
        integrity_audit
    )

except Exception:
    print(
        integrity_audit.to_string(
            index=False
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 162
)

print(
    "=== THESIS GLOBAL ANALYSIS — CELL 12 / STEP 6A FINAL RESULT ==="
)

print(
    "=" * 162
)

print(
    "Global analysis complete: True"
)

print(
    "\nPopulation:"
)

print(
    "Candidate projects screened: 25"
)

print(
    "Eligible/included projects: 24"
)

print(
    "Excluded projects: 1"
)

print(
    "Excluded project: Graylog2@graylog2-server"
)

print(
    "\nStudy unit:"
)

print(
    "Independent empirical unit: Project (N=24)"
)

print(
    "Repeated stochastic unit: 30 seeds within project"
)

print(
    "\nRQ1 primary result:"
)

print(
    "APFDc significant noisy-vs-clean comparisons:",
    rq1_primary_significant_total,
    "/ 32"
)

print(
    "First significant noise by technique:",
    actual_rq1_first_sig
)

print(
    "\nRQ2 primary result:"
)

print(
    "Comparator: LatestFail"
)

print(
    "First observed crossover by technique:",
    latestfail_threshold_json
)

print(
    "Interpolation allowed: False"
)

print(
    "\nRQ3 primary result:"
)

print(
    "Primary robustness winner:",
    str(
        rq3[
            "PrimaryRobustnessWinner"
        ]
    )
)

print(
    "Winner APFDc at 50%:",
    float(
        rq3_winner_row[
            "APFDcAt50"
        ]
    )
)

print(
    "Winner retention at 50%:",
    float(
        rq3_winner_row[
            "MeanRetentionPctAPFDcAt50"
        ]
    )
)

print(
    "Winner significantly better than all competitors at 50% by Nemenyi:",
    bool(
        rq3[
            "PrimaryWinnerNemenyiSignificantAgainstAllCompetitorsAt50"
        ]
    )
)

print(
    "\nFrozen RQ package roots:"
)

print(
    "RQ1:",
    EXPECTED_RQ1_PACKAGE_ROOT_SHA256
)

print(
    "RQ2:",
    EXPECTED_RQ2_PACKAGE_ROOT_SHA256
)

print(
    "RQ3:",
    EXPECTED_RQ3_PACKAGE_ROOT_SHA256
)

print(
    "\nFinal global package:"
)

print(
    "Files:",
    package_files
)

print(
    "Bytes:",
    package_bytes
)

print(
    "Global package root SHA-256:",
    package_root_sha
)

print(
    "\nIsolation:"
)

print(
    "New inferential tests executed: 0"
)

print(
    "New p-values computed: False"
)

print(
    "Project outputs modified: False"
)

print(
    "Completion registry modified: False"
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    )
)

print(
    "Failed checks:",
    len(
        failed_validation
    )
)

print(
    "\nFinal global checkpoint:"
)

print(
    CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    checkpoint_sha
)

print(
    "\nNext required step: NONE — GLOBAL ANALYSIS COMPLETE"
)

print(
    "\nSTATUS:",
    FINAL_STATUS
)

print(
    "=" * 162
)


=== THESIS GLOBAL ANALYSIS — CELL 12 / STEP 6A: FINAL CROSS-RQ CONSOLIDATION + GLOBAL RESULTS PACKAGE ===

Global Analysis Step 6A pre-freeze validation:


,Check,Expected,Actual,Pass
0,Step-0 checkpoint SHA-256,b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d...,b0e43922e4c935d0e845e281fb9c83e1b8e6750661356d...,True
1,Step-1B checkpoint SHA-256,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,eb616561b9b53f3d823b0f5e3c6a1c183745cb4d8d74b3...,True
2,Step-2 checkpoint SHA-256,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,f3c72f598f9ae55b1ba474fb0d2ee1905eb69b4927b128...,True
3,RQ1 checkpoint SHA-256,b4f3d10b77acb39200496542e8256d6213981fd6363254...,b4f3d10b77acb39200496542e8256d6213981fd6363254...,True
4,RQ2 checkpoint SHA-256,8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d35...,8a540f0fecc9324ea574cf6ea744343ba98cbf70a33d35...,True
5,RQ3 checkpoint SHA-256,17ceb2c1e2b50ac5aacad8539d40f41ed7b913ee236594...,17ceb2c1e2b50ac5aacad8539d40f41ed7b913ee236594...,True
6,Completion registry SHA-256,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,dc5cdc752d89661c0b41adc5680509034ded1c64f2f419...,True
7,Completion registry rows,24,24,True
8,RQ1 package root SHA-256,5b7c49e4096c11edd927052807658e5a196e58b5c7302e...,5b7c49e4096c11edd927052807658e5a196e58b5c7302e...,True
9,RQ2 package root SHA-256,cec947adcb77b912dc217ad7012f6b97a760932906c147...,cec947adcb77b912dc217ad7012f6b97a760932906c147...,True



FINAL cross-RQ primary-results table:


,RQ,PrimaryMetric,PrimaryResultType,PrimaryResult,PrimaryInferentialContext,InterpretationBoundary
0,RQ1,APFDc,noisy-vs-clean degradation onset,"{""LightGBM"":5,""NaiveBayes"":50,""RandomForest"":5...",25/32 Bonferroni-significant APFDc comparisons...,paired project-level inference; seeds not trea...
1,RQ2,APFDc,first observed crossover vs LatestFail,"{""LightGBM"":""5"",""NaiveBayes"":""5"",""RandomForest...",descriptive threshold analysis; 0 formal RQ2 h...,observed tested grid only; interpolation forbi...
2,RQ3,APFDc,robustness winner at highest tested noise,NaiveBayes,50% Friedman Holm-significant; 3/9 APFDc omnib...,winner defined by pre-declared 50% absolute AP...



RQ1 APFDc-vs-APFD sensitivity:


...


RQ3 APFDc-vs-APFD sensitivity:


,Metric,HolmSignificantFriedmanTestsOf9,StressRegionBestAbsoluteTechnique,StressRegionBestAbsoluteMean,StressRegionBestAverageRankTechnique,StressRegionBestAverageRank
0,APFDc,3,NaiveBayes,0.517922,NaiveBayes,2.222222
1,APFD,9,NaiveBayes,0.588123,NaiveBayes,1.666667



Global integrity audit:


,Artifact,SourcePackageRoot,CopiedPackageRoot,ExpectedPackageRootSHA256,RecomputedSourcePackageRootSHA256,CopiedPackageRootSHA256,SourceManifestFiles,SourceManifestBytes,SourceMissingFiles,SourceSizeMismatches,SourceSHAMismatches,CopiedMissingFiles,CopiedSizeMismatches,CopiedSHAMismatches,IntegrityPass
0,RQ1,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,5b7c49e4096c11edd927052807658e5a196e58b5c7302e...,5b7c49e4096c11edd927052807658e5a196e58b5c7302e...,5b7c49e4096c11edd927052807658e5a196e58b5c7302e...,14,1197454,0,0,0,0,0,0,True
1,RQ2,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,cec947adcb77b912dc217ad7012f6b97a760932906c147...,cec947adcb77b912dc217ad7012f6b97a760932906c147...,cec947adcb77b912dc217ad7012f6b97a760932906c147...,18,1546867,0,0,0,0,0,0,True
2,RQ3,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,77f25e4a0755a437e44ffcba5a86a360cd9736b63ab02c...,77f25e4a0755a437e44ffcba5a86a360cd9736b63ab02c...,77f25e4a0755a437e44ffcba5a86a360cd9736b63ab02c...,15,1167758,0,0,0,0,0,0,True



=== THESIS GLOBAL ANALYSIS — CELL 12 / STEP 6A FINAL RESULT ===
Global analysis complete: True

Population:
Candidate projects screened: 25
Eligible/included projects: 24
Excluded projects: 1
Excluded project: Graylog2@graylog2-server

Study unit:
Independent empirical unit: Project (N=24)
Repeated stochastic unit: 30 seeds within project

RQ1 primary result:
APFDc significant noisy-vs-clean comparisons: 25 / 32
First significant noise by technique: {'RandomForest': 5, 'XGBoost': 5, 'LightGBM': 5, 'NaiveBayes': 50}

RQ2 primary result:
Comparator: LatestFail
First observed crossover by technique: {'RandomForest': '5', 'XGBoost': '5', 'LightGBM': '5', 'NaiveBayes': '5'}
Interpolation allowed: False

RQ3 primary result:
Primary robustness winner: NaiveBayes
Winner APFDc at 50%: 0.4920519147301996
Winner retention at 50%: 87.23803558481858
Winner significantly better than all competitors at 50% by Nemenyi: False

Frozen RQ package roots:
RQ1: 5b7c49e4096c11edd927052807658e5a196e58b5c7302